<a href="https://colab.research.google.com/github/IhabAltekreeti/vaultify/blob/main/vaultify_compact_full_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# VAULTIFY COMPACT - Cell 1: Install all runtime dependencies

%pip install -q \
    qdrant-client \
    groq \
    docling \
    "rapidocr<3.8" \
    onnxruntime \
    "mcp[cli]" \
    flask \
    flask-sqlalchemy \
    flask-login \
    flask-wtf \
    flask-migrate \
    flask-limiter \
    email-validator \
    filetype \
    python-dotenv \
    sentence-transformers

print("✅ Vaultify runtime dependencies installed.")


✅ Vaultify runtime dependencies installed.


In [4]:
# VAULTIFY COMPACT - Cell 2: Load secrets and define the central configuration

from pathlib import Path

from google.colab import userdata

REQUIRED_SECRETS = [
    "QDRANT_URL",
    "QDRANT_API_KEY",
    "GROQ_API_KEY",
]

OPTIONAL_SECRETS = [
    "NGROK_TOKEN",
]

loaded_secrets = {}
missing_secrets = []

for secret_name in REQUIRED_SECRETS + OPTIONAL_SECRETS:
    try:
        secret_value = userdata.get(secret_name)
    except Exception:
        secret_value = None

    if secret_value:
        loaded_secrets[secret_name] = secret_value
        print(f"✅ {secret_name}: Found")
    elif secret_name in REQUIRED_SECRETS:
        missing_secrets.append(secret_name)
        print(f"❌ {secret_name}: Missing")
    else:
        print(f"ℹ️ {secret_name}: Optional and not configured")

if missing_secrets:
    raise RuntimeError(
        "Missing required Colab Secrets: "
        + ", ".join(missing_secrets)
    )

QDRANT_URL = loaded_secrets["QDRANT_URL"]
QDRANT_API_KEY = loaded_secrets["QDRANT_API_KEY"]
GROQ_API_KEY = loaded_secrets["GROQ_API_KEY"]
NGROK_TOKEN = loaded_secrets.get("NGROK_TOKEN")

PROJECT_NAME = "Vaultify"
PROJECT_VERSION = "3.0.0"

COLLECTION_NAME = "vaultify_v3_documents"

TENANT_ID_FIELD = "tenant_id"
DOCUMENT_HASH_FIELD = "document_hash"
FILENAME_FIELD = "filename"
CHUNK_INDEX_FIELD = "chunk_index"
CHUNK_TYPE_FIELD = "chunk_type"
TEXT_FIELD = "text"

EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
VECTOR_SIZE = 384
LLM_MODEL = "llama-3.3-70b-versatile"

PROJECT_ROOT = Path("/content/vaultify_v3")
UPLOADS_DIR = PROJECT_ROOT / "uploads"
CACHE_DIR = PROJECT_ROOT / "cache"

for directory in [PROJECT_ROOT, UPLOADS_DIR, CACHE_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print("\n✅ Vaultify configuration loaded.")
print(f"📁 Runtime workspace: {PROJECT_ROOT}")


✅ QDRANT_URL: Found
✅ QDRANT_API_KEY: Found
✅ GROQ_API_KEY: Found
✅ NGROK_TOKEN: Found

✅ Vaultify configuration loaded.
📁 Runtime workspace: /content/vaultify_v3


In [5]:
# VAULTIFY COMPACT - Cell 3: Verify the runtime and initialize cloud clients

import platform
import sys

import torch
from groq import Groq
from qdrant_client import QdrantClient

print(f"🐍 Python: {sys.version.split()[0]}")
print(f"🖥️ Platform: {platform.platform()}")
print(f"⚡ CUDA available: {torch.cuda.is_available()}")

if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU is unavailable. Select Runtime > Change runtime type > T4 GPU."
    )

print(f"🎮 GPU: {torch.cuda.get_device_name(0)}")

qdrant = QdrantClient(
    url=QDRANT_URL,
    api_key=QDRANT_API_KEY,
    timeout=60,
)

existing_collections = {
    collection.name
    for collection in qdrant.get_collections().collections
}

print("✅ Qdrant Cloud connection successful.")
print(f"📦 Collections: {sorted(existing_collections) or 'None'}")

groq_client = Groq(api_key=GROQ_API_KEY)

available_model_ids = {
    model.id
    for model in groq_client.models.list().data
}

if LLM_MODEL not in available_model_ids:
    raise RuntimeError(
        f"Groq model is unavailable: {LLM_MODEL}"
    )

print("✅ Groq Cloud connection successful.")
print(f"🤖 LLM model: {LLM_MODEL}")


🐍 Python: 3.12.13
🖥️ Platform: Linux-6.6.122+-x86_64-with-glibc2.35
⚡ CUDA available: True
🎮 GPU: Tesla T4
✅ Qdrant Cloud connection successful.
📦 Collections: ['vaultify_documents', 'vaultify_v3_documents']
✅ Groq Cloud connection successful.
🤖 LLM model: llama-3.3-70b-versatile


In [6]:
# VAULTIFY COMPACT - Cell 4: Load the embedding model and prepare Qdrant

from sentence_transformers import SentenceTransformer
from qdrant_client.models import (
    Distance,
    PayloadSchemaType,
    VectorParams,
)

embedding_model = SentenceTransformer(
    EMBEDDING_MODEL_NAME,
    device="cuda",
)

validation_embeddings = embedding_model.encode(
    [
        "Vaultify searches private company documents.",
        "Financial reports contain tables and numerical data.",
    ],
    normalize_embeddings=True,
    convert_to_numpy=True,
    show_progress_bar=False,
)

if validation_embeddings.shape[1] != VECTOR_SIZE:
    raise RuntimeError(
        "The embedding dimension does not match the configured vector size."
    )

existing_collections = {
    collection.name
    for collection in qdrant.get_collections().collections
}

if COLLECTION_NAME not in existing_collections:
    qdrant.create_collection(
        collection_name=COLLECTION_NAME,
        vectors_config=VectorParams(
            size=VECTOR_SIZE,
            distance=Distance.COSINE,
        ),
    )
    print(f"✅ Created collection: {COLLECTION_NAME}")
else:
    print(f"ℹ️ Collection already exists: {COLLECTION_NAME}")

collection_info = qdrant.get_collection(COLLECTION_NAME)

for field_name in [TENANT_ID_FIELD, DOCUMENT_HASH_FIELD]:
    if field_name not in (collection_info.payload_schema or {}):
        qdrant.create_payload_index(
            collection_name=COLLECTION_NAME,
            field_name=field_name,
            field_schema=PayloadSchemaType.KEYWORD,
            wait=True,
        )
        print(f"✅ Created payload index: {field_name}")
    else:
        print(f"ℹ️ Payload index already exists: {field_name}")

print(f"📐 Embedding shape: {validation_embeddings.shape}")
print("✅ Embedding and Qdrant setup completed.")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

ℹ️ Collection already exists: vaultify_v3_documents
ℹ️ Payload index already exists: tenant_id
ℹ️ Payload index already exists: document_hash
📐 Embedding shape: (2, 384)
✅ Embedding and Qdrant setup completed.


In [7]:
# VAULTIFY COMPACT - Cell 5: Inspect existing tenant data and choose the next path

from qdrant_client.models import (
    FieldCondition,
    Filter,
    MatchValue,
)

DEMO_TENANTS = {
    "company_a": {
        "tenant_id": "demo_apple_tenant",
        "display_name": "Apple Financial Demo",
    },
    "company_b": {
        "tenant_id": "demo_tesla_tenant",
        "display_name": "Tesla Financial Demo",
    },
}

tenant_counts = {}

for tenant_key, tenant in DEMO_TENANTS.items():
    tenant_filter = Filter(
        must=[
            FieldCondition(
                key=TENANT_ID_FIELD,
                match=MatchValue(
                    value=tenant["tenant_id"]
                ),
            )
        ]
    )

    tenant_counts[tenant_key] = qdrant.count(
        collection_name=COLLECTION_NAME,
        count_filter=tenant_filter,
        exact=True,
    ).count

total_point_count = qdrant.count(
    collection_name=COLLECTION_NAME,
    exact=True,
).count

print("📊 Existing Qdrant data:")
print(f"   Apple tenant: {tenant_counts['company_a']} chunks")
print(f"   Tesla tenant: {tenant_counts['company_b']} chunks")
print(f"   Entire collection: {total_point_count} points")

SEED_DATA_READY = (
    tenant_counts["company_a"] > 0
    and tenant_counts["company_b"] > 0
)

if SEED_DATA_READY:
    print("\n✅ Existing demo vectors were found.")
    print("⏭️ Seed PDF parsing and indexing can be skipped.")
    print("➡️ Next: restore retrieval, MCP, Flask, and Cloudflare services.")
else:
    print("\n⚠️ Demo vectors are incomplete or missing.")
    print("➡️ Next: run the compact ingestion cells before starting services.")


📊 Existing Qdrant data:
   Apple tenant: 745 chunks
   Tesla tenant: 140 chunks
   Entire collection: 1178 points

✅ Existing demo vectors were found.
⏭️ Seed PDF parsing and indexing can be skipped.
➡️ Next: restore retrieval, MCP, Flask, and Cloudflare services.


In [8]:
# VAULTIFY COMPACT - Cell 6: Restore tenant-scoped retrieval and grounded answers

from qdrant_client.models import FieldCondition, Filter, MatchValue

if not SEED_DATA_READY:
    raise RuntimeError(
        "The demo vectors are missing. Run the ingestion path before retrieval."
    )

RETRIEVAL_CANDIDATE_LIMIT = 12
FINAL_CONTEXT_LIMIT = 6
MAX_ANSWER_TOKENS = 400


def search_tenant_documents(
    query,
    tenant_id,
    limit=RETRIEVAL_CANDIDATE_LIMIT,
):
    """
    Search only the document chunks that belong to one trusted tenant.
    """
    cleaned_query = query.strip()

    if not cleaned_query:
        raise ValueError("The search query cannot be empty.")

    if not tenant_id:
        raise ValueError("A tenant ID is required.")

    if limit <= 0:
        raise ValueError("The retrieval limit must be greater than zero.")

    query_vector = embedding_model.encode(
        cleaned_query,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    )

    if query_vector.shape[0] != VECTOR_SIZE:
        raise RuntimeError(
            "The query embedding dimension does not match Qdrant."
        )

    tenant_filter = Filter(
        must=[
            FieldCondition(
                key=TENANT_ID_FIELD,
                match=MatchValue(value=tenant_id),
            )
        ]
    )

    response = qdrant.query_points(
        collection_name=COLLECTION_NAME,
        query=query_vector.tolist(),
        query_filter=tenant_filter,
        limit=limit,
        with_payload=True,
    )

    return response.points


def build_retrieval_context(
    results,
    context_limit=FINAL_CONTEXT_LIMIT,
):
    """
    Convert Qdrant results into a structured context for the language model.
    """
    context_sections = []

    for result_index, result in enumerate(
        results[:context_limit],
        start=1,
    ):
        payload = result.payload or {}
        chunk_text = str(
            payload.get(TEXT_FIELD, "")
        ).strip()

        if not chunk_text:
            continue

        context_sections.append(
            "\n".join(
                [
                    f"[Context {result_index}]",
                    f"File: {payload.get(FILENAME_FIELD, 'Unknown file')}",
                    f"Section: {payload.get('section', 'Unknown section')}",
                    f"Chunk type: {payload.get(CHUNK_TYPE_FIELD, 'unknown')}",
                    f"Similarity score: {float(result.score):.4f}",
                    "Content:",
                    chunk_text,
                ]
            )
        )

    return "\n\n---\n\n".join(context_sections)


def answer_tenant_question(
    question,
    tenant_id,
    retrieval_limit=RETRIEVAL_CANDIDATE_LIMIT,
):
    """
    Retrieve tenant-isolated evidence and generate an evidence-only answer.
    """
    cleaned_question = question.strip()

    if not cleaned_question:
        raise ValueError("The question cannot be empty.")

    results = search_tenant_documents(
        query=cleaned_question,
        tenant_id=tenant_id,
        limit=retrieval_limit,
    )

    context = build_retrieval_context(results)

    if not context:
        return {
            "answer": (
                "The requested information was not found "
                "in the tenant's indexed documents."
            ),
            "results": results,
        }

    system_prompt = """
You are Vaultify, a document-grounded business assistant.

Rules:
1. Use only the supplied document context.
2. Never use outside knowledge or guess missing facts.
3. Keep company names, reporting periods, units, and totals exact.
4. When evidence is insufficient, clearly say the information was not found.
5. Do not reveal or discuss documents that are absent from the supplied context.
6. Give a concise answer and mention the source file or section when useful.
""".strip()

    user_prompt = f"""
Question:
{cleaned_question}

Document context:
{context}
""".strip()

    response = groq_client.chat.completions.create(
        model=LLM_MODEL,
        messages=[
            {
                "role": "system",
                "content": system_prompt,
            },
            {
                "role": "user",
                "content": user_prompt,
            },
        ],
        temperature=0,
        max_tokens=MAX_ANSWER_TOKENS,
    )

    answer = response.choices[0].message.content.strip()

    if not answer:
        raise RuntimeError("Groq returned an empty answer.")

    return {
        "answer": answer,
        "results": results,
    }


grounded_tests = [
    {
        "name": "Apple FY2025 net sales",
        "tenant_id": DEMO_TENANTS["company_a"]["tenant_id"],
        "question": (
            "What were Apple's total net sales "
            "in fiscal year 2025?"
        ),
        "expected_filename": "apple_fy2025_10k.pdf",
    },
    {
        "name": "Tesla Q4 2025 revenue",
        "tenant_id": DEMO_TENANTS["company_b"]["tenant_id"],
        "question": (
            "What was Tesla's total revenue "
            "in the fourth quarter of 2025?"
        ),
        "expected_filename": "tesla_q4_2025_update.pdf",
    },
]

grounded_test_outputs = {}

for test in grounded_tests:
    result = answer_tenant_question(
        question=test["question"],
        tenant_id=test["tenant_id"],
    )

    returned_filenames = {
        item.payload.get(FILENAME_FIELD)
        for item in result["results"]
        if item.payload
    }

    unexpected_filenames = (
        returned_filenames
        - {test["expected_filename"]}
    )

    if unexpected_filenames:
        raise RuntimeError(
            "Tenant isolation failed. Unexpected files: "
            f"{sorted(unexpected_filenames)}"
        )

    grounded_test_outputs[test["name"]] = result

    print("\n" + "=" * 88)
    print(f"🧪 {test['name']}")
    print(f"🏢 Tenant: {test['tenant_id']}")
    print(f"📄 Retrieved files: {sorted(returned_filenames)}")
    print("🤖 Answer:")
    print(result["answer"])

print("\n✅ Tenant-scoped retrieval restored.")
print("✅ Apple and Tesla grounded-answer tests passed.")
print("✅ No cross-tenant document appeared in either retrieval result.")



🧪 Apple FY2025 net sales
🏢 Tenant: demo_apple_tenant
📄 Retrieved files: ['apple_fy2025_10k.pdf']
🤖 Answer:
According to the document context, specifically in [Context 3] File: apple_fy2025_10k.pdf, Section: Note 2 - Revenue, Apple's total net sales in fiscal year 2025 were $416,161. 

Source: [Context 3], Section: Note 2 - Revenue, Table: Total net sales.

🧪 Tesla Q4 2025 revenue
🏢 Tenant: demo_tesla_tenant
📄 Retrieved files: ['tesla_q4_2025_update.pdf']
🤖 Answer:
Tesla's total revenue in the fourth quarter of 2025 was $24,901 million. 
Source: File: tesla_q4_2025_update.pdf, Section: (Unaudited) table.

✅ Tenant-scoped retrieval restored.
✅ Apple and Tesla grounded-answer tests passed.
✅ No cross-tenant document appeared in either retrieval result.


In [9]:
# VAULTIFY COMPACT - Cell 7: Define and test the tenant-bound MCP tool in memory

import json
from importlib.metadata import version
from typing import Any

from mcp.server.fastmcp import FastMCP
from mcp.server.fastmcp.exceptions import ToolError
from mcp.server.transport_security import TransportSecuritySettings
from mcp.shared.memory import (
    create_connected_server_and_client_session,
)

MCP_HOST = "127.0.0.1"
MCP_PORT = 8003
MCP_SERVER_NAME = "Vaultify"
MCP_RETRIEVAL_LIMIT = 5

ACTIVE_DEMO_TENANT_KEY = "company_a"
ACTIVE_DEMO_TENANT_ID = (
    DEMO_TENANTS[ACTIVE_DEMO_TENANT_KEY]["tenant_id"]
)
ACTIVE_DEMO_FILENAME = "apple_fy2025_10k.pdf"

LOCAL_MCP_URL = f"http://{MCP_HOST}:{MCP_PORT}/mcp"

vaultify_mcp = FastMCP(
    MCP_SERVER_NAME,
    host=MCP_HOST,
    port=MCP_PORT,
    stateless_http=True,
    json_response=True,
    transport_security=TransportSecuritySettings(
        enable_dns_rebinding_protection=False,
    ),
)


def serialize_retrieval_sources(results):
    """
    Convert Qdrant results into non-sensitive MCP source metadata.
    """
    sources = []

    for result in results:
        payload = result.payload or {}

        sources.append(
            {
                "filename": payload.get(
                    FILENAME_FIELD,
                    "Unknown file",
                ),
                "section": payload.get(
                    "section",
                    "Unknown section",
                ),
                "chunk_type": payload.get(
                    CHUNK_TYPE_FIELD,
                    "unknown",
                ),
                "chunk_index": payload.get(
                    CHUNK_INDEX_FIELD,
                ),
                "similarity_score": round(
                    float(result.score),
                    4,
                ),
            }
        )

    return sources


def parse_mcp_tool_result(tool_result):
    """
    Parse structured MCP output or the JSON text fallback.
    """
    if tool_result.isError:
        error_messages = [
            item.text
            for item in tool_result.content
            if hasattr(item, "text")
        ]

        raise RuntimeError(
            "The MCP tool returned an error. "
            + " ".join(error_messages)
        )

    structured = tool_result.structuredContent

    if isinstance(structured, dict):
        if isinstance(structured.get("result"), dict):
            return structured["result"]

        return structured

    text_blocks = [
        item.text.strip()
        for item in tool_result.content
        if hasattr(item, "text")
        and item.text.strip()
    ]

    for text_block in text_blocks:
        try:
            parsed = json.loads(text_block)
        except json.JSONDecodeError:
            continue

        if isinstance(parsed, dict):
            if isinstance(parsed.get("result"), dict):
                return parsed["result"]

            return parsed

    raise RuntimeError(
        "The MCP response could not be parsed as structured JSON."
    )


def validate_mcp_payload(payload):
    """
    Validate the answer, trusted tenant, and tenant-owned source files.
    """
    answer = str(payload.get("answer", "")).strip()
    tenant_id = payload.get("tenant_id")
    sources = payload.get("sources", [])

    if not answer:
        raise RuntimeError("The MCP response contained no answer.")

    if tenant_id != ACTIVE_DEMO_TENANT_ID:
        raise RuntimeError(
            "The MCP response returned the wrong tenant."
        )

    if not isinstance(sources, list):
        raise RuntimeError("The MCP sources field is invalid.")

    unexpected_files = {
        source.get("filename")
        for source in sources
        if isinstance(source, dict)
        and source.get("filename") != ACTIVE_DEMO_FILENAME
    }

    if unexpected_files:
        raise RuntimeError(
            "The MCP response leaked unexpected files: "
            f"{sorted(unexpected_files)}"
        )

    return {
        "answer": answer,
        "tenant_id": tenant_id,
        "sources": sources,
    }


@vaultify_mcp.tool()
def ask_documents(question: str) -> dict[str, Any]:
    """
    Answer using only the server-configured tenant's indexed documents.
    """
    cleaned_question = question.strip()

    if not cleaned_question:
        raise ToolError("The question cannot be empty.")

    try:
        result = answer_tenant_question(
            question=cleaned_question,
            tenant_id=ACTIVE_DEMO_TENANT_ID,
            retrieval_limit=MCP_RETRIEVAL_LIMIT,
        )

    except Exception as error:
        raise ToolError(
            "Vaultify could not process the document question."
        ) from error

    return {
        "answer": result["answer"],
        "tenant_id": ACTIVE_DEMO_TENANT_ID,
        "sources": serialize_retrieval_sources(
            result["results"]
        ),
    }


MCP_TEST_QUESTION = (
    "What were Apple's total net sales "
    "in fiscal year 2025?"
)

async with create_connected_server_and_client_session(
    vaultify_mcp,
    raise_exceptions=True,
) as session:
    tools = await session.list_tools()
    tool_names = [tool.name for tool in tools.tools]

    if "ask_documents" not in tool_names:
        raise RuntimeError("The ask_documents tool was not registered.")

    in_memory_tool_result = await session.call_tool(
        "ask_documents",
        {"question": MCP_TEST_QUESTION},
    )

in_memory_payload = validate_mcp_payload(
    parse_mcp_tool_result(
        in_memory_tool_result
    )
)

print(f"✅ MCP SDK version: {version('mcp')}")
print(f"✅ Registered tools: {tool_names}")
print(f"✅ Bound tenant: {in_memory_payload['tenant_id']}")
print(f"✅ Returned sources: {len(in_memory_payload['sources'])}")
print("\n🤖 In-memory MCP answer:")
print(in_memory_payload["answer"])


✅ MCP SDK version: 1.28.1
✅ Registered tools: ['ask_documents']
✅ Bound tenant: demo_apple_tenant
✅ Returned sources: 5

🤖 In-memory MCP answer:
Apple's total net sales in fiscal year 2025 were $416,161. 
This information was found in the file apple_fy2025_10k.pdf, Section: Note 2 - Revenue.


In [10]:
# VAULTIFY COMPACT - Cell 8: Start and test the local Streamable HTTP MCP server

import socket
import threading
import time

from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client


def is_port_open(host, port):
    """
    Return True when a TCP server is listening on the target port.
    """
    with socket.socket(
        socket.AF_INET,
        socket.SOCK_STREAM,
    ) as connection_socket:
        connection_socket.settimeout(0.5)

        return (
            connection_socket.connect_ex((host, port))
            == 0
        )


def run_vaultify_mcp_server():
    """
    Run the stateless JSON Streamable HTTP server.
    """
    vaultify_mcp.run(
        transport="streamable-http"
    )


if is_port_open(MCP_HOST, MCP_PORT):
    print(
        f"ℹ️ Port {MCP_PORT} is already open. "
        "The existing local MCP server will be reused."
    )
else:
    mcp_server_thread = threading.Thread(
        target=run_vaultify_mcp_server,
        name="vaultify-mcp-server",
        daemon=True,
    )

    mcp_server_thread.start()

    startup_deadline = time.time() + 15

    while (
        time.time() < startup_deadline
        and not is_port_open(MCP_HOST, MCP_PORT)
    ):
        time.sleep(0.25)

if not is_port_open(MCP_HOST, MCP_PORT):
    raise RuntimeError(
        f"The MCP server did not start on port {MCP_PORT}."
    )


async def test_local_mcp_endpoint():
    """
    Run initialization, discovery, and one tool call through local HTTP.
    """
    async with streamablehttp_client(
        LOCAL_MCP_URL
    ) as (
        read_stream,
        write_stream,
        _,
    ):
        async with ClientSession(
            read_stream,
            write_stream,
        ) as session:
            initialization = await session.initialize()
            tools = await session.list_tools()
            tool_names = [
                tool.name
                for tool in tools.tools
            ]

            tool_result = await session.call_tool(
                "ask_documents",
                {"question": MCP_TEST_QUESTION},
            )

            return {
                "server_name": initialization.serverInfo.name,
                "server_version": initialization.serverInfo.version,
                "tool_names": tool_names,
                "tool_result": tool_result,
            }


local_test = await test_local_mcp_endpoint()

local_payload = validate_mcp_payload(
    parse_mcp_tool_result(
        local_test["tool_result"]
    )
)

print("✅ Local Streamable HTTP MCP test passed.")
print(f"🌐 Endpoint: {LOCAL_MCP_URL}")
print(
    f"📦 Server: "
    f"{local_test['server_name']} "
    f"{local_test['server_version']}"
)
print(f"🛠️ Tools: {local_test['tool_names']}")
print(f"🔐 Tenant: {local_payload['tenant_id']}")
print(f"📚 Sources: {len(local_payload['sources'])}")


INFO:     Started server process [1485]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8003 (Press CTRL+C to quit)


INFO:     127.0.0.1:42472 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:42472 - "POST /mcp HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:42482 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:42482 - "POST /mcp HTTP/1.1" 200 OK
✅ Local Streamable HTTP MCP test passed.
🌐 Endpoint: http://127.0.0.1:8003/mcp
📦 Server: Vaultify 1.28.1
🛠️ Tools: ['ask_documents']
🔐 Tenant: demo_apple_tenant
📚 Sources: 5


In [11]:
# VAULTIFY COMPACT - Cell 9: Start a Cloudflare Quick Tunnel for MCP

import re
import shutil
import subprocess
import threading
import time

CLOUDFLARED_BINARY = "cloudflared"
CLOUDFLARE_STARTUP_TIMEOUT_SECONDS = 60


def install_cloudflared_if_missing():
    """
    Install the Cloudflare tunnel client in the Colab runtime.
    """
    if shutil.which(CLOUDFLARED_BINARY):
        return

    install_command = """
set -e
wget -q \
  https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb \
  -O /tmp/cloudflared.deb
dpkg -i /tmp/cloudflared.deb >/dev/null
rm -f /tmp/cloudflared.deb
"""

    subprocess.run(
        ["bash", "-lc", install_command],
        check=True,
    )


def stream_cloudflared_logs(process, storage):
    """
    Store Cloudflare logs while displaying useful startup messages.
    """
    if process.stderr is None:
        return

    for line in iter(process.stderr.readline, ""):
        cleaned_line = line.strip()

        if cleaned_line:
            storage.append(cleaned_line)
            print(f"[cloudflared] {cleaned_line}")

        if process.poll() is not None:
            break


def extract_trycloudflare_url(log_lines):
    """
    Extract the generated public hostname from Cloudflare logs.
    """
    pattern = re.compile(
        r"https://[a-zA-Z0-9-]+"
        r"\.trycloudflare\.com"
    )

    for line in log_lines:
        match = pattern.search(line)

        if match:
            return match.group(0)

    return None


def tunnel_is_registered(log_lines):
    """
    Return True after Cloudflare registers the tunnel connection.
    """
    return any(
        "Registered tunnel connection" in line
        for line in log_lines
    )


install_cloudflared_if_missing()

if not is_port_open(MCP_HOST, MCP_PORT):
    raise RuntimeError(
        "The local MCP server is not running."
    )

if (
    "cloudflare_tunnel_process" in globals()
    and cloudflare_tunnel_process.poll() is None
):
    cloudflare_tunnel_process.terminate()

    try:
        cloudflare_tunnel_process.wait(timeout=5)
    except subprocess.TimeoutExpired:
        cloudflare_tunnel_process.kill()
        cloudflare_tunnel_process.wait(timeout=5)

cloudflare_tunnel_logs = []

cloudflare_tunnel_process = subprocess.Popen(
    [
        CLOUDFLARED_BINARY,
        "tunnel",
        "--url",
        f"http://{MCP_HOST}:{MCP_PORT}",
        "--no-autoupdate",
    ],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.PIPE,
    text=True,
    bufsize=1,
)

cloudflare_log_thread = threading.Thread(
    target=stream_cloudflared_logs,
    args=(
        cloudflare_tunnel_process,
        cloudflare_tunnel_logs,
    ),
    name="vaultify-cloudflare-log-reader",
    daemon=True,
)

cloudflare_log_thread.start()

startup_deadline = (
    time.time()
    + CLOUDFLARE_STARTUP_TIMEOUT_SECONDS
)

CLOUDFLARE_PUBLIC_URL = None

while time.time() < startup_deadline:
    if cloudflare_tunnel_process.poll() is not None:
        raise RuntimeError(
            "cloudflared stopped unexpectedly. "
            f"Recent logs: {cloudflare_tunnel_logs[-10:]}"
        )

    CLOUDFLARE_PUBLIC_URL = extract_trycloudflare_url(
        cloudflare_tunnel_logs
    )

    if (
        CLOUDFLARE_PUBLIC_URL
        and tunnel_is_registered(
            cloudflare_tunnel_logs
        )
    ):
        break

    time.sleep(0.25)

if not CLOUDFLARE_PUBLIC_URL:
    raise RuntimeError(
        "Cloudflare did not provide a public URL."
    )

if not tunnel_is_registered(
    cloudflare_tunnel_logs
):
    raise RuntimeError(
        "The URL was created, but the tunnel was not registered."
    )

PUBLIC_MCP_URL = (
    CLOUDFLARE_PUBLIC_URL.rstrip("/")
    + "/mcp"
)

print("\n✅ Cloudflare Quick Tunnel started.")
print(f"🌍 Public URL: {CLOUDFLARE_PUBLIC_URL}")
print(f"🔗 Public MCP endpoint: {PUBLIC_MCP_URL}")


[cloudflared] 2026-07-23T15:16:44Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
[cloudflared] 2026-07-23T15:16:44Z INF Requesting new quick Tunnel on trycloudflare.com...
[cloudflared] 2026-07-23T15:16:48Z INF +--------------------------------------------------------------------------------------------+
[cloudflared] 2026-07-23T15:16:48Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
[cloudflared] 2026-07-23T

In [13]:
# VAULTIFY COMPACT - Cell 10: Resolve tunnel DNS and verify the public MCP endpoint

import asyncio
import socket
from urllib.parse import urlparse

import requests

PUBLIC_TEST_ATTEMPTS = 12
PUBLIC_TEST_RETRY_SECONDS = 3


def ensure_public_hostname_resolves(public_url):
    """
    Resolve a new Quick Tunnel hostname, with DNS-over-HTTPS as a fallback.
    """
    hostname = urlparse(public_url).hostname

    if not hostname:
        raise RuntimeError("The public tunnel hostname is invalid.")

    try:
        resolved_ip = socket.gethostbyname(hostname)
        print(f"✅ DNS resolved normally: {hostname} → {resolved_ip}")
        return
    except socket.gaierror:
        print("ℹ️ Normal DNS is not ready. Trying DNS-over-HTTPS.")

    dns_response = requests.get(
        "https://dns.google/resolve",
        params={
            "name": hostname,
            "type": "A",
        },
        timeout=15,
    )

    dns_response.raise_for_status()

    answers = dns_response.json().get(
        "Answer",
        [],
    )

    ipv4_addresses = [
        answer["data"]
        for answer in answers
        if answer.get("type") == 1
    ]

    if not ipv4_addresses:
        raise RuntimeError(
            "The Quick Tunnel hostname has no IPv4 DNS answer yet."
        )

    resolved_ip = ipv4_addresses[0]
    hosts_entry = f"{resolved_ip} {hostname}\n"

    with open(
        "/etc/hosts",
        "r",
        encoding="utf-8",
    ) as hosts_file:
        existing_hosts = hosts_file.read()

    if hostname not in existing_hosts:
        with open(
            "/etc/hosts",
            "a",
            encoding="utf-8",
        ) as hosts_file:
            hosts_file.write(hosts_entry)

    print(
        f"✅ Added temporary DNS fallback: "
        f"{hostname} → {resolved_ip}"
    )


async def call_public_mcp_once():
    """
    Run one complete MCP request through the public Cloudflare endpoint.
    """
    async with streamablehttp_client(
        PUBLIC_MCP_URL
    ) as (
        read_stream,
        write_stream,
        _,
    ):
        async with ClientSession(
            read_stream,
            write_stream,
        ) as session:
            initialization = await session.initialize()
            tools = await session.list_tools()
            tool_names = [
                tool.name
                for tool in tools.tools
            ]

            if "ask_documents" not in tool_names:
                raise RuntimeError(
                    "The ask_documents tool is missing publicly."
                )

            tool_result = await session.call_tool(
                "ask_documents",
                {"question": MCP_TEST_QUESTION},
            )

            return {
                "server_name": initialization.serverInfo.name,
                "server_version": initialization.serverInfo.version,
                "tool_names": tool_names,
                "tool_result": tool_result,
            }


ensure_public_hostname_resolves(
    CLOUDFLARE_PUBLIC_URL
)

public_test = None
last_public_error = None

for attempt_number in range(
    1,
    PUBLIC_TEST_ATTEMPTS + 1,
):
    try:
        public_test = await call_public_mcp_once()
        break

    except BaseException as error:
        last_public_error = error

        print(
            f"⏳ Public MCP attempt "
            f"{attempt_number}/"
            f"{PUBLIC_TEST_ATTEMPTS} failed: "
            f"{type(error).__name__}: {error}"
        )

        if attempt_number < PUBLIC_TEST_ATTEMPTS:
            await asyncio.sleep(
                PUBLIC_TEST_RETRY_SECONDS
            )

if public_test is None:
    raise RuntimeError(
        "The public MCP endpoint remained unreachable."
    ) from last_public_error

public_payload = validate_mcp_payload(
    parse_mcp_tool_result(
        public_test["tool_result"]
    )
)

print("\n✅ Public MCP test passed.")
print(
    f"📦 Server: "
    f"{public_test['server_name']} "
    f"{public_test['server_version']}"
)
print(f"🛠️ Tools: {public_test['tool_names']}")
print(f"🔐 Tenant: {public_payload['tenant_id']}")
print(f"📚 Sources: {len(public_payload['sources'])}")
print("\n🤖 Public answer:")
print(public_payload["answer"])

print("\n🚀 Claude Custom Connector endpoint:")
print(PUBLIC_MCP_URL)
print(
    "\n⚠️ This Quick Tunnel is temporary and works only "
    "while the Colab runtime remains connected."
)


✅ DNS resolved normally: developing-ultimate-bizarre-putting.trycloudflare.com → 104.16.230.132
INFO:     34.16.167.140:0 - "POST /mcp HTTP/1.1" 200 OK
INFO:     34.16.167.140:0 - "POST /mcp HTTP/1.1" 202 Accepted
INFO:     34.16.167.140:0 - "POST /mcp HTTP/1.1" 200 OK
INFO:     34.16.167.140:0 - "POST /mcp HTTP/1.1" 200 OK

✅ Public MCP test passed.
📦 Server: Vaultify 1.28.1
🛠️ Tools: ['ask_documents']
🔐 Tenant: demo_apple_tenant
📚 Sources: 5

🤖 Public answer:
According to the table in Section: Note 2 - Revenue of the apple_fy2025_10k.pdf file, Apple's total net sales in fiscal year 2025 were $416,161. (Source: Context 3, Section: Note 2 - Revenue)

🚀 Claude Custom Connector endpoint:
https://developing-ultimate-bizarre-putting.trycloudflare.com/mcp

⚠️ This Quick Tunnel is temporary and works only while the Colab runtime remains connected.


In [14]:
# VAULTIFY COMPACT - Cell 11: Create the compact Flask frontend files

from pathlib import Path
import textwrap

WEB_PROJECT_ROOT = Path("/content/vaultify_compact_web")
WEB_TEMPLATES_DIR = WEB_PROJECT_ROOT / "templates"
WEB_STATIC_DIR = WEB_PROJECT_ROOT / "static"
WEB_UPLOADS_DIR = WEB_PROJECT_ROOT / "uploads"
WEB_INSTANCE_DIR = WEB_PROJECT_ROOT / "instance"

for directory in [
    WEB_TEMPLATES_DIR,
    WEB_STATIC_DIR,
    WEB_UPLOADS_DIR,
    WEB_INSTANCE_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)


def write_text_file(path, content):
    """
    Write one UTF-8 project file after removing shared indentation.
    """
    path.write_text(
        textwrap.dedent(content).strip() + "\n",
        encoding="utf-8",
    )


write_text_file(
    WEB_STATIC_DIR / "vaultify.css",
    r"""
    :root {
        color-scheme: dark;
        --bg: #080b12;
        --panel: #101522;
        --border: #283044;
        --text: #f5f7ff;
        --muted: #9da8c2;
        --accent: #8066ff;
        --success: #39e6bd;
        --danger: #ff6f91;
        --warning: #ffc96b;
        --max: 1120px;
    }

    * {
        box-sizing: border-box;
    }

    html {
        background: var(--bg);
    }

    body {
        margin: 0;
        min-height: 100vh;
        background:
            radial-gradient(
                circle at 15% 0%,
                rgba(62, 70, 155, 0.18),
                transparent 30rem
            ),
            var(--bg);
        color: var(--text);
        font-family:
            Inter,
            ui-sans-serif,
            system-ui,
            -apple-system,
            BlinkMacSystemFont,
            "Segoe UI",
            sans-serif;
        font-size: 15px;
        line-height: 1.5;
    }

    a {
        color: #a9b5ff;
        text-decoration: none;
    }

    a:hover {
        color: #d2d8ff;
    }

    button,
    input,
    select {
        font: inherit;
    }

    .shell {
        width: min(var(--max), calc(100% - 32px));
        margin: 0 auto;
    }

    .topbar {
        position: sticky;
        top: 0;
        z-index: 20;
        border-bottom: 1px solid rgba(40, 48, 68, 0.85);
        background: rgba(8, 11, 18, 0.92);
        backdrop-filter: blur(16px);
    }

    .nav {
        min-height: 64px;
        display: flex;
        align-items: center;
        gap: 18px;
    }

    .brand {
        color: white;
        font-size: 20px;
        font-weight: 800;
        letter-spacing: -0.04em;
        margin-right: auto;
    }

    .nav-links {
        display: flex;
        align-items: center;
        gap: 6px;
        flex-wrap: wrap;
        justify-content: flex-end;
    }

    .nav-link,
    .nav-button {
        border: 0;
        background: transparent;
        color: #b5bdd2;
        border-radius: 10px;
        padding: 8px 11px;
        cursor: pointer;
    }

    .nav-link:hover,
    .nav-link.active,
    .nav-button:hover {
        color: white;
        background: rgba(128, 102, 255, 0.13);
    }

    .org-switcher {
        max-width: 190px;
        min-width: 150px;
    }

    .page {
        padding: 38px 0 60px;
    }

    .eyebrow {
        margin: 0 0 10px;
        color: #8298ff;
        font-size: 12px;
        font-weight: 800;
        letter-spacing: 0.14em;
        text-transform: uppercase;
    }

    h1,
    h2,
    h3,
    p {
        margin-top: 0;
    }

    h1 {
        margin-bottom: 8px;
        font-size: clamp(32px, 5vw, 50px);
        line-height: 1.03;
        letter-spacing: -0.045em;
    }

    h2 {
        font-size: 22px;
        letter-spacing: -0.02em;
    }

    .lead,
    .muted {
        color: var(--muted);
    }

    .lead {
        margin-bottom: 26px;
        max-width: 740px;
        font-size: 16px;
    }

    .grid {
        display: grid;
        gap: 16px;
    }

    .stats {
        grid-template-columns: repeat(4, minmax(0, 1fr));
    }

    .two-column {
        grid-template-columns:
            minmax(0, 1.35fr)
            minmax(280px, 0.65fr);
    }

    .panel,
    .stat,
    .auth-card {
        border: 1px solid var(--border);
        background:
            linear-gradient(
                180deg,
                rgba(17, 22, 36, 0.98),
                rgba(13, 18, 30, 0.98)
            );
        box-shadow: 0 18px 55px rgba(0, 0, 0, 0.2);
    }

    .panel {
        border-radius: 18px;
        padding: 24px;
    }

    .stat {
        min-height: 112px;
        border-radius: 16px;
        padding: 19px;
    }

    .stat-label {
        color: #8ea1c7;
        font-size: 12px;
        letter-spacing: 0.08em;
        text-transform: uppercase;
    }

    .stat-value {
        margin-top: 10px;
        font-size: 23px;
        font-weight: 800;
        overflow-wrap: anywhere;
    }

    .flash-stack {
        display: grid;
        gap: 10px;
        margin-bottom: 18px;
    }

    .flash {
        border: 1px solid var(--border);
        border-radius: 12px;
        padding: 12px 14px;
        background: #11192a;
    }

    .flash.success {
        border-color: rgba(57, 230, 189, 0.38);
        color: var(--success);
    }

    .flash.error {
        border-color: rgba(255, 111, 145, 0.38);
        color: var(--danger);
    }

    .flash.warning {
        border-color: rgba(255, 201, 107, 0.38);
        color: var(--warning);
    }

    label {
        display: block;
        margin-bottom: 7px;
        font-size: 13px;
        font-weight: 700;
    }

    .field {
        margin-bottom: 15px;
    }

    input,
    select {
        width: 100%;
        min-height: 44px;
        border: 1px solid #35405a;
        border-radius: 11px;
        background: #0b1020;
        color: white;
        padding: 10px 12px;
        outline: none;
    }

    input:focus,
    select:focus {
        border-color: var(--accent);
        box-shadow:
            0 0 0 3px rgba(128, 102, 255, 0.16);
    }

    input:-webkit-autofill {
        -webkit-text-fill-color: white;
        box-shadow:
            0 0 0 1000px #172036 inset;
        transition:
            background-color 9999s ease-out;
    }

    .button {
        display: inline-flex;
        min-height: 42px;
        align-items: center;
        justify-content: center;
        border: 0;
        border-radius: 11px;
        padding: 10px 16px;
        background:
            linear-gradient(
                90deg,
                #7565ff,
                #9866ff
            );
        color: white;
        font-weight: 800;
        cursor: pointer;
    }

    .button:hover {
        filter: brightness(1.08);
        color: white;
    }

    .button.secondary {
        border: 1px solid var(--border);
        background: #171e2e;
    }

    .button.danger {
        background: #542139;
    }

    .button.small {
        min-height: 34px;
        padding: 7px 11px;
        font-size: 13px;
    }

    .button-row {
        display: flex;
        flex-wrap: wrap;
        gap: 10px;
        align-items: center;
    }

    .auth-wrap {
        min-height: calc(100vh - 65px);
        display: grid;
        place-items: center;
        padding: 24px 16px;
    }

    .auth-card {
        width: min(450px, 100%);
        border-radius: 18px;
        padding: 26px;
    }

    .auth-card h1 {
        font-size: clamp(30px, 7vw, 41px);
    }

    .auth-card .button {
        width: 100%;
    }

    .table-wrap {
        overflow-x: auto;
    }

    table {
        width: 100%;
        border-collapse: collapse;
    }

    th,
    td {
        border-bottom: 1px solid var(--border);
        padding: 13px 10px;
        text-align: left;
        vertical-align: top;
    }

    th {
        color: #9caccc;
        font-size: 12px;
        letter-spacing: 0.06em;
        text-transform: uppercase;
    }

    .status {
        display: inline-flex;
        border-radius: 999px;
        padding: 4px 9px;
        background: rgba(128, 102, 255, 0.14);
        color: #bcb5ff;
        font-size: 12px;
        font-weight: 800;
    }

    .status.ready {
        background: rgba(57, 230, 189, 0.12);
        color: var(--success);
    }

    .status.failed {
        background: rgba(255, 111, 145, 0.12);
        color: var(--danger);
    }

    .answer {
        margin-top: 18px;
        border-left: 3px solid var(--accent);
        border-radius: 0 12px 12px 0;
        background: #0a1020;
        padding: 18px;
        white-space: pre-wrap;
    }

    .source-list {
        margin: 12px 0 0;
        padding-left: 20px;
        color: var(--muted);
    }

    .empty {
        border: 1px dashed #35405a;
        border-radius: 16px;
        padding: 34px 20px;
        text-align: center;
        color: var(--muted);
    }

    code {
        color: #d8dbff;
        overflow-wrap: anywhere;
    }

    @media (max-width: 900px) {
        .stats,
        .two-column {
            grid-template-columns: 1fr 1fr;
        }

        .nav {
            align-items: flex-start;
            padding: 14px 0;
            flex-wrap: wrap;
        }

        .brand {
            width: 100%;
        }

        .nav-links {
            width: 100%;
            justify-content: flex-start;
        }
    }

    @media (max-width: 620px), (max-height: 720px) {
        .page {
            padding-top: 24px;
        }

        .stats,
        .two-column {
            grid-template-columns: 1fr;
        }

        .panel,
        .auth-card {
            padding: 19px;
        }

        .auth-wrap {
            place-items: start center;
            padding-top: 20px;
        }

        h1 {
            font-size: 34px;
        }
    }
    """,
)

write_text_file(
    WEB_TEMPLATES_DIR / "base.html",
    r"""
    <!doctype html>
    <html lang="en">
    <head>
        <meta charset="utf-8">
        <meta
            name="viewport"
            content="width=device-width, initial-scale=1"
        >
        <title>{{ title or "Vaultify" }}</title>
        <link
            rel="stylesheet"
            href="{{ url_for('static', filename='vaultify.css') }}"
        >
    </head>
    <body>
        <header class="topbar">
            <div class="shell nav">
                <a
                    class="brand"
                    href="{{
                        url_for('dashboard')
                        if current_user.is_authenticated
                        else url_for('login')
                    }}"
                >
                    Vaultify
                </a>

                {% if current_user.is_authenticated %}
                <form
                    method="post"
                    action="{{ url_for('switch_organization') }}"
                >
                    <input
                        type="hidden"
                        name="csrf_token"
                        value="{{ csrf_token() }}"
                    >

                    <select
                        class="org-switcher"
                        name="organization_id"
                        aria-label="Active organization"
                        onchange="this.form.submit()"
                    >
                        {% for membership in memberships %}
                        <option
                            value="{{ membership.organization.id }}"
                            {% if
                                current_org
                                and membership.organization.id
                                == current_org.id
                            %}selected{% endif %}
                        >
                            {{ membership.organization.name }}
                        </option>
                        {% endfor %}
                    </select>
                </form>

                <nav
                    class="nav-links"
                    aria-label="Primary navigation"
                >
                    <a
                        class="nav-link
                        {% if request.endpoint == 'dashboard' %}
                        active
                        {% endif %}"
                        href="{{ url_for('dashboard') }}"
                    >
                        Dashboard
                    </a>

                    <a
                        class="nav-link
                        {% if request.endpoint in
                            ['documents', 'upload_document']
                        %}
                        active
                        {% endif %}"
                        href="{{ url_for('documents') }}"
                    >
                        Documents
                    </a>

                    <a
                        class="nav-link
                        {% if request.endpoint == 'organization' %}
                        active
                        {% endif %}"
                        href="{{ url_for('organization') }}"
                    >
                        Organization
                    </a>

                    <span class="nav-link">
                        {{ current_user.display_name }}
                    </span>

                    <form
                        method="post"
                        action="{{ url_for('logout') }}"
                    >
                        <input
                            type="hidden"
                            name="csrf_token"
                            value="{{ csrf_token() }}"
                        >

                        <button
                            class="nav-button"
                            type="submit"
                        >
                            Sign out
                        </button>
                    </form>
                </nav>
                {% endif %}
            </div>
        </header>

        {% if current_user.is_authenticated %}
        <main class="shell page">
        {% else %}
        <main>
        {% endif %}

            {% with
                messages =
                get_flashed_messages(
                    with_categories=true
                )
            %}
                {% if messages %}
                <div
                    class="
                    {% if current_user.is_authenticated %}
                    flash-stack
                    {% else %}
                    shell flash-stack
                    {% endif %}
                    "
                >
                    {% for category, message in messages %}
                    <div class="flash {{ category }}">
                        {{ message }}
                    </div>
                    {% endfor %}
                </div>
                {% endif %}
            {% endwith %}

            {% block content %}{% endblock %}
        </main>
    </body>
    </html>
    """,
)

write_text_file(
    WEB_TEMPLATES_DIR / "login.html",
    r"""
    {% extends "base.html" %}

    {% block content %}
    <section class="auth-wrap">
        <div class="auth-card">
            <p class="eyebrow">Welcome back</p>
            <h1>Sign in to Vaultify</h1>

            <p class="lead">
                Access your organization's
                private document workspace.
            </p>

            <form method="post">
                <input
                    type="hidden"
                    name="csrf_token"
                    value="{{ csrf_token() }}"
                >

                <div class="field">
                    <label for="email">
                        Email address
                    </label>

                    <input
                        id="email"
                        name="email"
                        type="email"
                        autocomplete="email"
                        required
                    >
                </div>

                <div class="field">
                    <label for="password">
                        Password
                    </label>

                    <input
                        id="password"
                        name="password"
                        type="password"
                        autocomplete="current-password"
                        required
                    >
                </div>

                <button
                    class="button"
                    type="submit"
                >
                    Sign in
                </button>
            </form>

            <p
                class="muted"
                style="margin-top:16px"
            >
                New to Vaultify?
                <a href="{{ url_for('register') }}">
                    Create an account
                </a>
            </p>
        </div>
    </section>
    {% endblock %}
    """,
)

write_text_file(
    WEB_TEMPLATES_DIR / "register.html",
    r"""
    {% extends "base.html" %}

    {% block content %}
    <section class="auth-wrap">
        <div class="auth-card">
            <p class="eyebrow">
                Start using Vaultify
            </p>

            <h1>Create your account</h1>

            <p class="lead">
                Your first secure organization
                will be created automatically.
            </p>

            <form method="post">
                <input
                    type="hidden"
                    name="csrf_token"
                    value="{{ csrf_token() }}"
                >

                <div class="field">
                    <label for="display_name">
                        Full name
                    </label>

                    <input
                        id="display_name"
                        name="display_name"
                        maxlength="100"
                        required
                    >
                </div>

                <div class="field">
                    <label for="organization_name">
                        Organization name
                    </label>

                    <input
                        id="organization_name"
                        name="organization_name"
                        maxlength="120"
                        required
                    >
                </div>

                <div class="field">
                    <label for="email">
                        Email address
                    </label>

                    <input
                        id="email"
                        name="email"
                        type="email"
                        autocomplete="email"
                        required
                    >
                </div>

                <div class="field">
                    <label for="password">
                        Password
                    </label>

                    <input
                        id="password"
                        name="password"
                        type="password"
                        minlength="10"
                        autocomplete="new-password"
                        required
                    >
                </div>

                <div class="field">
                    <label for="confirm_password">
                        Confirm password
                    </label>

                    <input
                        id="confirm_password"
                        name="confirm_password"
                        type="password"
                        minlength="10"
                        autocomplete="new-password"
                        required
                    >
                </div>

                <button
                    class="button"
                    type="submit"
                >
                    Create account
                </button>
            </form>

            <p
                class="muted"
                style="margin-top:16px"
            >
                Already have an account?
                <a href="{{ url_for('login') }}">
                    Sign in
                </a>
            </p>
        </div>
    </section>
    {% endblock %}
    """,
)

write_text_file(
    WEB_TEMPLATES_DIR / "dashboard.html",
    r"""
    {% extends "base.html" %}

    {% block content %}
    <p class="eyebrow">
        Private AI workspace
    </p>

    <h1>
        Welcome, {{ current_user.display_name }}
    </h1>

    <p class="lead">
        Your tenant context is resolved
        from your authenticated organization membership.
    </p>

    <section class="grid stats">
        <article class="stat">
            <div class="stat-label">
                Organization
            </div>

            <div class="stat-value">
                {{ current_org.name }}
            </div>
        </article>

        <article class="stat">
            <div class="stat-label">
                Documents
            </div>

            <div class="stat-value">
                {{ document_count }}
            </div>
        </article>

        <article class="stat">
            <div class="stat-label">
                Ready chunks
            </div>

            <div class="stat-value">
                {{ chunk_count }}
            </div>
        </article>

        <article class="stat">
            <div class="stat-label">
                Role
            </div>

            <div class="stat-value">
                {{ current_membership.role|title }}
            </div>
        </article>
    </section>

    <section
        class="grid two-column"
        style="margin-top:18px"
    >
        <article class="panel">
            <h2>Ask your documents</h2>

            <p class="muted">
                Answers are restricted to the
                active organization's Qdrant tenant.
            </p>

            <form
                method="post"
                action="{{ url_for('ask') }}"
            >
                <input
                    type="hidden"
                    name="csrf_token"
                    value="{{ csrf_token() }}"
                >

                <div class="field">
                    <label for="question">
                        Question
                    </label>

                    <input
                        id="question"
                        name="question"
                        value="{{ question or '' }}"
                        placeholder="What was total revenue in Q4 2025?"
                        required
                    >
                </div>

                <button
                    class="button"
                    type="submit"
                >
                    Ask Vaultify
                </button>
            </form>

            {% if answer %}
            <div class="answer">
                {{ answer }}
            </div>

            {% if sources %}
            <ul class="source-list">
                {% for source in sources %}
                <li>
                    {{ source.filename }}
                    · {{ source.section }}
                    · score
                    {{ "%.4f"|format(source.score) }}
                </li>
                {% endfor %}
            </ul>
            {% endif %}
            {% endif %}
        </article>

        <aside class="panel">
            <h2>Trusted tenant</h2>

            <p class="muted">
                This value comes from the verified membership,
                not from a user-controlled form field.
            </p>

            <code>
                {{ current_org.tenant_id }}
            </code>

            <div
                class="button-row"
                style="margin-top:18px"
            >
                <a
                    class="button secondary"
                    href="{{ url_for('documents') }}"
                >
                    Open documents
                </a>
            </div>
        </aside>
    </section>
    {% endblock %}
    """,
)

write_text_file(
    WEB_TEMPLATES_DIR / "documents.html",
    r"""
    {% extends "base.html" %}

    {% block content %}
    <div
        class="button-row"
        style="justify-content:space-between"
    >
        <div>
            <p class="eyebrow">
                Knowledge base
            </p>

            <h1>Documents</h1>

            <p class="lead">
                Files belonging to
                {{ current_org.name }}.
            </p>
        </div>

        {% if documents %}
        <a
            class="button"
            href="{{ url_for('upload_document') }}"
        >
            Upload PDF
        </a>
        {% endif %}
    </div>

    {% if documents %}
    <section class="panel table-wrap">
        <table>
            <thead>
                <tr>
                    <th>Document</th>
                    <th>Status</th>
                    <th>Chunks</th>
                    <th>Uploaded</th>
                    <th>Actions</th>
                </tr>
            </thead>

            <tbody>
                {% for document in documents %}
                <tr>
                    <td>
                        <strong>
                            {{ document.original_filename }}
                        </strong>
                        <br>

                        <span class="muted">
                            {{ document.size_bytes }} bytes
                        </span>
                    </td>

                    <td>
                        <span
                            class="status {{ document.status }}"
                        >
                            {{ document.status }}
                        </span>
                    </td>

                    <td>
                        {{ document.chunk_count }}
                    </td>

                    <td>
                        {{
                            document.created_at.strftime(
                                "%Y-%m-%d %H:%M"
                            )
                        }}
                    </td>

                    <td>
                        <div class="button-row">
                            {% if document.status == "failed" %}
                            <form
                                method="post"
                                action="{{
                                    url_for(
                                        'retry_document',
                                        document_id=document.id
                                    )
                                }}"
                            >
                                <input
                                    type="hidden"
                                    name="csrf_token"
                                    value="{{ csrf_token() }}"
                                >

                                <button
                                    class="button secondary small"
                                    type="submit"
                                >
                                    Retry
                                </button>
                            </form>
                            {% endif %}

                            <form
                                method="post"
                                action="{{
                                    url_for(
                                        'delete_document',
                                        document_id=document.id
                                    )
                                }}"
                            >
                                <input
                                    type="hidden"
                                    name="csrf_token"
                                    value="{{ csrf_token() }}"
                                >

                                <button
                                    class="button danger small"
                                    type="submit"
                                >
                                    Delete
                                </button>
                            </form>
                        </div>
                    </td>
                </tr>
                {% endfor %}
            </tbody>
        </table>
    </section>

    {% else %}
    <section class="empty">
        <h2>No documents yet</h2>

        <p>
            Upload the first PDF
            for this organization.
        </p>

        <a
            class="button"
            href="{{ url_for('upload_document') }}"
        >
            Upload first document
        </a>
    </section>
    {% endif %}
    {% endblock %}
    """,
)

write_text_file(
    WEB_TEMPLATES_DIR / "upload.html",
    r"""
    {% extends "base.html" %}

    {% block content %}
    <p class="eyebrow">
        Secure ingestion
    </p>

    <h1>Upload a PDF</h1>

    <p class="lead">
        The file is validated, parsed,
        embedded, and indexed only under
        {{ current_org.name }}.
    </p>

    <section
        class="panel"
        style="max-width:720px"
    >
        <form
            method="post"
            enctype="multipart/form-data"
        >
            <input
                type="hidden"
                name="csrf_token"
                value="{{ csrf_token() }}"
            >

            <div class="field">
                <label for="file">
                    PDF document
                </label>

                <input
                    id="file"
                    name="file"
                    type="file"
                    accept=".pdf,application/pdf"
                    required
                >
            </div>

            <button
                class="button"
                type="submit"
            >
                Upload and index
            </button>
        </form>

        <p
            class="muted"
            style="margin-top:16px"
        >
            The compact version currently
            processes uploads synchronously.
            Large reports can take one or two minutes.
        </p>
    </section>
    {% endblock %}
    """,
)

write_text_file(
    WEB_TEMPLATES_DIR / "organization.html",
    r"""
    {% extends "base.html" %}

    {% block content %}
    <p class="eyebrow">
        Organization management
    </p>

    <h1>
        {{ current_org.name }}
    </h1>

    <p class="lead">
        This page shows the trusted organization
        context used by documents, retrieval, and MCP.
    </p>

    <section class="grid stats">
        <article class="stat">
            <div class="stat-label">
                Your role
            </div>

            <div class="stat-value">
                {{ current_membership.role|title }}
            </div>
        </article>

        <article class="stat">
            <div class="stat-label">
                Organization slug
            </div>

            <div class="stat-value">
                {{ current_org.slug }}
            </div>
        </article>

        <article
            class="stat"
            style="grid-column:span 2"
        >
            <div class="stat-label">
                Trusted tenant
            </div>

            <div class="stat-value">
                <code>
                    {{ current_org.tenant_id }}
                </code>
            </div>
        </article>
    </section>
    {% endblock %}
    """,
)

print(
    "✅ Compact Flask templates "
    "and responsive CSS created."
)
print(
    f"📁 Web project root: "
    f"{WEB_PROJECT_ROOT}"
)

✅ Compact Flask templates and responsive CSS created.
📁 Web project root: /content/vaultify_compact_web


In [15]:
# VAULTIFY COMPACT - Cell 12: Create the compact Flask backend

import textwrap

WEB_BACKEND_PATH = (
    WEB_PROJECT_ROOT
    / "vaultify_web_backend.py"
)

WEB_BACKEND_CODE = r"""
import hashlib
import re
import secrets
import unicodedata
import uuid
from datetime import datetime, timezone
from pathlib import Path

import filetype
import torch
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import (
    AcceleratorDevice,
    AcceleratorOptions,
    PdfPipelineOptions,
)
from docling.document_converter import (
    DocumentConverter,
    PdfFormatOption,
)
from flask import (
    Flask,
    abort,
    flash,
    redirect,
    render_template,
    request,
    session,
    url_for,
)
from flask_login import (
    LoginManager,
    UserMixin,
    current_user,
    login_required,
    login_user,
    logout_user,
)
from flask_sqlalchemy import SQLAlchemy
from flask_wtf.csrf import CSRFProtect
from qdrant_client.models import (
    FieldCondition,
    Filter,
    FilterSelector,
    MatchValue,
    PointStruct,
)
from sqlalchemy import UniqueConstraint
from werkzeug.security import (
    check_password_hash,
    generate_password_hash,
)
from werkzeug.utils import secure_filename


db = SQLAlchemy()
login_manager = LoginManager()
csrf = CSRFProtect()


def utc_now():
    return datetime.now(timezone.utc)


class User(UserMixin, db.Model):
    __tablename__ = "users"

    id = db.Column(
        db.Integer,
        primary_key=True,
    )

    email = db.Column(
        db.String(255),
        unique=True,
        index=True,
        nullable=False,
    )

    password_hash = db.Column(
        db.String(255),
        nullable=False,
    )

    display_name = db.Column(
        db.String(100),
        nullable=False,
    )

    is_active_account = db.Column(
        db.Boolean,
        nullable=False,
        default=True,
    )

    created_at = db.Column(
        db.DateTime(timezone=True),
        nullable=False,
        default=utc_now,
    )

    memberships = db.relationship(
        "Membership",
        back_populates="user",
        cascade="all, delete-orphan",
    )

    @property
    def is_active(self):
        return self.is_active_account

    def set_password(self, password):
        self.password_hash = (
            generate_password_hash(
                password
            )
        )

    def check_password(self, password):
        return check_password_hash(
            self.password_hash,
            password,
        )


class Organization(db.Model):
    __tablename__ = "organizations"

    id = db.Column(
        db.Integer,
        primary_key=True,
    )

    tenant_id = db.Column(
        db.String(80),
        unique=True,
        index=True,
        nullable=False,
        default=lambda: (
            f"tenant_{secrets.token_hex(12)}"
        ),
    )

    name = db.Column(
        db.String(120),
        nullable=False,
    )

    slug = db.Column(
        db.String(140),
        unique=True,
        index=True,
        nullable=False,
    )

    created_at = db.Column(
        db.DateTime(timezone=True),
        nullable=False,
        default=utc_now,
    )

    memberships = db.relationship(
        "Membership",
        back_populates="organization",
        cascade="all, delete-orphan",
    )

    documents = db.relationship(
        "Document",
        back_populates="organization",
        cascade="all, delete-orphan",
    )


class Membership(db.Model):
    __tablename__ = "memberships"

    __table_args__ = (
        UniqueConstraint(
            "user_id",
            "organization_id",
            name=(
                "uq_membership_"
                "user_organization"
            ),
        ),
    )

    id = db.Column(
        db.Integer,
        primary_key=True,
    )

    user_id = db.Column(
        db.Integer,
        db.ForeignKey(
            "users.id",
            ondelete="CASCADE",
        ),
        nullable=False,
        index=True,
    )

    organization_id = db.Column(
        db.Integer,
        db.ForeignKey(
            "organizations.id",
            ondelete="CASCADE",
        ),
        nullable=False,
        index=True,
    )

    role = db.Column(
        db.String(20),
        nullable=False,
        default="member",
    )

    user = db.relationship(
        "User",
        back_populates="memberships",
    )

    organization = db.relationship(
        "Organization",
        back_populates="memberships",
    )


class Document(db.Model):
    __tablename__ = "documents"

    __table_args__ = (
        UniqueConstraint(
            "organization_id",
            "document_hash",
            name=(
                "uq_document_"
                "organization_hash"
            ),
        ),
    )

    id = db.Column(
        db.Integer,
        primary_key=True,
    )

    organization_id = db.Column(
        db.Integer,
        db.ForeignKey(
            "organizations.id",
            ondelete="CASCADE",
        ),
        nullable=False,
        index=True,
    )

    original_filename = db.Column(
        db.String(255),
        nullable=False,
    )

    stored_filename = db.Column(
        db.String(255),
        nullable=False,
        unique=True,
    )

    storage_path = db.Column(
        db.String(500),
        nullable=False,
    )

    mime_type = db.Column(
        db.String(120),
        nullable=False,
    )

    size_bytes = db.Column(
        db.Integer,
        nullable=False,
    )

    document_hash = db.Column(
        db.String(64),
        nullable=False,
        index=True,
    )

    status = db.Column(
        db.String(20),
        nullable=False,
        default="uploaded",
    )

    chunk_count = db.Column(
        db.Integer,
        nullable=False,
        default=0,
    )

    error_message = db.Column(
        db.String(500),
    )

    created_at = db.Column(
        db.DateTime(timezone=True),
        nullable=False,
        default=utc_now,
    )

    indexed_at = db.Column(
        db.DateTime(timezone=True),
    )

    organization = db.relationship(
        "Organization",
        back_populates="documents",
    )


class QueryLog(db.Model):
    __tablename__ = "query_logs"

    id = db.Column(
        db.Integer,
        primary_key=True,
    )

    organization_id = db.Column(
        db.Integer,
        db.ForeignKey(
            "organizations.id",
            ondelete="CASCADE",
        ),
        nullable=False,
        index=True,
    )

    user_id = db.Column(
        db.Integer,
        db.ForeignKey(
            "users.id",
            ondelete="SET NULL",
        ),
        nullable=True,
        index=True,
    )

    question = db.Column(
        db.Text,
        nullable=False,
    )

    answer = db.Column(
        db.Text,
        nullable=False,
    )

    source_count = db.Column(
        db.Integer,
        nullable=False,
        default=0,
    )

    created_at = db.Column(
        db.DateTime(timezone=True),
        nullable=False,
        default=utc_now,
    )


@login_manager.user_loader
def load_user(user_id):
    try:
        return db.session.get(
            User,
            int(user_id),
        )
    except (TypeError, ValueError):
        return None


def normalize_email(value):
    return (
        value or ""
    ).strip().lower()


def create_slug(value):
    normalized = unicodedata.normalize(
        "NFKD",
        value or "",
    )

    ascii_value = normalized.encode(
        "ascii",
        "ignore",
    ).decode("ascii")

    slug = re.sub(
        r"[^a-zA-Z0-9]+",
        "-",
        ascii_value,
    ).strip("-").lower()

    return slug or "organization"


def unique_organization_slug(name):
    base_slug = create_slug(name)
    candidate = base_slug
    counter = 1

    while Organization.query.filter_by(
        slug=candidate
    ).first():
        counter += 1
        candidate = (
            f"{base_slug}-{counter}"
        )

    return candidate


def password_is_strong(password):
    return (
        len(password or "") >= 10
        and any(
            character.isalpha()
            for character in password
        )
        and any(
            character.isdigit()
            for character in password
        )
    )


def get_memberships_for_user(user_id):
    return (
        Membership.query
        .filter_by(user_id=user_id)
        .order_by(Membership.id.asc())
        .all()
    )


def resolve_active_membership():
    if not current_user.is_authenticated:
        return None

    memberships = get_memberships_for_user(
        current_user.id
    )

    if not memberships:
        logout_user()
        session.pop(
            "organization_id",
            None,
        )
        return None

    selected_id = session.get(
        "organization_id"
    )

    for membership in memberships:
        if (
            membership.organization_id
            == selected_id
        ):
            return membership

    membership = memberships[0]

    session["organization_id"] = (
        membership.organization_id
    )

    return membership


def require_management_role(membership):
    if membership.role not in {
        "owner",
        "admin",
    }:
        abort(403)


def build_qdrant_document_filter(
    tenant_id,
    document_hash,
):
    return Filter(
        must=[
            FieldCondition(
                key="tenant_id",
                match=MatchValue(
                    value=tenant_id
                ),
            ),
            FieldCondition(
                key="document_hash",
                match=MatchValue(
                    value=document_hash
                ),
            ),
        ]
    )


def delete_document_vectors(
    services,
    tenant_id,
    document_hash,
):
    services["qdrant"].delete(
        collection_name=(
            services["collection_name"]
        ),
        points_selector=FilterSelector(
            filter=(
                build_qdrant_document_filter(
                    tenant_id,
                    document_hash,
                )
            )
        ),
        wait=True,
    )


def token_count(model, text):
    return len(
        model.tokenizer.encode(
            text,
            add_special_tokens=True,
        )
    )


def split_text_by_tokens(
    model,
    text,
    max_tokens=220,
    overlap_tokens=30,
):
    token_ids = model.tokenizer.encode(
        text,
        add_special_tokens=False,
    )

    if not token_ids:
        return []

    pieces = []
    start = 0

    while start < len(token_ids):
        end = min(
            start + max_tokens,
            len(token_ids),
        )

        piece = model.tokenizer.decode(
            token_ids[start:end],
            skip_special_tokens=True,
        ).strip()

        if piece:
            pieces.append(piece)

        if end >= len(token_ids):
            break

        start = max(
            0,
            end - overlap_tokens,
        )

    return pieces


def split_large_table(
    model,
    header,
    table_lines,
    max_tokens=220,
):
    if len(table_lines) >= 2:
        prefix_lines = table_lines[:2]
        body_lines = table_lines[2:]
    else:
        prefix_lines = []
        body_lines = table_lines

    table_chunks = []
    current_lines = list(
        prefix_lines
    )

    for row in body_lines:
        candidate_lines = (
            current_lines + [row]
        )

        candidate = "\n".join(
            [
                header,
                "",
                *candidate_lines,
            ]
        ).strip()

        if (
            current_lines
            and token_count(
                model,
                candidate,
            )
            > max_tokens
        ):
            current_text = "\n".join(
                [
                    header,
                    "",
                    *current_lines,
                ]
            ).strip()

            if current_text:
                table_chunks.append(
                    current_text
                )

            current_lines = (
                list(prefix_lines)
                + [row]
            )
        else:
            current_lines.append(row)

    final_text = "\n".join(
        [
            header,
            "",
            *current_lines,
        ]
    ).strip()

    if final_text:
        table_chunks.append(
            final_text
        )

    return table_chunks


def smart_chunk_markdown(
    model,
    markdown_text,
):
    lines = markdown_text.splitlines()
    chunks = []
    text_buffer = []
    current_header = ""
    index = 0

    def flush_text():
        nonlocal text_buffer

        text = "\n".join(
            text_buffer
        ).strip()

        text_buffer = []

        if not text:
            return

        for piece in split_text_by_tokens(
            model,
            text,
        ):
            chunks.append(
                {
                    "text": piece,
                    "chunk_type": "text",
                    "section": (
                        current_header
                        or "Document text"
                    ),
                }
            )

    while index < len(lines):
        line = lines[index]

        if line.strip().startswith("#"):
            current_header = (
                line.strip()
            )

            text_buffer.append(line)
            index += 1
            continue

        if line.strip().startswith("|"):
            flush_text()

            table_lines = []

            while (
                index < len(lines)
                and lines[index]
                .strip()
                .startswith("|")
            ):
                table_lines.append(
                    lines[index]
                )
                index += 1

            full_table = "\n".join(
                [
                    current_header,
                    "",
                    *table_lines,
                ]
            ).strip()

            if (
                token_count(
                    model,
                    full_table,
                )
                <= 220
            ):
                table_pieces = [
                    full_table
                ]
            else:
                table_pieces = (
                    split_large_table(
                        model,
                        current_header,
                        table_lines,
                    )
                )

            for table_piece in table_pieces:
                chunks.append(
                    {
                        "text": table_piece,
                        "chunk_type": "table",
                        "section": (
                            current_header
                            or "Document table"
                        ),
                    }
                )

            continue

        text_buffer.append(line)
        index += 1

    flush_text()

    return chunks


def ingest_document(
    document,
    services,
):
    document.status = "parsing"
    document.error_message = None
    db.session.commit()

    try:
        pipeline_options = (
            PdfPipelineOptions()
        )

        if torch.cuda.is_available():
            pipeline_options.accelerator_options = (
                AcceleratorOptions(
                    num_threads=4,
                    device=(
                        AcceleratorDevice.CUDA
                    ),
                )
            )

        converter = DocumentConverter(
            format_options={
                InputFormat.PDF:
                PdfFormatOption(
                    pipeline_options=(
                        pipeline_options
                    )
                )
            }
        )

        result = converter.convert(
            document.storage_path
        )

        markdown_text = (
            result.document
            .export_to_markdown()
        )

        chunks = smart_chunk_markdown(
            services["embedding_model"],
            markdown_text,
        )

        if not chunks:
            raise RuntimeError(
                "The PDF produced "
                "no indexable chunks."
            )

        document.status = "indexing"
        db.session.commit()

        texts = [
            chunk["text"]
            for chunk in chunks
        ]

        vectors = (
            services["embedding_model"]
            .encode(
                texts,
                batch_size=64,
                normalize_embeddings=True,
                convert_to_numpy=True,
                show_progress_bar=True,
            )
        )

        delete_document_vectors(
            services,
            document.organization.tenant_id,
            document.document_hash,
        )

        points = []

        for (
            chunk_index,
            (chunk, vector),
        ) in enumerate(
            zip(
                chunks,
                vectors,
            )
        ):
            point_id = str(
                uuid.uuid5(
                    uuid.NAMESPACE_URL,
                    (
                        f"{document.organization.tenant_id}:"
                        f"{document.document_hash}:"
                        f"{chunk_index}"
                    ),
                )
            )

            points.append(
                PointStruct(
                    id=point_id,
                    vector=vector.tolist(),
                    payload={
                        "tenant_id": (
                            document.organization
                            .tenant_id
                        ),
                        "document_hash": (
                            document.document_hash
                        ),
                        "filename": (
                            document.original_filename
                        ),
                        "chunk_index": (
                            chunk_index
                        ),
                        "chunk_type": (
                            chunk["chunk_type"]
                        ),
                        "section": (
                            chunk["section"]
                        ),
                        "text": (
                            chunk["text"]
                        ),
                    },
                )
            )

        for batch_start in range(
            0,
            len(points),
            100,
        ):
            services["qdrant"].upsert(
                collection_name=(
                    services["collection_name"]
                ),
                points=points[
                    batch_start:
                    batch_start + 100
                ],
                wait=True,
            )

        document.status = "ready"
        document.chunk_count = len(
            points
        )
        document.indexed_at = utc_now()
        document.error_message = None

        db.session.commit()

        return len(points)

    except Exception as error:
        try:
            delete_document_vectors(
                services,
                (
                    document.organization
                    .tenant_id
                ),
                document.document_hash,
            )
        except Exception:
            pass

        document.status = "failed"
        document.chunk_count = 0

        document.error_message = (
            f"{type(error).__name__}: "
            f"{error}"
        )[:500]

        db.session.commit()
        raise


def serialize_sources(
    results,
    limit=5,
):
    sources = []

    for result in results[:limit]:
        payload = (
            result.payload or {}
        )

        sources.append(
            {
                "filename": payload.get(
                    "filename",
                    "Unknown file",
                ),
                "section": payload.get(
                    "section",
                    "Unknown section",
                ),
                "score": float(
                    result.score
                ),
            }
        )

    return sources


def create_app(
    services,
    root_path,
    secret_key,
):
    root_path = Path(root_path)

    app = Flask(
        __name__,
        template_folder=str(
            root_path / "templates"
        ),
        static_folder=str(
            root_path / "static"
        ),
    )

    app.config.update(
        SECRET_KEY=secret_key,
        SQLALCHEMY_DATABASE_URI=(
            "sqlite:///"
            + str(
                root_path
                / "instance"
                / "vaultify.db"
            )
        ),
        SQLALCHEMY_TRACK_MODIFICATIONS=False,
        MAX_CONTENT_LENGTH=(
            25 * 1024 * 1024
        ),
        UPLOAD_FOLDER=str(
            root_path / "uploads"
        ),
        SESSION_COOKIE_HTTPONLY=True,
        SESSION_COOKIE_SAMESITE="Lax",
        SESSION_COOKIE_SECURE=False,
        VAULTIFY_SERVICES=services,
    )

    db.init_app(app)
    login_manager.init_app(app)
    csrf.init_app(app)

    login_manager.login_view = "login"

    login_manager.login_message = (
        "Please sign in to continue."
    )

    login_manager.login_message_category = (
        "warning"
    )

    @app.context_processor
    def inject_navigation_context():
        if not current_user.is_authenticated:
            return {
                "memberships": [],
                "current_membership": None,
                "current_org": None,
            }

        memberships = (
            get_memberships_for_user(
                current_user.id
            )
        )

        current_membership = (
            resolve_active_membership()
        )

        return {
            "memberships": memberships,
            "current_membership": (
                current_membership
            ),
            "current_org": (
                current_membership.organization
                if current_membership
                else None
            ),
        }

    @app.errorhandler(413)
    def upload_too_large(_error):
        flash(
            (
                "The selected file exceeds "
                "the 25 MB limit."
            ),
            "error",
        )

        return redirect(
            url_for(
                "upload_document"
            )
        )

    @app.get("/health")
    def health():
        return {
            "status": "ok",
            "service": "Vaultify",
            "phase": 3,
        }

    @app.route(
        "/register",
        methods=["GET", "POST"],
    )
    def register():
        if current_user.is_authenticated:
            return redirect(
                url_for("dashboard")
            )

        if request.method == "POST":
            display_name = (
                request.form.get(
                    "display_name"
                )
                or ""
            ).strip()

            organization_name = (
                request.form.get(
                    "organization_name"
                )
                or ""
            ).strip()

            email = normalize_email(
                request.form.get("email")
            )

            password = (
                request.form.get(
                    "password"
                )
                or ""
            )

            confirm_password = (
                request.form.get(
                    "confirm_password"
                )
                or ""
            )

            if (
                not display_name
                or not organization_name
                or not email
            ):
                flash(
                    "All fields are required.",
                    "error",
                )

                return render_template(
                    "register.html"
                )

            if (
                password
                != confirm_password
            ):
                flash(
                    (
                        "Password confirmation "
                        "does not match."
                    ),
                    "error",
                )

                return render_template(
                    "register.html"
                )

            if not password_is_strong(
                password
            ):
                flash(
                    (
                        "Use at least 10 characters "
                        "with letters and numbers."
                    ),
                    "error",
                )

                return render_template(
                    "register.html"
                )

            if User.query.filter_by(
                email=email
            ).first():
                flash(
                    (
                        "An account with that "
                        "email already exists."
                    ),
                    "error",
                )

                return render_template(
                    "register.html"
                )

            user = User(
                email=email,
                display_name=display_name,
            )

            user.set_password(
                password
            )

            organization = Organization(
                name=organization_name,
                slug=(
                    unique_organization_slug(
                        organization_name
                    )
                ),
            )

            membership = Membership(
                user=user,
                organization=organization,
                role="owner",
            )

            db.session.add_all(
                [
                    user,
                    organization,
                    membership,
                ]
            )

            db.session.commit()

            login_user(user)

            session[
                "organization_id"
            ] = organization.id

            flash(
                (
                    "Your Vaultify account "
                    "was created successfully."
                ),
                "success",
            )

            return redirect(
                url_for("dashboard")
            )

        return render_template(
            "register.html",
            title=(
                "Create account — Vaultify"
            ),
        )

    @app.route(
        "/login",
        methods=["GET", "POST"],
    )
    def login():
        if current_user.is_authenticated:
            return redirect(
                url_for("dashboard")
            )

        if request.method == "POST":
            email = normalize_email(
                request.form.get("email")
            )

            password = (
                request.form.get(
                    "password"
                )
                or ""
            )

            user = User.query.filter_by(
                email=email
            ).first()

            if (
                user is None
                or not user.is_active
                or not user.check_password(
                    password
                )
            ):
                flash(
                    (
                        "Invalid email address "
                        "or password."
                    ),
                    "error",
                )

                return render_template(
                    "login.html"
                )

            login_user(user)

            membership = (
                Membership.query
                .filter_by(
                    user_id=user.id
                )
                .order_by(
                    Membership.id.asc()
                )
                .first()
            )

            if membership is None:
                logout_user()

                flash(
                    (
                        "This account has no "
                        "organization membership."
                    ),
                    "error",
                )

                return render_template(
                    "login.html"
                )

            session[
                "organization_id"
            ] = (
                membership.organization_id
            )

            return redirect(
                url_for("dashboard")
            )

        return render_template(
            "login.html",
            title=(
                "Sign in — Vaultify"
            ),
        )

    @app.post("/logout")
    @login_required
    def logout():
        logout_user()
        session.clear()

        flash(
            "You have been signed out.",
            "success",
        )

        return redirect(
            url_for("login")
        )

    @app.post(
        "/organizations/switch"
    )
    @login_required
    def switch_organization():
        try:
            organization_id = int(
                request.form.get(
                    "organization_id",
                    "",
                )
            )
        except ValueError:
            abort(400)

        membership = (
            Membership.query.filter_by(
                user_id=current_user.id,
                organization_id=(
                    organization_id
                ),
            ).first()
        )

        if membership is None:
            abort(403)

        session[
            "organization_id"
        ] = (
            membership.organization_id
        )

        return redirect(
            request.referrer
            or url_for("dashboard")
        )

    @app.get("/")
    def index():
        if current_user.is_authenticated:
            return redirect(
                url_for("dashboard")
            )

        return redirect(
            url_for("login")
        )

    @app.get("/dashboard")
    @login_required
    def dashboard():
        membership = (
            resolve_active_membership()
        )

        if membership is None:
            return redirect(
                url_for("login")
            )

        organization = (
            membership.organization
        )

        document_count = (
            Document.query.filter_by(
                organization_id=(
                    organization.id
                )
            ).count()
        )

        chunk_count = (
            db.session.query(
                db.func.coalesce(
                    db.func.sum(
                        Document.chunk_count
                    ),
                    0,
                )
            )
            .filter(
                Document.organization_id
                == organization.id,
                Document.status
                == "ready",
            )
            .scalar()
        )

        return render_template(
            "dashboard.html",
            title=(
                "Dashboard — Vaultify"
            ),
            current_membership=(
                membership
            ),
            current_org=organization,
            document_count=(
                document_count
            ),
            chunk_count=int(
                chunk_count or 0
            ),
            question=None,
            answer=None,
            sources=[],
        )

    @app.post("/ask")
    @login_required
    def ask():
        membership = (
            resolve_active_membership()
        )

        if membership is None:
            abort(403)

        question = (
            request.form.get(
                "question"
            )
            or ""
        ).strip()

        if not question:
            flash(
                (
                    "Enter a question "
                    "first."
                ),
                "error",
            )

            return redirect(
                url_for("dashboard")
            )

        services = app.config[
            "VAULTIFY_SERVICES"
        ]

        result = services[
            "answer_tenant_question"
        ](
            question=question,
            tenant_id=(
                membership.organization
                .tenant_id
            ),
        )

        answer = result["answer"]

        sources = serialize_sources(
            result["results"]
        )

        db.session.add(
            QueryLog(
                organization_id=(
                    membership
                    .organization_id
                ),
                user_id=current_user.id,
                question=question,
                answer=answer,
                source_count=len(
                    sources
                ),
            )
        )

        db.session.commit()

        document_count = (
            Document.query.filter_by(
                organization_id=(
                    membership
                    .organization_id
                )
            ).count()
        )

        chunk_count = (
            db.session.query(
                db.func.coalesce(
                    db.func.sum(
                        Document.chunk_count
                    ),
                    0,
                )
            )
            .filter(
                Document.organization_id
                == membership.organization_id,
                Document.status
                == "ready",
            )
            .scalar()
        )

        return render_template(
            "dashboard.html",
            title=(
                "Dashboard — Vaultify"
            ),
            current_membership=(
                membership
            ),
            current_org=(
                membership.organization
            ),
            document_count=(
                document_count
            ),
            chunk_count=int(
                chunk_count or 0
            ),
            question=question,
            answer=answer,
            sources=sources,
        )

    @app.get("/documents")
    @login_required
    def documents():
        membership = (
            resolve_active_membership()
        )

        if membership is None:
            abort(403)

        organization_documents = (
            Document.query
            .filter_by(
                organization_id=(
                    membership
                    .organization_id
                )
            )
            .order_by(
                Document.created_at.desc()
            )
            .all()
        )

        return render_template(
            "documents.html",
            title=(
                "Documents — Vaultify"
            ),
            current_membership=(
                membership
            ),
            current_org=(
                membership.organization
            ),
            documents=(
                organization_documents
            ),
        )

    @app.route(
        "/documents/upload",
        methods=["GET", "POST"],
    )
    @login_required
    def upload_document():
        membership = (
            resolve_active_membership()
        )

        if membership is None:
            abort(403)

        require_management_role(
            membership
        )

        if request.method == "POST":
            uploaded_file = (
                request.files.get("file")
            )

            if (
                uploaded_file is None
                or not uploaded_file.filename
            ):
                flash(
                    (
                        "Please select "
                        "a PDF document."
                    ),
                    "error",
                )

                return render_template(
                    "upload.html",
                    current_membership=(
                        membership
                    ),
                    current_org=(
                        membership.organization
                    ),
                )

            original_filename = (
                uploaded_file.filename
                .strip()
            )

            safe_filename = (
                secure_filename(
                    original_filename
                )
            )

            if (
                not safe_filename
                or not safe_filename
                .lower()
                .endswith(".pdf")
            ):
                flash(
                    (
                        "Only PDF documents "
                        "are supported."
                    ),
                    "error",
                )

                return render_template(
                    "upload.html",
                    current_membership=(
                        membership
                    ),
                    current_org=(
                        membership.organization
                    ),
                )

            file_bytes = (
                uploaded_file.read()
            )

            if not file_bytes:
                flash(
                    (
                        "The selected file "
                        "is empty."
                    ),
                    "error",
                )

                return render_template(
                    "upload.html",
                    current_membership=(
                        membership
                    ),
                    current_org=(
                        membership.organization
                    ),
                )

            if not file_bytes.startswith(
                b"%PDF-"
            ):
                flash(
                    (
                        "The selected file "
                        "is not a valid PDF."
                    ),
                    "error",
                )

                return render_template(
                    "upload.html",
                    current_membership=(
                        membership
                    ),
                    current_org=(
                        membership.organization
                    ),
                )

            guessed_type = filetype.guess(
                file_bytes[:8192]
            )

            if (
                guessed_type
                and guessed_type.mime
                != "application/pdf"
            ):
                flash(
                    (
                        "The selected file "
                        "signature is not PDF."
                    ),
                    "error",
                )

                return render_template(
                    "upload.html",
                    current_membership=(
                        membership
                    ),
                    current_org=(
                        membership.organization
                    ),
                )

            document_hash = (
                hashlib.sha256(
                    file_bytes
                ).hexdigest()
            )

            existing_document = (
                Document.query.filter_by(
                    organization_id=(
                        membership
                        .organization_id
                    ),
                    document_hash=(
                        document_hash
                    ),
                ).first()
            )

            if existing_document:
                flash(
                    (
                        "This document already "
                        "exists in the organization."
                    ),
                    "warning",
                )

                return redirect(
                    url_for("documents")
                )

            stored_filename = (
                f"{uuid.uuid4().hex}.pdf"
            )

            storage_path = (
                Path(
                    app.config[
                        "UPLOAD_FOLDER"
                    ]
                )
                / stored_filename
            )

            storage_path.write_bytes(
                file_bytes
            )

            document = Document(
                organization_id=(
                    membership
                    .organization_id
                ),
                original_filename=(
                    original_filename
                ),
                stored_filename=(
                    stored_filename
                ),
                storage_path=str(
                    storage_path
                ),
                mime_type=(
                    "application/pdf"
                ),
                size_bytes=len(
                    file_bytes
                ),
                document_hash=(
                    document_hash
                ),
                status="uploaded",
            )

            db.session.add(document)
            db.session.commit()

            try:
                chunk_count = (
                    ingest_document(
                        document,
                        app.config[
                            "VAULTIFY_SERVICES"
                        ],
                    )
                )

                flash(
                    (
                        f"{original_filename} "
                        f"was indexed successfully "
                        f"with {chunk_count} chunks."
                    ),
                    "success",
                )

            except Exception:
                flash(
                    (
                        "The PDF was stored, "
                        "but indexing failed safely. "
                        "Use Retry after checking "
                        "the runtime logs."
                    ),
                    "error",
                )

            return redirect(
                url_for("documents")
            )

        return render_template(
            "upload.html",
            title=(
                "Upload PDF — Vaultify"
            ),
            current_membership=(
                membership
            ),
            current_org=(
                membership.organization
            ),
        )

    @app.post(
        "/documents/<int:document_id>/retry"
    )
    @login_required
    def retry_document(document_id):
        membership = (
            resolve_active_membership()
        )

        if membership is None:
            abort(403)

        require_management_role(
            membership
        )

        document = (
            Document.query.filter_by(
                id=document_id,
                organization_id=(
                    membership
                    .organization_id
                ),
            ).first_or_404()
        )

        try:
            chunk_count = (
                ingest_document(
                    document,
                    app.config[
                        "VAULTIFY_SERVICES"
                    ],
                )
            )

            flash(
                (
                    "The document was indexed "
                    f"with {chunk_count} chunks."
                ),
                "success",
            )

        except Exception:
            flash(
                (
                    "The retry failed safely."
                ),
                "error",
            )

        return redirect(
            url_for("documents")
        )

    @app.post(
        "/documents/<int:document_id>/delete"
    )
    @login_required
    def delete_document(document_id):
        membership = (
            resolve_active_membership()
        )

        if membership is None:
            abort(403)

        require_management_role(
            membership
        )

        document = (
            Document.query.filter_by(
                id=document_id,
                organization_id=(
                    membership
                    .organization_id
                ),
            ).first_or_404()
        )

        services = app.config[
            "VAULTIFY_SERVICES"
        ]

        delete_document_vectors(
            services,
            (
                membership.organization
                .tenant_id
            ),
            document.document_hash,
        )

        storage_path = Path(
            document.storage_path
        )

        db.session.delete(document)
        db.session.commit()

        if storage_path.exists():
            storage_path.unlink()

        flash(
            (
                "The document was deleted "
                "from storage and Qdrant."
            ),
            "success",
        )

        return redirect(
            url_for("documents")
        )

    @app.get("/organization")
    @login_required
    def organization():
        membership = (
            resolve_active_membership()
        )

        if membership is None:
            abort(403)

        return render_template(
            "organization.html",
            title=(
                "Organization — Vaultify"
            ),
            current_membership=(
                membership
            ),
            current_org=(
                membership.organization
            ),
        )

    return app
"""

WEB_BACKEND_PATH.write_text(
    textwrap.dedent(
        WEB_BACKEND_CODE
    ).strip() + "\n",
    encoding="utf-8",
)

compile(
    WEB_BACKEND_PATH.read_text(
        encoding="utf-8"
    ),
    str(WEB_BACKEND_PATH),
    "exec",
)

print(
    "✅ Compact Flask backend created."
)
print(
    f"📄 Backend module: "
    f"{WEB_BACKEND_PATH}"
)

✅ Compact Flask backend created.
📄 Backend module: /content/vaultify_compact_web/vaultify_web_backend.py


In [16]:
# VAULTIFY COMPACT - Cell 13: Initialize Flask safely after every runtime restart

import hashlib
import importlib
import secrets
import sys
from pathlib import Path

from qdrant_client.models import (
    FieldCondition,
    Filter,
    MatchValue,
)

# ---------------------------------------------------------------------
# 1. Verify that Cell 12 created the backend module
# ---------------------------------------------------------------------

WEB_BACKEND_PATH = (
    WEB_PROJECT_ROOT
    / "vaultify_web_backend.py"
)

if not WEB_BACKEND_PATH.exists():
    raise RuntimeError(
        "The Flask backend file is missing. "
        "Run Cell 12 before Cell 13."
    )

# ---------------------------------------------------------------------
# 2. Add the generated backend folder to Python's import path
# ---------------------------------------------------------------------

web_project_path = str(
    WEB_PROJECT_ROOT.resolve()
)

if web_project_path not in sys.path:
    sys.path.insert(
        0,
        web_project_path,
    )

# Remove an old cached module after rerunning Cell 12
if "vaultify_web_backend" in sys.modules:
    del sys.modules[
        "vaultify_web_backend"
    ]

vaultify_web_backend = (
    importlib.import_module(
        "vaultify_web_backend"
    )
)

from vaultify_web_backend import (
    Document,
    Membership,
    Organization,
    User,
    create_app,
    db,
)

print(
    "✅ Flask backend module imported successfully."
)

# ---------------------------------------------------------------------
# 3. Load or create the Flask secret key
# ---------------------------------------------------------------------

try:
    FLASK_SECRET_KEY = userdata.get(
        "FLASK_SECRET_KEY"
    )
except Exception:
    FLASK_SECRET_KEY = None

if not FLASK_SECRET_KEY:
    FLASK_SECRET_KEY = (
        secrets.token_urlsafe(48)
    )

    print(
        "ℹ️ FLASK_SECRET_KEY is not configured. "
        "A temporary runtime key was generated."
    )
else:
    print(
        "✅ FLASK_SECRET_KEY loaded from Colab Secrets."
    )

# ---------------------------------------------------------------------
# 4. Connect Flask to the already initialized Vaultify services
# ---------------------------------------------------------------------

WEB_SERVICES = {
    "qdrant": qdrant,
    "embedding_model": embedding_model,
    "answer_tenant_question": (
        answer_tenant_question
    ),
    "collection_name": COLLECTION_NAME,
}

vaultify_web_app = create_app(
    services=WEB_SERVICES,
    root_path=WEB_PROJECT_ROOT,
    secret_key=FLASK_SECRET_KEY,
)

print(
    "✅ Compact Flask application created."
)

# ---------------------------------------------------------------------
# 5. Create the temporary Apple demo account safely
# ---------------------------------------------------------------------

DEMO_LOGIN_EMAIL = (
    "runtime-demo@vaultify.local"
)

DEMO_LOGIN_PASSWORD = (
    "Vaultify-"
    + secrets.token_urlsafe(9)
    + "-9"
)

DEMO_ORGANIZATION_NAME = (
    "Vaultify Apple Demo"
)

DEMO_ORGANIZATION_SLUG = (
    "vaultify-apple-demo"
)

DEMO_TENANT_ID = (
    DEMO_TENANTS[
        "company_a"
    ]["tenant_id"]
)

SEED_METADATA_FILENAME = (
    "seed-apple-demo-metadata-only.pdf"
)

SEED_METADATA_HASH = hashlib.sha256(
    b"vaultify-apple-demo-metadata-only"
).hexdigest()

with vaultify_web_app.app_context():
    # Clear a possible failed transaction from an earlier execution.
    db.session.rollback()

    db.create_all()

    demo_organization = (
        Organization.query.filter_by(
            tenant_id=DEMO_TENANT_ID
        ).first()
    )

    if demo_organization is None:
        demo_organization = Organization(
            tenant_id=DEMO_TENANT_ID,
            name=DEMO_ORGANIZATION_NAME,
            slug=DEMO_ORGANIZATION_SLUG,
        )

        db.session.add(
            demo_organization
        )

    demo_user = User.query.filter_by(
        email=DEMO_LOGIN_EMAIL
    ).first()

    if demo_user is None:
        demo_user = User(
            email=DEMO_LOGIN_EMAIL,
            display_name="Demo User",
        )

        # Password must exist before SQLAlchemy inserts the row.
        demo_user.set_password(
            DEMO_LOGIN_PASSWORD
        )

        db.session.add(
            demo_user
        )

    else:
        demo_user.display_name = (
            "Demo User"
        )

        demo_user.set_password(
            DEMO_LOGIN_PASSWORD
        )

    # Generate database IDs only after required fields are populated.
    db.session.flush()

    demo_membership = (
        Membership.query.filter_by(
            user_id=demo_user.id,
            organization_id=(
                demo_organization.id
            ),
        ).first()
    )

    if demo_membership is None:
        demo_membership = Membership(
            user_id=demo_user.id,
            organization_id=(
                demo_organization.id
            ),
            role="owner",
        )

        db.session.add(
            demo_membership
        )

    seed_document = (
        Document.query.filter_by(
            organization_id=(
                demo_organization.id
            ),
            stored_filename=(
                SEED_METADATA_FILENAME
            ),
        ).first()
    )

    if seed_document is None:
        seed_document = Document(
            organization_id=(
                demo_organization.id
            ),
            original_filename=(
                "apple_fy2025_10k.pdf"
            ),
            stored_filename=(
                SEED_METADATA_FILENAME
            ),
            storage_path=(
                "/content/"
                "vaultify_compact_web/"
                "seed/"
                "apple_fy2025_10k.pdf"
            ),
            mime_type=(
                "application/pdf"
            ),
            size_bytes=0,
            document_hash=(
                SEED_METADATA_HASH
            ),
            status="ready",
            chunk_count=(
                tenant_counts[
                    "company_a"
                ]
            ),
        )

        db.session.add(
            seed_document
        )

    else:
        seed_document.status = (
            "ready"
        )

        seed_document.chunk_count = (
            tenant_counts[
                "company_a"
            ]
        )

    db.session.commit()

print(
    "✅ Demo organization, user, "
    "membership, and metadata created."
)

# ---------------------------------------------------------------------
# 6. Run authentication and organization smoke tests
# ---------------------------------------------------------------------

original_testing_setting = (
    vaultify_web_app.config.get(
        "TESTING",
        False,
    )
)

original_csrf_setting = (
    vaultify_web_app.config.get(
        "WTF_CSRF_ENABLED",
        True,
    )
)

try:
    vaultify_web_app.config[
        "TESTING"
    ] = True

    vaultify_web_app.config[
        "WTF_CSRF_ENABLED"
    ] = False

    with vaultify_web_app.test_client() as client:
        health_response = client.get(
            "/health"
        )

        login_response = client.post(
            "/login",
            data={
                "email": (
                    DEMO_LOGIN_EMAIL
                ),
                "password": (
                    DEMO_LOGIN_PASSWORD
                ),
            },
            follow_redirects=True,
        )

        dashboard_reached = (
            login_response.status_code
            == 200
            and b"Vaultify Apple Demo"
            in login_response.data
        )

        documents_response = client.get(
            "/documents"
        )

        organization_response = client.get(
            "/organization"
        )

        logout_response = client.post(
            "/logout",
            follow_redirects=True,
        )

finally:
    vaultify_web_app.config[
        "TESTING"
    ] = original_testing_setting

    vaultify_web_app.config[
        "WTF_CSRF_ENABLED"
    ] = original_csrf_setting

if health_response.status_code != 200:
    raise RuntimeError(
        "The Flask health check failed."
    )

if not dashboard_reached:
    raise RuntimeError(
        "The demo login did not reach the dashboard."
    )

if documents_response.status_code != 200:
    raise RuntimeError(
        "The documents page failed."
    )

if organization_response.status_code != 200:
    raise RuntimeError(
        "The organization page failed."
    )

if logout_response.status_code != 200:
    raise RuntimeError(
        "The logout flow failed."
    )

print(
    "✅ Authentication smoke test passed."
)

print(
    "✅ Trusted organization resolution passed."
)

print(
    "✅ Documents and organization pages passed."
)

print(
    "\n🔐 Temporary demo account:"
)

print(
    f"   Email: {DEMO_LOGIN_EMAIL}"
)

print(
    f"   Password: {DEMO_LOGIN_PASSWORD}"
)

print(
    "\n⚠️ This password and SQLite database "
    "belong only to the current Colab runtime."
)

✅ Flask backend module imported successfully.
ℹ️ FLASK_SECRET_KEY is not configured. A temporary runtime key was generated.
✅ Compact Flask application created.
✅ Demo organization, user, membership, and metadata created.
✅ Authentication smoke test passed.
✅ Trusted organization resolution passed.
✅ Documents and organization pages passed.

🔐 Temporary demo account:
   Email: runtime-demo@vaultify.local
   Password: Vaultify-xt0rz8-YqwRg-9

⚠️ This password and SQLite database belong only to the current Colab runtime.


In [17]:
# VAULTIFY COMPACT - Cell 14: Start and test the local Flask web server

import socket
import threading
import time

import requests
from werkzeug.serving import (
    make_server,
)

WEB_HOST = "127.0.0.1"

WEB_PORT_CANDIDATES = range(
    5000,
    5011,
)


def web_port_is_open(
    host,
    port,
):
    """
    Return True when a TCP server is listening on the target port.
    """
    with socket.socket(
        socket.AF_INET,
        socket.SOCK_STREAM,
    ) as connection_socket:
        connection_socket.settimeout(
            0.5
        )

        return (
            connection_socket.connect_ex(
                (
                    host,
                    port,
                )
            )
            == 0
        )


def find_available_web_port():
    """
    Return the first available port for the Flask web server.
    """
    for candidate_port in (
        WEB_PORT_CANDIDATES
    ):
        if not web_port_is_open(
            WEB_HOST,
            candidate_port,
        ):
            return candidate_port

    raise RuntimeError(
        "No available Vaultify web port "
        "was found between 5000 and 5010."
    )


if (
    "vaultify_web_server"
    in globals()
    and vaultify_web_server
    is not None
):
    try:
        vaultify_web_server.shutdown()

        print(
            "ℹ️ Previous Vaultify "
            "web server stopped."
        )

    except Exception as error:
        print(
            "⚠️ Previous web server "
            "shutdown warning: "
            f"{error}"
        )


WEB_PORT = (
    find_available_web_port()
)

LOCAL_WEB_URL = (
    f"http://{WEB_HOST}:{WEB_PORT}"
)

vaultify_web_server = make_server(
    WEB_HOST,
    WEB_PORT,
    vaultify_web_app,
    threaded=True,
)

vaultify_web_server_thread = (
    threading.Thread(
        target=(
            vaultify_web_server
            .serve_forever
        ),
        name="vaultify-web-server",
        daemon=True,
    )
)

vaultify_web_server_thread.start()

startup_deadline = (
    time.time() + 15
)

while (
    time.time()
    < startup_deadline
    and not web_port_is_open(
        WEB_HOST,
        WEB_PORT,
    )
):
    time.sleep(0.25)

if not web_port_is_open(
    WEB_HOST,
    WEB_PORT,
):
    raise RuntimeError(
        "The Vaultify web server "
        f"did not start on port "
        f"{WEB_PORT}."
    )

local_health_response = requests.get(
    f"{LOCAL_WEB_URL}/health",
    timeout=10,
)

if (
    local_health_response.status_code
    != 200
    or local_health_response
    .json()
    .get("status")
    != "ok"
):
    raise RuntimeError(
        "The local Vaultify "
        "web health check failed."
    )

print(
    "✅ Vaultify web server "
    "started locally."
)
print(
    f"🌐 Local web URL: "
    f"{LOCAL_WEB_URL}"
)
print(
    f"🩺 Health: "
    f"{local_health_response.json()}"
)
print(
    "✅ The MCP server on port "
    "8003 was preserved."
)

INFO:werkzeug:127.0.0.1 - - [23/Jul/2026 15:23:29] "GET /health HTTP/1.1" 200 -


✅ Vaultify web server started locally.
🌐 Local web URL: http://127.0.0.1:5000
🩺 Health: {'phase': 3, 'service': 'Vaultify', 'status': 'ok'}
✅ The MCP server on port 8003 was preserved.


In [18]:
# VAULTIFY COMPACT - Cell 15: Create, expose, and verify the Flask website after every runtime restart
import re
import shutil
import socket
import subprocess
import threading
import time

from urllib.parse import urlparse

import requests


# ---------------------------------------------------------------------
# 1. Verify that the local Flask server is running
# ---------------------------------------------------------------------

if "LOCAL_WEB_URL" not in globals():
    raise RuntimeError(
        "LOCAL_WEB_URL is missing. "
        "Run Cell 14 before Cell 15."
    )

try:
    local_health_response = requests.get(
        f"{LOCAL_WEB_URL}/health",
        timeout=10,
    )

    local_health_response.raise_for_status()

except Exception as error:
    raise RuntimeError(
        "The local Flask website is not reachable. "
        "Run Cell 14 again."
    ) from error


if (
    local_health_response
    .json()
    .get("status")
    != "ok"
):
    raise RuntimeError(
        "The local Flask health response is invalid."
    )


print("✅ Local Flask website is healthy.")


# ---------------------------------------------------------------------
# 2. Install cloudflared when missing
# ---------------------------------------------------------------------

def install_cloudflared():
    """
    Install the Cloudflare tunnel client
    inside the current Colab runtime.
    """
    if shutil.which("cloudflared"):
        print("ℹ️ cloudflared is already installed.")
        return

    install_command = """
set -e

wget -q \
  https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb \
  -O /tmp/cloudflared.deb

dpkg -i /tmp/cloudflared.deb >/dev/null

rm -f /tmp/cloudflared.deb
"""

    subprocess.run(
        [
            "bash",
            "-lc",
            install_command,
        ],
        check=True,
    )

    print("✅ cloudflared installed.")


install_cloudflared()


# ---------------------------------------------------------------------
# 3. Stop a previous web tunnel when this cell is rerun
# ---------------------------------------------------------------------

if (
    "web_cloudflare_tunnel_process"
    in globals()
    and web_cloudflare_tunnel_process
    is not None
    and web_cloudflare_tunnel_process.poll()
    is None
):
    print("ℹ️ Stopping the previous web tunnel.")

    web_cloudflare_tunnel_process.terminate()

    try:
        web_cloudflare_tunnel_process.wait(
            timeout=5
        )

    except subprocess.TimeoutExpired:
        web_cloudflare_tunnel_process.kill()

        web_cloudflare_tunnel_process.wait(
            timeout=5
        )


# ---------------------------------------------------------------------
# 4. Start a new Cloudflare Quick Tunnel
# ---------------------------------------------------------------------

web_cloudflare_logs = []


def read_cloudflare_logs(
    process,
    storage,
):
    """
    Read Cloudflare logs continuously
    without blocking the notebook.
    """
    if process.stderr is None:
        return

    for line in iter(
        process.stderr.readline,
        "",
    ):
        cleaned_line = line.strip()

        if cleaned_line:
            storage.append(
                cleaned_line
            )

            print(
                f"[web-cloudflared] "
                f"{cleaned_line}"
            )

        if process.poll() is not None:
            break


def extract_public_url(
    log_lines,
):
    """
    Extract the generated TryCloudflare URL.
    """
    pattern = re.compile(
        r"https://[a-zA-Z0-9-]+"
        r"\.trycloudflare\.com"
    )

    for line in log_lines:
        match = pattern.search(
            line
        )

        if match:
            return match.group(0)

    return None


def tunnel_registered(
    log_lines,
):
    """
    Check whether Cloudflare registered
    the tunnel connection.
    """
    return any(
        "Registered tunnel connection"
        in line
        for line in log_lines
    )


web_cloudflare_tunnel_process = (
    subprocess.Popen(
        [
            "cloudflared",
            "tunnel",
            "--url",
            LOCAL_WEB_URL,
            "--no-autoupdate",
        ],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.PIPE,
        text=True,
        bufsize=1,
    )
)


web_cloudflare_log_thread = (
    threading.Thread(
        target=read_cloudflare_logs,
        args=(
            web_cloudflare_tunnel_process,
            web_cloudflare_logs,
        ),
        name=(
            "vaultify-web-"
            "cloudflare-log-reader"
        ),
        daemon=True,
    )
)


web_cloudflare_log_thread.start()


TUNNEL_STARTUP_TIMEOUT = 60

startup_deadline = (
    time.time()
    + TUNNEL_STARTUP_TIMEOUT
)

VAULTIFY_WEB_PUBLIC_URL = None


while time.time() < startup_deadline:
    if (
        web_cloudflare_tunnel_process
        .poll()
        is not None
    ):
        raise RuntimeError(
            "cloudflared stopped unexpectedly. "
            f"Recent logs: "
            f"{web_cloudflare_logs[-10:]}"
        )

    VAULTIFY_WEB_PUBLIC_URL = (
        extract_public_url(
            web_cloudflare_logs
        )
    )

    if (
        VAULTIFY_WEB_PUBLIC_URL
        and tunnel_registered(
            web_cloudflare_logs
        )
    ):
        break

    time.sleep(0.25)


if not VAULTIFY_WEB_PUBLIC_URL:
    raise RuntimeError(
        "Cloudflare did not provide "
        "a public website URL."
    )


if not tunnel_registered(
    web_cloudflare_logs
):
    raise RuntimeError(
        "The public URL was created, "
        "but the tunnel connection "
        "was not registered."
    )


print("\n✅ Cloudflare web tunnel started.")
print(
    f"🌍 Public website: "
    f"{VAULTIFY_WEB_PUBLIC_URL}"
)


# ---------------------------------------------------------------------
# 5. Wait for the new hostname to appear in DNS
# ---------------------------------------------------------------------

def query_dns_over_https(
    hostname,
):
    """
    Query Google and Cloudflare DNS
    for the newly created tunnel hostname.
    """
    providers = [
        {
            "url": (
                "https://dns.google/resolve"
            ),
            "headers": {},
        },
        {
            "url": (
                "https://cloudflare-dns.com/"
                "dns-query"
            ),
            "headers": {
                "accept": (
                    "application/dns-json"
                )
            },
        },
    ]

    for provider in providers:
        try:
            response = requests.get(
                provider["url"],
                params={
                    "name": hostname,
                    "type": "A",
                },
                headers=(
                    provider["headers"]
                ),
                timeout=15,
            )

            response.raise_for_status()

            answers = (
                response.json()
                .get("Answer", [])
            )

            ipv4_addresses = [
                answer.get("data")
                for answer in answers
                if (
                    answer.get("type") == 1
                    and answer.get("data")
                )
            ]

            if ipv4_addresses:
                return ipv4_addresses[0]

        except Exception:
            continue

    return None


def add_hosts_entry(
    hostname,
    ipv4_address,
):
    """
    Add a temporary DNS mapping
    to the Colab virtual machine.
    """
    with open(
        "/etc/hosts",
        "r",
        encoding="utf-8",
    ) as hosts_file:
        existing_lines = (
            hosts_file.readlines()
        )

    filtered_lines = [
        line
        for line in existing_lines
        if hostname not in line
    ]

    filtered_lines.append(
        f"{ipv4_address} "
        f"{hostname}\n"
    )

    with open(
        "/etc/hosts",
        "w",
        encoding="utf-8",
    ) as hosts_file:
        hosts_file.writelines(
            filtered_lines
        )


hostname = urlparse(
    VAULTIFY_WEB_PUBLIC_URL
).hostname


if not hostname:
    raise RuntimeError(
        "The generated tunnel hostname "
        "is invalid."
    )


DNS_WAIT_SECONDS = 120
DNS_RETRY_SECONDS = 5

dns_deadline = (
    time.time()
    + DNS_WAIT_SECONDS
)

resolved_ip = None
dns_attempt = 0


while time.time() < dns_deadline:
    dns_attempt += 1

    if (
        web_cloudflare_tunnel_process
        .poll()
        is not None
    ):
        raise RuntimeError(
            "The Cloudflare tunnel stopped "
            "while waiting for DNS."
        )

    try:
        resolved_ip = (
            socket.gethostbyname(
                hostname
            )
        )

        print(
            f"✅ DNS resolved normally: "
            f"{hostname} → {resolved_ip}"
        )

        break

    except socket.gaierror:
        resolved_ip = (
            query_dns_over_https(
                hostname
            )
        )

        if resolved_ip:
            add_hosts_entry(
                hostname,
                resolved_ip,
            )

            print(
                f"✅ DNS-over-HTTPS resolved: "
                f"{hostname} → {resolved_ip}"
            )

            break

    print(
        f"⏳ Waiting for tunnel DNS "
        f"(attempt {dns_attempt})..."
    )

    time.sleep(
        DNS_RETRY_SECONDS
    )


if not resolved_ip:
    raise RuntimeError(
        "The Cloudflare hostname did not "
        "become available in DNS."
    )


# ---------------------------------------------------------------------
# 6. Verify the website through the public tunnel
# ---------------------------------------------------------------------

PUBLIC_TEST_ATTEMPTS = 20
PUBLIC_TEST_RETRY_SECONDS = 3

public_health_response = None
last_public_error = None


for attempt_number in range(
    1,
    PUBLIC_TEST_ATTEMPTS + 1,
):
    try:
        public_health_response = (
            requests.get(
                (
                    VAULTIFY_WEB_PUBLIC_URL
                    + "/health"
                ),
                timeout=15,
            )
        )

        if (
            public_health_response
            .status_code
            == 200
            and public_health_response
            .json()
            .get("status")
            == "ok"
        ):
            break

        last_public_error = RuntimeError(
            "Unexpected public health "
            "status: "
            f"{public_health_response.status_code}"
        )

    except Exception as error:
        last_public_error = error

    print(
        f"⏳ Public website test "
        f"{attempt_number}/"
        f"{PUBLIC_TEST_ATTEMPTS}..."
    )

    if (
        attempt_number
        < PUBLIC_TEST_ATTEMPTS
    ):
        time.sleep(
            PUBLIC_TEST_RETRY_SECONDS
        )


if (
    public_health_response is None
    or public_health_response.status_code
    != 200
):
    raise RuntimeError(
        "The public Vaultify website "
        "remained unreachable."
    ) from last_public_error


# ---------------------------------------------------------------------
# 7. Print the final runtime URLs
# ---------------------------------------------------------------------

print(
    "\n✅ Public Vaultify website "
    "health check passed."
)

print(
    f"🌍 Website: "
    f"{VAULTIFY_WEB_PUBLIC_URL}"
)

print(
    f"📝 Register: "
    f"{VAULTIFY_WEB_PUBLIC_URL}/register"
)

print(
    f"🔐 Login: "
    f"{VAULTIFY_WEB_PUBLIC_URL}/login"
)

print(
    f"📊 Dashboard: "
    f"{VAULTIFY_WEB_PUBLIC_URL}/dashboard"
)

print(
    f"📄 Documents: "
    f"{VAULTIFY_WEB_PUBLIC_URL}/documents"
)

print(
    f"🏢 Organization: "
    f"{VAULTIFY_WEB_PUBLIC_URL}/organization"
)


print(
    "\n🔐 Temporary demo account:"
)

print(
    f"   Email: "
    f"{DEMO_LOGIN_EMAIL}"
)

print(
    f"   Password: "
    f"{DEMO_LOGIN_PASSWORD}"
)


if "PUBLIC_MCP_URL" in globals():
    print(
        "\n🔗 MCP endpoint:"
    )

    print(
        PUBLIC_MCP_URL
    )


print(
    "\n⚠️ The website URL is temporary "
    "and remains active only while "
    "this Colab runtime is connected."
)

INFO:werkzeug:127.0.0.1 - - [23/Jul/2026 15:23:33] "GET /health HTTP/1.1" 200 -


✅ Local Flask website is healthy.
ℹ️ cloudflared is already installed.
[web-cloudflared] 2026-07-23T15:23:33Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
[web-cloudflared] 2026-07-23T15:23:33Z INF Requesting new quick Tunnel on trycloudflare.com...
[web-cloudflared] 2026-07-23T15:23:36Z INF +--------------------------------------------------------------------------------------------+
[web-cloudflared] 2026-07-23T15:23:36Z INF |  Your quick Tunnel has been cre

INFO:werkzeug:127.0.0.1 - - [23/Jul/2026 15:23:42] "GET /health HTTP/1.1" 200 -



✅ Public Vaultify website health check passed.
🌍 Website: https://lean-women-dad-achievement.trycloudflare.com
📝 Register: https://lean-women-dad-achievement.trycloudflare.com/register
🔐 Login: https://lean-women-dad-achievement.trycloudflare.com/login
📊 Dashboard: https://lean-women-dad-achievement.trycloudflare.com/dashboard
📄 Documents: https://lean-women-dad-achievement.trycloudflare.com/documents
🏢 Organization: https://lean-women-dad-achievement.trycloudflare.com/organization

🔐 Temporary demo account:
   Email: runtime-demo@vaultify.local
   Password: Vaultify-xt0rz8-YqwRg-9

🔗 MCP endpoint:
https://developing-ultimate-bizarre-putting.trycloudflare.com/mcp

⚠️ The website URL is temporary and remains active only while this Colab runtime is connected.


In [19]:
# VAULTIFY COMPACT - Cell 16:
# Audit old and web-upload Tesla chunks directly from Qdrant

from collections import Counter, defaultdict
import statistics

from qdrant_client.models import (
    FieldCondition,
    Filter,
    MatchValue,
)


OLD_TESLA_TENANT_ID = (
    DEMO_TENANTS["company_b"]["tenant_id"]
)


def load_filtered_points(
    qdrant_filter,
):
    """
    Load every Qdrant point matching one filter.
    """
    points = []
    next_offset = None

    while True:
        batch, next_offset = qdrant.scroll(
            collection_name=COLLECTION_NAME,
            scroll_filter=qdrant_filter,
            limit=256,
            offset=next_offset,
            with_payload=True,
            with_vectors=False,
        )

        points.extend(batch)

        if next_offset is None:
            break

    return points


def load_tenant_points(
    tenant_id,
):
    """
    Load every Qdrant point belonging to one tenant.
    """
    tenant_filter = Filter(
        must=[
            FieldCondition(
                key=TENANT_ID_FIELD,
                match=MatchValue(
                    value=tenant_id
                ),
            )
        ]
    )

    return load_filtered_points(
        tenant_filter
    )


def normalize_chunk_text(text):
    """
    Normalize whitespace for exact duplicate detection.
    """
    return " ".join(
        str(text or "").split()
    )


def count_tokens_without_truncation(text):
    """
    Count tokenizer IDs without allowing silent truncation.
    """
    tokenized = (
        embedding_model.tokenizer(
            str(text or ""),
            add_special_tokens=True,
            truncation=False,
            return_attention_mask=False,
            return_token_type_ids=False,
            verbose=False,
        )
    )

    return len(
        tokenized["input_ids"]
    )


def percentile(
    values,
    percentage,
):
    """
    Calculate one percentile without external dependencies.
    """
    if not values:
        return 0

    sorted_values = sorted(values)

    index = int(
        round(
            (len(sorted_values) - 1)
            * percentage
        )
    )

    return sorted_values[index]


# ---------------------------------------------------------------------
# 1. Recover the Tesla document hash from the validated seed tenant
# ---------------------------------------------------------------------

old_tesla_points = load_tenant_points(
    OLD_TESLA_TENANT_ID
)

if not old_tesla_points:
    raise RuntimeError(
        "The original Tesla seed points were not found in Qdrant."
    )

old_hash_counts = Counter(
    (point.payload or {}).get(
        DOCUMENT_HASH_FIELD
    )
    for point in old_tesla_points
    if (point.payload or {}).get(
        DOCUMENT_HASH_FIELD
    )
)

if not old_hash_counts:
    raise RuntimeError(
        "The original Tesla points contain no document hash."
    )

TESLA_DOCUMENT_HASH = (
    old_hash_counts.most_common(1)[0][0]
)

print(
    "✅ Tesla document hash recovered "
    "from the original seed tenant."
)
print(
    f"🔑 Hash: {TESLA_DOCUMENT_HASH}"
)


# ---------------------------------------------------------------------
# 2. Find every tenant containing the same Tesla PDF
# ---------------------------------------------------------------------

tesla_hash_filter = Filter(
    must=[
        FieldCondition(
            key=DOCUMENT_HASH_FIELD,
            match=MatchValue(
                value=TESLA_DOCUMENT_HASH
            ),
        )
    ]
)

all_tesla_document_points = (
    load_filtered_points(
        tesla_hash_filter
    )
)

points_by_tenant = defaultdict(list)

for point in all_tesla_document_points:
    payload = point.payload or {}

    tenant_id = payload.get(
        TENANT_ID_FIELD
    )

    if tenant_id:
        points_by_tenant[
            tenant_id
        ].append(point)


print("\n📦 Tesla copies found in Qdrant:")

for tenant_id, points in sorted(
    points_by_tenant.items(),
    key=lambda item: len(item[1]),
):
    filenames = sorted(
        {
            (point.payload or {}).get(
                FILENAME_FIELD,
                "Unknown file",
            )
            for point in points
        }
    )

    print(
        f"   {tenant_id}: "
        f"{len(points)} points "
        f"— {filenames}"
    )


web_tenant_candidates = {
    tenant_id: points
    for tenant_id, points
    in points_by_tenant.items()
    if tenant_id
    != OLD_TESLA_TENANT_ID
}

if not web_tenant_candidates:
    raise RuntimeError(
        "No separate web-upload Tesla tenant was found in Qdrant. "
        "The runtime database was deleted, and the web-upload vectors "
        "may also have been removed or written to another collection."
    )

# Select the largest separate Tesla tenant.
NEW_TESLA_TENANT_ID = max(
    web_tenant_candidates,
    key=lambda tenant_id: len(
        web_tenant_candidates[
            tenant_id
        ]
    ),
)

print(
    "\n✅ Web-upload Tesla tenant "
    "recovered directly from Qdrant."
)
print(
    f"🏢 Tenant: "
    f"{NEW_TESLA_TENANT_ID}"
)


# ---------------------------------------------------------------------
# 3. Audit utility
# ---------------------------------------------------------------------

def summarize_tenant_chunks(
    label,
    tenant_id,
):
    """
    Print chunk types, token sizes, filenames,
    hashes, and duplicate counts.
    """
    points = load_tenant_points(
        tenant_id
    )

    payloads = [
        point.payload or {}
        for point in points
    ]

    texts = [
        normalize_chunk_text(
            payload.get(TEXT_FIELD)
        )
        for payload in payloads
    ]

    texts = [
        text
        for text in texts
        if text
    ]

    token_counts = [
        count_tokens_without_truncation(
            text
        )
        for text in texts
    ]

    chunk_types = Counter(
        payload.get(
            CHUNK_TYPE_FIELD,
            "unknown",
        )
        for payload in payloads
    )

    filenames = sorted(
        {
            payload.get(
                FILENAME_FIELD,
                "Unknown file",
            )
            for payload in payloads
        }
    )

    document_hashes = sorted(
        {
            payload.get(
                DOCUMENT_HASH_FIELD,
                "Missing hash",
            )
            for payload in payloads
        }
    )

    duplicate_count = (
        len(texts)
        - len(set(texts))
    )

    oversized_256_count = sum(
        token_count > 256
        for token_count
        in token_counts
    )

    oversized_240_count = sum(
        token_count > 240
        for token_count
        in token_counts
    )

    print("\n" + "=" * 88)
    print(f"📊 {label}")
    print(f"🏢 Tenant: {tenant_id}")
    print(f"📦 Total points: {len(points)}")
    print(f"📄 Files: {filenames}")
    print(f"🔑 Document hashes: {document_hashes}")
    print(
        f"🧩 Chunk types: "
        f"{dict(chunk_types)}"
    )
    print(
        f"♻️ Exact duplicate chunks: "
        f"{duplicate_count}"
    )

    if token_counts:
        print(
            "📐 Token counts:"
            f" min={min(token_counts)},"
            f" median="
            f"{statistics.median(token_counts):.1f},"
            f" p95="
            f"{percentile(token_counts, 0.95)},"
            f" max={max(token_counts)}"
        )

    print(
        f"⚠️ Chunks above 240 tokens: "
        f"{oversized_240_count}"
    )

    print(
        f"⚠️ Chunks above 256 tokens: "
        f"{oversized_256_count}"
    )

    return {
        "points": points,
        "texts": texts,
        "token_counts": token_counts,
        "chunk_types": chunk_types,
        "duplicates": duplicate_count,
        "oversized_240": (
            oversized_240_count
        ),
        "oversized_256": (
            oversized_256_count
        ),
    }


# ---------------------------------------------------------------------
# 4. Compare both pipelines
# ---------------------------------------------------------------------

old_tesla_audit = (
    summarize_tenant_chunks(
        label=(
            "Original seeded "
            "Tesla pipeline"
        ),
        tenant_id=(
            OLD_TESLA_TENANT_ID
        ),
    )
)

new_tesla_audit = (
    summarize_tenant_chunks(
        label=(
            "Web-upload "
            "Tesla pipeline"
        ),
        tenant_id=(
            NEW_TESLA_TENANT_ID
        ),
    )
)


old_count = len(
    old_tesla_audit["points"]
)

new_count = len(
    new_tesla_audit["points"]
)


print("\n" + "=" * 88)
print("🔍 Comparison")

print(
    f"Original seeded pipeline: "
    f"{old_count} chunks"
)

print(
    f"Web-upload pipeline: "
    f"{new_count} chunks"
)

if old_count:
    print(
        "Web/seed chunk ratio: "
        f"{new_count / old_count:.2f}x"
    )


new_pipeline_is_safe = (
    new_tesla_audit[
        "duplicates"
    ]
    == 0
    and new_tesla_audit[
        "oversized_240"
    ]
    == 0
)


if new_pipeline_is_safe:
    print(
        "✅ The web-upload chunks contain "
        "no exact duplicates and stay "
        "within the 240-token safety limit."
    )

else:
    print(
        "⚠️ The web-upload chunk set "
        "requires correction before becoming "
        "the final ingestion strategy."
    )

[web-cloudflared] 2026-07-23T15:23:42Z INF +-------------------------------------------------------------------------------------+
[web-cloudflared] 2026-07-23T15:23:42Z INF |                               CONNECTIVITY PRE-CHECKS                               |
[web-cloudflared] 2026-07-23T15:23:42Z INF +-------------------------------------------------------------------------------------+
[web-cloudflared] 2026-07-23T15:23:42Z INF |  COMPONENT         TARGET                     STATUS  DETAILS                       |
[web-cloudflared] 2026-07-23T15:23:42Z INF |  DNS Resolution    region1.v2.argotunnel.com  PASS    DNS Resolved successfully     |
[web-cloudflared] 2026-07-23T15:23:42Z INF |  DNS Resolution    region2.v2.argotunnel.com  PASS    DNS Resolved successfully     |
[web-cloudflared] 2026-07-23T15:23:42Z INF |  UDP Connectivity  region1.v2.argotunnel.com  PASS    QUIC connection successful    |
[web-cloudflared] 2026-07-23T15:23:42Z INF |  UDP Connectivity  region2.v2.argotunn

In [20]:
# VAULTIFY COMPACT - Cell 17:
# Install a token-safe, table-aware, duplicate-free chunker

import hashlib
import re

import vaultify_web_backend


# Leave a safety margin below the model's 256-token maximum.
MAX_SAFE_CHUNK_TOKENS = min(
    240,
    embedding_model.max_seq_length - 16,
)

TOKENIZER = embedding_model.tokenizer


def tokenize_ids(
    text,
    *,
    add_special_tokens,
):
    """
    Tokenize through the public tokenizer API
    without truncating the input.
    """
    encoded = TOKENIZER(
        str(text or ""),
        add_special_tokens=add_special_tokens,
        truncation=False,
        return_attention_mask=False,
        return_token_type_ids=False,
        verbose=False,
    )

    token_ids = encoded["input_ids"]

    if (
        token_ids
        and isinstance(
            token_ids[0],
            (list, tuple),
        )
    ):
        token_ids = token_ids[0]

    return list(token_ids)


empty_raw_ids = tokenize_ids(
    "",
    add_special_tokens=False,
)

empty_complete_ids = tokenize_ids(
    "",
    add_special_tokens=True,
)

SPECIAL_TOKEN_COUNT = max(
    0,
    len(empty_complete_ids)
    - len(empty_raw_ids),
)

RAW_TOKEN_BUDGET = (
    MAX_SAFE_CHUNK_TOKENS
    - SPECIAL_TOKEN_COUNT
)

TEXT_OVERLAP_TOKENS = 30


if RAW_TOKEN_BUDGET < 32:
    raise RuntimeError(
        "The calculated raw token budget is unexpectedly small."
    )


def get_raw_token_ids(text):
    """
    Return token IDs without special tokens or truncation.
    """
    return tokenize_ids(
        text,
        add_special_tokens=False,
    )


def count_safe_tokens(text):
    """
    Count the complete tokenizer input,
    including special tokens and without truncation.
    """
    return len(
        tokenize_ids(
            text,
            add_special_tokens=True,
        )
    )


def decode_token_ids(token_ids):
    """
    Convert tokenizer IDs back into readable text.
    """
    return TOKENIZER.decode(
        token_ids,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=True,
    ).strip()


print(
    f"ℹ️ Tokenizer: "
    f"{TOKENIZER.__class__.__name__}"
)

print(
    f"ℹ️ Special-token overhead: "
    f"{SPECIAL_TOKEN_COUNT}"
)

print(
    f"ℹ️ Raw payload budget: "
    f"{RAW_TOKEN_BUDGET}"
)

def split_by_exact_tokens(
    text,
    overlap_tokens=0,
):
    """
    Split text into windows that can never exceed the embedding limit.
    """
    text = str(text or "").strip()

    if not text:
        return []

    raw_ids = get_raw_token_ids(text)

    if len(raw_ids) <= RAW_TOKEN_BUDGET:
        return [text]

    overlap_tokens = max(
        0,
        min(
            overlap_tokens,
            RAW_TOKEN_BUDGET - 1,
        ),
    )

    step_size = (
        RAW_TOKEN_BUDGET
        - overlap_tokens
    )

    pieces = []
    start_index = 0

    while start_index < len(raw_ids):
        end_index = min(
            start_index + RAW_TOKEN_BUDGET,
            len(raw_ids),
        )

        token_slice = raw_ids[
            start_index:end_index
        ]

        decoded_piece = decode_token_ids(
            token_slice
        )

        if decoded_piece:
            pieces.append(
                decoded_piece
            )

        if end_index >= len(raw_ids):
            break

        start_index += step_size

    return pieces


def normalize_chunk_for_dedup(text):
    """
    Normalize formatting differences before exact duplicate comparison.
    """
    return re.sub(
        r"\s+",
        " ",
        str(text or ""),
    ).strip()


def create_chunk_record(
    text,
    chunk_type,
    section,
):
    """
    Create one consistently structured chunk dictionary.
    """
    return {
        "text": str(text).strip(),
        "chunk_type": chunk_type,
        "section": (
            str(section).strip()
            or "Document content"
        ),
    }


def split_normal_text_safely(
    text,
    section,
):
    """
    Split ordinary document text with a small token overlap.
    """
    return [
        create_chunk_record(
            text=piece,
            chunk_type="text",
            section=section,
        )
        for piece in split_by_exact_tokens(
            text,
            overlap_tokens=(
                TEXT_OVERLAP_TOKENS
            ),
        )
        if piece.strip()
    ]


def identify_table_header_lines(
    table_lines,
):
    """
    Preserve the Markdown table header and separator when present.
    """
    if not table_lines:
        return [], []

    has_separator_row = (
        len(table_lines) >= 2
        and re.match(
            r"^\|\s*:?-{3,}",
            table_lines[1].strip(),
        )
    )

    header_line_count = (
        2 if has_separator_row else 1
    )

    return (
        table_lines[:header_line_count],
        table_lines[header_line_count:],
    )


def split_table_safely(
    table_lines,
    section,
):
    """
    Split a Markdown table by rows while repeating its header.

    A single oversized row is split directly by tokenizer IDs.
    """
    if not table_lines:
        return []

    header_lines, data_rows = (
        identify_table_header_lines(
            table_lines
        )
    )

    table_prefix = "\n".join(
        [
            str(section).strip(),
            "",
            *header_lines,
        ]
    ).strip()

    # Extremely large headers fall back to exact token windows.
    if (
        count_safe_tokens(table_prefix)
        >= MAX_SAFE_CHUNK_TOKENS
    ):
        entire_table = "\n".join(
            [
                table_prefix,
                *data_rows,
            ]
        ).strip()

        return [
            create_chunk_record(
                text=piece,
                chunk_type="table",
                section=section,
            )
            for piece in split_by_exact_tokens(
                entire_table,
                overlap_tokens=0,
            )
        ]

    if not data_rows:
        return [
            create_chunk_record(
                text=piece,
                chunk_type="table",
                section=section,
            )
            for piece in split_by_exact_tokens(
                table_prefix,
                overlap_tokens=0,
            )
        ]

    table_chunks = []
    current_rows = []

    def flush_current_rows():
        nonlocal current_rows

        if not current_rows:
            return

        table_text = "\n".join(
            [
                table_prefix,
                *current_rows,
            ]
        ).strip()

        for piece in split_by_exact_tokens(
            table_text,
            overlap_tokens=0,
        ):
            table_chunks.append(
                create_chunk_record(
                    text=piece,
                    chunk_type="table",
                    section=section,
                )
            )

        current_rows = []

    for row in data_rows:
        candidate_rows = (
            current_rows + [row]
        )

        candidate_text = "\n".join(
            [
                table_prefix,
                *candidate_rows,
            ]
        ).strip()

        if (
            count_safe_tokens(candidate_text)
            <= MAX_SAFE_CHUNK_TOKENS
        ):
            current_rows = candidate_rows
            continue

        flush_current_rows()

        single_row_text = "\n".join(
            [
                table_prefix,
                row,
            ]
        ).strip()

        if (
            count_safe_tokens(single_row_text)
            <= MAX_SAFE_CHUNK_TOKENS
        ):
            current_rows = [row]
            continue

        # The individual row itself is oversized.
        prefix_raw_ids = get_raw_token_ids(
            table_prefix
        )

        available_row_budget = (
            RAW_TOKEN_BUDGET
            - len(prefix_raw_ids)
            - 2
        )

        if available_row_budget < 16:
            for piece in split_by_exact_tokens(
                single_row_text,
                overlap_tokens=0,
            ):
                table_chunks.append(
                    create_chunk_record(
                        text=piece,
                        chunk_type="table",
                        section=section,
                    )
                )

            continue

        row_raw_ids = get_raw_token_ids(
            row
        )

        for start_index in range(
            0,
            len(row_raw_ids),
            available_row_budget,
        ):
            row_slice = row_raw_ids[
                start_index:
                start_index
                + available_row_budget
            ]

            decoded_row = decode_token_ids(
                row_slice
            )

            safe_row_text = "\n".join(
                [
                    table_prefix,
                    decoded_row,
                ]
            ).strip()

            table_chunks.append(
                create_chunk_record(
                    text=safe_row_text,
                    chunk_type="table",
                    section=section,
                )
            )

    flush_current_rows()

    return table_chunks


def enforce_final_token_safety(
    chunks,
):
    """
    Apply one final hard token check to every generated chunk.
    """
    safe_chunks = []

    for chunk in chunks:
        chunk_text = chunk["text"]

        if (
            count_safe_tokens(chunk_text)
            <= MAX_SAFE_CHUNK_TOKENS
        ):
            safe_chunks.append(chunk)
            continue

        for piece in split_by_exact_tokens(
            chunk_text,
            overlap_tokens=0,
        ):
            safe_chunks.append(
                create_chunk_record(
                    text=piece,
                    chunk_type=(
                        chunk["chunk_type"]
                    ),
                    section=(
                        chunk["section"]
                    ),
                )
            )

    return safe_chunks


def remove_exact_duplicate_chunks(
    chunks,
):
    """
    Remove repeated chunks before embeddings and Qdrant writes.
    """
    unique_chunks = []
    seen_hashes = set()

    for chunk in chunks:
        normalized_text = (
            normalize_chunk_for_dedup(
                chunk["text"]
            )
        )

        if not normalized_text:
            continue

        chunk_hash = hashlib.sha256(
            normalized_text.encode(
                "utf-8"
            )
        ).hexdigest()

        if chunk_hash in seen_hashes:
            continue

        seen_hashes.add(chunk_hash)

        unique_chunks.append(chunk)

    for chunk_index, chunk in enumerate(
        unique_chunks
    ):
        chunk["chunk_index"] = (
            chunk_index
        )

    return unique_chunks


def safe_smart_chunk_markdown(
    model,
    markdown_text,
):
    """
    Convert Markdown into table-aware, token-safe, duplicate-free chunks.
    """
    lines = str(
        markdown_text or ""
    ).splitlines()

    chunks = []
    current_section = (
        "Document introduction"
    )

    text_buffer = []

    def flush_text_buffer():
        nonlocal text_buffer

        buffered_text = "\n".join(
            text_buffer
        ).strip()

        if buffered_text:
            chunks.extend(
                split_normal_text_safely(
                    text=buffered_text,
                    section=current_section,
                )
            )

        text_buffer = []

    line_index = 0

    while line_index < len(lines):
        line = lines[line_index]
        stripped_line = line.strip()

        heading_match = re.match(
            r"^(#{1,6})\s+(.+)$",
            stripped_line,
        )

        if heading_match:
            flush_text_buffer()

            current_section = (
                heading_match
                .group(2)
                .strip()
            )

            text_buffer.append(
                stripped_line
            )

            line_index += 1
            continue

        if stripped_line.startswith("|"):
            flush_text_buffer()

            table_lines = []

            while (
                line_index < len(lines)
                and lines[line_index]
                .strip()
                .startswith("|")
            ):
                table_lines.append(
                    lines[line_index]
                )

                line_index += 1

            chunks.extend(
                split_table_safely(
                    table_lines=table_lines,
                    section=current_section,
                )
            )

            continue

        text_buffer.append(line)
        line_index += 1

    flush_text_buffer()

    chunks = enforce_final_token_safety(
        chunks
    )

    chunks = remove_exact_duplicate_chunks(
        chunks
    )

    oversized_chunks = [
        chunk
        for chunk in chunks
        if (
            count_safe_tokens(
                chunk["text"]
            )
            > MAX_SAFE_CHUNK_TOKENS
        )
    ]

    if oversized_chunks:
        raise RuntimeError(
            "Token safety validation failed. "
            f"{len(oversized_chunks)} oversized "
            "chunks remain."
        )

    return chunks


# Patch the generated Flask backend for this runtime.
vaultify_web_backend.smart_chunk_markdown = (
    safe_smart_chunk_markdown
)

vaultify_web_backend.ingest_document.__globals__[
    "smart_chunk_markdown"
] = safe_smart_chunk_markdown


# ---------------------------------------------------------------------
# Synthetic safety test
# ---------------------------------------------------------------------

synthetic_markdown = "\n".join(
    [
        "# Synthetic Safety Test",
        "",
        " ".join(
            ["financial-data"] * 900
        ),
        "",
        "| Metric | Value |",
        "|---|---|",
        (
            "| Oversized row | "
            + " ".join(
                ["123456"] * 700
            )
            + " |"
        ),
        (
            "| Oversized row | "
            + " ".join(
                ["123456"] * 700
            )
            + " |"
        ),
    ]
)

synthetic_chunks = (
    safe_smart_chunk_markdown(
        embedding_model,
        synthetic_markdown,
    )
)

synthetic_token_counts = [
    count_safe_tokens(
        chunk["text"]
    )
    for chunk in synthetic_chunks
]

synthetic_normalized_texts = [
    normalize_chunk_for_dedup(
        chunk["text"]
    )
    for chunk in synthetic_chunks
]

synthetic_duplicate_count = (
    len(synthetic_normalized_texts)
    - len(set(synthetic_normalized_texts))
)

if max(synthetic_token_counts) > (
    MAX_SAFE_CHUNK_TOKENS
):
    raise RuntimeError(
        "Synthetic token safety test failed."
    )

if synthetic_duplicate_count != 0:
    raise RuntimeError(
        "Synthetic duplicate-removal test failed."
    )


print("✅ Safe web chunker installed.")
print(
    f"✅ Maximum allowed tokens: "
    f"{MAX_SAFE_CHUNK_TOKENS}"
)
print(
    f"✅ Synthetic chunks created: "
    f"{len(synthetic_chunks)}"
)
print(
    f"✅ Synthetic maximum tokens: "
    f"{max(synthetic_token_counts)}"
)
print(
    "✅ Synthetic oversized chunks: 0"
)
print(
    "✅ Synthetic exact duplicates: 0"
)
print(
    "\n⏭️ No real Qdrant vectors were changed yet."
)

ℹ️ Tokenizer: BertTokenizer
ℹ️ Special-token overhead: 2
ℹ️ Raw payload budget: 238
✅ Safe web chunker installed.
✅ Maximum allowed tokens: 240
✅ Synthetic chunks created: 8
✅ Synthetic maximum tokens: 240
✅ Synthetic oversized chunks: 0
✅ Synthetic exact duplicates: 0

⏭️ No real Qdrant vectors were changed yet.


In [28]:
# VAULTIFY COMPACT - Cell 18:
# Generic PDF ingestion dry-run with the safe chunker

from collections import Counter
from pathlib import Path

import hashlib
import json
import statistics
import time

import filetype
import torch

from google.colab import files
from werkzeug.utils import secure_filename

from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import (
    AcceleratorDevice,
    AcceleratorOptions,
    PdfPipelineOptions,
)
from docling.document_converter import (
    DocumentConverter,
    PdfFormatOption,
)


# ---------------------------------------------------------------------
# 1. Verify required runtime components
# ---------------------------------------------------------------------

required_runtime_names = [
    "safe_smart_chunk_markdown",
    "count_safe_tokens",
    "normalize_chunk_for_dedup",
    "MAX_SAFE_CHUNK_TOKENS",
    "embedding_model",
]

missing_runtime_names = [
    name
    for name in required_runtime_names
    if name not in globals()
]

if missing_runtime_names:
    raise RuntimeError(
        "Required components are missing: "
        + ", ".join(missing_runtime_names)
        + ". Run Cell 17 before Cell 18."
    )


# ---------------------------------------------------------------------
# 2. Upload one PDF
# ---------------------------------------------------------------------

print("📄 Select one PDF for the generic ingestion dry-run.")

uploaded_files = files.upload()

if len(uploaded_files) != 1:
    raise RuntimeError(
        "Upload exactly one PDF for this dry-run."
    )

original_filename, pdf_bytes = next(
    iter(uploaded_files.items())
)

safe_filename = secure_filename(
    original_filename
)

if (
    not safe_filename
    or not safe_filename.lower().endswith(".pdf")
):
    raise RuntimeError(
        "The selected file must have a .pdf extension."
    )

if not pdf_bytes:
    raise RuntimeError(
        "The selected PDF is empty."
    )

MAX_DRY_RUN_FILE_SIZE = (
    25 * 1024 * 1024
)

if len(pdf_bytes) > MAX_DRY_RUN_FILE_SIZE:
    raise RuntimeError(
        "The PDF exceeds the current 25 MB application limit."
    )

if not pdf_bytes.startswith(
    b"%PDF-"
):
    raise RuntimeError(
        "The selected file does not contain a valid PDF signature."
    )

detected_type = filetype.guess(
    pdf_bytes[:8192]
)

if (
    detected_type is not None
    and detected_type.mime
    != "application/pdf"
):
    raise RuntimeError(
        "The selected file MIME signature is not PDF."
    )


# ---------------------------------------------------------------------
# 3. Store the PDF temporarily
# ---------------------------------------------------------------------

DRY_RUN_ROOT = Path(
    "/content/vaultify_dry_run"
)

DRY_RUN_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

DRY_RUN_PDF_PATH = (
    DRY_RUN_ROOT
    / safe_filename
)

DRY_RUN_PDF_PATH.write_bytes(
    pdf_bytes
)

DRY_RUN_DOCUMENT_HASH = hashlib.sha256(
    pdf_bytes
).hexdigest()

DRY_RUN_ORIGINAL_FILENAME = (
    original_filename
)

print("\n✅ PDF validation passed.")
print(
    f"📄 Filename: "
    f"{DRY_RUN_ORIGINAL_FILENAME}"
)
print(
    f"💾 Size: "
    f"{len(pdf_bytes) / (1024 * 1024):.2f} MB"
)
print(
    f"🔑 SHA-256: "
    f"{DRY_RUN_DOCUMENT_HASH}"
)


# ---------------------------------------------------------------------
# 4. Parse with Docling
# ---------------------------------------------------------------------

print("\n⏳ Stage 1/3 — Parsing PDF with Docling...")

pipeline_options = PdfPipelineOptions()

if torch.cuda.is_available():
    pipeline_options.accelerator_options = (
        AcceleratorOptions(
            num_threads=4,
            device=(
                AcceleratorDevice.CUDA
            ),
        )
    )

document_converter = DocumentConverter(
    format_options={
        InputFormat.PDF:
        PdfFormatOption(
            pipeline_options=(
                pipeline_options
            )
        )
    }
)

parse_started_at = time.perf_counter()

conversion_result = (
    document_converter.convert(
        str(DRY_RUN_PDF_PATH)
    )
)

DRY_RUN_MARKDOWN = (
    conversion_result.document
    .export_to_markdown()
)

parse_duration_seconds = (
    time.perf_counter()
    - parse_started_at
)

if not DRY_RUN_MARKDOWN.strip():
    raise RuntimeError(
        "Docling produced no Markdown content."
    )

markdown_output_path = (
    DRY_RUN_ROOT
    / (
        DRY_RUN_PDF_PATH.stem
        + ".md"
    )
)

markdown_output_path.write_text(
    DRY_RUN_MARKDOWN,
    encoding="utf-8",
)

print(
    f"✅ Docling parsing completed "
    f"in {parse_duration_seconds:.1f} seconds."
)
print(
    f"📝 Markdown characters: "
    f"{len(DRY_RUN_MARKDOWN):,}"
)


# ---------------------------------------------------------------------
# 5. Apply the safe generic chunker
# ---------------------------------------------------------------------

print(
    "\n⏳ Stage 2/3 — "
    "Creating token-safe chunks..."
)

chunking_started_at = (
    time.perf_counter()
)

DRY_RUN_CHUNKS = (
    safe_smart_chunk_markdown(
        embedding_model,
        DRY_RUN_MARKDOWN,
    )
)

chunking_duration_seconds = (
    time.perf_counter()
    - chunking_started_at
)

if not DRY_RUN_CHUNKS:
    raise RuntimeError(
        "The safe chunker produced no chunks."
    )

print(
    f"✅ Chunking completed "
    f"in {chunking_duration_seconds:.1f} seconds."
)


# ---------------------------------------------------------------------
# 6. Build the quality report
# ---------------------------------------------------------------------

print(
    "\n⏳ Stage 3/3 — "
    "Auditing chunk quality..."
)


def dry_run_percentile(
    values,
    percentile,
):
    if not values:
        return 0

    ordered_values = sorted(
        values
    )

    index = round(
        (
            len(ordered_values)
            - 1
        )
        * percentile
    )

    return ordered_values[
        index
    ]


chunk_token_counts = [
    count_safe_tokens(
        chunk["text"]
    )
    for chunk in DRY_RUN_CHUNKS
]

normalized_chunk_texts = [
    normalize_chunk_for_dedup(
        chunk["text"]
    )
    for chunk in DRY_RUN_CHUNKS
]

exact_duplicate_count = (
    len(normalized_chunk_texts)
    - len(
        set(
            normalized_chunk_texts
        )
    )
)

empty_chunk_count = sum(
    not text
    for text in normalized_chunk_texts
)

oversized_240_count = sum(
    token_count
    > MAX_SAFE_CHUNK_TOKENS
    for token_count
    in chunk_token_counts
)

oversized_256_count = sum(
    token_count > 256
    for token_count
    in chunk_token_counts
)

tiny_chunk_count = sum(
    token_count < 20
    for token_count
    in chunk_token_counts
)

chunk_type_counts = Counter(
    chunk.get(
        "chunk_type",
        "unknown",
    )
    for chunk in DRY_RUN_CHUNKS
)

section_counts = Counter(
    chunk.get(
        "section",
        "Unknown section",
    )
    for chunk in DRY_RUN_CHUNKS
)

DRY_RUN_REPORT = {
    "filename": (
        DRY_RUN_ORIGINAL_FILENAME
    ),
    "document_hash": (
        DRY_RUN_DOCUMENT_HASH
    ),
    "file_size_bytes": len(
        pdf_bytes
    ),
    "parse_seconds": round(
        parse_duration_seconds,
        2,
    ),
    "chunking_seconds": round(
        chunking_duration_seconds,
        2,
    ),
    "markdown_characters": len(
        DRY_RUN_MARKDOWN
    ),
    "total_chunks": len(
        DRY_RUN_CHUNKS
    ),
    "chunk_types": dict(
        chunk_type_counts
    ),
    "token_min": min(
        chunk_token_counts
    ),
    "token_median": (
        statistics.median(
            chunk_token_counts
        )
    ),
    "token_p95": (
        dry_run_percentile(
            chunk_token_counts,
            0.95,
        )
    ),
    "token_max": max(
        chunk_token_counts
    ),
    "chunks_above_240": (
        oversized_240_count
    ),
    "chunks_above_256": (
        oversized_256_count
    ),
    "exact_duplicates": (
        exact_duplicate_count
    ),
    "empty_chunks": (
        empty_chunk_count
    ),
    "tiny_chunks_below_20": (
        tiny_chunk_count
    ),
    "unique_sections": len(
        section_counts
    ),
}

DRY_RUN_READY_FOR_EVALUATION = (
    oversized_240_count == 0
    and oversized_256_count == 0
    and exact_duplicate_count == 0
    and empty_chunk_count == 0
)


# ---------------------------------------------------------------------
# 7. Save temporary audit artifacts
# ---------------------------------------------------------------------

chunks_output_path = (
    DRY_RUN_ROOT
    / (
        DRY_RUN_PDF_PATH.stem
        + "_safe_chunks.json"
    )
)

report_output_path = (
    DRY_RUN_ROOT
    / (
        DRY_RUN_PDF_PATH.stem
        + "_dry_run_report.json"
    )
)

chunks_output_path.write_text(
    json.dumps(
        DRY_RUN_CHUNKS,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

report_output_path.write_text(
    json.dumps(
        DRY_RUN_REPORT,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)


# ---------------------------------------------------------------------
# 8. Print the final report
# ---------------------------------------------------------------------

print("\n" + "=" * 88)
print("📊 GENERIC INGESTION DRY-RUN REPORT")

print(
    f"📄 File: "
    f"{DRY_RUN_REPORT['filename']}"
)

print(
    f"📦 Total chunks: "
    f"{DRY_RUN_REPORT['total_chunks']}"
)

print(
    f"🧩 Chunk types: "
    f"{DRY_RUN_REPORT['chunk_types']}"
)

print(
    "📐 Token distribution:"
    f" min={DRY_RUN_REPORT['token_min']},"
    f" median="
    f"{DRY_RUN_REPORT['token_median']:.1f},"
    f" p95="
    f"{DRY_RUN_REPORT['token_p95']},"
    f" max="
    f"{DRY_RUN_REPORT['token_max']}"
)

print(
    f"⚠️ Chunks above 240 tokens: "
    f"{DRY_RUN_REPORT['chunks_above_240']}"
)

print(
    f"⚠️ Chunks above 256 tokens: "
    f"{DRY_RUN_REPORT['chunks_above_256']}"
)

print(
    f"♻️ Exact duplicates: "
    f"{DRY_RUN_REPORT['exact_duplicates']}"
)

print(
    f"🕳️ Empty chunks: "
    f"{DRY_RUN_REPORT['empty_chunks']}"
)

print(
    f"🔹 Tiny chunks below 20 tokens: "
    f"{DRY_RUN_REPORT['tiny_chunks_below_20']}"
)

print(
    f"📚 Unique sections: "
    f"{DRY_RUN_REPORT['unique_sections']}"
)

print("\nMost frequent sections:")

for section, count in (
    section_counts.most_common(10)
):
    print(
        f"   {count:>4} chunks — "
        f"{section}"
    )


print("\nLongest generated chunks:")

longest_chunk_indexes = sorted(
    range(
        len(DRY_RUN_CHUNKS)
    ),
    key=lambda index: (
        chunk_token_counts[index]
    ),
    reverse=True,
)[:5]

for chunk_index in (
    longest_chunk_indexes
):
    chunk = DRY_RUN_CHUNKS[
        chunk_index
    ]

    preview = " ".join(
        chunk["text"].split()
    )[:180]

    print(
        f"   Chunk {chunk_index}"
        f" — {chunk_token_counts[chunk_index]} tokens"
        f" — {chunk['chunk_type']}"
        f" — {chunk['section']}"
    )

    print(
        f"      {preview}..."
    )


print("\n" + "=" * 88)

if DRY_RUN_READY_FOR_EVALUATION:
    print(
        "✅ Dry-run passed all hard safety checks."
    )

    print(
        "➡️ The chunks are ready for "
        "retrieval-quality evaluation in Cell 19."
    )

else:
    print(
        "❌ Dry-run failed one or more hard checks."
    )

    print(
        "⛔ Do not generate embeddings "
        "or write these chunks to Qdrant."
    )


print(
    f"\n📁 Temporary artifacts: "
    f"{DRY_RUN_ROOT}"
)

print(
    "⏭️ No embeddings were generated."
)

print(
    "⏭️ No Qdrant points were created, "
    "updated, or deleted."
)

📄 Select one PDF for the generic ingestion dry-run.


[INFO] 2026-07-23 15:47:16,170 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-07-23 15:47:16,186 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_PP-OCRv4_det_infer.onnx
[INFO] 2026-07-23 15:47:16,186 [RapidOCR] main.py:53: Using /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_PP-OCRv4_det_infer.onnx


Saving _10-K-2025-As-Filed.pdf to _10-K-2025-As-Filed.pdf

✅ PDF validation passed.
📄 Filename: _10-K-2025-As-Filed.pdf
💾 Size: 1.77 MB
🔑 SHA-256: 3eb270b22acb7d8d8e9c32a43dc221dec3345f8e4bca02755fdbc1ee16c823de

⏳ Stage 1/3 — Parsing PDF with Docling...


[INFO] 2026-07-23 15:47:16,292 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-07-23 15:47:16,296 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_infer.onnx
[INFO] 2026-07-23 15:47:16,297 [RapidOCR] main.py:53: Using /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_infer.onnx
[INFO] 2026-07-23 15:47:16,349 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-07-23 15:47:16,380 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_PP-OCRv4_rec_infer.onnx
[INFO] 2026-07-23 15:47:16,381 [RapidOCR] main.py:53: Using /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_PP-OCRv4_rec_infer.onnx


Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

✅ Docling parsing completed in 90.2 seconds.
📝 Markdown characters: 342,871

⏳ Stage 2/3 — Creating token-safe chunks...
✅ Chunking completed in 1.8 seconds.

⏳ Stage 3/3 — Auditing chunk quality...

📊 GENERIC INGESTION DRY-RUN REPORT
📄 File: _10-K-2025-As-Filed.pdf
📦 Total chunks: 565
🧩 Chunk types: {'text': 400, 'table': 165}
📐 Token distribution: min=4, median=186.0, p95=240, max=240
⚠️ Chunks above 240 tokens: 0
⚠️ Chunks above 256 tokens: 0
♻️ Exact duplicates: 0
🕳️ Empty chunks: 0
🔹 Tiny chunks below 20 tokens: 55
📚 Unique sections: 234

Most frequent sections:
     31 chunks — CONSOLIDATED STATEMENTS OF CASH FLOWS
     21 chunks — (3) Exhibits required by Item 601 of Regulation S-K  (1)
     18 chunks — Note 13 - Segment Information and Geographic Data
     17 chunks — Term Debt
     12 chunks — Cash, Cash Equivalents and Marketable Securities
      9 chunks — Events of Default
      8 chunks — The  Company's  business  can  be  impacted  by  political  events,  trade  and  othe

In [29]:
# VAULTIFY COMPACT - Cell 18A:
# Preserve each document dry-run result for later comparison

if "DRY_RUN_RESULTS" not in globals():
    DRY_RUN_RESULTS = {}

DRY_RUN_RESULTS[
    DRY_RUN_DOCUMENT_HASH
] = {
    "filename": DRY_RUN_ORIGINAL_FILENAME,
    "document_hash": DRY_RUN_DOCUMENT_HASH,
    "report": dict(DRY_RUN_REPORT),
    "chunks": list(DRY_RUN_CHUNKS),
    "markdown": DRY_RUN_MARKDOWN,
}

print(
    "✅ Dry-run result preserved:"
    f" {DRY_RUN_ORIGINAL_FILENAME}"
)

print(
    f"📦 Stored documents: "
    f"{len(DRY_RUN_RESULTS)}"
)

for stored_result in (
    DRY_RUN_RESULTS.values()
):
    report = stored_result["report"]

    print(
        f"   {stored_result['filename']}"
        f" — {report['total_chunks']} chunks"
        f" — max {report['token_max']} tokens"
        f" — duplicates {report['exact_duplicates']}"
    )

✅ Dry-run result preserved: _10-K-2025-As-Filed.pdf
📦 Stored documents: 3
   TSLA-Q4-2025-Update.pdf — 258 chunks — max 240 tokens — duplicates 0
   nist.ai.100-1.pdf — 153 chunks — max 240 tokens — duplicates 0
   _10-K-2025-As-Filed.pdf — 565 chunks — max 240 tokens — duplicates 0


In [30]:
# VAULTIFY COMPACT - Cell 19:
# Generic in-memory retrieval benchmark for preserved dry-run documents

from collections import defaultdict

import numpy as np


# ---------------------------------------------------------------------
# 1. Validate preserved dry-run results
# ---------------------------------------------------------------------

if (
    "DRY_RUN_RESULTS" not in globals()
    or len(DRY_RUN_RESULTS) < 3
):
    raise RuntimeError(
        "Three preserved dry-run documents are required. "
        "Run Cell 18 and Cell 18A for Apple, Tesla, and NIST first."
    )


def find_preserved_document(filename_fragment):
    """
    Find one preserved dry-run document by a case-insensitive
    filename fragment.
    """
    filename_fragment = (
        filename_fragment.lower()
    )

    matches = [
        result
        for result in DRY_RUN_RESULTS.values()
        if filename_fragment
        in result["filename"].lower()
    ]

    if len(matches) != 1:
        raise RuntimeError(
            f"Expected exactly one document matching "
            f"'{filename_fragment}', found {len(matches)}."
        )

    return matches[0]


apple_result = find_preserved_document(
    "10-k"
)

tesla_result = find_preserved_document(
    "tsla"
)

nist_result = find_preserved_document(
    "nist"
)


DOCUMENT_BENCHMARKS = {
    "Apple 10-K": {
        "result": apple_result,
        "positive_queries": [
            {
                "question": (
                    "What were Apple's total net sales "
                    "in fiscal year 2025?"
                ),
                "expected_groups": [
                    ["416,161"],
                    [
                        "total net sales",
                        "net sales",
                    ],
                ],
            },
        ],
        "negative_queries": [
            (
                "What is the capital city "
                "of Australia?"
            ),
        ],
    },

    "Tesla Q4 2025": {
        "result": tesla_result,
        "positive_queries": [
            {
                "question": (
                    "What was Tesla's total revenue "
                    "in Q4 2025?"
                ),
                "expected_groups": [
                    ["24,901"],
                    [
                        "total revenue",
                        "total revenues",
                        "revenues",
                    ],
                ],
            },
        ],
        "negative_queries": [
            (
                "Who wrote the novel "
                "Pride and Prejudice?"
            ),
        ],
    },

    "NIST AI RMF": {
        "result": nist_result,
        "positive_queries": [
            {
                "question": (
                    "What are the four core functions "
                    "of the NIST AI Risk Management Framework?"
                ),
                "expected_groups": [
                    ["govern"],
                    ["map"],
                    ["measure"],
                    ["manage"],
                ],
            },
            {
                "question": (
                    "What characteristics are associated "
                    "with trustworthy AI systems?"
                ),
                "expected_groups": [
                    [
                        "valid and reliable",
                        "validity and reliability",
                    ],
                    ["safe"],
                    [
                        "secure and resilient",
                        "security and resilience",
                    ],
                    [
                        "accountable and transparent",
                        "accountability and transparency",
                    ],
                    [
                        "explainable and interpretable",
                        "explainability and interpretability",
                    ],
                    ["privacy"],
                    ["fair"],
                ],
            },
        ],
        "negative_queries": [
            (
                "How many goals did the winner "
                "of the 2018 World Cup final score?"
            ),
        ],
    },
}


# ---------------------------------------------------------------------
# 2. Create and cache document embeddings
# ---------------------------------------------------------------------

print("⏳ Creating in-memory embeddings...")

for document_label, benchmark in (
    DOCUMENT_BENCHMARKS.items()
):
    result = benchmark["result"]

    chunks = result["chunks"]

    chunk_texts = [
        chunk["text"]
        for chunk in chunks
    ]

    if "embeddings" not in result:
        print(
            f"   Embedding {document_label}: "
            f"{len(chunk_texts)} chunks"
        )

        result["embeddings"] = (
            embedding_model.encode(
                chunk_texts,
                batch_size=64,
                convert_to_numpy=True,
                normalize_embeddings=True,
                show_progress_bar=True,
            )
        )

    embeddings = result[
        "embeddings"
    ]

    if len(embeddings) != len(chunks):
        raise RuntimeError(
            f"Embedding count mismatch "
            f"for {document_label}."
        )

print("✅ In-memory embeddings are ready.")


# ---------------------------------------------------------------------
# 3. Generic semantic retrieval
# ---------------------------------------------------------------------

def retrieve_from_preserved_document(
    result,
    question,
    top_k=6,
):
    """
    Retrieve the most similar dry-run chunks using cosine similarity.
    Embeddings are normalized, so dot product equals cosine similarity.
    """
    query_embedding = (
        embedding_model.encode(
            [question],
            convert_to_numpy=True,
            normalize_embeddings=True,
            show_progress_bar=False,
        )[0]
    )

    document_embeddings = (
        result["embeddings"]
    )

    similarity_scores = (
        document_embeddings
        @ query_embedding
    )

    top_indexes = np.argsort(
        similarity_scores
    )[::-1][:top_k]

    retrieved = []

    for rank, chunk_index in enumerate(
        top_indexes,
        start=1,
    ):
        chunk = result["chunks"][
            int(chunk_index)
        ]

        retrieved.append(
            {
                "rank": rank,
                "chunk_index": int(
                    chunk_index
                ),
                "score": float(
                    similarity_scores[
                        chunk_index
                    ]
                ),
                "text": chunk["text"],
                "section": chunk.get(
                    "section",
                    "Unknown section",
                ),
                "chunk_type": chunk.get(
                    "chunk_type",
                    "unknown",
                ),
            }
        )

    return retrieved


def expected_groups_found(
    retrieved_chunks,
    expected_groups,
):
    """
    Every expected group must have at least one matching alternative
    somewhere in the retrieved context.
    """
    combined_context = " ".join(
        item["text"]
        for item in retrieved_chunks
    ).lower()

    group_results = []

    for alternatives in expected_groups:
        matched_alternative = next(
            (
                alternative
                for alternative
                in alternatives
                if alternative.lower()
                in combined_context
            ),
            None,
        )

        group_results.append(
            {
                "alternatives": alternatives,
                "matched": (
                    matched_alternative
                ),
            }
        )

    passed = all(
        group["matched"]
        is not None
        for group in group_results
    )

    return passed, group_results


# ---------------------------------------------------------------------
# 4. Print chunk-density diagnostics
# ---------------------------------------------------------------------

print("\n" + "=" * 92)
print("📐 CHUNK DENSITY DIAGNOSTICS")

for document_label, benchmark in (
    DOCUMENT_BENCHMARKS.items()
):
    result = benchmark["result"]
    chunks = result["chunks"]

    token_counts = [
        count_safe_tokens(
            chunk["text"]
        )
        for chunk in chunks
    ]

    saturated_count = sum(
        token_count
        == MAX_SAFE_CHUNK_TOKENS
        for token_count
        in token_counts
    )

    tiny_count = sum(
        token_count < 20
        for token_count
        in token_counts
    )

    saturated_ratio = (
        saturated_count
        / len(token_counts)
        * 100
    )

    tiny_ratio = (
        tiny_count
        / len(token_counts)
        * 100
    )

    print(
        f"\n{document_label}"
    )

    print(
        f"   Total chunks: "
        f"{len(chunks)}"
    )

    print(
        f"   Exactly 240 tokens: "
        f"{saturated_count} "
        f"({saturated_ratio:.1f}%)"
    )

    print(
        f"   Below 20 tokens: "
        f"{tiny_count} "
        f"({tiny_ratio:.1f}%)"
    )


# ---------------------------------------------------------------------
# 5. Positive retrieval benchmarks
# ---------------------------------------------------------------------

positive_results = []

print("\n" + "=" * 92)
print("🔎 POSITIVE RETRIEVAL BENCHMARKS")

for document_label, benchmark in (
    DOCUMENT_BENCHMARKS.items()
):
    for test_case in benchmark[
        "positive_queries"
    ]:
        question = test_case[
            "question"
        ]

        retrieved = (
            retrieve_from_preserved_document(
                result=benchmark["result"],
                question=question,
                top_k=6,
            )
        )

        passed, group_results = (
            expected_groups_found(
                retrieved,
                test_case[
                    "expected_groups"
                ],
            )
        )

        positive_results.append(
            {
                "document": (
                    document_label
                ),
                "question": question,
                "passed": passed,
                "top_score": (
                    retrieved[0]["score"]
                ),
                "retrieved": retrieved,
                "group_results": (
                    group_results
                ),
            }
        )

        status = (
            "✅ PASS"
            if passed
            else "❌ FAIL"
        )

        print(
            f"\n{status} — "
            f"{document_label}"
        )

        print(
            f"Question: {question}"
        )

        print(
            f"Top score: "
            f"{retrieved[0]['score']:.4f}"
        )

        print(
            "Expected evidence:"
        )

        for group in group_results:
            print(
                f"   "
                f"{'✅' if group['matched'] else '❌'} "
                f"{group['alternatives']}"
                f" → "
                f"{group['matched']}"
            )

        print(
            "Top retrieved chunks:"
        )

        for item in retrieved[:3]:
            preview = " ".join(
                item["text"].split()
            )[:180]

            print(
                f"   #{item['rank']} "
                f"score={item['score']:.4f} "
                f"type={item['chunk_type']} "
                f"section={item['section']}"
            )

            print(
                f"      {preview}..."
            )


# ---------------------------------------------------------------------
# 6. Negative-query score diagnostics
# ---------------------------------------------------------------------

negative_scores = defaultdict(
    list
)

print("\n" + "=" * 92)
print("🚫 NEGATIVE QUERY SCORE DIAGNOSTICS")

for document_label, benchmark in (
    DOCUMENT_BENCHMARKS.items()
):
    for question in benchmark[
        "negative_queries"
    ]:
        retrieved = (
            retrieve_from_preserved_document(
                result=benchmark["result"],
                question=question,
                top_k=3,
            )
        )

        top_score = retrieved[0][
            "score"
        ]

        negative_scores[
            document_label
        ].append(top_score)

        print(
            f"\n{document_label}"
        )

        print(
            f"Question: {question}"
        )

        print(
            f"Highest irrelevant similarity: "
            f"{top_score:.4f}"
        )

        print(
            f"Top unrelated section: "
            f"{retrieved[0]['section']}"
        )


# ---------------------------------------------------------------------
# 7. Final benchmark summary
# ---------------------------------------------------------------------

passed_positive_tests = sum(
    result["passed"]
    for result in positive_results
)

total_positive_tests = len(
    positive_results
)

CELL_19_RETRIEVAL_PASSED = (
    passed_positive_tests
    == total_positive_tests
)

CELL_19_RESULTS = {
    "positive_results": (
        positive_results
    ),
    "negative_scores": dict(
        negative_scores
    ),
    "passed": (
        CELL_19_RETRIEVAL_PASSED
    ),
}

print("\n" + "=" * 92)
print("📊 CELL 19 SUMMARY")

print(
    f"Positive retrieval tests passed: "
    f"{passed_positive_tests}"
    f"/{total_positive_tests}"
)

if CELL_19_RETRIEVAL_PASSED:
    print(
        "✅ Every expected answer signal "
        "was found inside the top-6 retrieved chunks."
    )

    print(
        "➡️ The new chunk sets are ready "
        "for old-vs-new retrieval comparison."
    )

else:
    print(
        "❌ At least one expected answer signal "
        "was not found in the top-6 results."
    )

    print(
        "⛔ Do not replace existing "
        "Qdrant vectors yet."
    )

print(
    "\n⏭️ No Qdrant points were created, "
    "updated, or deleted."
)

⏳ Creating in-memory embeddings...
   Embedding Apple 10-K: 565 chunks


Batches:   0%|          | 0/9 [00:00<?, ?it/s]

   Embedding Tesla Q4 2025: 258 chunks


Batches:   0%|          | 0/5 [00:00<?, ?it/s]

   Embedding NIST AI RMF: 153 chunks


Batches:   0%|          | 0/3 [00:00<?, ?it/s]

✅ In-memory embeddings are ready.

📐 CHUNK DENSITY DIAGNOSTICS

Apple 10-K
   Total chunks: 565
   Exactly 240 tokens: 147 (26.0%)
   Below 20 tokens: 55 (9.7%)

Tesla Q4 2025
   Total chunks: 258
   Exactly 240 tokens: 30 (11.6%)
   Below 20 tokens: 18 (7.0%)

NIST AI RMF
   Total chunks: 153
   Exactly 240 tokens: 93 (60.8%)
   Below 20 tokens: 9 (5.9%)

🔎 POSITIVE RETRIEVAL BENCHMARKS

❌ FAIL — Apple 10-K
Question: What were Apple's total net sales in fiscal year 2025?
Top score: 0.6956
Expected evidence:
   ❌ ['416,161'] → None
   ✅ ['total net sales', 'net sales'] → net sales
Top retrieved chunks:
   #1 score=0.6956 type=text section=Mac
      ## Mac Mac net sales increased during 2025 compared to 2024 primarily due to higher net sales of laptops and desktops....
   #2 score=0.6613 type=text section=iPhone
      ## iPhone iPhone net sales increased during 2025 compared to 2024 due to higher net sales of Pro models....
   #3 score=0.6350 type=text section=Products and Services Perf

In [31]:
# VAULTIFY COMPACT - Cell 19A:
# Diagnose where known evidence ranks inside a dry-run document

import re

import numpy as np


def normalize_evidence_text(text):
    """
    Normalize text so values such as 416,161 and 416161 can match.
    """
    normalized = str(text or "").lower()

    normalized = re.sub(
        r"\s+",
        " ",
        normalized,
    ).strip()

    compact = re.sub(
        r"[^a-z0-9]+",
        "",
        normalized,
    )

    return normalized, compact


def diagnose_evidence_rank(
    result,
    question,
    evidence_terms,
    label,
    preview_neighbors=True,
):
    """
    Find every chunk containing expected evidence and report
    its semantic retrieval rank.
    """
    chunks = result["chunks"]
    embeddings = result["embeddings"]

    query_embedding = embedding_model.encode(
        [question],
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    )[0]

    similarity_scores = (
        embeddings
        @ query_embedding
    )

    ranked_indexes = np.argsort(
        similarity_scores
    )[::-1]

    rank_by_index = {
        int(chunk_index): rank
        for rank, chunk_index
        in enumerate(
            ranked_indexes,
            start=1,
        )
    }

    normalized_terms = [
        normalize_evidence_text(term)
        for term in evidence_terms
    ]

    matching_indexes = []

    for chunk_index, chunk in enumerate(
        chunks
    ):
        normalized_text, compact_text = (
            normalize_evidence_text(
                chunk["text"]
            )
        )

        has_match = any(
            (
                normalized_term
                in normalized_text
            )
            or (
                compact_term
                and compact_term
                in compact_text
            )
            for (
                normalized_term,
                compact_term,
            )
            in normalized_terms
        )

        if has_match:
            matching_indexes.append(
                chunk_index
            )

    print("\n" + "=" * 92)
    print(f"🔬 EVIDENCE RANK DIAGNOSTIC — {label}")
    print(f"Question: {question}")
    print(f"Evidence terms: {evidence_terms}")
    print(
        f"Matching chunks found: "
        f"{len(matching_indexes)}"
    )

    if not matching_indexes:
        print(
            "❌ Expected evidence does not appear "
            "inside any generated chunk."
        )

        print(
            "This suggests a Docling extraction "
            "or chunk-construction issue."
        )

        return []

    diagnostic_results = []

    for chunk_index in matching_indexes:
        chunk = chunks[chunk_index]

        rank = rank_by_index[
            chunk_index
        ]

        score = float(
            similarity_scores[
                chunk_index
            ]
        )

        diagnostic_results.append(
            {
                "chunk_index": chunk_index,
                "rank": rank,
                "score": score,
                "section": chunk.get(
                    "section",
                    "Unknown section",
                ),
                "chunk_type": chunk.get(
                    "chunk_type",
                    "unknown",
                ),
            }
        )

    diagnostic_results.sort(
        key=lambda item: item["rank"]
    )

    for result_item in (
        diagnostic_results[:10]
    ):
        chunk_index = result_item[
            "chunk_index"
        ]

        chunk = chunks[
            chunk_index
        ]

        preview = " ".join(
            chunk["text"].split()
        )[:500]

        print(
            f"\nRank #{result_item['rank']}"
            f" — score={result_item['score']:.4f}"
            f" — chunk={chunk_index}"
            f" — type={result_item['chunk_type']}"
            f" — section={result_item['section']}"
        )

        print(
            f"   {preview}..."
        )

        if preview_neighbors:
            for neighbor_index in [
                chunk_index - 1,
                chunk_index + 1,
            ]:
                if (
                    neighbor_index < 0
                    or neighbor_index
                    >= len(chunks)
                ):
                    continue

                neighbor = chunks[
                    neighbor_index
                ]

                neighbor_preview = " ".join(
                    neighbor["text"].split()
                )[:260]

                print(
                    f"   ↳ Neighbor {neighbor_index}"
                    f" — {neighbor.get('chunk_type')}"
                    f" — {neighbor.get('section')}"
                )

                print(
                    f"      {neighbor_preview}..."
                )

    best_rank = diagnostic_results[
        0
    ]["rank"]

    print("\nInterpretation:")

    if best_rank <= 6:
        print(
            "✅ Evidence is already within top-6. "
            "The original benchmark matching logic "
            "may need inspection."
        )

    elif best_rank <= 20:
        print(
            "⚠️ Evidence exists and is reasonably close, "
            "but dense retrieval needs reranking."
        )

    else:
        print(
            "⚠️ Evidence exists but dense semantic retrieval "
            "ranks it too low. Hybrid lexical + vector "
            "retrieval is recommended."
        )

    return diagnostic_results


APPLE_EVIDENCE_DIAGNOSTIC = (
    diagnose_evidence_rank(
        result=apple_result,
        question=(
            "What were Apple's total net sales "
            "in fiscal year 2025?"
        ),
        evidence_terms=[
            "416,161",
            "416161",
        ],
        label="Apple FY2025 total net sales",
    )
)


🔬 EVIDENCE RANK DIAGNOSTIC — Apple FY2025 total net sales
Question: What were Apple's total net sales in fiscal year 2025?
Evidence terms: ['416,161', '416161']
Matching chunks found: 6

Rank #27 — score=0.5195 — chunk=289 — type=table — section=Note 2 - Revenue
   Note 2 - Revenue | | 2025 | 2024 | 2023 | |----------------------------------------------------------------------------------------------------|-----------|-----------|-----------| | Services (1) | 109,158 | 96,169 | 85,200 | | Total net sales | $ 416,161 | $ 391,035 | $ 383,285 | | Portion of total net sales that was included in deferred revenue as of the beginning of the period | $ 8,229 | $ 7,728 | $ 8,169 |...
   ↳ Neighbor 288 — table — Note 2 - Revenue
      Note 2 - Revenue | | 2025 | 2024 | 2023 | |----------------------------------------------------------------------------------------------------|-----------|-----------|-----------| | iPhone | $ 209,586 | $ 201,183 | $ 200,583 | | Mac | 33,708 | 29,984 | 29,357...


In [32]:
# VAULTIFY COMPACT - Cell 19B:
# Generic hybrid retrieval using Dense + BM25 + Reciprocal Rank Fusion

from collections import Counter, defaultdict
import html
import math
import re

import numpy as np


if "DOCUMENT_BENCHMARKS" not in globals():
    raise RuntimeError(
        "Run Cell 19 before Cell 19B."
    )


HYBRID_STOPWORDS = {
    "a", "an", "and", "are", "as", "at",
    "be", "been", "by", "for", "from",
    "had", "has", "have", "how", "in",
    "is", "it", "its", "of", "on", "or",
    "s", "that", "the", "their", "this",
    "to", "was", "were", "what", "which",
    "who", "with",
}


QUANTITATIVE_TERMS = {
    "assets",
    "capacity",
    "cash",
    "cost",
    "costs",
    "earnings",
    "expense",
    "expenses",
    "income",
    "liabilities",
    "margin",
    "margins",
    "percent",
    "percentage",
    "revenue",
    "revenues",
    "sales",
    "total",
}


def hybrid_tokenize(text):
    """
    Normalize text into lexical tokens while preserving
    years and financial values.
    """
    normalized = html.unescape(
        str(text or "")
    ).lower()

    # Convert 416,161 into 416161.
    normalized = re.sub(
        r"(?<=\d),(?=\d)",
        "",
        normalized,
    )

    tokens = re.findall(
        r"[a-z0-9]+",
        normalized,
    )

    return [
        token
        for token in tokens
        if token not in HYBRID_STOPWORDS
    ]


def build_bm25_index(chunks):
    """
    Build a lightweight in-memory BM25 index.
    """
    document_tokens = [
        hybrid_tokenize(
            chunk["text"]
        )
        for chunk in chunks
    ]

    term_frequencies = [
        Counter(tokens)
        for tokens in document_tokens
    ]

    document_lengths = np.array(
        [
            len(tokens)
            for tokens in document_tokens
        ],
        dtype=np.float32,
    )

    average_document_length = float(
        document_lengths.mean()
    ) if len(document_lengths) else 0.0

    postings = defaultdict(list)

    for document_index, frequencies in enumerate(
        term_frequencies
    ):
        for term, frequency in frequencies.items():
            postings[term].append(
                (
                    document_index,
                    frequency,
                )
            )

    document_frequency = {
        term: len(entries)
        for term, entries in postings.items()
    }

    return {
        "document_tokens": document_tokens,
        "document_lengths": document_lengths,
        "average_document_length": (
            average_document_length
        ),
        "postings": dict(postings),
        "document_frequency": (
            document_frequency
        ),
        "document_count": len(chunks),
    }


def calculate_bm25_scores(
    index,
    question,
    k1=1.5,
    b=0.75,
):
    """
    Calculate BM25 lexical relevance scores.
    """
    query_tokens = hybrid_tokenize(
        question
    )

    query_term_counts = Counter(
        query_tokens
    )

    scores = np.zeros(
        index["document_count"],
        dtype=np.float32,
    )

    average_length = max(
        index["average_document_length"],
        1.0,
    )

    for term, query_frequency in (
        query_term_counts.items()
    ):
        postings = index[
            "postings"
        ].get(term, [])

        document_frequency = index[
            "document_frequency"
        ].get(term, 0)

        if document_frequency == 0:
            continue

        inverse_document_frequency = math.log(
            1.0
            + (
                (
                    index["document_count"]
                    - document_frequency
                    + 0.5
                )
                / (
                    document_frequency
                    + 0.5
                )
            )
        )

        for document_index, term_frequency in postings:
            document_length = index[
                "document_lengths"
            ][document_index]

            denominator = (
                term_frequency
                + k1
                * (
                    1.0
                    - b
                    + b
                    * document_length
                    / average_length
                )
            )

            score = (
                inverse_document_frequency
                * (
                    term_frequency
                    * (k1 + 1.0)
                )
                / denominator
            )

            scores[document_index] += (
                score
                * query_frequency
            )

    return scores


def create_rank_array(scores):
    """
    Convert scores into one-based ranks.
    """
    order = np.argsort(
        scores
    )[::-1]

    ranks = np.empty(
        len(scores),
        dtype=np.int32,
    )

    ranks[order] = np.arange(
        1,
        len(scores) + 1,
    )

    return ranks


def build_query_phrases(
    query_tokens,
):
    """
    Create meaningful query bigrams and trigrams.
    """
    phrases = []

    for phrase_size in (3, 2):
        for start_index in range(
            0,
            len(query_tokens)
            - phrase_size
            + 1,
        ):
            phrase = " ".join(
                query_tokens[
                    start_index:
                    start_index
                    + phrase_size
                ]
            )

            phrases.append(
                phrase
            )

    return phrases


def retrieve_hybrid(
    result,
    question,
    top_k=6,
    rrf_constant=60,
):
    """
    Combine dense and BM25 rankings using Reciprocal Rank Fusion,
    then apply generic lexical, year, phrase, and table signals.
    """
    chunks = result["chunks"]

    if "bm25_index" not in result:
        result["bm25_index"] = (
            build_bm25_index(
                chunks
            )
        )

    query_embedding = embedding_model.encode(
        [question],
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    )[0]

    dense_scores = (
        result["embeddings"]
        @ query_embedding
    )

    lexical_scores = calculate_bm25_scores(
        result["bm25_index"],
        question,
    )

    dense_ranks = create_rank_array(
        dense_scores
    )

    lexical_ranks = create_rank_array(
        lexical_scores
    )

    query_tokens = hybrid_tokenize(
        question
    )

    unique_query_tokens = set(
        query_tokens
    )

    query_phrases = build_query_phrases(
        query_tokens
    )

    query_years = {
        token
        for token in query_tokens
        if re.fullmatch(
            r"(?:19|20)\d{2}",
            token,
        )
    }

    is_quantitative_query = bool(
        unique_query_tokens
        & QUANTITATIVE_TERMS
    )

    hybrid_scores = np.zeros(
        len(chunks),
        dtype=np.float32,
    )

    feature_details = []

    for chunk_index, chunk in enumerate(
        chunks
    ):
        chunk_tokens = set(
            result["bm25_index"][
                "document_tokens"
            ][chunk_index]
        )

        chunk_normalized = " ".join(
            result["bm25_index"][
                "document_tokens"
            ][chunk_index]
        )

        reciprocal_rank_score = (
            1.0
            / (
                rrf_constant
                + dense_ranks[
                    chunk_index
                ]
            )
            + 1.0
            / (
                rrf_constant
                + lexical_ranks[
                    chunk_index
                ]
            )
        )

        if unique_query_tokens:
            lexical_coverage = (
                len(
                    unique_query_tokens
                    & chunk_tokens
                )
                / len(
                    unique_query_tokens
                )
            )
        else:
            lexical_coverage = 0.0

        phrase_hits = sum(
            phrase in chunk_normalized
            for phrase in query_phrases
        )

        phrase_bonus = min(
            phrase_hits,
            2,
        ) * 0.004

        year_bonus = 0.0

        if (
            query_years
            and query_years.issubset(
                chunk_tokens
            )
        ):
            year_bonus = 0.004

        table_bonus = 0.0

        if (
            is_quantitative_query
            and chunk.get(
                "chunk_type"
            ) == "table"
        ):
            table_bonus = 0.002

        coverage_bonus = (
            lexical_coverage
            * 0.010
        )

        final_score = (
            reciprocal_rank_score
            + coverage_bonus
            + phrase_bonus
            + year_bonus
            + table_bonus
        )

        hybrid_scores[
            chunk_index
        ] = final_score

        feature_details.append(
            {
                "dense_rank": int(
                    dense_ranks[
                        chunk_index
                    ]
                ),
                "lexical_rank": int(
                    lexical_ranks[
                        chunk_index
                    ]
                ),
                "dense_score": float(
                    dense_scores[
                        chunk_index
                    ]
                ),
                "lexical_score": float(
                    lexical_scores[
                        chunk_index
                    ]
                ),
                "coverage": float(
                    lexical_coverage
                ),
                "phrase_hits": int(
                    phrase_hits
                ),
                "year_bonus": (
                    year_bonus
                ),
                "table_bonus": (
                    table_bonus
                ),
            }
        )

    top_indexes = np.argsort(
        hybrid_scores
    )[::-1][:top_k]

    results = []

    for rank, chunk_index in enumerate(
        top_indexes,
        start=1,
    ):
        chunk_index = int(
            chunk_index
        )

        chunk = chunks[
            chunk_index
        ]

        features = feature_details[
            chunk_index
        ]

        results.append(
            {
                "rank": rank,
                "chunk_index": (
                    chunk_index
                ),
                "hybrid_score": float(
                    hybrid_scores[
                        chunk_index
                    ]
                ),
                "dense_rank": features[
                    "dense_rank"
                ],
                "lexical_rank": features[
                    "lexical_rank"
                ],
                "dense_score": features[
                    "dense_score"
                ],
                "lexical_score": features[
                    "lexical_score"
                ],
                "coverage": features[
                    "coverage"
                ],
                "text": chunk["text"],
                "section": chunk.get(
                    "section",
                    "Unknown section",
                ),
                "chunk_type": chunk.get(
                    "chunk_type",
                    "unknown",
                ),
            }
        )

    return results


def hybrid_expected_groups_found(
    retrieved_chunks,
    expected_groups,
):
    combined_context = " ".join(
        item["text"]
        for item in retrieved_chunks
    ).lower()

    group_results = []

    for alternatives in expected_groups:
        matched = next(
            (
                alternative
                for alternative
                in alternatives
                if alternative.lower()
                in combined_context
            ),
            None,
        )

        group_results.append(
            {
                "alternatives": (
                    alternatives
                ),
                "matched": matched,
            }
        )

    passed = all(
        result["matched"]
        is not None
        for result in group_results
    )

    return passed, group_results


# ---------------------------------------------------------------------
# Build reusable indexes
# ---------------------------------------------------------------------

print("⏳ Building generic BM25 indexes...")

for document_label, benchmark in (
    DOCUMENT_BENCHMARKS.items()
):
    result = benchmark["result"]

    result["bm25_index"] = (
        build_bm25_index(
            result["chunks"]
        )
    )

    print(
        f"   ✅ {document_label}: "
        f"{len(result['chunks'])} chunks"
    )


# ---------------------------------------------------------------------
# Run positive hybrid benchmarks
# ---------------------------------------------------------------------

HYBRID_POSITIVE_RESULTS = []

print("\n" + "=" * 92)
print("🔀 HYBRID RETRIEVAL BENCHMARKS")

for document_label, benchmark in (
    DOCUMENT_BENCHMARKS.items()
):
    for test_case in benchmark[
        "positive_queries"
    ]:
        retrieved = retrieve_hybrid(
            result=benchmark["result"],
            question=test_case[
                "question"
            ],
            top_k=6,
        )

        passed, group_results = (
            hybrid_expected_groups_found(
                retrieved,
                test_case[
                    "expected_groups"
                ],
            )
        )

        HYBRID_POSITIVE_RESULTS.append(
            {
                "document": (
                    document_label
                ),
                "question": test_case[
                    "question"
                ],
                "passed": passed,
                "retrieved": retrieved,
                "groups": group_results,
            }
        )

        status = (
            "✅ PASS"
            if passed
            else "❌ FAIL"
        )

        print(
            f"\n{status} — "
            f"{document_label}"
        )

        print(
            f"Question: "
            f"{test_case['question']}"
        )

        for group in group_results:
            print(
                f"   "
                f"{'✅' if group['matched'] else '❌'} "
                f"{group['alternatives']}"
                f" → {group['matched']}"
            )

        print(
            "Top hybrid chunks:"
        )

        for item in retrieved[:3]:
            preview = " ".join(
                item["text"].split()
            )[:200]

            print(
                f"   #{item['rank']} "
                f"hybrid={item['hybrid_score']:.4f} "
                f"dense_rank={item['dense_rank']} "
                f"bm25_rank={item['lexical_rank']} "
                f"type={item['chunk_type']} "
                f"section={item['section']}"
            )

            print(
                f"      {preview}..."
            )


# ---------------------------------------------------------------------
# Negative-query diagnostics
# ---------------------------------------------------------------------

print("\n" + "=" * 92)
print("🚫 HYBRID NEGATIVE QUERY DIAGNOSTICS")

HYBRID_NEGATIVE_RESULTS = []

for document_label, benchmark in (
    DOCUMENT_BENCHMARKS.items()
):
    for question in benchmark[
        "negative_queries"
    ]:
        retrieved = retrieve_hybrid(
            result=benchmark["result"],
            question=question,
            top_k=3,
        )

        top_result = retrieved[0]

        HYBRID_NEGATIVE_RESULTS.append(
            {
                "document": document_label,
                "question": question,
                "top_result": top_result,
            }
        )

        print(
            f"\n{document_label}"
        )

        print(
            f"Question: {question}"
        )

        print(
            f"Hybrid score: "
            f"{top_result['hybrid_score']:.4f}"
        )

        print(
            f"Dense score: "
            f"{top_result['dense_score']:.4f}"
        )

        print(
            f"BM25 score: "
            f"{top_result['lexical_score']:.4f}"
        )

        print(
            f"Query coverage: "
            f"{top_result['coverage']:.2f}"
        )

        print(
            f"Top unrelated section: "
            f"{top_result['section']}"
        )


# ---------------------------------------------------------------------
# Final summary
# ---------------------------------------------------------------------

hybrid_passed_count = sum(
    result["passed"]
    for result in HYBRID_POSITIVE_RESULTS
)

hybrid_total_count = len(
    HYBRID_POSITIVE_RESULTS
)

CELL_19B_HYBRID_PASSED = (
    hybrid_passed_count
    == hybrid_total_count
)

CELL_19B_RESULTS = {
    "positive_results": (
        HYBRID_POSITIVE_RESULTS
    ),
    "negative_results": (
        HYBRID_NEGATIVE_RESULTS
    ),
    "passed": (
        CELL_19B_HYBRID_PASSED
    ),
}

print("\n" + "=" * 92)
print("📊 CELL 19B SUMMARY")

print(
    f"Hybrid positive tests passed: "
    f"{hybrid_passed_count}"
    f"/{hybrid_total_count}"
)

if CELL_19B_HYBRID_PASSED:
    print(
        "✅ Hybrid retrieval found every expected "
        "answer signal inside the top-6 results."
    )

    print(
        "➡️ Continue to old-vs-new retrieval "
        "comparison before replacing vectors."
    )

else:
    print(
        "❌ At least one hybrid retrieval "
        "test still failed."
    )

    print(
        "⛔ Do not replace existing "
        "Qdrant vectors."
    )

print(
    "\n⏭️ No Qdrant points were created, "
    "updated, or deleted."
)

⏳ Building generic BM25 indexes...
   ✅ Apple 10-K: 565 chunks
   ✅ Tesla Q4 2025: 258 chunks
   ✅ NIST AI RMF: 153 chunks

🔀 HYBRID RETRIEVAL BENCHMARKS

✅ PASS — Apple 10-K
Question: What were Apple's total net sales in fiscal year 2025?
   ✅ ['416,161'] → 416,161
   ✅ ['total net sales', 'net sales'] → total net sales
Top hybrid chunks:
   #1 hybrid=0.0469 dense_rank=4 bm25_rank=2 type=text section=Note 2 - Revenue
      (1) Services net sales include amortization of the deferred value of services bundled in the sales price of certain products. The Company's proportion of net sales by disaggregated revenue source was ...
   #2 hybrid=0.0468 dense_rank=27 bm25_rank=4 type=table section=Note 2 - Revenue
      Note 2 - Revenue | | 2025 | 2024 | 2023 | |----------------------------------------------------------------------------------------------------|-----------|-----------|-----------| | Services (1) | 10...
   #3 hybrid=0.0451 dense_rank=32 bm25_rank=9 type=table section=Operating E

In [33]:
# VAULTIFY COMPACT - Cell 19C:
# Fair old-vs-new chunking comparison using the same hybrid retriever

from collections import Counter
import statistics

import numpy as np

from qdrant_client.models import (
    FieldCondition,
    Filter,
    MatchValue,
)


# ---------------------------------------------------------------------
# 1. Verify previous benchmark components
# ---------------------------------------------------------------------

required_names = [
    "retrieve_hybrid",
    "build_bm25_index",
    "hybrid_expected_groups_found",
    "apple_result",
    "tesla_result",
    "embedding_model",
    "qdrant",
    "COLLECTION_NAME",
    "DEMO_TENANTS",
    "TENANT_ID_FIELD",
    "TEXT_FIELD",
]

missing_names = [
    name
    for name in required_names
    if name not in globals()
]

if missing_names:
    raise RuntimeError(
        "Missing required components: "
        + ", ".join(missing_names)
        + ". Run Cells 16–19B first."
    )


# ---------------------------------------------------------------------
# 2. Load one complete seed tenant from Qdrant
# ---------------------------------------------------------------------

def load_seed_chunks_from_qdrant(
    tenant_id,
):
    """
    Load every text payload for one validated seed tenant.
    """
    points = []
    next_offset = None

    tenant_filter = Filter(
        must=[
            FieldCondition(
                key=TENANT_ID_FIELD,
                match=MatchValue(
                    value=tenant_id
                ),
            )
        ]
    )

    while True:
        batch, next_offset = qdrant.scroll(
            collection_name=COLLECTION_NAME,
            scroll_filter=tenant_filter,
            limit=256,
            offset=next_offset,
            with_payload=True,
            with_vectors=False,
        )

        points.extend(batch)

        if next_offset is None:
            break

    chunks = []

    for point in points:
        payload = point.payload or {}

        chunk_text = str(
            payload.get(
                TEXT_FIELD,
                "",
            )
        ).strip()

        if not chunk_text:
            continue

        chunks.append(
            {
                "text": chunk_text,
                "chunk_type": payload.get(
                    CHUNK_TYPE_FIELD,
                    "unknown",
                ),
               "section": payload.get(
    "section",
    "Unknown section",
),
"chunk_index": payload.get(
    CHUNK_INDEX_FIELD,
    len(chunks),
),
            }
        )

    if not chunks:
        raise RuntimeError(
            f"No seed chunks found for tenant: "
            f"{tenant_id}"
        )

    return chunks


# ---------------------------------------------------------------------
# 3. Convert chunks into the same in-memory retrieval format
# ---------------------------------------------------------------------

def prepare_comparison_result(
    label,
    chunks,
):
    """
    Create normalized embeddings and BM25 index using exactly
    the same settings for old and new chunk sets.
    """
    print(
        f"⏳ Preparing {label}: "
        f"{len(chunks)} chunks"
    )

    chunk_texts = [
        chunk["text"]
        for chunk in chunks
    ]

    embeddings = embedding_model.encode(
        chunk_texts,
        batch_size=64,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=True,
    )

    result = {
        "filename": label,
        "chunks": chunks,
        "embeddings": embeddings,
    }

    result["bm25_index"] = (
        build_bm25_index(
            chunks
        )
    )

    return result


print("📦 Loading validated seed chunks from Qdrant...")

old_apple_chunks = (
    load_seed_chunks_from_qdrant(
        DEMO_TENANTS[
            "company_a"
        ]["tenant_id"]
    )
)

old_tesla_chunks = (
    load_seed_chunks_from_qdrant(
        DEMO_TENANTS[
            "company_b"
        ]["tenant_id"]
    )
)

print(
    f"✅ Old Apple chunks: "
    f"{len(old_apple_chunks)}"
)

print(
    f"✅ Old Tesla chunks: "
    f"{len(old_tesla_chunks)}"
)


OLD_APPLE_RESULT = (
    prepare_comparison_result(
        "Old Apple seed",
        old_apple_chunks,
    )
)

NEW_APPLE_RESULT = (
    prepare_comparison_result(
        "New Apple safe",
        apple_result["chunks"],
    )
)

OLD_TESLA_RESULT = (
    prepare_comparison_result(
        "Old Tesla seed",
        old_tesla_chunks,
    )
)

NEW_TESLA_RESULT = (
    prepare_comparison_result(
        "New Tesla safe",
        tesla_result["chunks"],
    )
)


# ---------------------------------------------------------------------
# 4. Define comparison questions
# ---------------------------------------------------------------------

COMPARISON_TESTS = {
    "Apple": {
        "old": OLD_APPLE_RESULT,
        "new": NEW_APPLE_RESULT,
        "queries": [
            {
                "question": (
                    "What were Apple's total net sales "
                    "in fiscal year 2025?"
                ),
                "expected_groups": [
                    ["416,161", "416161"],
                    ["total net sales"],
                ],
            },
            {
                "question": (
                    "What were Apple's services net sales "
                    "in fiscal year 2025?"
                ),
                "expected_groups": [
                    ["109,158", "109158"],
                    ["services"],
                ],
            },
            {
                "question": (
                    "What were Apple's iPhone net sales "
                    "in fiscal year 2025?"
                ),
                "expected_groups": [
                    ["209,586", "209586"],
                    ["iphone"],
                ],
            },
        ],
    },

    "Tesla": {
        "old": OLD_TESLA_RESULT,
        "new": NEW_TESLA_RESULT,
        "queries": [
            {
                "question": (
                    "What was Tesla's total revenue "
                    "in Q4 2025?"
                ),
                "expected_groups": [
                    ["24,901", "24901"],
                    [
                        "total revenue",
                        "total revenues",
                    ],
                ],
            },
            {
                "question": (
                    "What was Tesla's automotive revenue "
                    "in Q4 2025?"
                ),
                "expected_groups": [
                    ["automotive revenues"],
                    ["q4-2025", "q4 2025"],
                ],
            },
            {
                "question": (
                    "What was Tesla's energy generation "
                    "and storage revenue in Q4 2025?"
                ),
                "expected_groups": [
                    [
                        "energy generation and storage",
                        "energy generation & storage",
                    ],
                    ["q4-2025", "q4 2025"],
                ],
            },
            {
                "question": (
                    "What was Tesla's GAAP operating income "
                    "in Q4 2025?"
                ),
                "expected_groups": [
                    ["operating income"],
                    ["q4-2025", "q4 2025"],
                ],
            },
        ],
    },
}


# ---------------------------------------------------------------------
# 5. Find the best rank containing all expected evidence
# ---------------------------------------------------------------------

def normalize_comparison_text(text):
    normalized = " ".join(
        str(text or "").lower().split()
    )

    compact = "".join(
        character
        for character in normalized
        if character.isalnum()
    )

    return normalized, compact


def chunk_contains_expected_groups(
    text,
    expected_groups,
):
    normalized_text, compact_text = (
        normalize_comparison_text(
            text
        )
    )

    for alternatives in expected_groups:
        group_found = False

        for alternative in alternatives:
            (
                normalized_alternative,
                compact_alternative,
            ) = normalize_comparison_text(
                alternative
            )

            if (
                normalized_alternative
                in normalized_text
                or (
                    compact_alternative
                    and compact_alternative
                    in compact_text
                )
            ):
                group_found = True
                break

        if not group_found:
            return False

    return True


def evaluate_one_chunk_set(
    result,
    question,
    expected_groups,
    retrieval_depth=20,
):
    retrieved = retrieve_hybrid(
        result=result,
        question=question,
        top_k=retrieval_depth,
    )

    best_complete_rank = None

    for item in retrieved:
        if chunk_contains_expected_groups(
            item["text"],
            expected_groups,
        ):
            best_complete_rank = item[
                "rank"
            ]
            break

    context_passed, group_results = (
        hybrid_expected_groups_found(
            retrieved[:6],
            expected_groups,
        )
    )

    return {
        "retrieved": retrieved,
        "best_complete_rank": (
            best_complete_rank
        ),
        "top_6_context_passed": (
            context_passed
        ),
        "group_results": (
            group_results
        ),
    }


# ---------------------------------------------------------------------
# 6. Run fair old-vs-new comparison
# ---------------------------------------------------------------------

OLD_VS_NEW_RESULTS = []

print("\n" + "=" * 96)
print("⚖️ OLD VS NEW CHUNKING — SAME HYBRID RETRIEVER")

for document_label, test_config in (
    COMPARISON_TESTS.items()
):
    print(
        f"\n{'=' * 96}"
    )

    print(
        f"📄 {document_label}"
    )

    for test_case in test_config[
        "queries"
    ]:
        question = test_case[
            "question"
        ]

        expected_groups = test_case[
            "expected_groups"
        ]

        old_evaluation = (
            evaluate_one_chunk_set(
                result=test_config["old"],
                question=question,
                expected_groups=(
                    expected_groups
                ),
            )
        )

        new_evaluation = (
            evaluate_one_chunk_set(
                result=test_config["new"],
                question=question,
                expected_groups=(
                    expected_groups
                ),
            )
        )

        old_rank = old_evaluation[
            "best_complete_rank"
        ]

        new_rank = new_evaluation[
            "best_complete_rank"
        ]

        if old_rank is None:
            old_rank_label = ">20"
        else:
            old_rank_label = str(
                old_rank
            )

        if new_rank is None:
            new_rank_label = ">20"
        else:
            new_rank_label = str(
                new_rank
            )

        if (
            new_rank is not None
            and (
                old_rank is None
                or new_rank < old_rank
            )
        ):
            winner = "NEW"

        elif (
            old_rank is not None
            and (
                new_rank is None
                or old_rank < new_rank
            )
        ):
            winner = "OLD"

        else:
            winner = "TIE"

        OLD_VS_NEW_RESULTS.append(
            {
                "document": (
                    document_label
                ),
                "question": question,
                "old_rank": old_rank,
                "new_rank": new_rank,
                "winner": winner,
                "old_top_6_passed": (
                    old_evaluation[
                        "top_6_context_passed"
                    ]
                ),
                "new_top_6_passed": (
                    new_evaluation[
                        "top_6_context_passed"
                    ]
                ),
            }
        )

        print(
            f"\nQuestion: {question}"
        )

        print(
            f"   Old complete-evidence rank: "
            f"{old_rank_label}"
        )

        print(
            f"   New complete-evidence rank: "
            f"{new_rank_label}"
        )

        print(
            f"   Old top-6 context pass: "
            f"{old_evaluation['top_6_context_passed']}"
        )

        print(
            f"   New top-6 context pass: "
            f"{new_evaluation['top_6_context_passed']}"
        )

        print(
            f"   Winner: {winner}"
        )

        print(
            "   Old top result:"
        )

        old_top = old_evaluation[
            "retrieved"
        ][0]

        print(
            f"      rank=1 "
            f"dense={old_top['dense_rank']} "
            f"bm25={old_top['lexical_rank']} "
            f"section={old_top['section']}"
        )

        print(
            "      "
            + " ".join(
                old_top["text"].split()
            )[:180]
            + "..."
        )

        print(
            "   New top result:"
        )

        new_top = new_evaluation[
            "retrieved"
        ][0]

        print(
            f"      rank=1 "
            f"dense={new_top['dense_rank']} "
            f"bm25={new_top['lexical_rank']} "
            f"section={new_top['section']}"
        )

        print(
            "      "
            + " ".join(
                new_top["text"].split()
            )[:180]
            + "..."
        )


# ---------------------------------------------------------------------
# 7. Final comparison summary
# ---------------------------------------------------------------------

new_wins = sum(
    result["winner"] == "NEW"
    for result in OLD_VS_NEW_RESULTS
)

old_wins = sum(
    result["winner"] == "OLD"
    for result in OLD_VS_NEW_RESULTS
)

ties = sum(
    result["winner"] == "TIE"
    for result in OLD_VS_NEW_RESULTS
)

new_top_6_passes = sum(
    result["new_top_6_passed"]
    for result in OLD_VS_NEW_RESULTS
)

old_top_6_passes = sum(
    result["old_top_6_passed"]
    for result in OLD_VS_NEW_RESULTS
)

total_tests = len(
    OLD_VS_NEW_RESULTS
)

CELL_19C_RESULTS = {
    "results": OLD_VS_NEW_RESULTS,
    "new_wins": new_wins,
    "old_wins": old_wins,
    "ties": ties,
    "new_top_6_passes": (
        new_top_6_passes
    ),
    "old_top_6_passes": (
        old_top_6_passes
    ),
    "total_tests": total_tests,
}

print("\n" + "=" * 96)
print("📊 CELL 19C SUMMARY")

print(
    f"New chunker wins: "
    f"{new_wins}"
)

print(
    f"Old chunker wins: "
    f"{old_wins}"
)

print(
    f"Ties: "
    f"{ties}"
)

print(
    f"Old top-6 passes: "
    f"{old_top_6_passes}"
    f"/{total_tests}"
)

print(
    f"New top-6 passes: "
    f"{new_top_6_passes}"
    f"/{total_tests}"
)

CELL_19C_NEW_CHUNKER_APPROVED = (
    new_top_6_passes
    >= old_top_6_passes
    and new_wins >= old_wins
)

if CELL_19C_NEW_CHUNKER_APPROVED:
    print(
        "✅ The new safe chunker is at least as reliable "
        "as the old chunker under the same hybrid retriever."
    )

    print(
        "➡️ It is eligible for a staged Qdrant migration."
    )

else:
    print(
        "❌ The new safe chunker did not match "
        "the old retrieval quality."
    )

    print(
        "⛔ Do not replace existing vectors."
    )

print(
    "\n⏭️ No Qdrant points were created, "
    "updated, or deleted."
)

📦 Loading validated seed chunks from Qdrant...
✅ Old Apple chunks: 745
✅ Old Tesla chunks: 140
⏳ Preparing Old Apple seed: 745 chunks


Batches:   0%|          | 0/12 [00:00<?, ?it/s]

⏳ Preparing New Apple safe: 565 chunks


Batches:   0%|          | 0/9 [00:00<?, ?it/s]

⏳ Preparing Old Tesla seed: 140 chunks


Batches:   0%|          | 0/3 [00:00<?, ?it/s]

⏳ Preparing New Tesla safe: 258 chunks


Batches:   0%|          | 0/5 [00:00<?, ?it/s]


⚖️ OLD VS NEW CHUNKING — SAME HYBRID RETRIEVER

📄 Apple

Question: What were Apple's total net sales in fiscal year 2025?
   Old complete-evidence rank: 1
   New complete-evidence rank: 2
   Old top-6 context pass: True
   New top-6 context pass: True
   Winner: OLD
   Old top result:
      rank=1 dense=3 bm25=13 section=Note 2 - Revenue
      Section: Note 2 - Revenue Table columns: | | 2025 | 2024 | 2023 | | iPhone | $ 209,586 | $ 201,183 | $ 200,583 | | Mac | 33,708 | 29,984 | 29,357 | | iPad | 28,023 | 26,694 | 28,30...
   New top result:
      rank=1 dense=4 bm25=2 section=Note 2 - Revenue
      (1) Services net sales include amortization of the deferred value of services bundled in the sales price of certain products. The Company's proportion of net sales by disaggregated...

Question: What were Apple's services net sales in fiscal year 2025?
   Old complete-evidence rank: 7
   New complete-evidence rank: 13
   Old top-6 context pass: False
   New top-6 context pass: False
   Wi

In [34]:
# VAULTIFY COMPACT - Cell 20A:
# Build the canonical seed-style, token-safe chunker V2
# and rechunk the preserved Markdown documents.

from collections import Counter
import hashlib
import re
import statistics


# ---------------------------------------------------------------------
# 1. Runtime validation
# ---------------------------------------------------------------------

if (
    "DRY_RUN_RESULTS" not in globals()
    or len(DRY_RUN_RESULTS) < 3
):
    raise RuntimeError(
        "Preserved Apple, Tesla, and NIST dry-run results "
        "were not found. Run Cell 18 and Cell 18A first."
    )


V2_TOKENIZER = embedding_model.tokenizer

V2_MAX_CHUNK_TOKENS = min(
    240,
    embedding_model.max_seq_length - 16,
)

V2_SPECIAL_TOKEN_BUFFER = 2

V2_TOKEN_PAYLOAD_LIMIT = (
    V2_MAX_CHUNK_TOKENS
    - V2_SPECIAL_TOKEN_BUFFER
)

# Preserve section and column information without
# allowing the prefix to consume the whole chunk.
V2_MAX_TABLE_PREFIX_TOKENS = min(
    96,
    V2_TOKEN_PAYLOAD_LIMIT // 2,
)

# These are the validated seed-pipeline text settings.
V2_TEXT_CHUNK_SIZE = 800
V2_TEXT_CHUNK_OVERLAP = 120


# ---------------------------------------------------------------------
# 2. Token utilities
# ---------------------------------------------------------------------

def v2_get_raw_token_ids(text):
    encoded = V2_TOKENIZER(
        str(text or ""),
        add_special_tokens=False,
        truncation=False,
        return_attention_mask=False,
        return_token_type_ids=False,
        verbose=False,
    )

    token_ids = encoded["input_ids"]

    if (
        token_ids
        and isinstance(
            token_ids[0],
            (list, tuple),
        )
    ):
        token_ids = token_ids[0]

    return list(token_ids)


def v2_count_raw_tokens(text):
    return len(
        v2_get_raw_token_ids(text)
    )


def v2_count_embedding_tokens(text):
    encoded = V2_TOKENIZER(
        str(text or ""),
        add_special_tokens=True,
        truncation=False,
        return_attention_mask=False,
        return_token_type_ids=False,
        verbose=False,
    )

    token_ids = encoded["input_ids"]

    if (
        token_ids
        and isinstance(
            token_ids[0],
            (list, tuple),
        )
    ):
        token_ids = token_ids[0]

    return len(token_ids)


def v2_decode_token_slice(token_ids):
    return V2_TOKENIZER.decode(
        token_ids,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=True,
    ).strip()


def v2_truncate_to_token_limit(
    text,
    token_limit,
):
    text = str(text or "").strip()

    if not text or token_limit <= 0:
        return ""

    token_ids = v2_get_raw_token_ids(
        text
    )

    if len(token_ids) <= token_limit:
        return text

    return v2_decode_token_slice(
        token_ids[:token_limit]
    )


def v2_split_to_token_safe_text(
    text,
    token_limit=V2_TOKEN_PAYLOAD_LIMIT,
):
    text = str(text or "").strip()

    if not text:
        return []

    token_ids = v2_get_raw_token_ids(
        text
    )

    if len(token_ids) <= token_limit:
        return [text]

    safe_pieces = []

    for start_index in range(
        0,
        len(token_ids),
        token_limit,
    ):
        token_slice = token_ids[
            start_index:
            start_index + token_limit
        ]

        decoded_piece = (
            v2_decode_token_slice(
                token_slice
            )
        )

        if decoded_piece:
            safe_pieces.append(
                decoded_piece
            )

    return safe_pieces


# ---------------------------------------------------------------------
# 3. Table helpers
# ---------------------------------------------------------------------

def v2_is_markdown_separator_row(line):
    stripped_line = str(
        line or ""
    ).strip()

    if not stripped_line.startswith("|"):
        return False

    cells = [
        cell.strip()
        for cell in stripped_line
        .strip("|")
        .split("|")
    ]

    if not cells:
        return False

    return all(
        bool(
            re.fullmatch(
                r":?-{3,}:?",
                cell,
            )
        )
        for cell in cells
    )


def v2_build_safe_table_prefix(
    section,
    header_line,
):
    """
    Give tables explicit semantic labels and remove
    token-expensive Markdown separator rows.
    """
    section = (
        str(section or "").strip()
        or "Unknown section"
    )

    header_line = str(
        header_line or ""
    ).strip()

    section_label = (
        f"Section: {section}\n"
        "Table columns:\n"
    )

    section_token_count = (
        v2_count_raw_tokens(
            section_label
        )
    )

    if (
        section_token_count
        >= V2_MAX_TABLE_PREFIX_TOKENS
    ):
        return v2_truncate_to_token_limit(
            section_label,
            V2_MAX_TABLE_PREFIX_TOKENS,
        )

    available_header_tokens = (
        V2_MAX_TABLE_PREFIX_TOKENS
        - section_token_count
    )

    compact_header = (
        v2_truncate_to_token_limit(
            header_line,
            available_header_tokens,
        )
    )

    return (
        section_label
        + compact_header
    ).strip()


def v2_create_chunk(
    text,
    chunk_type,
    section,
):
    return {
        "text": str(text or "").strip(),
        "chunk_type": chunk_type,
        "section": (
            str(section or "").strip()
            or "Document content"
        ),
    }


# ---------------------------------------------------------------------
# 4. Text chunking
# ---------------------------------------------------------------------

def v2_split_normal_text(
    text,
    section,
):
    text = str(text or "").strip()

    if not text:
        return []

    chunks = []
    start_index = 0

    while start_index < len(text):
        end_index = min(
            start_index
            + V2_TEXT_CHUNK_SIZE,
            len(text),
        )

        character_piece = text[
            start_index:end_index
        ].strip()

        for safe_piece in (
            v2_split_to_token_safe_text(
                character_piece
            )
        ):
            chunks.append(
                v2_create_chunk(
                    text=safe_piece,
                    chunk_type="text",
                    section=section,
                )
            )

        if end_index >= len(text):
            break

        next_start_index = (
            end_index
            - V2_TEXT_CHUNK_OVERLAP
        )

        start_index = max(
            next_start_index,
            start_index + 1,
        )

    return chunks


# ---------------------------------------------------------------------
# 5. Table chunking
# ---------------------------------------------------------------------

def v2_split_oversized_table_row(
    row,
    table_prefix,
    section,
):
    prefix_with_newline = (
        table_prefix.rstrip()
        + "\n"
    )

    prefix_token_count = (
        v2_count_raw_tokens(
            prefix_with_newline
        )
    )

    available_row_tokens = (
        V2_TOKEN_PAYLOAD_LIMIT
        - prefix_token_count
    )

    if available_row_tokens <= 0:
        raise RuntimeError(
            "The table prefix consumed the complete token budget. "
            f"Section: {section}"
        )

    row_token_ids = (
        v2_get_raw_token_ids(
            row
        )
    )

    chunks = []

    for start_index in range(
        0,
        len(row_token_ids),
        available_row_tokens,
    ):
        token_slice = row_token_ids[
            start_index:
            start_index
            + available_row_tokens
        ]

        decoded_row = (
            v2_decode_token_slice(
                token_slice
            )
        )

        if not decoded_row:
            continue

        chunk_text = (
            prefix_with_newline
            + decoded_row
        ).strip()

        if (
            v2_count_embedding_tokens(
                chunk_text
            )
            > V2_MAX_CHUNK_TOKENS
        ):
            raise RuntimeError(
                "An oversized table-row piece "
                "was generated unexpectedly."
            )

        chunks.append(
            v2_create_chunk(
                text=chunk_text,
                chunk_type="table",
                section=section,
            )
        )

    return chunks


def v2_split_table(
    table_lines,
    section,
):
    if not table_lines:
        return []

    cleaned_lines = [
        line.strip()
        for line in table_lines
        if line.strip()
    ]

    if not cleaned_lines:
        return []

    header_line = cleaned_lines[0]

    if (
        len(cleaned_lines) > 1
        and v2_is_markdown_separator_row(
            cleaned_lines[1]
        )
    ):
        data_rows = cleaned_lines[2:]
    else:
        data_rows = cleaned_lines[1:]

    table_prefix = (
        v2_build_safe_table_prefix(
            section=section,
            header_line=header_line,
        )
    )

    chunks = []
    current_rows = []

    def build_table_text(rows):
        return (
            table_prefix.rstrip()
            + "\n"
            + "\n".join(rows)
        ).strip()

    def flush_current_rows():
        nonlocal current_rows

        if not current_rows:
            return

        chunk_text = build_table_text(
            current_rows
        )

        token_count = (
            v2_count_embedding_tokens(
                chunk_text
            )
        )

        if token_count > V2_MAX_CHUNK_TOKENS:
            raise RuntimeError(
                "An internal table chunk exceeded "
                "the 240-token limit."
            )

        chunks.append(
            v2_create_chunk(
                text=chunk_text,
                chunk_type="table",
                section=section,
            )
        )

        current_rows = []

    if not data_rows:
        for safe_piece in (
            v2_split_to_token_safe_text(
                table_prefix
            )
        ):
            chunks.append(
                v2_create_chunk(
                    text=safe_piece,
                    chunk_type="table",
                    section=section,
                )
            )

        return chunks

    for row in data_rows:
        candidate_rows = (
            current_rows + [row]
        )

        candidate_text = (
            build_table_text(
                candidate_rows
            )
        )

        if (
            v2_count_embedding_tokens(
                candidate_text
            )
            <= V2_MAX_CHUNK_TOKENS
        ):
            current_rows = candidate_rows
            continue

        flush_current_rows()

        single_row_text = (
            build_table_text([row])
        )

        if (
            v2_count_embedding_tokens(
                single_row_text
            )
            <= V2_MAX_CHUNK_TOKENS
        ):
            current_rows = [row]

        else:
            chunks.extend(
                v2_split_oversized_table_row(
                    row=row,
                    table_prefix=table_prefix,
                    section=section,
                )
            )

    flush_current_rows()

    return chunks


# ---------------------------------------------------------------------
# 6. Canonical Markdown chunker
# ---------------------------------------------------------------------

def canonical_chunk_markdown_v2(
    model,
    markdown_text,
):
    """
    Canonical Vaultify chunker:

    - validated seed-style table serialization
    - explicit section and column labels
    - separator-row removal
    - strict 240-token maximum
    - oversized-row splitting
    - exact duplicate removal
    """
    lines = str(
        markdown_text or ""
    ).splitlines()

    chunks = []
    current_section = (
        "Document introduction"
    )

    text_buffer = []

    def flush_text_buffer():
        nonlocal text_buffer

        buffered_text = "\n".join(
            text_buffer
        ).strip()

        if buffered_text:
            chunks.extend(
                v2_split_normal_text(
                    text=buffered_text,
                    section=current_section,
                )
            )

        text_buffer = []

    line_index = 0

    while line_index < len(lines):
        line = lines[line_index]
        stripped_line = line.strip()

        heading_match = re.match(
            r"^(#{1,6})\s+(.+)$",
            stripped_line,
        )

        if heading_match:
            flush_text_buffer()

            current_section = (
                heading_match
                .group(2)
                .strip()
            )

            text_buffer.append(
                stripped_line
            )

            line_index += 1
            continue

        if stripped_line.startswith("|"):
            flush_text_buffer()

            table_lines = []

            while (
                line_index < len(lines)
                and lines[line_index]
                .strip()
                .startswith("|")
            ):
                table_lines.append(
                    lines[line_index]
                )

                line_index += 1

            chunks.extend(
                v2_split_table(
                    table_lines=table_lines,
                    section=current_section,
                )
            )

            continue

        text_buffer.append(line)
        line_index += 1

    flush_text_buffer()

    # Exact deduplication.
    unique_chunks = []
    seen_hashes = set()

    for chunk in chunks:
        normalized_text = re.sub(
            r"\s+",
            " ",
            chunk["text"],
        ).strip()

        if not normalized_text:
            continue

        chunk_hash = hashlib.sha256(
            normalized_text.encode(
                "utf-8"
            )
        ).hexdigest()

        if chunk_hash in seen_hashes:
            continue

        seen_hashes.add(
            chunk_hash
        )

        chunk["chunk_index"] = len(
            unique_chunks
        )

        unique_chunks.append(
            chunk
        )

    oversized_chunks = [
        chunk
        for chunk in unique_chunks
        if (
            v2_count_embedding_tokens(
                chunk["text"]
            )
            > V2_MAX_CHUNK_TOKENS
        )
    ]

    if oversized_chunks:
        raise RuntimeError(
            f"Canonical chunker generated "
            f"{len(oversized_chunks)} oversized chunks."
        )

    return unique_chunks


# ---------------------------------------------------------------------
# 7. Rechunk preserved documents without reparsing
# ---------------------------------------------------------------------

DRY_RUN_RESULTS_V2 = {}

print(
    "⏳ Rechunking preserved Markdown "
    "with Canonical Chunker V2..."
)

for (
    document_hash,
    preserved_result,
) in DRY_RUN_RESULTS.items():

    filename = preserved_result[
        "filename"
    ]

    markdown_text = preserved_result[
        "markdown"
    ]

    chunks = canonical_chunk_markdown_v2(
        embedding_model,
        markdown_text,
    )

    token_counts = [
        v2_count_embedding_tokens(
            chunk["text"]
        )
        for chunk in chunks
    ]

    chunk_types = Counter(
        chunk["chunk_type"]
        for chunk in chunks
    )

    normalized_texts = [
        re.sub(
            r"\s+",
            " ",
            chunk["text"],
        ).strip()
        for chunk in chunks
    ]

    duplicate_count = (
        len(normalized_texts)
        - len(set(normalized_texts))
    )

    oversized_count = sum(
        token_count
        > V2_MAX_CHUNK_TOKENS
        for token_count
        in token_counts
    )

    report = {
        "filename": filename,
        "total_chunks": len(chunks),
        "chunk_types": dict(
            chunk_types
        ),
        "token_min": min(
            token_counts
        ),
        "token_median": (
            statistics.median(
                token_counts
            )
        ),
        "token_p95": sorted(
            token_counts
        )[
            round(
                (len(token_counts) - 1)
                * 0.95
            )
        ],
        "token_max": max(
            token_counts
        ),
        "exact_duplicates": (
            duplicate_count
        ),
        "oversized": (
            oversized_count
        ),
    }

    DRY_RUN_RESULTS_V2[
        document_hash
    ] = {
        "filename": filename,
        "document_hash": (
            document_hash
        ),
        "markdown": markdown_text,
        "chunks": chunks,
        "report": report,
    }

    print(
        f"\n✅ {filename}"
    )

    print(
        f"   Total chunks: "
        f"{report['total_chunks']}"
    )

    print(
        f"   Types: "
        f"{report['chunk_types']}"
    )

    print(
        "   Tokens:"
        f" median={report['token_median']:.1f},"
        f" p95={report['token_p95']},"
        f" max={report['token_max']}"
    )

    print(
        f"   Oversized: "
        f"{report['oversized']}"
    )

    print(
        f"   Exact duplicates: "
        f"{report['exact_duplicates']}"
    )


print("\n" + "=" * 88)
print("✅ Canonical Chunker V2 dry-run completed.")
print(
    f"📦 Documents processed: "
    f"{len(DRY_RUN_RESULTS_V2)}"
)
print("⏭️ Existing web backend was not patched.")
print("⏭️ No embeddings were generated.")
print("⏭️ No Qdrant points were changed.")

⏳ Rechunking preserved Markdown with Canonical Chunker V2...

✅ TSLA-Q4-2025-Update.pdf
   Total chunks: 136
   Types: {'text': 67, 'table': 69}
   Tokens: median=156.5, p95=238, max=240
   Oversized: 0
   Exact duplicates: 0

✅ nist.ai.100-1.pdf
   Total chunks: 183
   Types: {'text': 151, 'table': 32}
   Tokens: median=148.0, p95=207, max=240
   Oversized: 0
   Exact duplicates: 0

✅ _10-K-2025-As-Filed.pdf
   Total chunks: 599
   Types: {'text': 512, 'table': 87}
   Tokens: median=133.0, p95=215, max=240
   Oversized: 0
   Exact duplicates: 0

✅ Canonical Chunker V2 dry-run completed.
📦 Documents processed: 3
⏭️ Existing web backend was not patched.
⏭️ No embeddings were generated.
⏭️ No Qdrant points were changed.


In [35]:
# VAULTIFY COMPACT - Cell 20A:
# Build the canonical seed-style, token-safe chunker V2
# and rechunk the preserved Markdown documents.

from collections import Counter
import hashlib
import re
import statistics


# ---------------------------------------------------------------------
# 1. Runtime validation
# ---------------------------------------------------------------------

if (
    "DRY_RUN_RESULTS" not in globals()
    or len(DRY_RUN_RESULTS) < 3
):
    raise RuntimeError(
        "Preserved Apple, Tesla, and NIST dry-run results "
        "were not found. Run Cell 18 and Cell 18A first."
    )


V2_TOKENIZER = embedding_model.tokenizer

V2_MAX_CHUNK_TOKENS = min(
    240,
    embedding_model.max_seq_length - 16,
)

V2_SPECIAL_TOKEN_BUFFER = 2

V2_TOKEN_PAYLOAD_LIMIT = (
    V2_MAX_CHUNK_TOKENS
    - V2_SPECIAL_TOKEN_BUFFER
)

# Preserve section and column information without
# allowing the prefix to consume the whole chunk.
V2_MAX_TABLE_PREFIX_TOKENS = min(
    96,
    V2_TOKEN_PAYLOAD_LIMIT // 2,
)

# These are the validated seed-pipeline text settings.
V2_TEXT_CHUNK_SIZE = 800
V2_TEXT_CHUNK_OVERLAP = 120


# ---------------------------------------------------------------------
# 2. Token utilities
# ---------------------------------------------------------------------

def v2_get_raw_token_ids(text):
    encoded = V2_TOKENIZER(
        str(text or ""),
        add_special_tokens=False,
        truncation=False,
        return_attention_mask=False,
        return_token_type_ids=False,
        verbose=False,
    )

    token_ids = encoded["input_ids"]

    if (
        token_ids
        and isinstance(
            token_ids[0],
            (list, tuple),
        )
    ):
        token_ids = token_ids[0]

    return list(token_ids)


def v2_count_raw_tokens(text):
    return len(
        v2_get_raw_token_ids(text)
    )


def v2_count_embedding_tokens(text):
    encoded = V2_TOKENIZER(
        str(text or ""),
        add_special_tokens=True,
        truncation=False,
        return_attention_mask=False,
        return_token_type_ids=False,
        verbose=False,
    )

    token_ids = encoded["input_ids"]

    if (
        token_ids
        and isinstance(
            token_ids[0],
            (list, tuple),
        )
    ):
        token_ids = token_ids[0]

    return len(token_ids)


def v2_decode_token_slice(token_ids):
    return V2_TOKENIZER.decode(
        token_ids,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=True,
    ).strip()


def v2_truncate_to_token_limit(
    text,
    token_limit,
):
    text = str(text or "").strip()

    if not text or token_limit <= 0:
        return ""

    token_ids = v2_get_raw_token_ids(
        text
    )

    if len(token_ids) <= token_limit:
        return text

    return v2_decode_token_slice(
        token_ids[:token_limit]
    )


def v2_split_to_token_safe_text(
    text,
    token_limit=V2_TOKEN_PAYLOAD_LIMIT,
):
    text = str(text or "").strip()

    if not text:
        return []

    token_ids = v2_get_raw_token_ids(
        text
    )

    if len(token_ids) <= token_limit:
        return [text]

    safe_pieces = []

    for start_index in range(
        0,
        len(token_ids),
        token_limit,
    ):
        token_slice = token_ids[
            start_index:
            start_index + token_limit
        ]

        decoded_piece = (
            v2_decode_token_slice(
                token_slice
            )
        )

        if decoded_piece:
            safe_pieces.append(
                decoded_piece
            )

    return safe_pieces


# ---------------------------------------------------------------------
# 3. Table helpers
# ---------------------------------------------------------------------

def v2_is_markdown_separator_row(line):
    stripped_line = str(
        line or ""
    ).strip()

    if not stripped_line.startswith("|"):
        return False

    cells = [
        cell.strip()
        for cell in stripped_line
        .strip("|")
        .split("|")
    ]

    if not cells:
        return False

    return all(
        bool(
            re.fullmatch(
                r":?-{3,}:?",
                cell,
            )
        )
        for cell in cells
    )


def v2_build_safe_table_prefix(
    section,
    header_line,
):
    """
    Give tables explicit semantic labels and remove
    token-expensive Markdown separator rows.
    """
    section = (
        str(section or "").strip()
        or "Unknown section"
    )

    header_line = str(
        header_line or ""
    ).strip()

    section_label = (
        f"Section: {section}\n"
        "Table columns:\n"
    )

    section_token_count = (
        v2_count_raw_tokens(
            section_label
        )
    )

    if (
        section_token_count
        >= V2_MAX_TABLE_PREFIX_TOKENS
    ):
        return v2_truncate_to_token_limit(
            section_label,
            V2_MAX_TABLE_PREFIX_TOKENS,
        )

    available_header_tokens = (
        V2_MAX_TABLE_PREFIX_TOKENS
        - section_token_count
    )

    compact_header = (
        v2_truncate_to_token_limit(
            header_line,
            available_header_tokens,
        )
    )

    return (
        section_label
        + compact_header
    ).strip()


def v2_create_chunk(
    text,
    chunk_type,
    section,
):
    return {
        "text": str(text or "").strip(),
        "chunk_type": chunk_type,
        "section": (
            str(section or "").strip()
            or "Document content"
        ),
    }


# ---------------------------------------------------------------------
# 4. Text chunking
# ---------------------------------------------------------------------

def v2_split_normal_text(
    text,
    section,
):
    text = str(text or "").strip()

    if not text:
        return []

    chunks = []
    start_index = 0

    while start_index < len(text):
        end_index = min(
            start_index
            + V2_TEXT_CHUNK_SIZE,
            len(text),
        )

        character_piece = text[
            start_index:end_index
        ].strip()

        for safe_piece in (
            v2_split_to_token_safe_text(
                character_piece
            )
        ):
            chunks.append(
                v2_create_chunk(
                    text=safe_piece,
                    chunk_type="text",
                    section=section,
                )
            )

        if end_index >= len(text):
            break

        next_start_index = (
            end_index
            - V2_TEXT_CHUNK_OVERLAP
        )

        start_index = max(
            next_start_index,
            start_index + 1,
        )

    return chunks


# ---------------------------------------------------------------------
# 5. Table chunking
# ---------------------------------------------------------------------

def v2_split_oversized_table_row(
    row,
    table_prefix,
    section,
):
    prefix_with_newline = (
        table_prefix.rstrip()
        + "\n"
    )

    prefix_token_count = (
        v2_count_raw_tokens(
            prefix_with_newline
        )
    )

    available_row_tokens = (
        V2_TOKEN_PAYLOAD_LIMIT
        - prefix_token_count
    )

    if available_row_tokens <= 0:
        raise RuntimeError(
            "The table prefix consumed the complete token budget. "
            f"Section: {section}"
        )

    row_token_ids = (
        v2_get_raw_token_ids(
            row
        )
    )

    chunks = []

    for start_index in range(
        0,
        len(row_token_ids),
        available_row_tokens,
    ):
        token_slice = row_token_ids[
            start_index:
            start_index
            + available_row_tokens
        ]

        decoded_row = (
            v2_decode_token_slice(
                token_slice
            )
        )

        if not decoded_row:
            continue

        chunk_text = (
            prefix_with_newline
            + decoded_row
        ).strip()

        if (
            v2_count_embedding_tokens(
                chunk_text
            )
            > V2_MAX_CHUNK_TOKENS
        ):
            raise RuntimeError(
                "An oversized table-row piece "
                "was generated unexpectedly."
            )

        chunks.append(
            v2_create_chunk(
                text=chunk_text,
                chunk_type="table",
                section=section,
            )
        )

    return chunks


def v2_split_table(
    table_lines,
    section,
):
    if not table_lines:
        return []

    cleaned_lines = [
        line.strip()
        for line in table_lines
        if line.strip()
    ]

    if not cleaned_lines:
        return []

    header_line = cleaned_lines[0]

    if (
        len(cleaned_lines) > 1
        and v2_is_markdown_separator_row(
            cleaned_lines[1]
        )
    ):
        data_rows = cleaned_lines[2:]
    else:
        data_rows = cleaned_lines[1:]

    table_prefix = (
        v2_build_safe_table_prefix(
            section=section,
            header_line=header_line,
        )
    )

    chunks = []
    current_rows = []

    def build_table_text(rows):
        return (
            table_prefix.rstrip()
            + "\n"
            + "\n".join(rows)
        ).strip()

    def flush_current_rows():
        nonlocal current_rows

        if not current_rows:
            return

        chunk_text = build_table_text(
            current_rows
        )

        token_count = (
            v2_count_embedding_tokens(
                chunk_text
            )
        )

        if token_count > V2_MAX_CHUNK_TOKENS:
            raise RuntimeError(
                "An internal table chunk exceeded "
                "the 240-token limit."
            )

        chunks.append(
            v2_create_chunk(
                text=chunk_text,
                chunk_type="table",
                section=section,
            )
        )

        current_rows = []

    if not data_rows:
        for safe_piece in (
            v2_split_to_token_safe_text(
                table_prefix
            )
        ):
            chunks.append(
                v2_create_chunk(
                    text=safe_piece,
                    chunk_type="table",
                    section=section,
                )
            )

        return chunks

    for row in data_rows:
        candidate_rows = (
            current_rows + [row]
        )

        candidate_text = (
            build_table_text(
                candidate_rows
            )
        )

        if (
            v2_count_embedding_tokens(
                candidate_text
            )
            <= V2_MAX_CHUNK_TOKENS
        ):
            current_rows = candidate_rows
            continue

        flush_current_rows()

        single_row_text = (
            build_table_text([row])
        )

        if (
            v2_count_embedding_tokens(
                single_row_text
            )
            <= V2_MAX_CHUNK_TOKENS
        ):
            current_rows = [row]

        else:
            chunks.extend(
                v2_split_oversized_table_row(
                    row=row,
                    table_prefix=table_prefix,
                    section=section,
                )
            )

    flush_current_rows()

    return chunks


# ---------------------------------------------------------------------
# 6. Canonical Markdown chunker
# ---------------------------------------------------------------------

def canonical_chunk_markdown_v2(
    model,
    markdown_text,
):
    """
    Canonical Vaultify chunker:

    - validated seed-style table serialization
    - explicit section and column labels
    - separator-row removal
    - strict 240-token maximum
    - oversized-row splitting
    - exact duplicate removal
    """
    lines = str(
        markdown_text or ""
    ).splitlines()

    chunks = []
    current_section = (
        "Document introduction"
    )

    text_buffer = []

    def flush_text_buffer():
        nonlocal text_buffer

        buffered_text = "\n".join(
            text_buffer
        ).strip()

        if buffered_text:
            chunks.extend(
                v2_split_normal_text(
                    text=buffered_text,
                    section=current_section,
                )
            )

        text_buffer = []

    line_index = 0

    while line_index < len(lines):
        line = lines[line_index]
        stripped_line = line.strip()

        heading_match = re.match(
            r"^(#{1,6})\s+(.+)$",
            stripped_line,
        )

        if heading_match:
            flush_text_buffer()

            current_section = (
                heading_match
                .group(2)
                .strip()
            )

            text_buffer.append(
                stripped_line
            )

            line_index += 1
            continue

        if stripped_line.startswith("|"):
            flush_text_buffer()

            table_lines = []

            while (
                line_index < len(lines)
                and lines[line_index]
                .strip()
                .startswith("|")
            ):
                table_lines.append(
                    lines[line_index]
                )

                line_index += 1

            chunks.extend(
                v2_split_table(
                    table_lines=table_lines,
                    section=current_section,
                )
            )

            continue

        text_buffer.append(line)
        line_index += 1

    flush_text_buffer()

    # Exact deduplication.
    unique_chunks = []
    seen_hashes = set()

    for chunk in chunks:
        normalized_text = re.sub(
            r"\s+",
            " ",
            chunk["text"],
        ).strip()

        if not normalized_text:
            continue

        chunk_hash = hashlib.sha256(
            normalized_text.encode(
                "utf-8"
            )
        ).hexdigest()

        if chunk_hash in seen_hashes:
            continue

        seen_hashes.add(
            chunk_hash
        )

        chunk["chunk_index"] = len(
            unique_chunks
        )

        unique_chunks.append(
            chunk
        )

    oversized_chunks = [
        chunk
        for chunk in unique_chunks
        if (
            v2_count_embedding_tokens(
                chunk["text"]
            )
            > V2_MAX_CHUNK_TOKENS
        )
    ]

    if oversized_chunks:
        raise RuntimeError(
            f"Canonical chunker generated "
            f"{len(oversized_chunks)} oversized chunks."
        )

    return unique_chunks


# ---------------------------------------------------------------------
# 7. Rechunk preserved documents without reparsing
# ---------------------------------------------------------------------

DRY_RUN_RESULTS_V2 = {}

print(
    "⏳ Rechunking preserved Markdown "
    "with Canonical Chunker V2..."
)

for (
    document_hash,
    preserved_result,
) in DRY_RUN_RESULTS.items():

    filename = preserved_result[
        "filename"
    ]

    markdown_text = preserved_result[
        "markdown"
    ]

    chunks = canonical_chunk_markdown_v2(
        embedding_model,
        markdown_text,
    )

    token_counts = [
        v2_count_embedding_tokens(
            chunk["text"]
        )
        for chunk in chunks
    ]

    chunk_types = Counter(
        chunk["chunk_type"]
        for chunk in chunks
    )

    normalized_texts = [
        re.sub(
            r"\s+",
            " ",
            chunk["text"],
        ).strip()
        for chunk in chunks
    ]

    duplicate_count = (
        len(normalized_texts)
        - len(set(normalized_texts))
    )

    oversized_count = sum(
        token_count
        > V2_MAX_CHUNK_TOKENS
        for token_count
        in token_counts
    )

    report = {
        "filename": filename,
        "total_chunks": len(chunks),
        "chunk_types": dict(
            chunk_types
        ),
        "token_min": min(
            token_counts
        ),
        "token_median": (
            statistics.median(
                token_counts
            )
        ),
        "token_p95": sorted(
            token_counts
        )[
            round(
                (len(token_counts) - 1)
                * 0.95
            )
        ],
        "token_max": max(
            token_counts
        ),
        "exact_duplicates": (
            duplicate_count
        ),
        "oversized": (
            oversized_count
        ),
    }

    DRY_RUN_RESULTS_V2[
        document_hash
    ] = {
        "filename": filename,
        "document_hash": (
            document_hash
        ),
        "markdown": markdown_text,
        "chunks": chunks,
        "report": report,
    }

    print(
        f"\n✅ {filename}"
    )

    print(
        f"   Total chunks: "
        f"{report['total_chunks']}"
    )

    print(
        f"   Types: "
        f"{report['chunk_types']}"
    )

    print(
        "   Tokens:"
        f" median={report['token_median']:.1f},"
        f" p95={report['token_p95']},"
        f" max={report['token_max']}"
    )

    print(
        f"   Oversized: "
        f"{report['oversized']}"
    )

    print(
        f"   Exact duplicates: "
        f"{report['exact_duplicates']}"
    )


print("\n" + "=" * 88)
print("✅ Canonical Chunker V2 dry-run completed.")
print(
    f"📦 Documents processed: "
    f"{len(DRY_RUN_RESULTS_V2)}"
)
print("⏭️ Existing web backend was not patched.")
print("⏭️ No embeddings were generated.")
print("⏭️ No Qdrant points were changed.")

⏳ Rechunking preserved Markdown with Canonical Chunker V2...

✅ TSLA-Q4-2025-Update.pdf
   Total chunks: 136
   Types: {'text': 67, 'table': 69}
   Tokens: median=156.5, p95=238, max=240
   Oversized: 0
   Exact duplicates: 0

✅ nist.ai.100-1.pdf
   Total chunks: 183
   Types: {'text': 151, 'table': 32}
   Tokens: median=148.0, p95=207, max=240
   Oversized: 0
   Exact duplicates: 0

✅ _10-K-2025-As-Filed.pdf
   Total chunks: 599
   Types: {'text': 512, 'table': 87}
   Tokens: median=133.0, p95=215, max=240
   Oversized: 0
   Exact duplicates: 0

✅ Canonical Chunker V2 dry-run completed.
📦 Documents processed: 3
⏭️ Existing web backend was not patched.
⏭️ No embeddings were generated.
⏭️ No Qdrant points were changed.


In [36]:
# VAULTIFY COMPACT - Cell 20B:
# Compare Canonical Chunker V2 against the validated seed pipeline
# using the same hybrid retrieval system.

import re

import numpy as np


# ---------------------------------------------------------------------
# 1. Validate runtime dependencies
# ---------------------------------------------------------------------

required_names = [
    "DRY_RUN_RESULTS_V2",
    "OLD_APPLE_RESULT",
    "OLD_TESLA_RESULT",
    "embedding_model",
    "build_bm25_index",
    "retrieve_hybrid",
]

missing_names = [
    name
    for name in required_names
    if name not in globals()
]

if missing_names:
    raise RuntimeError(
        "Missing required components: "
        + ", ".join(missing_names)
        + ". Run Cells 19B, 19C, and 20A first."
    )


# ---------------------------------------------------------------------
# 2. Locate Canonical V2 documents
# ---------------------------------------------------------------------

def find_v2_document(filename_fragment):
    filename_fragment = (
        filename_fragment.lower()
    )

    matches = [
        result
        for result
        in DRY_RUN_RESULTS_V2.values()
        if filename_fragment
        in result["filename"].lower()
    ]

    if len(matches) != 1:
        raise RuntimeError(
            f"Expected one Canonical V2 document matching "
            f"'{filename_fragment}', found {len(matches)}."
        )

    return matches[0]


v2_apple_document = find_v2_document(
    "10-k"
)

v2_tesla_document = find_v2_document(
    "tsla"
)


# ---------------------------------------------------------------------
# 3. Prepare Canonical V2 retrieval objects
# ---------------------------------------------------------------------

def prepare_v2_retrieval_result(
    label,
    preserved_document,
):
    chunks = preserved_document[
        "chunks"
    ]

    print(
        f"⏳ Preparing {label}: "
        f"{len(chunks)} chunks"
    )

    embeddings = embedding_model.encode(
        [
            chunk["text"]
            for chunk in chunks
        ],
        batch_size=64,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=True,
    )

    result = {
        "filename": label,
        "chunks": chunks,
        "embeddings": embeddings,
    }

    result["bm25_index"] = (
        build_bm25_index(
            chunks
        )
    )

    return result


V2_APPLE_RESULT = (
    prepare_v2_retrieval_result(
        label="Canonical V2 Apple",
        preserved_document=(
            v2_apple_document
        ),
    )
)

V2_TESLA_RESULT = (
    prepare_v2_retrieval_result(
        label="Canonical V2 Tesla",
        preserved_document=(
            v2_tesla_document
        ),
    )
)


# ---------------------------------------------------------------------
# 4. Benchmark questions
# ---------------------------------------------------------------------

V2_COMPARISON_TESTS = {
    "Apple": {
        "old": OLD_APPLE_RESULT,
        "v2": V2_APPLE_RESULT,
        "queries": [
            {
                "question": (
                    "What were Apple's total net sales "
                    "in fiscal year 2025?"
                ),
                "expected_groups": [
                    [
                        "416,161",
                        "416161",
                    ],
                    [
                        "total net sales",
                    ],
                ],
            },
            {
                "question": (
                    "What were Apple's services net sales "
                    "in fiscal year 2025?"
                ),
                "expected_groups": [
                    [
                        "109,158",
                        "109158",
                    ],
                    [
                        "services",
                    ],
                ],
            },
            {
                "question": (
                    "What were Apple's iPhone net sales "
                    "in fiscal year 2025?"
                ),
                "expected_groups": [
                    [
                        "209,586",
                        "209586",
                    ],
                    [
                        "iphone",
                    ],
                ],
            },
        ],
    },

    "Tesla": {
        "old": OLD_TESLA_RESULT,
        "v2": V2_TESLA_RESULT,
        "queries": [
            {
                "question": (
                    "What was Tesla's total revenue "
                    "in Q4 2025?"
                ),
                "expected_groups": [
                    [
                        "24,901",
                        "24901",
                    ],
                    [
                        "total revenue",
                        "total revenues",
                    ],
                ],
            },
            {
                "question": (
                    "What was Tesla's automotive revenue "
                    "in Q4 2025?"
                ),
                "expected_groups": [
                    [
                        "automotive revenues",
                        "automotive revenue",
                    ],
                    [
                        "q4-2025",
                        "q4 2025",
                    ],
                ],
            },
            {
                "question": (
                    "What was Tesla's energy generation "
                    "and storage revenue in Q4 2025?"
                ),
                "expected_groups": [
                    [
                        "energy generation and storage",
                        "energy generation & storage",
                    ],
                    [
                        "q4-2025",
                        "q4 2025",
                    ],
                ],
            },
            {
                "question": (
                    "What was Tesla's GAAP operating income "
                    "in Q4 2025?"
                ),
                "expected_groups": [
                    [
                        "gaap operating income",
                        "operating income",
                    ],
                    [
                        "1.4b",
                        "$1.4b",
                        "1.4 b",
                    ],
                    [
                        "q4",
                    ],
                ],
            },
        ],
    },
}


# ---------------------------------------------------------------------
# 5. Evidence matching
# ---------------------------------------------------------------------

def normalize_v2_evidence(text):
    normalized = " ".join(
        str(text or "")
        .lower()
        .split()
    )

    compact = re.sub(
        r"[^a-z0-9]+",
        "",
        normalized,
    )

    return normalized, compact


def v2_text_contains_groups(
    text,
    expected_groups,
):
    normalized_text, compact_text = (
        normalize_v2_evidence(
            text
        )
    )

    for alternatives in expected_groups:
        group_found = False

        for alternative in alternatives:
            (
                normalized_alternative,
                compact_alternative,
            ) = normalize_v2_evidence(
                alternative
            )

            if (
                normalized_alternative
                in normalized_text
                or (
                    compact_alternative
                    and compact_alternative
                    in compact_text
                )
            ):
                group_found = True
                break

        if not group_found:
            return False

    return True


def evaluate_v2_chunk_set(
    result,
    question,
    expected_groups,
    retrieval_depth=20,
):
    retrieved = retrieve_hybrid(
        result=result,
        question=question,
        top_k=retrieval_depth,
    )

    complete_evidence_rank = None

    for item in retrieved:
        if v2_text_contains_groups(
            item["text"],
            expected_groups,
        ):
            complete_evidence_rank = (
                item["rank"]
            )
            break

    combined_top_6_context = " ".join(
        item["text"]
        for item in retrieved[:6]
    )

    top_6_context_passed = (
        v2_text_contains_groups(
            combined_top_6_context,
            expected_groups,
        )
    )

    return {
        "retrieved": retrieved,
        "complete_evidence_rank": (
            complete_evidence_rank
        ),
        "top_6_context_passed": (
            top_6_context_passed
        ),
    }


# ---------------------------------------------------------------------
# 6. Run comparison
# ---------------------------------------------------------------------

CELL_20B_RESULTS = []

print("\n" + "=" * 96)
print(
    "⚖️ VALIDATED SEED VS CANONICAL V2 "
    "— SAME HYBRID RETRIEVER"
)

for document_label, config in (
    V2_COMPARISON_TESTS.items()
):
    print("\n" + "=" * 96)
    print(f"📄 {document_label}")

    for test_case in config[
        "queries"
    ]:
        question = test_case[
            "question"
        ]

        expected_groups = test_case[
            "expected_groups"
        ]

        old_evaluation = (
            evaluate_v2_chunk_set(
                result=config["old"],
                question=question,
                expected_groups=(
                    expected_groups
                ),
            )
        )

        v2_evaluation = (
            evaluate_v2_chunk_set(
                result=config["v2"],
                question=question,
                expected_groups=(
                    expected_groups
                ),
            )
        )

        old_rank = old_evaluation[
            "complete_evidence_rank"
        ]

        v2_rank = v2_evaluation[
            "complete_evidence_rank"
        ]

        if (
            v2_rank is not None
            and (
                old_rank is None
                or v2_rank < old_rank
            )
        ):
            winner = "V2"

        elif (
            old_rank is not None
            and (
                v2_rank is None
                or old_rank < v2_rank
            )
        ):
            winner = "SEED"

        else:
            winner = "TIE"

        CELL_20B_RESULTS.append(
            {
                "document": (
                    document_label
                ),
                "question": question,
                "seed_rank": old_rank,
                "v2_rank": v2_rank,
                "winner": winner,
                "seed_top_6": (
                    old_evaluation[
                        "top_6_context_passed"
                    ]
                ),
                "v2_top_6": (
                    v2_evaluation[
                        "top_6_context_passed"
                    ]
                ),
            }
        )

        old_rank_label = (
            str(old_rank)
            if old_rank is not None
            else ">20"
        )

        v2_rank_label = (
            str(v2_rank)
            if v2_rank is not None
            else ">20"
        )

        print(
            f"\nQuestion: {question}"
        )

        print(
            f"   Seed complete-evidence rank: "
            f"{old_rank_label}"
        )

        print(
            f"   V2 complete-evidence rank: "
            f"{v2_rank_label}"
        )

        print(
            f"   Seed top-6 context pass: "
            f"{old_evaluation['top_6_context_passed']}"
        )

        print(
            f"   V2 top-6 context pass: "
            f"{v2_evaluation['top_6_context_passed']}"
        )

        print(
            f"   Winner: {winner}"
        )

        seed_top = old_evaluation[
            "retrieved"
        ][0]

        v2_top = v2_evaluation[
            "retrieved"
        ][0]

        print(
            "   Seed top result:"
        )

        print(
            f"      dense={seed_top['dense_rank']} "
            f"bm25={seed_top['lexical_rank']} "
            f"section={seed_top['section']}"
        )

        print(
            "      "
            + " ".join(
                seed_top["text"].split()
            )[:170]
            + "..."
        )

        print(
            "   V2 top result:"
        )

        print(
            f"      dense={v2_top['dense_rank']} "
            f"bm25={v2_top['lexical_rank']} "
            f"section={v2_top['section']}"
        )

        print(
            "      "
            + " ".join(
                v2_top["text"].split()
            )[:170]
            + "..."
        )


# ---------------------------------------------------------------------
# 7. Summary and approval gate
# ---------------------------------------------------------------------

v2_wins = sum(
    result["winner"] == "V2"
    for result in CELL_20B_RESULTS
)

seed_wins = sum(
    result["winner"] == "SEED"
    for result in CELL_20B_RESULTS
)

ties = sum(
    result["winner"] == "TIE"
    for result in CELL_20B_RESULTS
)

seed_top_6_passes = sum(
    result["seed_top_6"]
    for result in CELL_20B_RESULTS
)

v2_top_6_passes = sum(
    result["v2_top_6"]
    for result in CELL_20B_RESULTS
)

total_tests = len(
    CELL_20B_RESULTS
)

CELL_20B_V2_APPROVED = (
    v2_top_6_passes
    >= seed_top_6_passes
    and v2_wins >= seed_wins
)

print("\n" + "=" * 96)
print("📊 CELL 20B SUMMARY")

print(
    f"Canonical V2 wins: "
    f"{v2_wins}"
)

print(
    f"Validated seed wins: "
    f"{seed_wins}"
)

print(
    f"Ties: "
    f"{ties}"
)

print(
    f"Seed top-6 passes: "
    f"{seed_top_6_passes}"
    f"/{total_tests}"
)

print(
    f"Canonical V2 top-6 passes: "
    f"{v2_top_6_passes}"
    f"/{total_tests}"
)

if CELL_20B_V2_APPROVED:
    print(
        "✅ Canonical V2 matched or exceeded "
        "the validated seed retrieval quality."
    )

    print(
        "➡️ Canonical V2 is eligible to become "
        "the web ingestion chunker."
    )

else:
    print(
        "❌ Canonical V2 still introduced "
        "a measurable retrieval regression."
    )

    print(
        "⛔ Do not patch the web backend yet."
    )

print(
    "\n⏭️ No Qdrant points were created, "
    "updated, or deleted."
)

⏳ Preparing Canonical V2 Apple: 599 chunks


Batches:   0%|          | 0/10 [00:00<?, ?it/s]

⏳ Preparing Canonical V2 Tesla: 136 chunks


Batches:   0%|          | 0/3 [00:00<?, ?it/s]


⚖️ VALIDATED SEED VS CANONICAL V2 — SAME HYBRID RETRIEVER

📄 Apple

Question: What were Apple's total net sales in fiscal year 2025?
   Seed complete-evidence rank: 1
   V2 complete-evidence rank: 1
   Seed top-6 context pass: True
   V2 top-6 context pass: True
   Winner: TIE
   Seed top result:
      dense=3 bm25=13 section=Note 2 - Revenue
      Section: Note 2 - Revenue Table columns: | | 2025 | 2024 | 2023 | | iPhone | $ 209,586 | $ 201,183 | $ 200,583 | | Mac | 33,708 | 29,984 | 29,357 | | iPad | 28,023 | 26,6...
   V2 top result:
      dense=3 bm25=11 section=Note 2 - Revenue
      Section: Note 2 - Revenue Table columns: | | 2025 | 2024 | 2023 | | iPhone | $ 209,586 | $ 201,183 | $ 200,583 | | Mac | 33,708 | 29,984 | 29,357 | | iPad | 28,023 | 26,6...

Question: What were Apple's services net sales in fiscal year 2025?
   Seed complete-evidence rank: 7
   V2 complete-evidence rank: 7
   Seed top-6 context pass: False
   V2 top-6 context pass: False
   Winner: TIE
   Seed top r

In [37]:
# VAULTIFY COMPACT - Cell 20C:
# Activate Canonical Chunker V2 for the current web runtime

import vaultify_web_backend


required_names = [
    "canonical_chunk_markdown_v2",
    "CELL_20B_V2_APPROVED",
    "DRY_RUN_RESULTS_V2",
]

missing_names = [
    name
    for name in required_names
    if name not in globals()
]

if missing_names:
    raise RuntimeError(
        "Missing required components: "
        + ", ".join(missing_names)
        + ". Run Cells 20A and 20B first."
    )


if not CELL_20B_V2_APPROVED:
    raise RuntimeError(
        "Canonical V2 did not pass the retrieval approval gate."
    )


# Keep references to the previous runtime chunker
# so that the patch can be inspected or reverted.
PREVIOUS_WEB_CHUNKER = getattr(
    vaultify_web_backend,
    "smart_chunk_markdown",
    None,
)


# Patch the module-level function.
vaultify_web_backend.smart_chunk_markdown = (
    canonical_chunk_markdown_v2
)


# Patch the exact global reference used by ingest_document().
vaultify_web_backend.ingest_document.__globals__[
    "smart_chunk_markdown"
] = canonical_chunk_markdown_v2


# ---------------------------------------------------------------------
# Verify that every backend reference points to Canonical V2
# ---------------------------------------------------------------------

module_patch_ok = (
    vaultify_web_backend.smart_chunk_markdown
    is canonical_chunk_markdown_v2
)

ingestion_patch_ok = (
    vaultify_web_backend
    .ingest_document
    .__globals__[
        "smart_chunk_markdown"
    ]
    is canonical_chunk_markdown_v2
)


if not module_patch_ok:
    raise RuntimeError(
        "The backend module chunker was not patched correctly."
    )

if not ingestion_patch_ok:
    raise RuntimeError(
        "The ingestion function still references the old chunker."
    )


# ---------------------------------------------------------------------
# Lightweight regression smoke test using preserved Markdown
# ---------------------------------------------------------------------

smoke_test_results = []

for preserved_result in (
    DRY_RUN_RESULTS_V2.values()
):
    report = preserved_result[
        "report"
    ]

    smoke_test_results.append(
        {
            "filename": preserved_result[
                "filename"
            ],
            "chunks": report[
                "total_chunks"
            ],
            "maximum_tokens": report[
                "token_max"
            ],
            "oversized": report[
                "oversized"
            ],
            "duplicates": report[
                "exact_duplicates"
            ],
        }
    )


for result in smoke_test_results:
    if result["maximum_tokens"] > 240:
        raise RuntimeError(
            f"Oversized regression in "
            f"{result['filename']}."
        )

    if result["oversized"] != 0:
        raise RuntimeError(
            f"Oversized chunks detected in "
            f"{result['filename']}."
        )

    if result["duplicates"] != 0:
        raise RuntimeError(
            f"Duplicate chunks detected in "
            f"{result['filename']}."
        )


CANONICAL_V2_WEB_PATCH_ACTIVE = True


print("✅ Canonical Chunker V2 installed in the web backend.")
print("✅ ingest_document now uses Canonical V2.")
print("✅ Maximum chunk limit: 240 tokens.")
print("✅ Exact duplicate removal: active.")
print("✅ Table-aware seed-style serialization: active.")

print("\nValidated document results:")

for result in smoke_test_results:
    print(
        f"   {result['filename']}"
        f" — {result['chunks']} chunks"
        f" — max {result['maximum_tokens']} tokens"
        f" — oversized {result['oversized']}"
        f" — duplicates {result['duplicates']}"
    )

print(
    "\n⏭️ No Qdrant points were created, "
    "updated, or deleted."
)

print(
    "⏭️ The next uploaded PDF will use "
    "Canonical Chunker V2."
)

✅ Canonical Chunker V2 installed in the web backend.
✅ ingest_document now uses Canonical V2.
✅ Maximum chunk limit: 240 tokens.
✅ Exact duplicate removal: active.
✅ Table-aware seed-style serialization: active.

Validated document results:
   TSLA-Q4-2025-Update.pdf — 136 chunks — max 240 tokens — oversized 0 — duplicates 0
   nist.ai.100-1.pdf — 183 chunks — max 240 tokens — oversized 0 — duplicates 0
   _10-K-2025-As-Filed.pdf — 599 chunks — max 240 tokens — oversized 0 — duplicates 0

⏭️ No Qdrant points were created, updated, or deleted.
⏭️ The next uploaded PDF will use Canonical Chunker V2.


In [39]:
# VAULTIFY COMPACT - Cell 21A:
# Build the tenant document catalog for Retrieval Orchestrator V1
#
# READ-ONLY:
# - Reads Qdrant payloads for one tenant
# - Groups chunks by document
# - Detects document entities, aliases, sections, years, and chunk types
# - Does not generate embeddings
# - Does not modify Qdrant

from collections import Counter, defaultdict
from pathlib import Path
import re

from qdrant_client.models import (
    FieldCondition,
    Filter,
    MatchValue,
)


# ---------------------------------------------------------------------
# 1. Resolve runtime dependencies
# ---------------------------------------------------------------------

if "qdrant" not in globals():
    raise RuntimeError(
        "The Qdrant client was not found. "
        "Run the Qdrant connection cells first."
    )


ORCHESTRATOR_COLLECTION_NAME = globals().get(
    "COLLECTION_NAME",
    "vaultify_v3_documents",
)

ORCHESTRATOR_TENANT_FIELD = globals().get(
    "TENANT_ID_FIELD",
    "tenant_id",
)


# Prefer a previously selected orchestrator tenant.
if globals().get("ORCHESTRATOR_TENANT_ID"):
    ORCHESTRATOR_TENANT_ID = str(
        ORCHESTRATOR_TENANT_ID
    )

# Otherwise use the validated Apple demo tenant.
elif (
    "DEMO_TENANTS" in globals()
    and isinstance(DEMO_TENANTS, dict)
    and "company_a" in DEMO_TENANTS
):
    company_a_config = DEMO_TENANTS[
        "company_a"
    ]

    if isinstance(company_a_config, dict):
        ORCHESTRATOR_TENANT_ID = str(
            company_a_config.get(
                "tenant_id",
                "demo_apple_tenant",
            )
        )
    else:
        ORCHESTRATOR_TENANT_ID = (
            "demo_apple_tenant"
        )

else:
    ORCHESTRATOR_TENANT_ID = (
        "demo_apple_tenant"
    )


print(
    f"🏢 Orchestrator tenant: "
    f"{ORCHESTRATOR_TENANT_ID}"
)

print(
    f"📦 Qdrant collection: "
    f"{ORCHESTRATOR_COLLECTION_NAME}"
)


# ---------------------------------------------------------------------
# 2. Payload compatibility helpers
# ---------------------------------------------------------------------

def first_payload_value(
    payload,
    candidate_keys,
    default=None,
):
    """
    Return the first non-empty payload value among several
    compatible field names.
    """
    for key in candidate_keys:
        value = payload.get(key)

        if value is not None and value != "":
            return value

    return default


def payload_text(payload):
    return str(
        first_payload_value(
            payload,
            [
                "text",
                "content",
                "page_content",
                "chunk_text",
            ],
            "",
        )
    ).strip()


def payload_filename(payload):
    return str(
        first_payload_value(
            payload,
            [
                "filename",
                "file_name",
                "document_name",
                "source",
            ],
            "unknown_document",
        )
    ).strip()


def payload_document_hash(payload):
    value = first_payload_value(
        payload,
        [
            "document_hash",
            "file_hash",
            "sha256",
            "document_id",
        ],
        None,
    )

    if value is None:
        return None

    return str(value).strip()


def payload_section(payload):
    return str(
        first_payload_value(
            payload,
            [
                "section",
                "section_name",
                "heading",
                "title",
            ],
            "Unknown section",
        )
    ).strip()


def payload_chunk_type(payload):
    return str(
        first_payload_value(
            payload,
            [
                "chunk_type",
                "type",
                "content_type",
            ],
            "unknown",
        )
    ).strip().lower()


def payload_chunk_index(
    payload,
    fallback_index,
):
    value = first_payload_value(
        payload,
        [
            "chunk_index",
            "index",
            "chunk_id",
        ],
        fallback_index,
    )

    try:
        return int(value)
    except (TypeError, ValueError):
        return fallback_index


# ---------------------------------------------------------------------
# 3. Read every point belonging to the selected tenant
# ---------------------------------------------------------------------

def load_tenant_points(
    tenant_id,
    batch_size=256,
):
    tenant_filter = Filter(
        must=[
            FieldCondition(
                key=ORCHESTRATOR_TENANT_FIELD,
                match=MatchValue(
                    value=tenant_id
                ),
            )
        ]
    )

    points = []
    next_offset = None

    while True:
        batch, next_offset = qdrant.scroll(
            collection_name=(
                ORCHESTRATOR_COLLECTION_NAME
            ),
            scroll_filter=tenant_filter,
            limit=batch_size,
            offset=next_offset,
            with_payload=True,
            with_vectors=False,
        )

        points.extend(batch)

        if next_offset is None:
            break

    return points


print(
    "\n⏳ Reading tenant payloads "
    "from Qdrant..."
)

tenant_points = load_tenant_points(
    ORCHESTRATOR_TENANT_ID
)

if not tenant_points:
    raise RuntimeError(
        "No Qdrant points were found for tenant "
        f"'{ORCHESTRATOR_TENANT_ID}'."
    )

print(
    f"✅ Tenant points loaded: "
    f"{len(tenant_points)}"
)


# ---------------------------------------------------------------------
# 4. Convert Qdrant points into normalized chunk records
# ---------------------------------------------------------------------

ORCHESTRATOR_TENANT_CHUNKS = []

for fallback_index, point in enumerate(
    tenant_points
):
    payload = point.payload or {}

    text = payload_text(payload)

    if not text:
        continue

    filename = payload_filename(
        payload
    )

    document_hash = payload_document_hash(
        payload
    )

    ORCHESTRATOR_TENANT_CHUNKS.append(
        {
            "point_id": str(point.id),
            "tenant_id": (
                ORCHESTRATOR_TENANT_ID
            ),
            "filename": filename,
            "document_hash": (
                document_hash
            ),
            "chunk_index": (
                payload_chunk_index(
                    payload,
                    fallback_index,
                )
            ),
            "chunk_type": (
                payload_chunk_type(
                    payload
                )
            ),
            "section": (
                payload_section(
                    payload
                )
            ),
            "text": text,
            "payload": payload,
        }
    )


if not ORCHESTRATOR_TENANT_CHUNKS:
    raise RuntimeError(
        "Tenant points existed, but no usable "
        "chunk text was found."
    )


# ---------------------------------------------------------------------
# 5. Deterministic V1 entity and alias inference
# ---------------------------------------------------------------------

KNOWN_ENTITY_RULES = {
    "Apple": {
        "filename_terms": {
            "apple",
            "aapl",
        },
        "content_terms": {
            "apple inc",
        },
        "aliases": {
            "apple",
            "apple inc",
            "aapl",
        },
    },

    "Tesla": {
        "filename_terms": {
            "tesla",
            "tsla",
        },
        "content_terms": {
            "tesla inc",
        },
        "aliases": {
            "tesla",
            "tesla inc",
            "tsla",
        },
    },

    "NIST": {
        "filename_terms": {
            "nist",
        },
        "content_terms": {
            "national institute "
            "of standards and technology",
        },
        "aliases": {
            "nist",
            "national institute "
            "of standards and technology",
        },
    },
}


def normalize_catalog_text(text):
    normalized = str(
        text or ""
    ).lower()

    normalized = re.sub(
        r"[_\-]+",
        " ",
        normalized,
    )

    normalized = re.sub(
        r"\s+",
        " ",
        normalized,
    )

    return normalized.strip()


def infer_document_entities(
    filename,
    chunks,
):
    normalized_filename = (
        normalize_catalog_text(
            filename
        )
    )

    sample_content = " ".join(
        chunk["text"]
        for chunk in chunks[:20]
    )

    normalized_content = (
        normalize_catalog_text(
            sample_content
        )
    )

    detected_entities = []

    for entity, rule in (
        KNOWN_ENTITY_RULES.items()
    ):
        filename_match = any(
            term in normalized_filename
            for term in rule[
                "filename_terms"
            ]
        )

        content_match = any(
            term in normalized_content
            for term in rule[
                "content_terms"
            ]
        )

        if filename_match or content_match:
            detected_entities.append(
                entity
            )

    # Generic fallback for unknown future documents.
    if not detected_entities:
        fallback_name = Path(
            filename
        ).stem

        fallback_name = re.sub(
            r"[_\-]+",
            " ",
            fallback_name,
        )

        fallback_name = re.sub(
            r"\s+",
            " ",
            fallback_name,
        ).strip()

        detected_entities.append(
            fallback_name
            or "Unknown entity"
        )

    return detected_entities


def aliases_for_entities(
    entities,
):
    aliases = set()

    for entity in entities:
        rule = KNOWN_ENTITY_RULES.get(
            entity
        )

        if rule:
            aliases.update(
                rule["aliases"]
            )
        else:
            aliases.add(
                normalize_catalog_text(
                    entity
                )
            )

    return sorted(aliases)


# ---------------------------------------------------------------------
# 6. Group chunks by document
# ---------------------------------------------------------------------

document_groups = defaultdict(list)

for chunk in ORCHESTRATOR_TENANT_CHUNKS:
    document_hash = chunk[
        "document_hash"
    ]

    filename = chunk[
        "filename"
    ]

    # Prefer the SHA-256 document hash.
    # Fall back to filename for older seed payloads.
    document_key = (
        document_hash
        if document_hash
        else f"filename::{filename.lower()}"
    )

    document_groups[
        document_key
    ].append(chunk)


ORCHESTRATOR_DOCUMENT_CATALOG = {}
ORCHESTRATOR_ENTITY_REGISTRY = (
    defaultdict(
        lambda: {
            "aliases": set(),
            "document_keys": set(),
            "filenames": set(),
        }
    )
)


YEAR_PATTERN = re.compile(
    r"\b(?:19|20)\d{2}\b"
)


for document_key, chunks in (
    document_groups.items()
):
    chunks.sort(
        key=lambda chunk: (
            chunk["chunk_index"],
            chunk["point_id"],
        )
    )

    filename_counts = Counter(
        chunk["filename"]
        for chunk in chunks
    )

    filename = filename_counts.most_common(
        1
    )[0][0]

    document_hash = next(
        (
            chunk["document_hash"]
            for chunk in chunks
            if chunk["document_hash"]
        ),
        None,
    )

    chunk_type_counts = Counter(
        chunk["chunk_type"]
        for chunk in chunks
    )

    section_counts = Counter(
        chunk["section"]
        for chunk in chunks
        if chunk["section"]
    )

    year_counts = Counter()

    for chunk in chunks:
        year_counts.update(
            YEAR_PATTERN.findall(
                chunk["text"]
            )
        )

    entities = infer_document_entities(
        filename=filename,
        chunks=chunks,
    )

    aliases = aliases_for_entities(
        entities
    )

    catalog_entry = {
        "document_key": document_key,
        "filename": filename,
        "document_hash": (
            document_hash
        ),
        "tenant_id": (
            ORCHESTRATOR_TENANT_ID
        ),
        "chunk_count": len(chunks),
        "chunk_types": dict(
            chunk_type_counts
        ),
        "entities": entities,
        "aliases": aliases,
        "top_years": [
            {
                "year": year,
                "mentions": count,
            }
            for year, count
            in year_counts.most_common(10)
        ],
        "top_sections": [
            {
                "section": section,
                "chunks": count,
            }
            for section, count
            in section_counts.most_common(12)
        ],
        "chunks": chunks,
    }

    ORCHESTRATOR_DOCUMENT_CATALOG[
        document_key
    ] = catalog_entry

    for entity in entities:
        registry_entry = (
            ORCHESTRATOR_ENTITY_REGISTRY[
                entity
            ]
        )

        registry_entry[
            "aliases"
        ].update(aliases)

        registry_entry[
            "document_keys"
        ].add(document_key)

        registry_entry[
            "filenames"
        ].add(filename)


# Convert defaultdict and sets to stable dictionaries/lists.
ORCHESTRATOR_ENTITY_REGISTRY = {
    entity: {
        "aliases": sorted(
            metadata["aliases"]
        ),
        "document_keys": sorted(
            metadata["document_keys"]
        ),
        "filenames": sorted(
            metadata["filenames"]
        ),
    }
    for entity, metadata
    in ORCHESTRATOR_ENTITY_REGISTRY.items()
}


# ---------------------------------------------------------------------
# 7. Validate the mixed-corpus fixture
# ---------------------------------------------------------------------

catalog_entities = {
    entity
    for document
    in ORCHESTRATOR_DOCUMENT_CATALOG.values()
    for entity
    in document["entities"]
}

ORCHESTRATOR_MIXED_CORPUS_READY = (
    "Apple" in catalog_entities
    and "Tesla" in catalog_entities
)


# ---------------------------------------------------------------------
# 8. Print catalog summary
# ---------------------------------------------------------------------

print("\n" + "=" * 92)
print("📚 RETRIEVAL ORCHESTRATOR DOCUMENT CATALOG")

print(
    f"Tenant: "
    f"{ORCHESTRATOR_TENANT_ID}"
)

print(
    f"Documents: "
    f"{len(ORCHESTRATOR_DOCUMENT_CATALOG)}"
)

print(
    f"Usable chunks: "
    f"{len(ORCHESTRATOR_TENANT_CHUNKS)}"
)


for document_number, document in enumerate(
    ORCHESTRATOR_DOCUMENT_CATALOG.values(),
    start=1,
):
    print("\n" + "-" * 92)

    print(
        f"📄 Document {document_number}: "
        f"{document['filename']}"
    )

    print(
        f"   Document hash: "
        f"{document['document_hash'] or 'not available'}"
    )

    print(
        f"   Chunks: "
        f"{document['chunk_count']}"
    )

    print(
        f"   Types: "
        f"{document['chunk_types']}"
    )

    print(
        f"   Entities: "
        f"{document['entities']}"
    )

    print(
        f"   Aliases: "
        f"{document['aliases']}"
    )

    print(
        "   Most frequent years: "
        + ", ".join(
            f"{item['year']} "
            f"({item['mentions']})"
            for item
            in document["top_years"][:6]
        )
    )

    print(
        "   Top sections:"
    )

    for section_entry in (
        document["top_sections"][:5]
    ):
        print(
            f"      "
            f"{section_entry['chunks']:>3} chunks"
            f" — "
            f"{section_entry['section']}"
        )


print("\n" + "=" * 92)
print("🏷️ ENTITY REGISTRY")

for entity, metadata in (
    ORCHESTRATOR_ENTITY_REGISTRY.items()
):
    print(
        f"\n{entity}"
    )

    print(
        f"   Aliases: "
        f"{metadata['aliases']}"
    )

    print(
        f"   Documents: "
        f"{metadata['filenames']}"
    )


print("\n" + "=" * 92)

if ORCHESTRATOR_MIXED_CORPUS_READY:
    print(
        "✅ Mixed Apple + Tesla corpus detected."
    )

    print(
        "✅ Tenant document catalog is ready "
        "for Query Analyzer V1."
    )

else:
    print(
        "⚠️ Apple + Tesla mixed corpus was not "
        "fully detected."
    )

    print(
        "Review the entity registry before "
        "continuing to Cell 21B."
    )


print(
    "\n⏭️ No embeddings were generated."
)

print(
    "⏭️ No Qdrant points were created, "
    "updated, or deleted."
)

🏢 Orchestrator tenant: demo_apple_tenant
📦 Qdrant collection: vaultify_v3_documents

⏳ Reading tenant payloads from Qdrant...
✅ Tenant points loaded: 745

📚 RETRIEVAL ORCHESTRATOR DOCUMENT CATALOG
Tenant: demo_apple_tenant
Documents: 2
Usable chunks: 745

--------------------------------------------------------------------------------------------
📄 Document 1: TSLA-Q4-2025-Update.pdf
   Document hash: 05a5eb4fff82710fafa5639deb2127d531b297fd6f6d93af58e72a2869926a18
   Chunks: 136
   Types: {'text': 67, 'table': 69}
   Entities: ['Tesla']
   Aliases: ['tesla', 'tesla inc', 'tsla']
   Most frequent years: 2025 (136), 2022 (64), 2023 (53), 2024 (49), 2026 (16), 2021 (13)
   Top sections:
       25 chunks — R E C O N C I L I A T I O N O F G A A P T O N O N - G A A P F I N A N C I A L I N F O R M A T I O N
       11 chunks — (Unaudited)
        8 chunks — O P E R A T I O N A L   S U M M A R Y
        8 chunks — S T A T E M E N T   O F   O P E R A T I O N S
        7 chunks — F I N A N C I A

In [40]:
# VAULTIFY COMPACT - Cell 21B:
# Deterministic Query Analyzer V1 for Retrieval Orchestrator
#
# READ-ONLY:
# - Detects entities, metrics, periods, comparisons, and ambiguity
# - Produces entity-specific subqueries
# - Does not call Groq
# - Does not generate embeddings
# - Does not modify Qdrant

from copy import deepcopy
import re


# ---------------------------------------------------------------------
# 1. Validate Cell 21A outputs
# ---------------------------------------------------------------------

required_names = [
    "ORCHESTRATOR_DOCUMENT_CATALOG",
    "ORCHESTRATOR_ENTITY_REGISTRY",
    "ORCHESTRATOR_MIXED_CORPUS_READY",
]

missing_names = [
    name
    for name in required_names
    if name not in globals()
]

if missing_names:
    raise RuntimeError(
        "Missing Cell 21A components: "
        + ", ".join(missing_names)
    )


if not ORCHESTRATOR_ENTITY_REGISTRY:
    raise RuntimeError(
        "The entity registry is empty."
    )


# ---------------------------------------------------------------------
# 2. Analyzer configuration
# ---------------------------------------------------------------------

COMPARISON_PATTERNS = [
    r"\bcompare\b",
    r"\bcomparison\b",
    r"\bversus\b",
    r"\bvs\.?\b",
    r"\bcompared\s+with\b",
    r"\bcompared\s+to\b",
    r"\bdifference\s+between\b",
    r"\bhow\s+does\b.*\bcompare\b",
]


METRIC_RULES = [
    {
        "canonical": "total_net_sales",
        "label": "total net sales",
        "patterns": [
            r"\btotal\s+net\s+sales\b",
            r"\bnet\s+sales\s+total\b",
        ],
        "quantitative": True,
    },
    {
        "canonical": "services_net_sales",
        "label": "services net sales",
        "patterns": [
            r"\bservices?\s+net\s+sales\b",
            r"\bnet\s+sales\s+from\s+services\b",
        ],
        "quantitative": True,
    },
    {
        "canonical": "iphone_net_sales",
        "label": "iPhone net sales",
        "patterns": [
            r"\biphone\s+net\s+sales\b",
            r"\bnet\s+sales\s+from\s+iphone\b",
        ],
        "quantitative": True,
    },
    {
        "canonical": "automotive_revenue",
        "label": "automotive revenue",
        "patterns": [
            r"\bautomotive\s+revenues?\b",
            r"\btotal\s+automotive\s+revenues?\b",
        ],
        "quantitative": True,
    },
    {
        "canonical": "energy_storage_revenue",
        "label": (
            "energy generation and storage revenue"
        ),
        "patterns": [
            (
                r"\benergy\s+generation\s+"
                r"(?:and|&)\s+storage\s+revenues?\b"
            ),
            (
                r"\benergy\s+storage\s+revenues?\b"
            ),
        ],
        "quantitative": True,
    },
    {
        "canonical": "gaap_operating_income",
        "label": "GAAP operating income",
        "patterns": [
            r"\bgaap\s+operating\s+income\b",
        ],
        "quantitative": True,
    },
    {
        "canonical": "operating_income",
        "label": "operating income",
        "patterns": [
            r"\boperating\s+income\b",
        ],
        "quantitative": True,
    },
    {
        "canonical": "total_revenue",
        "label": "total revenue",
        "patterns": [
            r"\btotal\s+revenues?\b",
        ],
        "quantitative": True,
    },
    {
        "canonical": "net_sales",
        "label": "net sales",
        "patterns": [
            r"\bnet\s+sales\b",
        ],
        "quantitative": True,
    },
    {
        "canonical": "revenue",
        "label": "revenue",
        "patterns": [
            r"\brevenues?\b",
        ],
        "quantitative": True,
    },
]


DOCUMENT_SCOPE_PATTERNS = [
    r"\baccording\s+to\s+the\s+documents?\b",
    r"\bin\s+the\s+documents?\b",
    r"\bin\s+the\s+reports?\b",
    r"\bin\s+the\s+files?\b",
    r"\bsummarize\b",
    r"\bdocuments?\s+say\b",
    r"\breports?\s+say\b",
    r"\bacross\s+the\s+documents?\b",
]


# ---------------------------------------------------------------------
# 3. Normalization helpers
# ---------------------------------------------------------------------

def normalize_query_text(text):
    normalized = str(
        text or ""
    )

    normalized = (
        normalized
        .replace("’", "'")
        .replace("‘", "'")
        .replace("“", '"')
        .replace("”", '"')
    )

    normalized = normalized.lower()

    normalized = re.sub(
        r"\bfy\s*[- ]?(\d{4})\b",
        r"fiscal year \1",
        normalized,
    )

    normalized = re.sub(
        r"\bfiscal\s+(\d{4})\b",
        r"fiscal year \1",
        normalized,
    )

    normalized = re.sub(
        r"\bq\s*([1-4])\s*[- ]?(\d{4})\b",
        r"q\1 \2",
        normalized,
    )

    normalized = re.sub(
        r"\s+",
        " ",
        normalized,
    )

    return normalized.strip()


# ---------------------------------------------------------------------
# 4. Entity alias index
# ---------------------------------------------------------------------

def build_entity_alias_index(
    entity_registry,
):
    alias_records = []

    for entity, metadata in (
        entity_registry.items()
    ):
        aliases = set(
            metadata.get(
                "aliases",
                [],
            )
        )

        aliases.add(
            entity.lower()
        )

        for alias in aliases:
            normalized_alias = (
                normalize_query_text(
                    alias
                )
            )

            if not normalized_alias:
                continue

            alias_records.append(
                {
                    "entity": entity,
                    "alias": normalized_alias,
                }
            )

    # Prefer longer aliases such as "apple inc"
    # before shorter aliases such as "apple".
    alias_records.sort(
        key=lambda item: len(
            item["alias"]
        ),
        reverse=True,
    )

    return alias_records


ORCHESTRATOR_ENTITY_ALIAS_INDEX = (
    build_entity_alias_index(
        ORCHESTRATOR_ENTITY_REGISTRY
    )
)


def detect_entity_mentions(
    normalized_question,
):
    mentions = []
    occupied_ranges = []

    for alias_record in (
        ORCHESTRATOR_ENTITY_ALIAS_INDEX
    ):
        entity = alias_record[
            "entity"
        ]

        alias = alias_record[
            "alias"
        ]

        alias_pattern = (
            r"(?<![a-z0-9])"
            + re.escape(alias)
            + r"(?:'s)?"
            + r"(?![a-z0-9])"
        )

        for match in re.finditer(
            alias_pattern,
            normalized_question,
        ):
            start, end = match.span()

            overlaps_existing = any(
                start < occupied_end
                and end > occupied_start
                for (
                    occupied_start,
                    occupied_end,
                )
                in occupied_ranges
            )

            if overlaps_existing:
                continue

            mentions.append(
                {
                    "entity": entity,
                    "alias": alias,
                    "start": start,
                    "end": end,
                    "matched_text": (
                        match.group(0)
                    ),
                }
            )

            occupied_ranges.append(
                (start, end)
            )

    mentions.sort(
        key=lambda item: (
            item["start"],
            item["end"],
        )
    )

    return mentions


# ---------------------------------------------------------------------
# 5. Metric extraction
# ---------------------------------------------------------------------

def detect_metrics(text):
    normalized_text = (
        normalize_query_text(
            text
        )
    )

    detected_metrics = []
    occupied_spans = []

    for rule in METRIC_RULES:
        for pattern in rule[
            "patterns"
        ]:
            match = re.search(
                pattern,
                normalized_text,
            )

            if not match:
                continue

            start, end = match.span()

            overlaps_existing = any(
                start < existing_end
                and end > existing_start
                for (
                    existing_start,
                    existing_end,
                )
                in occupied_spans
            )

            if overlaps_existing:
                continue

            detected_metrics.append(
                {
                    "canonical": (
                        rule["canonical"]
                    ),
                    "label": rule[
                        "label"
                    ],
                    "quantitative": (
                        rule["quantitative"]
                    ),
                    "matched_text": (
                        match.group(0)
                    ),
                    "start": start,
                    "end": end,
                }
            )

            occupied_spans.append(
                (start, end)
            )

            break

    detected_metrics.sort(
        key=lambda item: (
            item["start"],
            item["end"],
        )
    )

    return detected_metrics


# ---------------------------------------------------------------------
# 6. Period extraction
# ---------------------------------------------------------------------

def detect_periods(text):
    normalized_text = (
        normalize_query_text(
            text
        )
    )

    periods = []
    occupied_spans = []

    period_patterns = [
        {
            "kind": "quarter",
            "pattern": (
                r"\bq([1-4])\s+"
                r"((?:19|20)\d{2})\b"
            ),
        },
        {
            "kind": "fiscal_year",
            "pattern": (
                r"\bfiscal\s+year\s+"
                r"((?:19|20)\d{2})\b"
            ),
        },
        {
            "kind": "calendar_year",
            "pattern": (
                r"\bcalendar\s+year\s+"
                r"((?:19|20)\d{2})\b"
            ),
        },
        {
            "kind": "year_unspecified",
            "pattern": (
                r"\b((?:19|20)\d{2})\b"
            ),
        },
    ]

    for period_rule in period_patterns:
        for match in re.finditer(
            period_rule["pattern"],
            normalized_text,
        ):
            start, end = match.span()

            overlaps_existing = any(
                start < existing_end
                and end > existing_start
                for (
                    existing_start,
                    existing_end,
                )
                in occupied_spans
            )

            if overlaps_existing:
                continue

            kind = period_rule[
                "kind"
            ]

            if kind == "quarter":
                quarter = int(
                    match.group(1)
                )

                year = int(
                    match.group(2)
                )

                label = (
                    f"Q{quarter} {year}"
                )

            else:
                quarter = None

                year = int(
                    match.group(1)
                )

                if kind == "fiscal_year":
                    label = (
                        f"fiscal year {year}"
                    )

                elif kind == "calendar_year":
                    label = (
                        f"calendar year {year}"
                    )

                else:
                    label = str(year)

            periods.append(
                {
                    "kind": kind,
                    "year": year,
                    "quarter": quarter,
                    "label": label,
                    "matched_text": (
                        match.group(0)
                    ),
                    "start": start,
                    "end": end,
                }
            )

            occupied_spans.append(
                (start, end)
            )

    periods.sort(
        key=lambda item: (
            item["start"],
            item["end"],
        )
    )

    return periods


# ---------------------------------------------------------------------
# 7. Comparison and document-scope detection
# ---------------------------------------------------------------------

def has_comparison_intent(
    normalized_question,
):
    return any(
        re.search(
            pattern,
            normalized_question,
        )
        for pattern
        in COMPARISON_PATTERNS
    )


def has_document_scope_intent(
    normalized_question,
):
    return any(
        re.search(
            pattern,
            normalized_question,
        )
        for pattern
        in DOCUMENT_SCOPE_PATTERNS
    )


# ---------------------------------------------------------------------
# 8. Entity-specific segment extraction
# ---------------------------------------------------------------------

def build_entity_segments(
    normalized_question,
    entity_mentions,
):
    """
    Split a multi-entity question into entity-specific spans.

    Example:
    Compare Apple's fiscal 2025 net sales
    with Tesla's Q4 2025 revenue.

    Apple segment:
    compare apple's fiscal year 2025 net sales with

    Tesla segment:
    tesla's q4 2025 revenue
    """
    if not entity_mentions:
        return []

    unique_mentions = []

    seen_entities = set()

    for mention in entity_mentions:
        if mention["entity"] in seen_entities:
            continue

        seen_entities.add(
            mention["entity"]
        )

        unique_mentions.append(
            mention
        )

    segments = []

    for mention_index, mention in enumerate(
        unique_mentions
    ):
        if mention_index == 0:
            segment_start = 0
        else:
            segment_start = mention[
                "start"
            ]

        if (
            mention_index + 1
            < len(unique_mentions)
        ):
            segment_end = unique_mentions[
                mention_index + 1
            ]["start"]
        else:
            segment_end = len(
                normalized_question
            )

        segment_text = (
            normalized_question[
                segment_start:segment_end
            ]
            .strip(
                " ,.;:-"
            )
        )

        segments.append(
            {
                "entity": mention[
                    "entity"
                ],
                "segment": segment_text,
            }
        )

    return segments


# ---------------------------------------------------------------------
# 9. Clarification generator
# ---------------------------------------------------------------------

def build_clarification_message(
    entities,
    metrics,
    periods,
    reason,
):
    entity_list = sorted(
        ORCHESTRATOR_ENTITY_REGISTRY
        .keys()
    )

    entity_text = " or ".join(
        entity_list
    )

    metric_label = (
        metrics[0]["label"]
        if metrics
        else "the requested information"
    )

    if reason == "missing_entity":
        return (
            f"Which company do you mean: "
            f"{entity_text}? Please also specify "
            f"the reporting period for "
            f"{metric_label}."
        )

    if reason == "multiple_entities_without_comparison":
        return (
            "Your question refers to multiple companies. "
            "Should I compare them, or answer for only "
            "one company?"
        )

    if reason == "unspecified_period_scope":
        return (
            "Please specify whether you mean a fiscal "
            "year, calendar year, or a particular quarter."
        )

    return (
        "Please specify the company, metric, "
        "and reporting period."
    )


# ---------------------------------------------------------------------
# 10. Main Query Analyzer V1
# ---------------------------------------------------------------------

def analyze_query_v1(question):
    original_question = str(
        question or ""
    ).strip()

    if not original_question:
        raise ValueError(
            "Question cannot be empty."
        )

    normalized_question = (
        normalize_query_text(
            original_question
        )
    )

    entity_mentions = (
        detect_entity_mentions(
            normalized_question
        )
    )

    entities = []

    for mention in entity_mentions:
        if mention["entity"] not in entities:
            entities.append(
                mention["entity"]
            )

    metrics = detect_metrics(
        normalized_question
    )

    periods = detect_periods(
        normalized_question
    )

    comparison_intent = (
        has_comparison_intent(
            normalized_question
        )
    )

    document_scope_intent = (
        has_document_scope_intent(
            normalized_question
        )
    )

    corpus_entities = sorted(
        ORCHESTRATOR_ENTITY_REGISTRY
        .keys()
    )

    query_type = None
    needs_clarification = False
    clarification_reason = None
    clarification_message = None
    subqueries = []

    # -------------------------------------------------------------
    # A. Comparison query
    # -------------------------------------------------------------

    if (
        comparison_intent
        and len(entities) >= 2
    ):
        query_type = "comparison"

        entity_segments = (
            build_entity_segments(
                normalized_question,
                entity_mentions,
            )
        )

        for segment_entry in (
            entity_segments
        ):
            segment_metrics = (
                detect_metrics(
                    segment_entry[
                        "segment"
                    ]
                )
            )

            segment_periods = (
                detect_periods(
                    segment_entry[
                        "segment"
                    ]
                )
            )

            subqueries.append(
                {
                    "entity": (
                        segment_entry[
                            "entity"
                        ]
                    ),
                    "metric": (
                        deepcopy(
                            segment_metrics[0]
                        )
                        if segment_metrics
                        else None
                    ),
                    "period": (
                        deepcopy(
                            segment_periods[0]
                        )
                        if segment_periods
                        else None
                    ),
                    "segment": (
                        segment_entry[
                            "segment"
                        ]
                    ),
                    "retrieval_query": (
                        segment_entry[
                            "segment"
                        ]
                    ),
                }
            )

    # -------------------------------------------------------------
    # B. Multiple named entities without comparison intent
    # -------------------------------------------------------------

    elif len(entities) >= 2:
        query_type = "ambiguous"
        needs_clarification = True

        clarification_reason = (
            "multiple_entities_without_comparison"
        )

        clarification_message = (
            build_clarification_message(
                entities=entities,
                metrics=metrics,
                periods=periods,
                reason=clarification_reason,
            )
        )

    # -------------------------------------------------------------
    # C. One explicit entity
    # -------------------------------------------------------------

    elif len(entities) == 1:
        query_type = "single_entity"

        subqueries.append(
            {
                "entity": entities[0],
                "metric": (
                    deepcopy(metrics[0])
                    if metrics
                    else None
                ),
                "period": (
                    deepcopy(periods[0])
                    if periods
                    else None
                ),
                "segment": (
                    normalized_question
                ),
                "retrieval_query": (
                    normalized_question
                ),
            }
        )

    # -------------------------------------------------------------
    # D. No entity in a mixed tenant, but financial/document metric
    # -------------------------------------------------------------

    elif (
        len(corpus_entities) > 1
        and (
            metrics
            or periods
        )
    ):
        query_type = "ambiguous"
        needs_clarification = True
        clarification_reason = (
            "missing_entity"
        )

        clarification_message = (
            build_clarification_message(
                entities=entities,
                metrics=metrics,
                periods=periods,
                reason=clarification_reason,
            )
        )

    # -------------------------------------------------------------
    # E. Generic document-wide question
    # -------------------------------------------------------------

    elif document_scope_intent:
        query_type = "corpus_general"

    # -------------------------------------------------------------
    # F. Likely unrelated / unsupported question
    # -------------------------------------------------------------

    else:
        query_type = (
            "no_answer_candidate"
        )

    # -------------------------------------------------------------
    # Detect comparison period-scope mismatch
    # -------------------------------------------------------------

    period_scope_mismatch = False
    period_scope_warning = None

    if query_type == "comparison":
        comparison_periods = [
            subquery["period"]
            for subquery in subqueries
            if subquery[
                "period"
            ] is not None
        ]

        period_kinds = {
            period["kind"]
            for period
            in comparison_periods
        }

        if (
            "quarter" in period_kinds
            and (
                "fiscal_year"
                in period_kinds
                or "calendar_year"
                in period_kinds
            )
        ):
            period_scope_mismatch = True

            period_scope_warning = (
                "The comparison mixes an annual figure "
                "with a quarterly figure, so the values "
                "are not directly period-equivalent."
            )

    analysis = {
        "original_question": (
            original_question
        ),
        "normalized_question": (
            normalized_question
        ),
        "query_type": query_type,
        "entities": entities,
        "entity_mentions": (
            entity_mentions
        ),
        "metrics": metrics,
        "periods": periods,
        "comparison_intent": (
            comparison_intent
        ),
        "document_scope_intent": (
            document_scope_intent
        ),
        "needs_clarification": (
            needs_clarification
        ),
        "clarification_reason": (
            clarification_reason
        ),
        "clarification_message": (
            clarification_message
        ),
        "subqueries": subqueries,
        "period_scope_mismatch": (
            period_scope_mismatch
        ),
        "period_scope_warning": (
            period_scope_warning
        ),
        "available_corpus_entities": (
            corpus_entities
        ),
    }

    return analysis


# ---------------------------------------------------------------------
# 11. Human-readable diagnostic output
# ---------------------------------------------------------------------

def print_query_analysis(
    analysis,
):
    print("\n" + "=" * 92)

    print(
        f"❓ Question: "
        f"{analysis['original_question']}"
    )

    print(
        f"🧭 Query type: "
        f"{analysis['query_type']}"
    )

    print(
        f"🏷️ Entities: "
        f"{analysis['entities']}"
    )

    metric_labels = [
        metric["label"]
        for metric
        in analysis["metrics"]
    ]

    print(
        f"📊 Metrics: "
        f"{metric_labels}"
    )

    period_labels = [
        period["label"]
        for period
        in analysis["periods"]
    ]

    print(
        f"📅 Periods: "
        f"{period_labels}"
    )

    print(
        f"⚖️ Comparison intent: "
        f"{analysis['comparison_intent']}"
    )

    if analysis[
        "subqueries"
    ]:
        print(
            "🔎 Planned subqueries:"
        )

        for index, subquery in enumerate(
            analysis["subqueries"],
            start=1,
        ):
            metric_label = (
                subquery["metric"][
                    "label"
                ]
                if subquery["metric"]
                else "not detected"
            )

            period_label = (
                subquery["period"][
                    "label"
                ]
                if subquery["period"]
                else "not detected"
            )

            print(
                f"   {index}. "
                f"Entity={subquery['entity']} | "
                f"Metric={metric_label} | "
                f"Period={period_label}"
            )

            print(
                f"      Retrieval query: "
                f"{subquery['retrieval_query']}"
            )

    if analysis[
        "period_scope_mismatch"
    ]:
        print(
            "⚠️ Period warning: "
            + analysis[
                "period_scope_warning"
            ]
        )

    if analysis[
        "needs_clarification"
    ]:
        print(
            "💬 Clarification: "
            + analysis[
                "clarification_message"
            ]
        )


# ---------------------------------------------------------------------
# 12. Regression fixture
# ---------------------------------------------------------------------

QUERY_ANALYZER_TEST_CASES = [
    {
        "question": (
            "What were Apple's total net sales "
            "in fiscal year 2025?"
        ),
        "expected_type": (
            "single_entity"
        ),
        "expected_entities": [
            "Apple"
        ],
    },
    {
        "question": (
            "What was Tesla's total revenue "
            "in Q4 2025?"
        ),
        "expected_type": (
            "single_entity"
        ),
        "expected_entities": [
            "Tesla"
        ],
    },
    {
        "question": (
            "Compare Apple's fiscal 2025 net sales "
            "with Tesla's Q4 2025 revenue."
        ),
        "expected_type": (
            "comparison"
        ),
        "expected_entities": [
            "Apple",
            "Tesla",
        ],
    },
    {
        "question": (
            "What was the total revenue in 2025?"
        ),
        "expected_type": (
            "ambiguous"
        ),
        "expected_entities": [],
    },
    {
        "question": (
            "Who wrote Pride and Prejudice?"
        ),
        "expected_type": (
            "no_answer_candidate"
        ),
        "expected_entities": [],
    },
]


QUERY_ANALYZER_TEST_RESULTS = []

print(
    "🧠 Running Query Analyzer V1 "
    "regression tests..."
)

for test_case in (
    QUERY_ANALYZER_TEST_CASES
):
    analysis = analyze_query_v1(
        test_case["question"]
    )

    type_passed = (
        analysis["query_type"]
        == test_case[
            "expected_type"
        ]
    )

    entities_passed = (
        analysis["entities"]
        == test_case[
            "expected_entities"
        ]
    )

    passed = (
        type_passed
        and entities_passed
    )

    QUERY_ANALYZER_TEST_RESULTS.append(
        {
            "question": (
                test_case["question"]
            ),
            "passed": passed,
            "analysis": analysis,
        }
    )

    print_query_analysis(
        analysis
    )

    print(
        f"\n{'✅ PASS' if passed else '❌ FAIL'}"
    )


QUERY_ANALYZER_V1_PASSED = all(
    result["passed"]
    for result
    in QUERY_ANALYZER_TEST_RESULTS
)


print("\n" + "=" * 92)
print("📊 CELL 21B SUMMARY")

passed_count = sum(
    result["passed"]
    for result
    in QUERY_ANALYZER_TEST_RESULTS
)

total_count = len(
    QUERY_ANALYZER_TEST_RESULTS
)

print(
    f"Analyzer tests passed: "
    f"{passed_count}/{total_count}"
)

if QUERY_ANALYZER_V1_PASSED:
    print(
        "✅ Query Analyzer V1 passed "
        "all regression tests."
    )

    print(
        "➡️ Ready for entity-routed "
        "hybrid retrieval in Cell 21C."
    )

else:
    print(
        "❌ Query Analyzer V1 requires "
        "adjustment before retrieval integration."
    )


print(
    "\n⏭️ No Groq request was made."
)

print(
    "⏭️ No embeddings were generated."
)

print(
    "⏭️ No Qdrant points were created, "
    "updated, or deleted."
)

🧠 Running Query Analyzer V1 regression tests...

❓ Question: What were Apple's total net sales in fiscal year 2025?
🧭 Query type: single_entity
🏷️ Entities: ['Apple']
📊 Metrics: ['total net sales']
📅 Periods: ['fiscal year 2025']
⚖️ Comparison intent: False
🔎 Planned subqueries:
   1. Entity=Apple | Metric=total net sales | Period=fiscal year 2025
      Retrieval query: what were apple's total net sales in fiscal year 2025?

✅ PASS

❓ Question: What was Tesla's total revenue in Q4 2025?
🧭 Query type: single_entity
🏷️ Entities: ['Tesla']
📊 Metrics: ['total revenue']
📅 Periods: ['Q4 2025']
⚖️ Comparison intent: False
🔎 Planned subqueries:
   1. Entity=Tesla | Metric=total revenue | Period=Q4 2025
      Retrieval query: what was tesla's total revenue in q4 2025?

✅ PASS

❓ Question: Compare Apple's fiscal 2025 net sales with Tesla's Q4 2025 revenue.
🧭 Query type: comparison
🏷️ Entities: ['Apple', 'Tesla']
📊 Metrics: ['net sales', 'revenue']
📅 Periods: ['fiscal year 2025', 'Q4 2025']
⚖️ Co

In [41]:
# VAULTIFY COMPACT - Cell 21C:
# Entity-Routed Hybrid Retrieval V1
#
# Uses:
# - Cell 21A tenant document catalog
# - Cell 21B Query Analyzer
# - Cell 19B hybrid retrieval functions
#
# READ-ONLY:
# - Builds in-memory embeddings and BM25 indexes
# - Routes each subquery only to the matching entity documents
# - Does not call Groq
# - Does not create, update, or delete Qdrant points

from collections import defaultdict
import re

import numpy as np


# ---------------------------------------------------------------------
# 1. Validate runtime dependencies
# ---------------------------------------------------------------------

required_names = [
    "ORCHESTRATOR_DOCUMENT_CATALOG",
    "ORCHESTRATOR_ENTITY_REGISTRY",
    "ORCHESTRATOR_TENANT_CHUNKS",
    "analyze_query_v1",
    "embedding_model",
    "build_bm25_index",
    "retrieve_hybrid",
]

missing_names = [
    name
    for name in required_names
    if name not in globals()
]

if missing_names:
    raise RuntimeError(
        "Missing required components: "
        + ", ".join(missing_names)
        + ". Run Cells 19B, 21A, and 21B first."
    )


if not QUERY_ANALYZER_V1_PASSED:
    raise RuntimeError(
        "Query Analyzer V1 did not pass its regression gate."
    )


# ---------------------------------------------------------------------
# 2. Generic metric query expansions
# ---------------------------------------------------------------------

ORCHESTRATOR_METRIC_EXPANSIONS = {
    "total_net_sales": [
        "total net sales",
        "net sales",
    ],
    "services_net_sales": [
        "services net sales",
        "services",
    ],
    "iphone_net_sales": [
        "iPhone net sales",
        "iPhone",
    ],
    "net_sales": [
        "net sales",
    ],
    "total_revenue": [
        "total revenue",
        "total revenues",
        "revenue",
        "revenues",
    ],
    "revenue": [
        "revenue",
        "revenues",
        "total revenue",
        "total revenues",
    ],
    "automotive_revenue": [
        "automotive revenue",
        "automotive revenues",
        "total automotive revenues",
    ],
    "energy_storage_revenue": [
        "energy generation and storage revenue",
        "energy generation and storage revenues",
        "energy storage revenue",
    ],
    "gaap_operating_income": [
        "GAAP operating income",
        "operating income",
    ],
    "operating_income": [
        "operating income",
    ],
}


# ---------------------------------------------------------------------
# 3. Build clean retrieval queries from analyzer plans
# ---------------------------------------------------------------------

def build_routed_retrieval_query(
    subquery,
):
    """
    Convert a Query Analyzer subquery into a concise,
    entity-specific retrieval query.

    This removes comparison wording such as "compare" and "with"
    so that each entity is searched independently.
    """
    entity = str(
        subquery.get(
            "entity",
            "",
        )
    ).strip()

    metric = subquery.get(
        "metric"
    )

    period = subquery.get(
        "period"
    )

    query_parts = []

    if entity:
        query_parts.append(
            entity
        )

    if period:
        period_label = str(
            period.get(
                "label",
                "",
            )
        ).strip()

        if period_label:
            query_parts.append(
                period_label
            )

    if metric:
        metric_name = metric.get(
            "canonical"
        )

        expansions = (
            ORCHESTRATOR_METRIC_EXPANSIONS.get(
                metric_name,
                [
                    metric.get(
                        "label",
                        "",
                    )
                ],
            )
        )

        query_parts.extend(
            expansion
            for expansion in expansions
            if expansion
        )

    else:
        segment = str(
            subquery.get(
                "segment",
                "",
            )
        ).strip()

        segment = re.sub(
            r"^\s*compare\s+",
            "",
            segment,
            flags=re.IGNORECASE,
        )

        segment = re.sub(
            r"\s+with\s*$",
            "",
            segment,
            flags=re.IGNORECASE,
        )

        if segment:
            query_parts.append(
                segment
            )

    retrieval_query = " ".join(
        query_parts
    )

    retrieval_query = re.sub(
        r"\s+",
        " ",
        retrieval_query,
    ).strip()

    return retrieval_query


# ---------------------------------------------------------------------
# 4. Collect chunks belonging to one entity
# ---------------------------------------------------------------------

def collect_entity_chunks(
    entity,
):
    """
    Collect all unique chunks belonging to documents registered
    under a particular entity.
    """
    registry_entry = (
        ORCHESTRATOR_ENTITY_REGISTRY.get(
            entity
        )
    )

    if not registry_entry:
        raise KeyError(
            f"Unknown entity: {entity}"
        )

    document_keys = registry_entry.get(
        "document_keys",
        [],
    )

    entity_chunks = []
    seen_point_ids = set()

    for document_key in document_keys:
        document = (
            ORCHESTRATOR_DOCUMENT_CATALOG.get(
                document_key
            )
        )

        if not document:
            continue

        for chunk in document[
            "chunks"
        ]:
            point_id = str(
                chunk.get(
                    "point_id",
                    "",
                )
            )

            if (
                point_id
                and point_id in seen_point_ids
            ):
                continue

            if point_id:
                seen_point_ids.add(
                    point_id
                )

            normalized_chunk = {
                "point_id": point_id,
                "tenant_id": chunk.get(
                    "tenant_id"
                ),
                "filename": chunk.get(
                    "filename",
                    document["filename"],
                ),
                "document_hash": chunk.get(
                    "document_hash",
                    document.get(
                        "document_hash"
                    ),
                ),
                "chunk_index": chunk.get(
                    "chunk_index"
                ),
                "chunk_type": chunk.get(
                    "chunk_type",
                    "unknown",
                ),
                "section": chunk.get(
                    "section",
                    "Unknown section",
                ),
                "text": chunk[
                    "text"
                ],
                "entity": entity,
            }

            entity_chunks.append(
                normalized_chunk
            )

    entity_chunks.sort(
        key=lambda chunk: (
            chunk["filename"],
            (
                chunk["chunk_index"]
                if isinstance(
                    chunk["chunk_index"],
                    int,
                )
                else 0
            ),
            chunk["point_id"],
        )
    )

    if not entity_chunks:
        raise RuntimeError(
            f"No usable chunks were found "
            f"for entity '{entity}'."
        )

    return entity_chunks


# ---------------------------------------------------------------------
# 5. Prepare one hybrid retrieval index per entity
# ---------------------------------------------------------------------

def prepare_entity_retrieval_index(
    entity,
):
    chunks = collect_entity_chunks(
        entity
    )

    print(
        f"⏳ Preparing {entity} retrieval index: "
        f"{len(chunks)} chunks"
    )

    chunk_texts = [
        chunk["text"]
        for chunk in chunks
    ]

    embeddings = embedding_model.encode(
        chunk_texts,
        batch_size=64,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=True,
    )

    if len(embeddings) != len(chunks):
        raise RuntimeError(
            f"Embedding count mismatch "
            f"for entity '{entity}'."
        )

    retrieval_result = {
        "entity": entity,
        "chunks": chunks,
        "embeddings": embeddings,
    }

    retrieval_result[
        "bm25_index"
    ] = build_bm25_index(
        chunks
    )

    return retrieval_result


ORCHESTRATOR_ENTITY_RETRIEVAL_INDEXES = {}


print(
    "🧭 Building entity-routed "
    "hybrid retrieval indexes..."
)

for entity in sorted(
    ORCHESTRATOR_ENTITY_REGISTRY.keys()
):
    ORCHESTRATOR_ENTITY_RETRIEVAL_INDEXES[
        entity
    ] = prepare_entity_retrieval_index(
        entity
    )


print(
    "\n✅ Entity retrieval indexes are ready."
)


# ---------------------------------------------------------------------
# 6. Retrieve from exactly one entity
# ---------------------------------------------------------------------

def retrieve_for_entity_v1(
    entity,
    retrieval_query,
    top_k=6,
):
    """
    Run hybrid retrieval only against documents belonging
    to the requested entity.
    """
    retrieval_index = (
        ORCHESTRATOR_ENTITY_RETRIEVAL_INDEXES.get(
            entity
        )
    )

    if not retrieval_index:
        raise KeyError(
            f"No retrieval index exists "
            f"for entity '{entity}'."
        )

    retrieved = retrieve_hybrid(
        result=retrieval_index,
        question=retrieval_query,
        top_k=top_k,
    )

    enriched_results = []

    for item in retrieved:
        chunk_index = item[
            "chunk_index"
        ]

        source_chunk = retrieval_index[
            "chunks"
        ][chunk_index]

        enriched_item = dict(
            item
        )

        enriched_item.update(
            {
                "entity": entity,
                "filename": source_chunk.get(
                    "filename",
                    "unknown_document",
                ),
                "document_hash": (
                    source_chunk.get(
                        "document_hash"
                    )
                ),
                "point_id": source_chunk.get(
                    "point_id"
                ),
                "source_chunk_index": (
                    source_chunk.get(
                        "chunk_index"
                    )
                ),
            }
        )

        enriched_results.append(
            enriched_item
        )

    return enriched_results


# ---------------------------------------------------------------------
# 7. Route an analyzed question
# ---------------------------------------------------------------------

def route_query_v1(
    question,
    top_k_per_entity=6,
):
    """
    Analyze the question and select the correct retrieval strategy.

    Behaviors:
    - single_entity: retrieve only from that entity
    - comparison: retrieve separately for every entity
    - ambiguous: return clarification without retrieval
    - no_answer_candidate: skip retrieval
    """
    analysis = analyze_query_v1(
        question
    )

    query_type = analysis[
        "query_type"
    ]

    routed_result = {
        "question": question,
        "analysis": analysis,
        "status": None,
        "routes": [],
        "clarification": None,
    }

    if query_type == "ambiguous":
        routed_result[
            "status"
        ] = "clarification_required"

        routed_result[
            "clarification"
        ] = analysis[
            "clarification_message"
        ]

        return routed_result

    if query_type == "no_answer_candidate":
        routed_result[
            "status"
        ] = "no_answer_candidate"

        return routed_result

    if query_type == "corpus_general":
        routed_result[
            "status"
        ] = "corpus_general_pending"

        return routed_result

    if query_type not in {
        "single_entity",
        "comparison",
    }:
        routed_result[
            "status"
        ] = "unsupported_query_type"

        return routed_result

    for subquery in analysis[
        "subqueries"
    ]:
        entity = subquery[
            "entity"
        ]

        retrieval_query = (
            build_routed_retrieval_query(
                subquery
            )
        )

        retrieved = retrieve_for_entity_v1(
            entity=entity,
            retrieval_query=(
                retrieval_query
            ),
            top_k=top_k_per_entity,
        )

        routed_result[
            "routes"
        ].append(
            {
                "entity": entity,
                "metric": subquery.get(
                    "metric"
                ),
                "period": subquery.get(
                    "period"
                ),
                "retrieval_query": (
                    retrieval_query
                ),
                "results": retrieved,
            }
        )

    if query_type == "comparison":
        routed_result[
            "status"
        ] = "comparison_retrieved"

    else:
        routed_result[
            "status"
        ] = "single_entity_retrieved"

    return routed_result


# ---------------------------------------------------------------------
# 8. Evidence-presence diagnostics
# ---------------------------------------------------------------------

def normalize_retrieval_evidence(
    text,
):
    normalized = str(
        text or ""
    ).lower()

    normalized = re.sub(
        r"\s+",
        " ",
        normalized,
    ).strip()

    compact = re.sub(
        r"[^a-z0-9]+",
        "",
        normalized,
    )

    return normalized, compact


def retrieval_context_contains_groups(
    retrieved_results,
    expected_groups,
):
    combined_context = " ".join(
        item["text"]
        for item in retrieved_results
    )

    (
        normalized_context,
        compact_context,
    ) = normalize_retrieval_evidence(
        combined_context
    )

    group_results = []

    for alternatives in expected_groups:
        matched = None

        for alternative in alternatives:
            (
                normalized_alternative,
                compact_alternative,
            ) = normalize_retrieval_evidence(
                alternative
            )

            if (
                normalized_alternative
                in normalized_context
                or (
                    compact_alternative
                    and compact_alternative
                    in compact_context
                )
            ):
                matched = alternative
                break

        group_results.append(
            {
                "alternatives": alternatives,
                "matched": matched,
            }
        )

    passed = all(
        group["matched"] is not None
        for group in group_results
    )

    return passed, group_results


# ---------------------------------------------------------------------
# 9. Human-readable routed retrieval output
# ---------------------------------------------------------------------

def print_routed_retrieval(
    routed_result,
):
    print("\n" + "=" * 96)

    print(
        f"❓ Question: "
        f"{routed_result['question']}"
    )

    print(
        f"🧭 Query type: "
        f"{routed_result['analysis']['query_type']}"
    )

    print(
        f"📍 Routing status: "
        f"{routed_result['status']}"
    )

    if routed_result[
        "clarification"
    ]:
        print(
            f"💬 Clarification: "
            f"{routed_result['clarification']}"
        )

    for route_number, route in enumerate(
        routed_result["routes"],
        start=1,
    ):
        print("\n" + "-" * 96)

        print(
            f"Route {route_number}"
        )

        print(
            f"   Entity: "
            f"{route['entity']}"
        )

        print(
            f"   Retrieval query: "
            f"{route['retrieval_query']}"
        )

        print(
            f"   Retrieved results: "
            f"{len(route['results'])}"
        )

        for item in route[
            "results"
        ][:3]:
            preview = " ".join(
                item["text"].split()
            )[:220]

            print(
                f"\n   #{item['rank']} "
                f"hybrid={item['hybrid_score']:.4f} "
                f"dense_rank={item['dense_rank']} "
                f"bm25_rank={item['lexical_rank']}"
            )

            print(
                f"      File: "
                f"{item['filename']}"
            )

            print(
                f"      Section: "
                f"{item['section']}"
            )

            print(
                f"      Type: "
                f"{item['chunk_type']}"
            )

            print(
                f"      {preview}..."
            )


# ---------------------------------------------------------------------
# 10. Regression tests
# ---------------------------------------------------------------------

ROUTED_RETRIEVAL_TEST_CASES = [
    {
        "name": "Apple single-entity retrieval",
        "question": (
            "What were Apple's total net sales "
            "in fiscal year 2025?"
        ),
        "expected_status": (
            "single_entity_retrieved"
        ),
        "expected_routes": [
            {
                "entity": "Apple",
                "expected_groups": [
                    [
                        "416,161",
                        "416161",
                    ],
                    [
                        "total net sales",
                    ],
                ],
            }
        ],
    },
    {
        "name": "Tesla single-entity retrieval",
        "question": (
            "What was Tesla's total revenue "
            "in Q4 2025?"
        ),
        "expected_status": (
            "single_entity_retrieved"
        ),
        "expected_routes": [
            {
                "entity": "Tesla",
                "expected_groups": [
                    [
                        "24,901",
                        "24901",
                    ],
                    [
                        "total revenue",
                        "total revenues",
                    ],
                ],
            }
        ],
    },
    {
        "name": "Apple and Tesla comparison routing",
        "question": (
            "Compare Apple's fiscal 2025 net sales "
            "with Tesla's Q4 2025 revenue."
        ),
        "expected_status": (
            "comparison_retrieved"
        ),
        "expected_routes": [
            {
                "entity": "Apple",
                "expected_groups": [
                    [
                        "416,161",
                        "416161",
                    ],
                    [
                        "total net sales",
                        "net sales",
                    ],
                ],
            },
            {
                "entity": "Tesla",
                "expected_groups": [
                    [
                        "24,901",
                        "24901",
                    ],
                    [
                        "total revenue",
                        "total revenues",
                    ],
                ],
            },
        ],
    },
    {
        "name": "Ambiguous query",
        "question": (
            "What was the total revenue in 2025?"
        ),
        "expected_status": (
            "clarification_required"
        ),
        "expected_routes": [],
    },
    {
        "name": "Unrelated query",
        "question": (
            "Who wrote Pride and Prejudice?"
        ),
        "expected_status": (
            "no_answer_candidate"
        ),
        "expected_routes": [],
    },
]


ROUTED_RETRIEVAL_TEST_RESULTS = []


print(
    "\n🧪 Running Entity-Routed "
    "Hybrid Retrieval V1 tests..."
)


for test_case in (
    ROUTED_RETRIEVAL_TEST_CASES
):
    routed_result = route_query_v1(
        test_case["question"],
        top_k_per_entity=6,
    )

    print_routed_retrieval(
        routed_result
    )

    status_passed = (
        routed_result["status"]
        == test_case[
            "expected_status"
        ]
    )

    route_checks = []

    for expected_route in (
        test_case["expected_routes"]
    ):
        matching_route = next(
            (
                route
                for route
                in routed_result[
                    "routes"
                ]
                if route["entity"]
                == expected_route[
                    "entity"
                ]
            ),
            None,
        )

        if matching_route is None:
            route_checks.append(
                {
                    "entity": (
                        expected_route[
                            "entity"
                        ]
                    ),
                    "passed": False,
                    "groups": [],
                }
            )

            continue

        (
            evidence_passed,
            group_results,
        ) = retrieval_context_contains_groups(
            matching_route["results"],
            expected_route[
                "expected_groups"
            ],
        )

        route_checks.append(
            {
                "entity": (
                    expected_route[
                        "entity"
                    ]
                ),
                "passed": (
                    evidence_passed
                ),
                "groups": (
                    group_results
                ),
            }
        )

    routes_passed = all(
        check["passed"]
        for check in route_checks
    )

    if not test_case[
        "expected_routes"
    ]:
        routes_passed = (
            len(
                routed_result["routes"]
            )
            == 0
        )

    test_passed = (
        status_passed
        and routes_passed
    )

    ROUTED_RETRIEVAL_TEST_RESULTS.append(
        {
            "name": test_case[
                "name"
            ],
            "passed": test_passed,
            "status_passed": (
                status_passed
            ),
            "routes_passed": (
                routes_passed
            ),
            "route_checks": (
                route_checks
            ),
            "result": routed_result,
        }
    )

    print("\nEvidence checks:")

    if route_checks:
        for route_check in route_checks:
            print(
                f"   "
                f"{'✅' if route_check['passed'] else '❌'} "
                f"{route_check['entity']}"
            )

            for group in route_check[
                "groups"
            ]:
                print(
                    f"      "
                    f"{'✅' if group['matched'] else '❌'} "
                    f"{group['alternatives']}"
                    f" → {group['matched']}"
                )

    else:
        print(
            "   No retrieval route expected."
        )

    print(
        f"\n"
        f"{'✅ PASS' if test_passed else '❌ FAIL'} "
        f"— {test_case['name']}"
    )


# ---------------------------------------------------------------------
# 11. Final summary
# ---------------------------------------------------------------------

ROUTED_RETRIEVAL_V1_PASSED = all(
    result["passed"]
    for result
    in ROUTED_RETRIEVAL_TEST_RESULTS
)


passed_count = sum(
    result["passed"]
    for result
    in ROUTED_RETRIEVAL_TEST_RESULTS
)

total_count = len(
    ROUTED_RETRIEVAL_TEST_RESULTS
)


print("\n" + "=" * 96)
print("📊 CELL 21C SUMMARY")

print(
    f"Routed retrieval tests passed: "
    f"{passed_count}/{total_count}"
)

if ROUTED_RETRIEVAL_V1_PASSED:
    print(
        "✅ Entity-Routed Hybrid Retrieval V1 "
        "passed all regression tests."
    )

    print(
        "✅ Apple and Tesla evidence can now "
        "be retrieved independently."
    )

    print(
        "✅ Ambiguous questions stop before retrieval."
    )

    print(
        "➡️ Ready for formal Evidence Verification "
        "in Cell 21D."
    )

else:
    print(
        "❌ Entity-routed retrieval still has "
        "a regression."
    )

    print(
        "⛔ Do not continue to answer generation yet."
    )


print(
    "\n⏭️ No Groq request was made."
)

print(
    "⏭️ No Qdrant points were created, "
    "updated, or deleted."
)

🧭 Building entity-routed hybrid retrieval indexes...
⏳ Preparing Apple retrieval index: 609 chunks


Batches:   0%|          | 0/10 [00:00<?, ?it/s]

⏳ Preparing Tesla retrieval index: 136 chunks


Batches:   0%|          | 0/3 [00:00<?, ?it/s]


✅ Entity retrieval indexes are ready.

🧪 Running Entity-Routed Hybrid Retrieval V1 tests...

❓ Question: What were Apple's total net sales in fiscal year 2025?
🧭 Query type: single_entity
📍 Routing status: single_entity_retrieved

------------------------------------------------------------------------------------------------
Route 1
   Entity: Apple
   Retrieval query: Apple fiscal year 2025 total net sales net sales
   Retrieved results: 6

   #1 hybrid=0.0497 dense_rank=5 bm25_rank=6
      File: apple_fy2025_10k.pdf
      Section: Note 2 - Revenue
      Type: text
      (1) Services net sales include amortization of the deferred value of services bundled in the sales price of certain products. The Company's proportion of net sales by disaggregated revenue source was generally consistent...

   #2 hybrid=0.0495 dense_rank=2 bm25_rank=13
      File: apple_fy2025_10k.pdf
      Section: Note 2 - Revenue
      Type: table
      Section: Note 2 - Revenue Table columns: | | 2025 | 2024 | 

In [42]:
# VAULTIFY COMPACT - Cell 21C.1:
# Improve aggregate financial metric expansion and rerun
# Entity-Routed Hybrid Retrieval V1 regression tests.
#
# READ-ONLY:
# - Reuses existing Apple and Tesla retrieval indexes
# - Does not regenerate document embeddings
# - Does not call Groq
# - Does not modify Qdrant

# ---------------------------------------------------------------------
# 1. Patch generic company-level financial metric expansions
# ---------------------------------------------------------------------

ORCHESTRATOR_METRIC_EXPANSIONS[
    "net_sales"
] = [
    "total net sales",
    "net sales",
]

ORCHESTRATOR_METRIC_EXPANSIONS[
    "revenue"
] = [
    "total revenue",
    "total revenues",
    "revenue",
    "revenues",
]


print(
    "✅ Aggregate financial metric expansions updated."
)

print(
    "   net_sales → total net sales + net sales"
)

print(
    "   revenue → total revenue + total revenues "
    "+ revenue + revenues"
)


# ---------------------------------------------------------------------
# 2. Verify the corrected comparison plan
# ---------------------------------------------------------------------

comparison_question = (
    "Compare Apple's fiscal 2025 net sales "
    "with Tesla's Q4 2025 revenue."
)

comparison_analysis = analyze_query_v1(
    comparison_question
)

print("\nCorrected routed queries:")

for subquery in comparison_analysis[
    "subqueries"
]:
    corrected_query = (
        build_routed_retrieval_query(
            subquery
        )
    )

    print(
        f"   {subquery['entity']}: "
        f"{corrected_query}"
    )


# ---------------------------------------------------------------------
# 3. Rerun the complete Cell 21C regression suite
# ---------------------------------------------------------------------

ROUTED_RETRIEVAL_PATCHED_RESULTS = []

print(
    "\n🧪 Rerunning Entity-Routed Hybrid "
    "Retrieval V1 regression tests..."
)


for test_case in ROUTED_RETRIEVAL_TEST_CASES:
    routed_result = route_query_v1(
        test_case["question"],
        top_k_per_entity=6,
    )

    print_routed_retrieval(
        routed_result
    )

    status_passed = (
        routed_result["status"]
        == test_case["expected_status"]
    )

    route_checks = []

    for expected_route in test_case[
        "expected_routes"
    ]:
        matching_route = next(
            (
                route
                for route
                in routed_result["routes"]
                if route["entity"]
                == expected_route["entity"]
            ),
            None,
        )

        if matching_route is None:
            route_checks.append(
                {
                    "entity": expected_route[
                        "entity"
                    ],
                    "passed": False,
                    "groups": [],
                }
            )

            continue

        (
            evidence_passed,
            group_results,
        ) = retrieval_context_contains_groups(
            matching_route["results"],
            expected_route[
                "expected_groups"
            ],
        )

        route_checks.append(
            {
                "entity": expected_route[
                    "entity"
                ],
                "passed": evidence_passed,
                "groups": group_results,
            }
        )

    if test_case["expected_routes"]:
        routes_passed = all(
            check["passed"]
            for check in route_checks
        )
    else:
        routes_passed = (
            len(routed_result["routes"])
            == 0
        )

    test_passed = (
        status_passed
        and routes_passed
    )

    ROUTED_RETRIEVAL_PATCHED_RESULTS.append(
        {
            "name": test_case["name"],
            "passed": test_passed,
            "status_passed": status_passed,
            "routes_passed": routes_passed,
            "route_checks": route_checks,
            "result": routed_result,
        }
    )

    print("\nEvidence checks:")

    if route_checks:
        for route_check in route_checks:
            print(
                f"   "
                f"{'✅' if route_check['passed'] else '❌'} "
                f"{route_check['entity']}"
            )

            for group in route_check[
                "groups"
            ]:
                print(
                    f"      "
                    f"{'✅' if group['matched'] else '❌'} "
                    f"{group['alternatives']}"
                    f" → {group['matched']}"
                )

    else:
        print(
            "   No retrieval route expected."
        )

    print(
        f"\n"
        f"{'✅ PASS' if test_passed else '❌ FAIL'} "
        f"— {test_case['name']}"
    )


# ---------------------------------------------------------------------
# 4. Final approval gate
# ---------------------------------------------------------------------

CELL_21C_ROUTED_RETRIEVAL_APPROVED = all(
    result["passed"]
    for result
    in ROUTED_RETRIEVAL_PATCHED_RESULTS
)

passed_count = sum(
    result["passed"]
    for result
    in ROUTED_RETRIEVAL_PATCHED_RESULTS
)

total_count = len(
    ROUTED_RETRIEVAL_PATCHED_RESULTS
)


print("\n" + "=" * 96)
print("📊 CELL 21C.1 SUMMARY")

print(
    f"Patched routed retrieval tests passed: "
    f"{passed_count}/{total_count}"
)

if CELL_21C_ROUTED_RETRIEVAL_APPROVED:
    print(
        "✅ Entity-Routed Hybrid Retrieval V1 "
        "passed the complete regression suite."
    )

    print(
        "✅ Apple and Tesla evidence can be "
        "retrieved independently for comparisons."
    )

    print(
        "➡️ Ready for Evidence Verification "
        "in Cell 21D."
    )

else:
    print(
        "❌ At least one routed retrieval "
        "test still failed."
    )

    print(
        "⛔ Do not continue to answer "
        "generation yet."
    )


print(
    "\n⏭️ Existing entity embeddings were reused."
)

print(
    "⏭️ No Groq request was made."
)

print(
    "⏭️ No Qdrant points were created, "
    "updated, or deleted."
)

✅ Aggregate financial metric expansions updated.
   net_sales → total net sales + net sales
   revenue → total revenue + total revenues + revenue + revenues

Corrected routed queries:
   Apple: Apple fiscal year 2025 total net sales net sales
   Tesla: Tesla Q4 2025 total revenue total revenues revenue revenues

🧪 Rerunning Entity-Routed Hybrid Retrieval V1 regression tests...

❓ Question: What were Apple's total net sales in fiscal year 2025?
🧭 Query type: single_entity
📍 Routing status: single_entity_retrieved

------------------------------------------------------------------------------------------------
Route 1
   Entity: Apple
   Retrieval query: Apple fiscal year 2025 total net sales net sales
   Retrieved results: 6

   #1 hybrid=0.0497 dense_rank=5 bm25_rank=6
      File: apple_fy2025_10k.pdf
      Section: Note 2 - Revenue
      Type: text
      (1) Services net sales include amortization of the deferred value of services bundled in the sales price of certain products. The Co

In [43]:
# VAULTIFY COMPACT - Cell 21D:
# Structured Evidence Verification V1
#
# Responsibilities:
# - Verifies entity, metric, period, and numeric evidence
# - Extracts period-specific values from table chunks
# - Selects the strongest evidence for each routed entity
# - Stops incomplete comparisons before answer generation
#
# READ-ONLY:
# - Reuses Cell 21C retrieval indexes and results
# - Does not call Groq
# - Does not generate document embeddings
# - Does not modify Qdrant

import html
import re


# ---------------------------------------------------------------------
# 1. Validate runtime dependencies
# ---------------------------------------------------------------------

required_names = [
    "route_query_v1",
    "ORCHESTRATOR_ENTITY_REGISTRY",
    "ORCHESTRATOR_METRIC_EXPANSIONS",
    "CELL_21C_ROUTED_RETRIEVAL_APPROVED",
]

missing_names = [
    name
    for name in required_names
    if name not in globals()
]

if missing_names:
    raise RuntimeError(
        "Missing required components: "
        + ", ".join(missing_names)
        + ". Run Cells 21A–21C.1 first."
    )


if not CELL_21C_ROUTED_RETRIEVAL_APPROVED:
    raise RuntimeError(
        "Entity-Routed Hybrid Retrieval V1 "
        "did not pass its approval gate."
    )


# ---------------------------------------------------------------------
# 2. Normalization helpers
# ---------------------------------------------------------------------

def normalize_evidence_phrase(text):
    normalized = html.unescape(
        str(text or "")
    ).lower()

    normalized = (
        normalized
        .replace("’", "'")
        .replace("–", "-")
        .replace("—", "-")
    )

    normalized = re.sub(
        r"[^a-z0-9.%$€£&+\-]+",
        " ",
        normalized,
    )

    normalized = re.sub(
        r"\s+",
        " ",
        normalized,
    )

    return normalized.strip()


def compact_evidence_phrase(text):
    return re.sub(
        r"[^a-z0-9]+",
        "",
        normalize_evidence_phrase(text),
    )


def normalize_table_cell(text):
    normalized = html.unescape(
        str(text or "")
    )

    normalized = re.sub(
        r"\s+",
        " ",
        normalized,
    )

    return normalized.strip()


# ---------------------------------------------------------------------
# 3. Metric aliases
# ---------------------------------------------------------------------

def metric_aliases_for_verification(
    metric,
):
    if not metric:
        return []

    canonical_name = metric.get(
        "canonical"
    )

    aliases = list(
        ORCHESTRATOR_METRIC_EXPANSIONS.get(
            canonical_name,
            [],
        )
    )

    label = metric.get(
        "label"
    )

    if label:
        aliases.append(label)

    unique_aliases = []
    seen_aliases = set()

    for alias in aliases:
        normalized_alias = (
            normalize_evidence_phrase(
                alias
            )
        )

        if (
            normalized_alias
            and normalized_alias
            not in seen_aliases
        ):
            seen_aliases.add(
                normalized_alias
            )

            unique_aliases.append(
                normalized_alias
            )

    # Longer and more specific aliases first.
    unique_aliases.sort(
        key=len,
        reverse=True,
    )

    return unique_aliases


def find_metric_alias_in_text(
    text,
    metric,
):
    normalized_text = (
        normalize_evidence_phrase(
            text
        )
    )

    for alias in metric_aliases_for_verification(
        metric
    ):
        pattern = (
            r"(?<![a-z0-9])"
            + re.escape(alias)
            + r"(?![a-z0-9])"
        )

        if re.search(
            pattern,
            normalized_text,
        ):
            return alias

    return None


# ---------------------------------------------------------------------
# 4. Period normalization
# ---------------------------------------------------------------------

def canonicalize_period_cell(
    cell,
):
    normalized_cell = (
        normalize_evidence_phrase(
            cell
        )
    )

    quarter_match = re.fullmatch(
        r"q([1-4])[\s\-]*((?:19|20)\d{2})",
        normalized_cell,
    )

    if quarter_match:
        return {
            "kind": "quarter",
            "quarter": int(
                quarter_match.group(1)
            ),
            "year": int(
                quarter_match.group(2)
            ),
            "label": (
                f"Q{quarter_match.group(1)} "
                f"{quarter_match.group(2)}"
            ),
        }

    year_match = re.fullmatch(
        r"((?:19|20)\d{2})",
        normalized_cell,
    )

    if year_match:
        return {
            "kind": "year",
            "quarter": None,
            "year": int(
                year_match.group(1)
            ),
            "label": year_match.group(1),
        }

    return None


def period_matches_header(
    requested_period,
    header_period,
):
    if not requested_period:
        return True

    if not header_period:
        return False

    requested_year = requested_period.get(
        "year"
    )

    requested_quarter = requested_period.get(
        "quarter"
    )

    if requested_year != header_period.get(
        "year"
    ):
        return False

    if requested_quarter is not None:
        return (
            requested_quarter
            == header_period.get(
                "quarter"
            )
        )

    return True


def period_signals_present(
    text,
    period,
):
    if not period:
        return True

    normalized_text = (
        normalize_evidence_phrase(
            text
        )
    )

    year = period.get(
        "year"
    )

    quarter = period.get(
        "quarter"
    )

    if year is None:
        return True

    if str(year) not in normalized_text:
        return False

    if quarter is not None:
        quarter_pattern = (
            rf"\bq\s*{quarter}\b"
        )

        return bool(
            re.search(
                quarter_pattern,
                normalized_text,
            )
        )

    return True


# ---------------------------------------------------------------------
# 5. Numeric value helpers
# ---------------------------------------------------------------------

NUMERIC_CELL_PATTERN = re.compile(
    r"""
    ^
    \s*
    \(?
    [\$€£]?
    \s*
    -?
    \d[\d,]*
    (?:\.\d+)?
    \s*
    (?:
        %
        |
        [kmb]
        |
        thousand
        |
        million
        |
        billion
    )?
    \s*
    \)?
    \s*
    $
    """,
    flags=(
        re.IGNORECASE
        | re.VERBOSE
    ),
)


NUMERIC_SEARCH_PATTERN = re.compile(
    r"""
    (?:
        [\$€£]
        \s*
    )?
    \(?
    -?
    \d[\d,]*
    (?:\.\d+)?
    \s*
    (?:
        %
        |
        [kmb]
        |
        thousand
        |
        million
        |
        billion
    )?
    \)?
    """,
    flags=(
        re.IGNORECASE
        | re.VERBOSE
    ),
)


def is_numeric_table_cell(
    cell,
):
    cleaned_cell = (
        normalize_table_cell(
            cell
        )
    )

    if not cleaned_cell:
        return False

    if cleaned_cell in {
        "-",
        "—",
        "–",
    }:
        return True

    return bool(
        NUMERIC_CELL_PATTERN.fullmatch(
            cleaned_cell
        )
    )


def compact_numeric_value(
    value,
):
    return re.sub(
        r"[^0-9a-z.%\-]+",
        "",
        normalize_evidence_phrase(
            value
        ),
    )


def infer_value_unit(
    chunk_text,
    raw_value,
):
    normalized_text = (
        normalize_evidence_phrase(
            chunk_text
        )
    )

    normalized_value = (
        normalize_evidence_phrase(
            raw_value
        )
    )

    if "%" in normalized_value:
        return "percent"

    if "billion" in normalized_value:
        return "billion"

    if re.search(
        r"\d(?:\.\d+)?\s*b\b",
        normalized_value,
    ):
        return "billion"

    if (
        "in millions" in normalized_text
        or "dollars in millions"
        in normalized_text
        or "$ in millions"
        in normalized_text
    ):
        return "USD millions"

    if "$" in str(raw_value):
        return "USD"

    return "document units"


# ---------------------------------------------------------------------
# 6. Table value extraction
# ---------------------------------------------------------------------

def metric_cell_candidates(
    cells,
    metric,
):
    aliases = (
        metric_aliases_for_verification(
            metric
        )
    )

    candidates = []

    for cell_index, cell in enumerate(
        cells
    ):
        normalized_cell = (
            normalize_evidence_phrase(
                cell
            )
        )

        if not normalized_cell:
            continue

        for alias_priority, alias in enumerate(
            aliases
        ):
            if normalized_cell == alias:
                match_strength = 3

            elif normalized_cell.startswith(
                alias
            ):
                match_strength = 2

            elif re.search(
                (
                    r"(?<![a-z0-9])"
                    + re.escape(alias)
                    + r"(?![a-z0-9])"
                ),
                normalized_cell,
            ):
                match_strength = 1

            else:
                continue

            candidates.append(
                {
                    "cell_index": (
                        cell_index
                    ),
                    "cell": cell,
                    "alias": alias,
                    "alias_priority": (
                        alias_priority
                    ),
                    "match_strength": (
                        match_strength
                    ),
                }
            )

    candidates.sort(
        key=lambda candidate: (
            candidate[
                "alias_priority"
            ],
            -candidate[
                "match_strength"
            ],
            candidate[
                "cell_index"
            ],
        )
    )

    return candidates


def extract_period_value_from_table(
    text,
    metric,
    period,
):
    """
    Extract the requested row/column intersection from a
    flattened Markdown table.

    Example:
    Header: 2025 | 2024 | 2023
    Row: Total net sales | 416,161 | 391,035 | 383,285
    """
    if "|" not in str(text):
        return None

    cells = [
        normalize_table_cell(cell)
        for cell in str(text).split("|")
    ]

    metric_candidates = (
        metric_cell_candidates(
            cells,
            metric,
        )
    )

    for metric_candidate in metric_candidates:
        metric_cell_index = (
            metric_candidate[
                "cell_index"
            ]
        )

        header_periods = []

        for header_index in range(
            0,
            metric_cell_index,
        ):
            parsed_period = (
                canonicalize_period_cell(
                    cells[header_index]
                )
            )

            if parsed_period:
                header_periods.append(
                    {
                        "cell_index": (
                            header_index
                        ),
                        "period": (
                            parsed_period
                        ),
                    }
                )

        # Remove repeated period labels while keeping order.
        unique_header_periods = []
        seen_period_keys = set()

        for header_entry in header_periods:
            period_key = (
                header_entry["period"][
                    "year"
                ],
                header_entry["period"][
                    "quarter"
                ],
            )

            if period_key in seen_period_keys:
                continue

            seen_period_keys.add(
                period_key
            )

            unique_header_periods.append(
                header_entry
            )

        if not unique_header_periods:
            continue

        requested_column_ordinal = next(
            (
                column_ordinal
                for column_ordinal,
                header_entry
                in enumerate(
                    unique_header_periods
                )
                if period_matches_header(
                    period,
                    header_entry[
                        "period"
                    ],
                )
            ),
            None,
        )

        if requested_column_ordinal is None:
            continue

        row_numeric_values = []

        for value_cell in cells[
            metric_cell_index + 1:
        ]:
            if is_numeric_table_cell(
                value_cell
            ):
                row_numeric_values.append(
                    value_cell
                )

                if (
                    len(row_numeric_values)
                    >= len(
                        unique_header_periods
                    )
                ):
                    break

            elif row_numeric_values:
                # A new textual row probably started.
                break

        if (
            requested_column_ordinal
            >= len(row_numeric_values)
        ):
            continue

        raw_value = row_numeric_values[
            requested_column_ordinal
        ]

        return {
            "value": raw_value,
            "value_compact": (
                compact_numeric_value(
                    raw_value
                )
            ),
            "unit": infer_value_unit(
                text,
                raw_value,
            ),
            "metric_alias": (
                metric_candidate[
                    "alias"
                ]
            ),
            "period_label": (
                unique_header_periods[
                    requested_column_ordinal
                ]["period"]["label"]
            ),
            "method": (
                "table_row_column"
            ),
        }

    return None


# ---------------------------------------------------------------------
# 7. Text fallback extraction
# ---------------------------------------------------------------------

def extract_value_from_text_window(
    text,
    metric,
    period,
):
    normalized_text = (
        normalize_evidence_phrase(
            text
        )
    )

    aliases = (
        metric_aliases_for_verification(
            metric
        )
    )

    requested_year = (
        period.get("year")
        if period
        else None
    )

    for alias in aliases:
        alias_match = re.search(
            (
                r"(?<![a-z0-9])"
                + re.escape(alias)
                + r"(?![a-z0-9])"
            ),
            normalized_text,
        )

        if not alias_match:
            continue

        window_start = max(
            0,
            alias_match.start() - 80,
        )

        window_end = min(
            len(normalized_text),
            alias_match.end() + 220,
        )

        evidence_window = (
            normalized_text[
                window_start:window_end
            ]
        )

        numeric_matches = list(
            NUMERIC_SEARCH_PATTERN.finditer(
                evidence_window
            )
        )

        for numeric_match in numeric_matches:
            raw_value = (
                numeric_match
                .group(0)
                .strip()
            )

            compact_value = (
                compact_numeric_value(
                    raw_value
                )
            )

            # Skip a bare reporting year.
            if (
                requested_year is not None
                and compact_value
                == str(requested_year)
            ):
                continue

            return {
                "value": raw_value,
                "value_compact": (
                    compact_value
                ),
                "unit": infer_value_unit(
                    text,
                    raw_value,
                ),
                "metric_alias": alias,
                "period_label": (
                    period.get(
                        "label"
                    )
                    if period
                    else None
                ),
                "method": (
                    "text_window"
                ),
            }

    return None


# ---------------------------------------------------------------------
# 8. Verify one retrieved chunk
# ---------------------------------------------------------------------

def verify_evidence_chunk_v1(
    item,
    metric,
    period,
):
    text = item.get(
        "text",
        "",
    )

    matched_metric_alias = (
        find_metric_alias_in_text(
            text,
            metric,
        )
    )

    period_present = (
        period_signals_present(
            text,
            period,
        )
    )

    extracted_value = (
        extract_period_value_from_table(
            text=text,
            metric=metric,
            period=period,
        )
    )

    if extracted_value is None:
        extracted_value = (
            extract_value_from_text_window(
                text=text,
                metric=metric,
                period=period,
            )
        )

    metric_present = (
        matched_metric_alias
        is not None
    )

    numeric_present = (
        extracted_value
        is not None
    )

    table_bonus = (
        1
        if item.get(
            "chunk_type"
        ) == "table"
        else 0
    )

    extraction_bonus = (
        4
        if (
            extracted_value
            and extracted_value[
                "method"
            ]
            == "table_row_column"
        )
        else (
            2
            if extracted_value
            else 0
        )
    )

    verification_score = (
        extraction_bonus
        + (
            2
            if metric_present
            else 0
        )
        + (
            2
            if period_present
            else 0
        )
        + table_bonus
        + max(
            0.0,
            1.0
            - (
                item.get(
                    "rank",
                    99,
                )
                - 1
            )
            * 0.10,
        )
    )

    verified = (
        metric_present
        and period_present
        and numeric_present
    )

    return {
        "verified": verified,
        "verification_score": (
            float(
                verification_score
            )
        ),
        "metric_present": (
            metric_present
        ),
        "matched_metric_alias": (
            matched_metric_alias
        ),
        "period_present": (
            period_present
        ),
        "numeric_present": (
            numeric_present
        ),
        "extracted_value": (
            extracted_value
        ),
        "source": {
            "entity": item.get(
                "entity"
            ),
            "filename": item.get(
                "filename"
            ),
            "document_hash": (
                item.get(
                    "document_hash"
                )
            ),
            "section": item.get(
                "section"
            ),
            "chunk_type": item.get(
                "chunk_type"
            ),
            "point_id": item.get(
                "point_id"
            ),
            "chunk_index": item.get(
                "source_chunk_index"
            ),
            "retrieval_rank": item.get(
                "rank"
            ),
            "hybrid_score": item.get(
                "hybrid_score"
            ),
            "dense_rank": item.get(
                "dense_rank"
            ),
            "bm25_rank": item.get(
                "lexical_rank"
            ),
        },
        "text": text,
    }


# ---------------------------------------------------------------------
# 9. Verify one entity route
# ---------------------------------------------------------------------

def verify_route_evidence_v1(
    route,
):
    entity = route.get(
        "entity"
    )

    metric = route.get(
        "metric"
    )

    period = route.get(
        "period"
    )

    candidate_verifications = [
        verify_evidence_chunk_v1(
            item=result,
            metric=metric,
            period=period,
        )
        for result in route.get(
            "results",
            []
        )
    ]

    candidate_verifications.sort(
        key=lambda candidate: (
            candidate[
                "verified"
            ],
            candidate[
                "verification_score"
            ],
        ),
        reverse=True,
    )

    best_candidate = (
        candidate_verifications[0]
        if candidate_verifications
        else None
    )

    route_verified = bool(
        best_candidate
        and best_candidate[
            "verified"
        ]
    )

    return {
        "entity": entity,
        "metric": metric,
        "period": period,
        "retrieval_query": route.get(
            "retrieval_query"
        ),
        "verified": route_verified,
        "best_evidence": (
            best_candidate
            if route_verified
            else None
        ),
        "best_candidate": (
            best_candidate
        ),
        "candidate_verifications": (
            candidate_verifications
        ),
    }


# ---------------------------------------------------------------------
# 10. Verify a complete routed question
# ---------------------------------------------------------------------

def verify_routed_question_v1(
    routed_result,
):
    query_type = routed_result[
        "analysis"
    ]["query_type"]

    result = {
        "question": routed_result[
            "question"
        ],
        "analysis": routed_result[
            "analysis"
        ],
        "routing_status": routed_result[
            "status"
        ],
        "verification_status": None,
        "route_verifications": [],
        "clarification": routed_result.get(
            "clarification"
        ),
        "period_scope_warning": (
            routed_result[
                "analysis"
            ].get(
                "period_scope_warning"
            )
        ),
    }

    if routed_result[
        "status"
    ] == "clarification_required":
        result[
            "verification_status"
        ] = "clarification_required"

        return result

    if routed_result[
        "status"
    ] == "no_answer_candidate":
        result[
            "verification_status"
        ] = "no_answer_candidate"

        return result

    if query_type not in {
        "single_entity",
        "comparison",
    }:
        result[
            "verification_status"
        ] = "unsupported"

        return result

    route_verifications = [
        verify_route_evidence_v1(
            route
        )
        for route in routed_result[
            "routes"
        ]
    ]

    result[
        "route_verifications"
    ] = route_verifications

    all_routes_verified = (
        bool(route_verifications)
        and all(
            route[
                "verified"
            ]
            for route
            in route_verifications
        )
    )

    if all_routes_verified:
        if query_type == "comparison":
            result[
                "verification_status"
            ] = "verified_comparison"

        else:
            result[
                "verification_status"
            ] = "verified_single_entity"

    else:
        result[
            "verification_status"
        ] = "insufficient_evidence"

    return result


def retrieve_and_verify_v1(
    question,
    top_k_per_entity=6,
):
    routed_result = route_query_v1(
        question,
        top_k_per_entity=(
            top_k_per_entity
        ),
    )

    return verify_routed_question_v1(
        routed_result
    )


# ---------------------------------------------------------------------
# 11. Human-readable verification output
# ---------------------------------------------------------------------

def print_evidence_verification(
    result,
):
    print("\n" + "=" * 96)

    print(
        f"❓ Question: "
        f"{result['question']}"
    )

    print(
        f"🧭 Query type: "
        f"{result['analysis']['query_type']}"
    )

    print(
        f"🛡️ Verification status: "
        f"{result['verification_status']}"
    )

    if result.get(
        "clarification"
    ):
        print(
            f"💬 Clarification: "
            f"{result['clarification']}"
        )

    if result.get(
        "period_scope_warning"
    ):
        print(
            f"⚠️ Period warning: "
            f"{result['period_scope_warning']}"
        )

    for route_verification in result[
        "route_verifications"
    ]:
        print("\n" + "-" * 96)

        print(
            f"Entity: "
            f"{route_verification['entity']}"
        )

        metric = route_verification.get(
            "metric"
        )

        period = route_verification.get(
            "period"
        )

        print(
            f"Metric: "
            f"{metric['label'] if metric else 'not detected'}"
        )

        print(
            f"Period: "
            f"{period['label'] if period else 'not detected'}"
        )

        print(
            f"Verified: "
            f"{route_verification['verified']}"
        )

        candidate = route_verification.get(
            "best_candidate"
        )

        if not candidate:
            print(
                "No evidence candidate found."
            )

            continue

        print(
            f"Metric present: "
            f"{candidate['metric_present']}"
        )

        print(
            f"Period present: "
            f"{candidate['period_present']}"
        )

        print(
            f"Numeric value present: "
            f"{candidate['numeric_present']}"
        )

        extracted_value = candidate.get(
            "extracted_value"
        )

        if extracted_value:
            print(
                f"Extracted value: "
                f"{extracted_value['value']}"
            )

            print(
                f"Unit: "
                f"{extracted_value['unit']}"
            )

            print(
                f"Extraction method: "
                f"{extracted_value['method']}"
            )

        source = candidate[
            "source"
        ]

        print(
            f"Source: "
            f"{source['filename']}"
            f" · {source['section']}"
            f" · retrieval rank "
            f"{source['retrieval_rank']}"
        )

        preview = " ".join(
            candidate["text"].split()
        )[:320]

        print(
            f"Evidence preview: "
            f"{preview}..."
        )


# ---------------------------------------------------------------------
# 12. Regression tests
# ---------------------------------------------------------------------

EVIDENCE_VERIFICATION_TEST_CASES = [
    {
        "name": "Apple verified value",
        "question": (
            "What were Apple's total net sales "
            "in fiscal year 2025?"
        ),
        "expected_status": (
            "verified_single_entity"
        ),
        "expected_values": {
            "Apple": [
                "416161",
            ],
        },
    },
    {
        "name": "Tesla verified value",
        "question": (
            "What was Tesla's total revenue "
            "in Q4 2025?"
        ),
        "expected_status": (
            "verified_single_entity"
        ),
        "expected_values": {
            "Tesla": [
                "24901",
            ],
        },
    },
    {
        "name": "Verified comparison",
        "question": (
            "Compare Apple's fiscal 2025 net sales "
            "with Tesla's Q4 2025 revenue."
        ),
        "expected_status": (
            "verified_comparison"
        ),
        "expected_values": {
            "Apple": [
                "416161",
            ],
            "Tesla": [
                "24901",
            ],
        },
    },
    {
        "name": "Clarification preserved",
        "question": (
            "What was the total revenue in 2025?"
        ),
        "expected_status": (
            "clarification_required"
        ),
        "expected_values": {},
    },
    {
        "name": "No-answer candidate preserved",
        "question": (
            "Who wrote Pride and Prejudice?"
        ),
        "expected_status": (
            "no_answer_candidate"
        ),
        "expected_values": {},
    },
]


EVIDENCE_VERIFICATION_TEST_RESULTS = []


print(
    "🛡️ Running Structured Evidence "
    "Verification V1 regression tests..."
)


for test_case in (
    EVIDENCE_VERIFICATION_TEST_CASES
):
    verification_result = (
        retrieve_and_verify_v1(
            test_case["question"],
            top_k_per_entity=6,
        )
    )

    print_evidence_verification(
        verification_result
    )

    status_passed = (
        verification_result[
            "verification_status"
        ]
        == test_case[
            "expected_status"
        ]
    )

    value_checks = []

    for (
        expected_entity,
        expected_values,
    ) in test_case[
        "expected_values"
    ].items():
        matching_route = next(
            (
                route
                for route
                in verification_result[
                    "route_verifications"
                ]
                if route["entity"]
                == expected_entity
            ),
            None,
        )

        extracted_compact_value = None

        if (
            matching_route
            and matching_route.get(
                "best_evidence"
            )
        ):
            extracted_value = (
                matching_route[
                    "best_evidence"
                ].get(
                    "extracted_value"
                )
            )

            if extracted_value:
                extracted_compact_value = (
                    extracted_value[
                        "value_compact"
                    ]
                )

        value_passed = (
            extracted_compact_value
            in expected_values
        )

        value_checks.append(
            {
                "entity": (
                    expected_entity
                ),
                "passed": (
                    value_passed
                ),
                "extracted": (
                    extracted_compact_value
                ),
                "expected": (
                    expected_values
                ),
            }
        )

    values_passed = all(
        check["passed"]
        for check in value_checks
    )

    if not test_case[
        "expected_values"
    ]:
        values_passed = True

    test_passed = (
        status_passed
        and values_passed
    )

    EVIDENCE_VERIFICATION_TEST_RESULTS.append(
        {
            "name": test_case[
                "name"
            ],
            "passed": test_passed,
            "status_passed": (
                status_passed
            ),
            "values_passed": (
                values_passed
            ),
            "value_checks": (
                value_checks
            ),
            "result": (
                verification_result
            ),
        }
    )

    if value_checks:
        print("\nValue checks:")

        for value_check in value_checks:
            print(
                f"   "
                f"{'✅' if value_check['passed'] else '❌'} "
                f"{value_check['entity']}: "
                f"extracted={value_check['extracted']} "
                f"expected={value_check['expected']}"
            )

    print(
        f"\n"
        f"{'✅ PASS' if test_passed else '❌ FAIL'} "
        f"— {test_case['name']}"
    )


# ---------------------------------------------------------------------
# 13. Final approval gate
# ---------------------------------------------------------------------

EVIDENCE_VERIFICATION_V1_PASSED = all(
    result["passed"]
    for result
    in EVIDENCE_VERIFICATION_TEST_RESULTS
)


passed_count = sum(
    result["passed"]
    for result
    in EVIDENCE_VERIFICATION_TEST_RESULTS
)

total_count = len(
    EVIDENCE_VERIFICATION_TEST_RESULTS
)


print("\n" + "=" * 96)
print("📊 CELL 21D SUMMARY")

print(
    f"Evidence verification tests passed: "
    f"{passed_count}/{total_count}"
)

if EVIDENCE_VERIFICATION_V1_PASSED:
    print(
        "✅ Structured Evidence Verification V1 "
        "passed all regression tests."
    )

    print(
        "✅ Apple and Tesla values were extracted "
        "from entity-specific evidence."
    )

    print(
        "✅ Incomplete comparisons will stop before "
        "answer generation."
    )

    print(
        "➡️ Ready for grounded answer generation "
        "in Cell 21E."
    )

else:
    print(
        "❌ Evidence verification requires "
        "adjustment."
    )

    print(
        "⛔ Do not continue to Groq answer "
        "generation yet."
    )


print(
    "\n⏭️ Existing retrieval indexes were reused."
)

print(
    "⏭️ No Groq request was made."
)

print(
    "⏭️ No Qdrant points were created, "
    "updated, or deleted."
)

🛡️ Running Structured Evidence Verification V1 regression tests...

❓ Question: What were Apple's total net sales in fiscal year 2025?
🧭 Query type: single_entity
🛡️ Verification status: verified_single_entity

------------------------------------------------------------------------------------------------
Entity: Apple
Metric: total net sales
Period: fiscal year 2025
Verified: True
Metric present: True
Period present: True
Numeric value present: True
Extracted value: $ 416,161
Unit: USD
Extraction method: table_row_column
Source: apple_fy2025_10k.pdf · Note 2 - Revenue · retrieval rank 2
Evidence preview: Section: Note 2 - Revenue Table columns: | | 2025 | 2024 | 2023 | | iPhone | $ 209,586 | $ 201,183 | $ 200,583 | | Mac | 33,708 | 29,984 | 29,357 | | iPad | 28,023 | 26,694 | 28,300 | | Wearables, Home and Accessories | 35,686 | 37,005 | 39,845 | | Services (1) | 109,158 | 96,169 | 85,200 | | Total net sales | $ 416,16...

Value checks:
   ✅ Apple: extracted=416161 expected=['416161'

In [44]:
# VAULTIFY COMPACT - Cell 21E:
# Grounded Answer Generation V1
#
# Responsibilities:
# - Uses only evidence verified by Cell 21D
# - Generates natural-language answers with Groq
# - Validates that required values remain in the answer
# - Falls back to deterministic output if generation fails
# - Returns clarification and no-answer responses without calling Groq
#
# READ-ONLY:
# - Does not modify Qdrant
# - Does not generate document embeddings

import re


# ---------------------------------------------------------------------
# 1. Validate runtime dependencies
# ---------------------------------------------------------------------

required_names = [
    "retrieve_and_verify_v1",
    "EVIDENCE_VERIFICATION_V1_PASSED",
    "ORCHESTRATOR_TENANT_ID",
]

missing_names = [
    name
    for name in required_names
    if name not in globals()
]

if missing_names:
    raise RuntimeError(
        "Missing required components: "
        + ", ".join(missing_names)
        + ". Run Cells 21A–21D first."
    )


if not EVIDENCE_VERIFICATION_V1_PASSED:
    raise RuntimeError(
        "Structured Evidence Verification V1 "
        "did not pass its approval gate."
    )


# ---------------------------------------------------------------------
# 2. Resolve the existing Groq client
# ---------------------------------------------------------------------

def resolve_answer_groq_client():
    candidates = [
        globals().get("groq_client"),
        getattr(
            globals().get(
                "vaultify_web_backend",
                None,
            ),
            "groq_client",
            None,
        ),
    ]

    for candidate in candidates:
        try:
            create_method = (
                candidate
                .chat
                .completions
                .create
            )

            if callable(create_method):
                return candidate

        except (AttributeError, TypeError):
            continue

    return None


ANSWER_GROQ_CLIENT = (
    resolve_answer_groq_client()
)

ANSWER_GROQ_MODEL = globals().get(
    "GROQ_MODEL",
    "llama-3.3-70b-versatile",
)


print(
    f"🤖 Answer model: "
    f"{ANSWER_GROQ_MODEL}"
)

print(
    "✅ Groq client available."
    if ANSWER_GROQ_CLIENT
    else (
        "⚠️ Groq client was not found. "
        "Deterministic answers will be used."
    )
)


# ---------------------------------------------------------------------
# 3. Formatting helpers
# ---------------------------------------------------------------------

def compact_answer_text(text):
    return re.sub(
        r"[^a-z0-9]+",
        "",
        str(text or "").lower(),
    )


def clean_currency_spacing(value):
    value = str(
        value or ""
    ).strip()

    value = re.sub(
        r"([$€£])\s+",
        r"\1",
        value,
    )

    return value


def format_verified_value(
    extracted_value,
):
    raw_value = clean_currency_spacing(
        extracted_value.get(
            "value",
            "",
        )
    )

    unit = extracted_value.get(
        "unit",
        "document units",
    )

    if unit == "USD millions":
        if raw_value.startswith("$"):
            return (
                f"{raw_value} million"
            )

        return (
            f"${raw_value} million"
        )

    if unit == "USD":
        if raw_value.startswith("$"):
            return raw_value

        return f"${raw_value}"

    if unit == "percent":
        return raw_value

    if unit == "billion":
        return (
            raw_value
            if "b" in raw_value.lower()
            else f"{raw_value} billion"
        )

    return raw_value


# ---------------------------------------------------------------------
# 4. Convert verified routes into structured fact packets
# ---------------------------------------------------------------------

def build_verified_fact_packet(
    verification_result,
):
    facts = []

    for route in verification_result.get(
        "route_verifications",
        [],
    ):
        best_evidence = route.get(
            "best_evidence"
        )

        if not best_evidence:
            continue

        extracted_value = (
            best_evidence.get(
                "extracted_value"
            )
        )

        if not extracted_value:
            continue

        metric = route.get(
            "metric"
        ) or {}

        period = route.get(
            "period"
        ) or {}

        source = best_evidence.get(
            "source",
            {},
        )

        facts.append(
            {
                "entity": route.get(
                    "entity"
                ),
                "metric": metric.get(
                    "label",
                    "requested metric",
                ),
                "period": period.get(
                    "label",
                    "requested period",
                ),
                "raw_value": (
                    extracted_value.get(
                        "value"
                    )
                ),
                "value_compact": (
                    extracted_value.get(
                        "value_compact"
                    )
                ),
                "display_value": (
                    format_verified_value(
                        extracted_value
                    )
                ),
                "unit": (
                    extracted_value.get(
                        "unit"
                    )
                ),
                "extraction_method": (
                    extracted_value.get(
                        "method"
                    )
                ),
                "filename": source.get(
                    "filename"
                ),
                "section": source.get(
                    "section"
                ),
                "retrieval_rank": (
                    source.get(
                        "retrieval_rank"
                    )
                ),
                "chunk_type": source.get(
                    "chunk_type"
                ),
            }
        )

    return facts


def build_source_cards(
    facts,
):
    return [
        {
            "entity": fact[
                "entity"
            ],
            "filename": fact[
                "filename"
            ],
            "section": fact[
                "section"
            ],
            "metric": fact[
                "metric"
            ],
            "period": fact[
                "period"
            ],
            "value": fact[
                "display_value"
            ],
            "retrieval_rank": fact[
                "retrieval_rank"
            ],
        }
        for fact in facts
    ]


# ---------------------------------------------------------------------
# 5. Deterministic grounded-answer fallback
# ---------------------------------------------------------------------

def build_deterministic_answer(
    verification_result,
    facts,
):
    status = verification_result[
        "verification_status"
    ]

    if status == "clarification_required":
        return verification_result.get(
            "clarification"
        ) or (
            "Please specify the company "
            "and reporting period."
        )

    if status == "no_answer_candidate":
        return (
            "I could not identify relevant evidence "
            "for this question in the active "
            "organization's documents."
        )

    if status == "insufficient_evidence":
        verified_entities = [
            route["entity"]
            for route in verification_result.get(
                "route_verifications",
                []
            )
            if route.get(
                "verified"
            )
        ]

        if verified_entities:
            return (
                "I found verified evidence for "
                + ", ".join(
                    verified_entities
                )
                + ", but there was not enough verified "
                "evidence to answer the complete question."
            )

        return (
            "I could not find sufficient verified "
            "evidence to answer this question."
        )

    if status == "verified_single_entity":
        fact = facts[0]

        return (
            f"{fact['entity']}'s "
            f"{fact['metric']} for "
            f"{fact['period']} was "
            f"{fact['display_value']}. "
            f"Source: {fact['filename']}, "
            f"section “{fact['section']}”."
        )

    if status == "verified_comparison":
        fact_sentences = [
            (
                f"{fact['entity']}'s "
                f"{fact['metric']} for "
                f"{fact['period']} was "
                f"{fact['display_value']}"
            )
            for fact in facts
        ]

        answer = (
            "; ".join(
                fact_sentences
            )
            + "."
        )

        warning = verification_result.get(
            "period_scope_warning"
        )

        if warning:
            answer += " " + warning

        answer += " Sources: " + "; ".join(
            (
                f"{fact['filename']}, "
                f"section “{fact['section']}”"
            )
            for fact in facts
        ) + "."

        return answer

    return (
        "The question could not be completed "
        "with the currently verified evidence."
    )


# ---------------------------------------------------------------------
# 6. Build constrained Groq prompt
# ---------------------------------------------------------------------

def build_grounded_generation_prompt(
    verification_result,
    facts,
):
    fact_lines = []

    for fact_number, fact in enumerate(
        facts,
        start=1,
    ):
        fact_lines.append(
            "\n".join(
                [
                    f"FACT {fact_number}",
                    (
                        f"Entity: "
                        f"{fact['entity']}"
                    ),
                    (
                        f"Metric: "
                        f"{fact['metric']}"
                    ),
                    (
                        f"Period: "
                        f"{fact['period']}"
                    ),
                    (
                        f"Verified value: "
                        f"{fact['display_value']}"
                    ),
                    (
                        f"Unit classification: "
                        f"{fact['unit']}"
                    ),
                    (
                        f"Source file: "
                        f"{fact['filename']}"
                    ),
                    (
                        f"Source section: "
                        f"{fact['section']}"
                    ),
                ]
            )
        )

    warning = verification_result.get(
        "period_scope_warning"
    )

    warning_block = (
        warning
        if warning
        else "None"
    )

    return (
        "USER QUESTION:\n"
        f"{verification_result['question']}\n\n"
        "VERIFIED FACTS:\n"
        + "\n\n".join(
            fact_lines
        )
        + "\n\n"
        "COMPARISON WARNING:\n"
        f"{warning_block}\n\n"
        "Write the final answer using only the "
        "verified facts above."
    )


GROUNDED_ANSWER_SYSTEM_PROMPT = """
You are Vaultify's grounded document-answering engine.

Rules:
1. Use only the verified facts supplied by the application.
2. Preserve every verified numeric value exactly.
3. Do not invent, estimate, convert, calculate, or add numbers.
4. Do not change the supplied units.
5. Do not introduce information from general knowledge.
6. For a comparison, include every verified entity.
7. When a comparison warning is supplied, clearly explain it.
8. Mention the source file and section briefly.
9. Do not mention prompts, context numbers, retrieval ranks, or internal systems.
10. Keep the answer concise and professional.
""".strip()


# ---------------------------------------------------------------------
# 7. Validate generated answers
# ---------------------------------------------------------------------

def validate_generated_answer(
    answer,
    verification_result,
    facts,
):
    compact_answer = (
        compact_answer_text(
            answer
        )
    )

    missing_values = []

    for fact in facts:
        required_value = fact.get(
            "value_compact"
        )

        if (
            required_value
            and required_value
            not in compact_answer
        ):
            missing_values.append(
                {
                    "entity": fact[
                        "entity"
                    ],
                    "value": required_value,
                }
            )

    warning_required = bool(
        verification_result.get(
            "period_scope_warning"
        )
    )

    warning_present = True

    if warning_required:
        has_comparison_phrase = any(
            phrase in str(
                answer
            ).lower()
            for phrase in [
                "not directly comparable",
                "not directly period-equivalent",
                "different reporting periods",
                "annual figure",
                "quarterly figure",
            ]
        )

        has_annual_and_quarter = (
            "annual" in str(
                answer
            ).lower()
            and (
                "quarter" in str(
                    answer
                ).lower()
            )
        )

        warning_present = (
            has_comparison_phrase
            or has_annual_and_quarter
        )

    passed = (
        not missing_values
        and warning_present
    )

    return {
        "passed": passed,
        "missing_values": (
            missing_values
        ),
        "warning_required": (
            warning_required
        ),
        "warning_present": (
            warning_present
        ),
    }


# ---------------------------------------------------------------------
# 8. Generate one final grounded answer
# ---------------------------------------------------------------------

def generate_grounded_answer_v1(
    verification_result,
    use_llm=True,
):
    verification_status = (
        verification_result[
            "verification_status"
        ]
    )

    facts = build_verified_fact_packet(
        verification_result
    )

    sources = build_source_cards(
        facts
    )

    gated_statuses = {
        "clarification_required",
        "no_answer_candidate",
        "insufficient_evidence",
        "unsupported",
    }

    if verification_status in gated_statuses:
        answer = build_deterministic_answer(
            verification_result,
            facts,
        )

        mapped_status = {
            "clarification_required": (
                "clarification_required"
            ),
            "no_answer_candidate": (
                "no_answer"
            ),
            "insufficient_evidence": (
                "insufficient_evidence"
            ),
            "unsupported": (
                "unsupported"
            ),
        }.get(
            verification_status,
            verification_status,
        )

        return {
            "status": mapped_status,
            "answer": answer,
            "facts": facts,
            "sources": sources,
            "generation_method": (
                "deterministic_gate"
            ),
            "llm_called": False,
            "generation_validation": (
                None
            ),
        }

    if not facts:
        answer = build_deterministic_answer(
            verification_result,
            facts,
        )

        return {
            "status": (
                "insufficient_evidence"
            ),
            "answer": answer,
            "facts": facts,
            "sources": sources,
            "generation_method": (
                "deterministic_fallback"
            ),
            "llm_called": False,
            "generation_validation": (
                None
            ),
        }

    deterministic_answer = (
        build_deterministic_answer(
            verification_result,
            facts,
        )
    )

    if (
        not use_llm
        or ANSWER_GROQ_CLIENT is None
    ):
        return {
            "status": "answered",
            "answer": (
                deterministic_answer
            ),
            "facts": facts,
            "sources": sources,
            "generation_method": (
                "deterministic_verified"
            ),
            "llm_called": False,
            "generation_validation": (
                validate_generated_answer(
                    deterministic_answer,
                    verification_result,
                    facts,
                )
            ),
        }

    user_prompt = (
        build_grounded_generation_prompt(
            verification_result,
            facts,
        )
    )

    try:
        response = (
            ANSWER_GROQ_CLIENT
            .chat
            .completions
            .create(
                model=ANSWER_GROQ_MODEL,
                messages=[
                    {
                        "role": "system",
                        "content": (
                            GROUNDED_ANSWER_SYSTEM_PROMPT
                        ),
                    },
                    {
                        "role": "user",
                        "content": (
                            user_prompt
                        ),
                    },
                ],
                temperature=0,
                max_tokens=350,
            )
        )

        generated_answer = str(
            response
            .choices[0]
            .message
            .content
            or ""
        ).strip()

        generation_validation = (
            validate_generated_answer(
                generated_answer,
                verification_result,
                facts,
            )
        )

        if (
            not generated_answer
            or not generation_validation[
                "passed"
            ]
        ):
            return {
                "status": "answered",
                "answer": (
                    deterministic_answer
                ),
                "facts": facts,
                "sources": sources,
                "generation_method": (
                    "deterministic_fallback"
                ),
                "llm_called": True,
                "generation_validation": (
                    generation_validation
                ),
            }

        return {
            "status": "answered",
            "answer": generated_answer,
            "facts": facts,
            "sources": sources,
            "generation_method": (
                "groq_verified"
            ),
            "llm_called": True,
            "generation_validation": (
                generation_validation
            ),
        }

    except Exception as error:
        return {
            "status": "answered",
            "answer": deterministic_answer,
            "facts": facts,
            "sources": sources,
            "generation_method": (
                "deterministic_fallback"
            ),
            "llm_called": True,
            "generation_error": str(
                error
            ),
            "generation_validation": (
                None
            ),
        }


# ---------------------------------------------------------------------
# 9. Unified question-answering service
# ---------------------------------------------------------------------

def answer_question_v2(
    tenant_id,
    question,
    use_llm=True,
):
    tenant_id = str(
        tenant_id or ""
    ).strip()

    if not tenant_id:
        raise ValueError(
            "tenant_id is required."
        )

    if (
        tenant_id
        != ORCHESTRATOR_TENANT_ID
    ):
        raise PermissionError(
            "The current orchestrator index belongs "
            "to a different tenant."
        )

    question = str(
        question or ""
    ).strip()

    if not question:
        raise ValueError(
            "Question cannot be empty."
        )

    verification_result = (
        retrieve_and_verify_v1(
            question,
            top_k_per_entity=6,
        )
    )

    generation_result = (
        generate_grounded_answer_v1(
            verification_result,
            use_llm=use_llm,
        )
    )

    return {
        "tenant_id": tenant_id,
        "question": question,
        "status": generation_result[
            "status"
        ],
        "answer": generation_result[
            "answer"
        ],
        "sources": generation_result[
            "sources"
        ],
        "facts": generation_result[
            "facts"
        ],
        "generation_method": (
            generation_result[
                "generation_method"
            ]
        ),
        "llm_called": (
            generation_result[
                "llm_called"
            ]
        ),
        "generation_validation": (
            generation_result.get(
                "generation_validation"
            )
        ),
        "verification": (
            verification_result
        ),
    }


# ---------------------------------------------------------------------
# 10. Human-readable result output
# ---------------------------------------------------------------------

def print_answer_v2(
    result,
):
    print("\n" + "=" * 96)

    print(
        f"❓ Question: "
        f"{result['question']}"
    )

    print(
        f"📍 Status: "
        f"{result['status']}"
    )

    print(
        f"🧠 Generation method: "
        f"{result['generation_method']}"
    )

    print(
        f"🤖 Groq called: "
        f"{result['llm_called']}"
    )

    print(
        "\n💬 Answer:"
    )

    print(
        result["answer"]
    )

    if result["sources"]:
        print(
            "\n📚 Verified sources:"
        )

        for source_number, source in enumerate(
            result["sources"],
            start=1,
        ):
            print(
                f"   {source_number}. "
                f"{source['entity']} — "
                f"{source['filename']} — "
                f"{source['section']} — "
                f"{source['value']}"
            )


# ---------------------------------------------------------------------
# 11. Regression tests
# ---------------------------------------------------------------------

GROUNDED_ANSWER_TEST_CASES = [
    {
        "name": "Apple grounded answer",
        "question": (
            "What were Apple's total net sales "
            "in fiscal year 2025?"
        ),
        "expected_status": "answered",
        "required_values": [
            "416161",
        ],
        "required_entities": [
            "Apple",
        ],
    },
    {
        "name": "Tesla grounded answer",
        "question": (
            "What was Tesla's total revenue "
            "in Q4 2025?"
        ),
        "expected_status": "answered",
        "required_values": [
            "24901",
        ],
        "required_entities": [
            "Tesla",
        ],
    },
    {
        "name": "Grounded comparison",
        "question": (
            "Compare Apple's fiscal 2025 net sales "
            "with Tesla's Q4 2025 revenue."
        ),
        "expected_status": "answered",
        "required_values": [
            "416161",
            "24901",
        ],
        "required_entities": [
            "Apple",
            "Tesla",
        ],
        "requires_period_warning": True,
    },
    {
        "name": "Clarification response",
        "question": (
            "What was the total revenue in 2025?"
        ),
        "expected_status": (
            "clarification_required"
        ),
        "required_values": [],
        "required_entities": [],
        "expects_no_llm": True,
    },
    {
        "name": "No-answer response",
        "question": (
            "Who wrote Pride and Prejudice?"
        ),
        "expected_status": "no_answer",
        "required_values": [],
        "required_entities": [],
        "expects_no_llm": True,
    },
]


GROUNDED_ANSWER_TEST_RESULTS = []


print(
    "\n🧪 Running Grounded Answer "
    "Generation V1 regression tests..."
)


for test_case in (
    GROUNDED_ANSWER_TEST_CASES
):
    answer_result = answer_question_v2(
        tenant_id=(
            ORCHESTRATOR_TENANT_ID
        ),
        question=test_case[
            "question"
        ],
        use_llm=True,
    )

    print_answer_v2(
        answer_result
    )

    status_passed = (
        answer_result["status"]
        == test_case[
            "expected_status"
        ]
    )

    compact_answer = compact_answer_text(
        answer_result["answer"]
    )

    values_passed = all(
        value in compact_answer
        for value in test_case[
            "required_values"
        ]
    )

    source_entities = {
        source["entity"]
        for source
        in answer_result[
            "sources"
        ]
    }

    entities_passed = all(
        entity in source_entities
        for entity in test_case[
            "required_entities"
        ]
    )

    warning_passed = True

    if test_case.get(
        "requires_period_warning"
    ):
        lower_answer = (
            answer_result[
                "answer"
            ].lower()
        )

        warning_passed = (
    (
        (
            "annual" in lower_answer
            or "fiscal year" in lower_answer
        )
        and (
            "quarter" in lower_answer
            or "q1" in lower_answer
            or "q2" in lower_answer
            or "q3" in lower_answer
            or "q4" in lower_answer
        )
    )
    or (
        "not directly comparable"
        in lower_answer
    )
    or (
        "not directly period-equivalent"
        in lower_answer
    )
    or (
        "different reporting periods"
        in lower_answer
    )
)

    llm_gate_passed = True

    if test_case.get(
        "expects_no_llm"
    ):
        llm_gate_passed = not (
            answer_result[
                "llm_called"
            ]
        )

    test_passed = all(
        [
            status_passed,
            values_passed,
            entities_passed,
            warning_passed,
            llm_gate_passed,
        ]
    )

    GROUNDED_ANSWER_TEST_RESULTS.append(
        {
            "name": test_case[
                "name"
            ],
            "passed": test_passed,
            "status_passed": (
                status_passed
            ),
            "values_passed": (
                values_passed
            ),
            "entities_passed": (
                entities_passed
            ),
            "warning_passed": (
                warning_passed
            ),
            "llm_gate_passed": (
                llm_gate_passed
            ),
            "result": answer_result,
        }
    )

    print(
        f"\n"
        f"{'✅ PASS' if test_passed else '❌ FAIL'} "
        f"— {test_case['name']}"
    )


# ---------------------------------------------------------------------
# 12. Final approval gate
# ---------------------------------------------------------------------

GROUNDED_ANSWER_V1_PASSED = all(
    result["passed"]
    for result
    in GROUNDED_ANSWER_TEST_RESULTS
)


passed_count = sum(
    result["passed"]
    for result
    in GROUNDED_ANSWER_TEST_RESULTS
)

total_count = len(
    GROUNDED_ANSWER_TEST_RESULTS
)


print("\n" + "=" * 96)
print("📊 CELL 21E SUMMARY")

print(
    f"Grounded answer tests passed: "
    f"{passed_count}/{total_count}"
)

if GROUNDED_ANSWER_V1_PASSED:
    print(
        "✅ Grounded Answer Generation V1 "
        "passed all regression tests."
    )

    print(
        "✅ Groq received only structured, "
        "verified evidence."
    )

    print(
        "✅ Clarification and no-answer cases "
        "did not call Groq."
    )

    print(
        "✅ Invalid model output automatically "
        "falls back to deterministic answers."
    )

    print(
        "➡️ Ready to connect answer_question_v2 "
        "to the web and MCP layers."
    )

else:
    print(
        "❌ Grounded answer generation "
        "requires adjustment."
    )

    print(
        "⛔ Do not patch the web or MCP "
        "backend yet."
    )


print(
    "\n⏭️ No Qdrant points were created, "
    "updated, or deleted."
)

🤖 Answer model: llama-3.3-70b-versatile
✅ Groq client available.

🧪 Running Grounded Answer Generation V1 regression tests...

❓ Question: What were Apple's total net sales in fiscal year 2025?
📍 Status: answered
🧠 Generation method: groq_verified
🤖 Groq called: True

💬 Answer:
According to the apple_fy2025_10k.pdf, Note 2 - Revenue, Apple's total net sales in fiscal year 2025 were $416,161.

📚 Verified sources:
   1. Apple — apple_fy2025_10k.pdf — Note 2 - Revenue — $416,161

✅ PASS — Apple grounded answer

❓ Question: What was Tesla's total revenue in Q4 2025?
📍 Status: answered
🧠 Generation method: groq_verified
🤖 Groq called: True

💬 Answer:
According to the TSLA-Q4-2025-Update.pdf, specifically the unaudited section, Tesla's total revenue in Q4 2025 was $24,901 million.

📚 Verified sources:
   1. Tesla — TSLA-Q4-2025-Update.pdf — (Unaudited) — $24,901 million

✅ PASS — Tesla grounded answer

❓ Question: Compare Apple's fiscal 2025 net sales with Tesla's Q4 2025 revenue.
📍 Status: 

In [45]:
# VAULTIFY COMPACT - Cell 21E.1:
# Context-Aware Financial Unit Resolution
#
# Fixes cases where a table value and its unit declaration
# are stored in neighboring chunks.
#
# Example:
# Value chunk: Total net sales | $416,161
# Nearby context: (In millions)
# Final verified value: $416,161 million
#
# READ-ONLY:
# - Reuses existing indexes
# - Does not call Groq
# - Does not modify Qdrant

from collections import defaultdict
import html
import re


# ---------------------------------------------------------------------
# 1. Validate dependencies
# ---------------------------------------------------------------------

required_names = [
    "verify_evidence_chunk_v1",
    "retrieve_and_verify_v1",
    "generate_grounded_answer_v1",
    "ORCHESTRATOR_ENTITY_RETRIEVAL_INDEXES",
    "metric_aliases_for_verification",
    "clean_currency_spacing",
]

missing_names = [
    name
    for name in required_names
    if name not in globals()
]

if missing_names:
    raise RuntimeError(
        "Missing required components: "
        + ", ".join(missing_names)
        + ". Run Cells 21A–21E first."
    )


# ---------------------------------------------------------------------
# 2. Monetary metric definitions
# ---------------------------------------------------------------------

MONETARY_METRIC_NAMES = {
    "total_net_sales",
    "services_net_sales",
    "iphone_net_sales",
    "net_sales",
    "total_revenue",
    "revenue",
    "automotive_revenue",
    "energy_storage_revenue",
    "gaap_operating_income",
    "operating_income",
}


def is_monetary_metric(metric):
    if not metric:
        return False

    canonical_name = metric.get(
        "canonical",
        "",
    )

    if canonical_name in MONETARY_METRIC_NAMES:
        return True

    metric_label = str(
        metric.get(
            "label",
            "",
        )
    ).lower()

    return any(
        term in metric_label
        for term in [
            "sales",
            "revenue",
            "income",
            "expense",
            "cost",
            "cash",
            "assets",
            "liabilities",
        ]
    )


# ---------------------------------------------------------------------
# 3. Explicit scale detection
# ---------------------------------------------------------------------

def normalize_unit_context_text(text):
    normalized = html.unescape(
        str(text or "")
    ).lower()

    normalized = (
        normalized
        .replace("’", "'")
        .replace("–", "-")
        .replace("—", "-")
    )

    normalized = re.sub(
        r"\s+",
        " ",
        normalized,
    )

    return normalized.strip()


UNIT_SCALE_PATTERNS = {
    "billions": [
        r"\bin\s+billions?\b",
        r"\bdollars?\s+in\s+billions?\b",
        r"\busd\s+in\s+billions?\b",
        r"\$\s+in\s+billions?\b",
    ],
    "millions": [
        r"\bin\s+millions?\b",
        r"\bdollars?\s+in\s+millions?\b",
        r"\busd\s+in\s+millions?\b",
        r"\$\s+in\s+millions?\b",
    ],
    "thousands": [
        r"\bin\s+thousands?\b",
        r"\bdollars?\s+in\s+thousands?\b",
        r"\busd\s+in\s+thousands?\b",
        r"\$\s+in\s+thousands?\b",
    ],
}


def detect_explicit_unit_scale(text):
    normalized_text = (
        normalize_unit_context_text(
            text
        )
    )

    for scale, patterns in (
        UNIT_SCALE_PATTERNS.items()
    ):
        for pattern in patterns:
            if re.search(
                pattern,
                normalized_text,
            ):
                return scale

    return None


# ---------------------------------------------------------------------
# 4. Locate nearby and related source chunks
# ---------------------------------------------------------------------

def unit_context_chunk_key(chunk):
    point_id = str(
        chunk.get(
            "point_id",
            "",
        )
    ).strip()

    if point_id:
        return (
            "point",
            point_id,
        )

    return (
        "chunk",
        chunk.get("filename"),
        chunk.get("chunk_index"),
        chunk.get("section"),
    )


def collect_unit_context_candidates(
    item,
    metric,
):
    """
    Build weighted context around one retrieved chunk.

    Priority:
    1. Current chunk
    2. Immediate neighboring chunks
    3. Same section
    4. Metric-related chunks with explicit units
    5. Other explicit unit declarations in the document
    """
    entity = item.get(
        "entity"
    )

    retrieval_index = (
        ORCHESTRATOR_ENTITY_RETRIEVAL_INDEXES.get(
            entity
        )
    )

    if not retrieval_index:
        return []

    source_chunks = retrieval_index[
        "chunks"
    ]

    filename = item.get(
        "filename"
    )

    section = item.get(
        "section"
    )

    source_chunk_index = item.get(
        "source_chunk_index"
    )

    metric_aliases = (
        metric_aliases_for_verification(
            metric
        )
    )

    candidates = {}

    def add_candidate(
        chunk,
        weight,
        reason,
    ):
        key = unit_context_chunk_key(
            chunk
        )

        previous = candidates.get(
            key
        )

        if (
            previous is None
            or weight > previous[
                "weight"
            ]
        ):
            candidates[key] = {
                "chunk": chunk,
                "weight": float(
                    weight
                ),
                "reason": reason,
            }

    # Current retrieved chunk.
    add_candidate(
        {
            "point_id": item.get(
                "point_id"
            ),
            "filename": filename,
            "chunk_index": (
                source_chunk_index
            ),
            "section": section,
            "text": item.get(
                "text",
                "",
            ),
        },
        weight=10,
        reason="current_chunk",
    )

    same_document_chunks = [
        chunk
        for chunk in source_chunks
        if chunk.get(
            "filename"
        ) == filename
    ]

    for chunk in same_document_chunks:
        chunk_index = chunk.get(
            "chunk_index"
        )

        chunk_text = chunk.get(
            "text",
            "",
        )

        chunk_section = chunk.get(
            "section"
        )

        # Immediate neighboring chunks.
        if (
            isinstance(
                source_chunk_index,
                int,
            )
            and isinstance(
                chunk_index,
                int,
            )
        ):
            distance = abs(
                chunk_index
                - source_chunk_index
            )

            if 0 < distance <= 4:
                add_candidate(
                    chunk,
                    weight=9 - distance,
                    reason=(
                        f"neighbor_distance_{distance}"
                    ),
                )

        # Same section.
        if (
            section
            and chunk_section == section
        ):
            add_candidate(
                chunk,
                weight=6,
                reason="same_section",
            )

        normalized_chunk_text = (
            normalize_unit_context_text(
                chunk_text
            )
        )

        has_metric_alias = any(
            alias in normalized_chunk_text
            for alias in metric_aliases
        )

        explicit_scale = (
            detect_explicit_unit_scale(
                chunk_text
            )
        )

        # Explicit unit declaration tied to the same metric.
        if (
            has_metric_alias
            and explicit_scale
        ):
            add_candidate(
                chunk,
                weight=7,
                reason="metric_and_unit",
            )

        # General document-level unit declaration.
        elif explicit_scale:
            add_candidate(
                chunk,
                weight=2,
                reason="document_unit_signal",
            )

    return list(
        candidates.values()
    )


# ---------------------------------------------------------------------
# 5. Resolve unit using weighted source context
# ---------------------------------------------------------------------

def resolve_contextual_financial_unit(
    item,
    metric,
    extracted_value,
):
    base_unit = str(
        extracted_value.get(
            "unit",
            "document units",
        )
    )

    raw_value = str(
        extracted_value.get(
            "value",
            "",
        )
    )

    # Already precise enough.
    if base_unit in {
        "USD millions",
        "USD billions",
        "USD thousands",
        "percent",
        "billion",
    }:
        return {
            "unit": base_unit,
            "method": (
                "existing_explicit_unit"
            ),
            "scale_scores": {},
            "support": [],
        }

    if not is_monetary_metric(
        metric
    ):
        return {
            "unit": base_unit,
            "method": (
                "non_monetary_metric"
            ),
            "scale_scores": {},
            "support": [],
        }

    context_candidates = (
        collect_unit_context_candidates(
            item=item,
            metric=metric,
        )
    )

    scale_scores = defaultdict(
        float
    )

    supporting_context = []

    for candidate in (
        context_candidates
    ):
        chunk = candidate[
            "chunk"
        ]

        scale = (
            detect_explicit_unit_scale(
                chunk.get(
                    "text",
                    "",
                )
            )
        )

        if not scale:
            continue

        weight = candidate[
            "weight"
        ]

        scale_scores[
            scale
        ] += weight

        supporting_context.append(
            {
                "scale": scale,
                "weight": weight,
                "reason": candidate[
                    "reason"
                ],
                "filename": chunk.get(
                    "filename"
                ),
                "section": chunk.get(
                    "section"
                ),
                "chunk_index": chunk.get(
                    "chunk_index"
                ),
            }
        )

    if not scale_scores:
        return {
            "unit": base_unit,
            "method": (
                "no_contextual_scale_found"
            ),
            "scale_scores": {},
            "support": [],
        }

    resolved_scale = max(
        scale_scores,
        key=scale_scores.get,
    )

    combined_context = " ".join(
        candidate["chunk"].get(
            "text",
            "",
        )
        for candidate in context_candidates
    )

    normalized_context = (
        normalize_unit_context_text(
            combined_context
        )
    )

    is_usd = (
        "$" in raw_value
        or base_unit == "USD"
        or "$" in combined_context
        or "usd" in normalized_context
        or "u.s. dollar"
        in normalized_context
        or "dollars in"
        in normalized_context
    )

    if is_usd:
        resolved_unit = (
            f"USD {resolved_scale}"
        )
    else:
        resolved_unit = (
            resolved_scale
        )

    return {
        "unit": resolved_unit,
        "method": (
            "weighted_source_context"
        ),
        "scale_scores": dict(
            scale_scores
        ),
        "support": sorted(
            supporting_context,
            key=lambda entry: (
                entry["weight"]
            ),
            reverse=True,
        )[:8],
    }


# ---------------------------------------------------------------------
# 6. Patch evidence verification
# ---------------------------------------------------------------------

# Preserve the original Cell 21D verifier only once.
if (
    "BASE_VERIFY_EVIDENCE_CHUNK_V1"
    not in globals()
):
    BASE_VERIFY_EVIDENCE_CHUNK_V1 = (
        verify_evidence_chunk_v1
    )


def verify_evidence_chunk_v1(
    item,
    metric,
    period,
):
    result = (
        BASE_VERIFY_EVIDENCE_CHUNK_V1(
            item=item,
            metric=metric,
            period=period,
        )
    )

    extracted_value = result.get(
        "extracted_value"
    )

    if not extracted_value:
        return result

    unit_resolution = (
        resolve_contextual_financial_unit(
            item=item,
            metric=metric,
            extracted_value=(
                extracted_value
            ),
        )
    )

    extracted_value[
        "unit"
    ] = unit_resolution[
        "unit"
    ]

    result[
        "unit_resolution"
    ] = unit_resolution

    return result


# ---------------------------------------------------------------------
# 7. Patch value display formatting
# ---------------------------------------------------------------------

def format_verified_value(
    extracted_value,
):
    raw_value = clean_currency_spacing(
        extracted_value.get(
            "value",
            "",
        )
    )

    unit = extracted_value.get(
        "unit",
        "document units",
    )

    lower_value = raw_value.lower()

    if unit.startswith("USD"):
        if not raw_value.startswith(
            "$"
        ):
            raw_value = (
                f"${raw_value}"
            )

    if unit == "USD millions":
        if "million" in lower_value:
            return raw_value

        return (
            f"{raw_value} million"
        )

    if unit == "USD billions":
        if "billion" in lower_value:
            return raw_value

        return (
            f"{raw_value} billion"
        )

    if unit == "USD thousands":
        if "thousand" in lower_value:
            return raw_value

        return (
            f"{raw_value} thousand"
        )

    if unit == "USD":
        return raw_value

    if unit == "percent":
        return raw_value

    if unit == "billion":
        if "billion" in lower_value:
            return raw_value

        return (
            f"{raw_value} billion"
        )

    if unit == "millions":
        return (
            raw_value
            if "million" in lower_value
            else f"{raw_value} million"
        )

    if unit == "billions":
        return (
            raw_value
            if "billion" in lower_value
            else f"{raw_value} billion"
        )

    if unit == "thousands":
        return (
            raw_value
            if "thousand" in lower_value
            else f"{raw_value} thousand"
        )

    return raw_value


# ---------------------------------------------------------------------
# 8. Unit regression tests
# ---------------------------------------------------------------------

UNIT_CONTEXT_TEST_CASES = [
    {
        "name": (
            "Apple million-dollar unit"
        ),
        "question": (
            "What were Apple's total net sales "
            "in fiscal year 2025?"
        ),
        "expected_status": (
            "verified_single_entity"
        ),
        "expected": {
            "Apple": {
                "value": "416161",
                "unit": "USD millions",
                "display": (
                    "$416,161 million"
                ),
            },
        },
    },
    {
        "name": (
            "Tesla million-dollar unit"
        ),
        "question": (
            "What was Tesla's total revenue "
            "in Q4 2025?"
        ),
        "expected_status": (
            "verified_single_entity"
        ),
        "expected": {
            "Tesla": {
                "value": "24901",
                "unit": "USD millions",
                "display": (
                    "$24,901 million"
                ),
            },
        },
    },
    {
        "name": (
            "Comparison units"
        ),
        "question": (
            "Compare Apple's fiscal 2025 net sales "
            "with Tesla's Q4 2025 revenue."
        ),
        "expected_status": (
            "verified_comparison"
        ),
        "expected": {
            "Apple": {
                "value": "416161",
                "unit": "USD millions",
                "display": (
                    "$416,161 million"
                ),
            },
            "Tesla": {
                "value": "24901",
                "unit": "USD millions",
                "display": (
                    "$24,901 million"
                ),
            },
        },
    },
]


UNIT_CONTEXT_TEST_RESULTS = []


print(
    "💱 Running Context-Aware Unit "
    "Resolution regression tests..."
)


for test_case in (
    UNIT_CONTEXT_TEST_CASES
):
    verification_result = (
        retrieve_and_verify_v1(
            test_case["question"],
            top_k_per_entity=6,
        )
    )

    status_passed = (
        verification_result[
            "verification_status"
        ]
        == test_case[
            "expected_status"
        ]
    )

    entity_checks = []

    for entity, expected in (
        test_case["expected"].items()
    ):
        matching_route = next(
            (
                route
                for route
                in verification_result[
                    "route_verifications"
                ]
                if route["entity"]
                == entity
            ),
            None,
        )

        extracted_value = None
        resolution = None

        if (
            matching_route
            and matching_route.get(
                "best_evidence"
            )
        ):
            best_evidence = (
                matching_route[
                    "best_evidence"
                ]
            )

            extracted_value = (
                best_evidence.get(
                    "extracted_value"
                )
            )

            resolution = (
                best_evidence.get(
                    "unit_resolution"
                )
            )

        actual_value = (
            extracted_value.get(
                "value_compact"
            )
            if extracted_value
            else None
        )

        actual_unit = (
            extracted_value.get(
                "unit"
            )
            if extracted_value
            else None
        )

        actual_display = (
            format_verified_value(
                extracted_value
            )
            if extracted_value
            else None
        )

        entity_passed = all(
            [
                actual_value
                == expected[
                    "value"
                ],
                actual_unit
                == expected[
                    "unit"
                ],
                actual_display
                == expected[
                    "display"
                ],
            ]
        )

        entity_checks.append(
            {
                "entity": entity,
                "passed": entity_passed,
                "actual_value": (
                    actual_value
                ),
                "actual_unit": (
                    actual_unit
                ),
                "actual_display": (
                    actual_display
                ),
                "resolution": resolution,
            }
        )

    entities_passed = all(
        check["passed"]
        for check in entity_checks
    )

    test_passed = (
        status_passed
        and entities_passed
    )

    UNIT_CONTEXT_TEST_RESULTS.append(
        {
            "name": test_case[
                "name"
            ],
            "passed": test_passed,
            "status_passed": (
                status_passed
            ),
            "entity_checks": (
                entity_checks
            ),
        }
    )

    print("\n" + "=" * 96)

    print(
        f"Test: "
        f"{test_case['name']}"
    )

    print(
        f"Verification status: "
        f"{verification_result['verification_status']}"
    )

    for check in entity_checks:
        print(
            f"\n"
            f"{'✅' if check['passed'] else '❌'} "
            f"{check['entity']}"
        )

        print(
            f"   Value: "
            f"{check['actual_value']}"
        )

        print(
            f"   Unit: "
            f"{check['actual_unit']}"
        )

        print(
            f"   Display: "
            f"{check['actual_display']}"
        )

        resolution = check.get(
            "resolution"
        ) or {}

        print(
            f"   Resolution method: "
            f"{resolution.get('method')}"
        )

        print(
            f"   Scale scores: "
            f"{resolution.get('scale_scores')}"
        )

    print(
        f"\n"
        f"{'✅ PASS' if test_passed else '❌ FAIL'}"
    )


# ---------------------------------------------------------------------
# 9. Deterministic answer formatting check
# ---------------------------------------------------------------------

print(
    "\n🧾 Checking final answer formatting "
    "without calling Groq..."
)

UNIT_FORMATTING_CHECKS = []

for test_case in (
    UNIT_CONTEXT_TEST_CASES
):
    verification_result = (
        retrieve_and_verify_v1(
            test_case["question"],
            top_k_per_entity=6,
        )
    )

    generation_result = (
        generate_grounded_answer_v1(
            verification_result,
            use_llm=False,
        )
    )

    required_displays = [
        expected["display"]
        for expected
        in test_case[
            "expected"
        ].values()
    ]

    formatting_passed = all(
        display
        in generation_result[
            "answer"
        ]
        for display in required_displays
    )

    UNIT_FORMATTING_CHECKS.append(
        formatting_passed
    )

    print(
        f"\n"
        f"{'✅' if formatting_passed else '❌'} "
        f"{test_case['name']}"
    )

    print(
        generation_result[
            "answer"
        ]
    )


# ---------------------------------------------------------------------
# 10. Final approval gate
# ---------------------------------------------------------------------

UNIT_CONTEXT_PATCH_APPROVED = (
    all(
        result["passed"]
        for result
        in UNIT_CONTEXT_TEST_RESULTS
    )
    and all(
        UNIT_FORMATTING_CHECKS
    )
)


passed_count = sum(
    result["passed"]
    for result
    in UNIT_CONTEXT_TEST_RESULTS
)

total_count = len(
    UNIT_CONTEXT_TEST_RESULTS
)


print("\n" + "=" * 96)
print("📊 CELL 21E.1 SUMMARY")

print(
    f"Unit verification tests passed: "
    f"{passed_count}/{total_count}"
)

if UNIT_CONTEXT_PATCH_APPROVED:
    print(
        "✅ Context-Aware Unit Resolution passed "
        "all regression tests."
    )

    print(
        "✅ Apple is now correctly represented "
        "as $416,161 million."
    )

    print(
        "✅ Tesla remains correctly represented "
        "as $24,901 million."
    )

    print(
        "➡️ Ready for web and MCP integration."
    )

else:
    print(
        "❌ Unit resolution still contains "
        "a regression."
    )

    print(
        "⛔ Do not connect the new answer service "
        "to web or MCP yet."
    )


print(
    "\n⏭️ No Groq request was made."
)

print(
    "⏭️ No Qdrant points were created, "
    "updated, or deleted."
)

💱 Running Context-Aware Unit Resolution regression tests...

Test: Apple million-dollar unit
Verification status: verified_single_entity

✅ Apple
   Value: 416161
   Unit: USD millions
   Display: $416,161 million
   Resolution method: weighted_source_context
   Scale scores: {'millions': 93.0, 'thousands': 4.0}

✅ PASS

Test: Tesla million-dollar unit
Verification status: verified_single_entity

✅ Tesla
   Value: 24901
   Unit: USD millions
   Display: $24,901 million
   Resolution method: existing_explicit_unit
   Scale scores: {}

✅ PASS

Test: Comparison units
Verification status: verified_comparison

✅ Apple
   Value: 416161
   Unit: USD millions
   Display: $416,161 million
   Resolution method: weighted_source_context
   Scale scores: {'millions': 93.0, 'thousands': 4.0}

✅ Tesla
   Value: 24901
   Unit: USD millions
   Display: $24,901 million
   Resolution method: existing_explicit_unit
   Scale scores: {}

✅ PASS

🧾 Checking final answer formatting without calling Groq...

✅ 

In [53]:
# VAULTIFY COMPACT - Cell 22A:
# Inspect the active Flask web integration contract before patching it.
#
# READ-ONLY:
# - Resolves the active Flask application
# - Lists all registered Flask routes
# - Locates the current question-answer route
# - Inspects route dependencies without resolving Flask LocalProxy objects
# - Inspects tenant-resolution and template-facing fields
# - Does not call Groq
# - Does not query or modify Qdrant
# - Does not modify Flask routes

import inspect
import re

from flask import Flask
from werkzeug.local import LocalProxy

import vaultify_web_backend


# ---------------------------------------------------------------------
# 1. Validate the approved orchestrator service
# ---------------------------------------------------------------------

REQUIRED_ORCHESTRATOR_NAMES = [
    "answer_question_v2",
    "GROUNDED_ANSWER_V1_PASSED",
    "UNIT_CONTEXT_PATCH_APPROVED",
]


missing_orchestrator_names = [
    name
    for name in REQUIRED_ORCHESTRATOR_NAMES
    if name not in globals()
]


if missing_orchestrator_names:
    raise RuntimeError(
        "Missing approved orchestrator components: "
        + ", ".join(missing_orchestrator_names)
        + ". Run Cells 21E and 21E.1 before Cell 22A."
    )


if not bool(GROUNDED_ANSWER_V1_PASSED):
    raise RuntimeError(
        "Grounded Answer Generation V1 did not pass "
        "its regression gate."
    )


if not bool(UNIT_CONTEXT_PATCH_APPROVED):
    raise RuntimeError(
        "Context-Aware Unit Resolution did not pass "
        "its regression gate."
    )


if not callable(answer_question_v2):
    raise TypeError(
        "answer_question_v2 exists but is not callable."
    )


print("=" * 96)
print("✅ APPROVED ORCHESTRATOR SERVICE")

print(
    "answer_question_v2"
    f"{inspect.signature(answer_question_v2)}"
)

print(
    f"GROUNDED_ANSWER_V1_PASSED: "
    f"{GROUNDED_ANSWER_V1_PASSED}"
)

print(
    f"UNIT_CONTEXT_PATCH_APPROVED: "
    f"{UNIT_CONTEXT_PATCH_APPROVED}"
)


# ---------------------------------------------------------------------
# 2. Safe inspection helpers
# ---------------------------------------------------------------------

def is_local_proxy(value):
    """
    Return True without resolving the wrapped Flask object.
    """

    return isinstance(
        value,
        LocalProxy,
    )


def safe_is_callable(value):
    """
    Check callability without dereferencing Flask LocalProxy objects.
    """

    if is_local_proxy(value):
        return False

    try:
        return callable(value)

    except Exception:
        return False


def safe_signature(value):
    """
    Return a callable signature without triggering Flask proxies.
    """

    if is_local_proxy(value):
        return "(LocalProxy; requires request context)"

    try:
        return str(
            inspect.signature(value)
        )

    except (
        TypeError,
        ValueError,
        RuntimeError,
    ):
        return "(signature unavailable)"


def safe_unwrap(value):
    """
    Unwrap decorated callables without resolving Flask proxies.
    """

    if is_local_proxy(value):
        return value

    try:
        return inspect.unwrap(value)

    except (
        TypeError,
        ValueError,
        RuntimeError,
    ):
        return value


def safe_source(value):
    """
    Return Python source code when available.
    """

    if is_local_proxy(value):
        return ""

    candidates = [
        value,
        safe_unwrap(value),
    ]

    for candidate in candidates:
        try:
            return inspect.getsource(
                candidate
            )

        except (
            TypeError,
            OSError,
            IOError,
            RuntimeError,
        ):
            continue

    return ""


def safe_location(value):
    """
    Return the source file without resolving Flask proxies.
    """

    if is_local_proxy(value):
        return "Flask LocalProxy"

    try:
        return (
            inspect.getsourcefile(value)
            or "dynamic source"
        )

    except (
        TypeError,
        OSError,
        RuntimeError,
    ):
        return "dynamic source"


def describe_object(value):
    """
    Produce a safe description for route dependencies.
    """

    if is_local_proxy(value):
        return (
            "Flask LocalProxy "
            "(not resolved outside request context)"
        )

    if safe_is_callable(value):
        return (
            "callable "
            f"{safe_signature(value)}"
        )

    if value is None:
        return "not found in function globals"

    return type(value).__name__


def collect_referenced_names(value):
    """
    Collect referenced global names from a callable's bytecode.
    """

    names = set()

    candidates = [
        value,
        safe_unwrap(value),
    ]

    for candidate in candidates:
        if is_local_proxy(candidate):
            continue

        code_object = getattr(
            candidate,
            "__code__",
            None,
        )

        if code_object is not None:
            names.update(
                code_object.co_names
            )

    return sorted(names)


def print_source_preview(
    value,
    maximum_lines=120,
):
    """
    Print a limited source preview.
    """

    source = safe_source(value)

    if not source:
        print(
            "   Source preview unavailable."
        )
        return

    source_lines = source.splitlines()

    for line in source_lines[
        :maximum_lines
    ]:
        print(
            "   " + line
        )

    omitted_count = (
        len(source_lines)
        - maximum_lines
    )

    if omitted_count > 0:
        print(
            f"   ... {omitted_count} "
            f"additional lines omitted"
        )


# ---------------------------------------------------------------------
# 3. Resolve the active Flask application
# ---------------------------------------------------------------------

WEB_BACKEND_MODULE = vaultify_web_backend

flask_app_candidates = []


def register_flask_candidate(
    source_name,
    candidate,
):
    """
    Register a Flask application without adding duplicates.
    """

    if not isinstance(
        candidate,
        Flask,
    ):
        return

    if any(
        record["app"] is candidate
        for record in flask_app_candidates
    ):
        return

    flask_app_candidates.append(
        {
            "source": source_name,
            "app": candidate,
        }
    )


# Preferred active application created in Cell 13.
register_flask_candidate(
    "notebook global: vaultify_web_app",
    globals().get(
        "vaultify_web_app"
    ),
)

# Optional fallback names.
register_flask_candidate(
    "notebook global: WEB_FLASK_APP",
    globals().get(
        "WEB_FLASK_APP"
    ),
)

register_flask_candidate(
    "notebook global: app",
    globals().get(
        "app"
    ),
)

register_flask_candidate(
    "vaultify_web_backend.vaultify_web_app",
    getattr(
        WEB_BACKEND_MODULE,
        "vaultify_web_app",
        None,
    ),
)

register_flask_candidate(
    "vaultify_web_backend.app",
    getattr(
        WEB_BACKEND_MODULE,
        "app",
        None,
    ),
)


# Scan notebook globals for additional Flask applications.
for global_name, global_value in list(
    globals().items()
):
    register_flask_candidate(
        f"notebook global scan: {global_name}",
        global_value,
    )


# Scan backend module values for additional Flask applications.
for module_name in dir(
    WEB_BACKEND_MODULE
):
    try:
        module_value = getattr(
            WEB_BACKEND_MODULE,
            module_name,
        )

    except Exception:
        continue

    register_flask_candidate(
        (
            "vaultify_web_backend scan: "
            f"{module_name}"
        ),
        module_value,
    )


if not flask_app_candidates:
    raise RuntimeError(
        "No active Flask application instance was found. "
        "Cell 12 writes the backend module, but Cell 13 must call "
        "create_app(...) and assign the result to vaultify_web_app."
    )


selected_candidate = (
    flask_app_candidates[0]
)

WEB_FLASK_APP = (
    selected_candidate["app"]
)

WEB_FLASK_APP_SOURCE = (
    selected_candidate["source"]
)


print("\n" + "=" * 96)
print("🌐 WEB BACKEND INTEGRATION INVENTORY")

print(
    f"Module: "
    f"{WEB_BACKEND_MODULE.__name__}"
)

print(
    f"Module file: "
    f"{getattr(WEB_BACKEND_MODULE, '__file__', 'dynamic module')}"
)

print(
    f"Selected Flask app source: "
    f"{WEB_FLASK_APP_SOURCE}"
)

print(
    f"Flask app name: "
    f"{WEB_FLASK_APP.name}"
)

print(
    f"Flask app import name: "
    f"{WEB_FLASK_APP.import_name}"
)

print(
    f"Detected Flask app candidates: "
    f"{len(flask_app_candidates)}"
)


for index, candidate_record in enumerate(
    flask_app_candidates,
    start=1,
):
    selected_marker = (
        " ← selected"
        if candidate_record["app"]
        is WEB_FLASK_APP
        else ""
    )

    print(
        f"   {index}. "
        f"{candidate_record['source']}"
        f"{selected_marker}"
    )


# ---------------------------------------------------------------------
# 4. List Flask routes
# ---------------------------------------------------------------------

print("\n" + "=" * 96)
print("🛣️ FLASK ROUTES")

WEB_ROUTE_INVENTORY = []


for rule in sorted(
    WEB_FLASK_APP.url_map.iter_rules(),
    key=lambda route: (
        route.rule,
        route.endpoint,
    ),
):
    methods = sorted(
        method
        for method in rule.methods
        if method not in {
            "HEAD",
            "OPTIONS",
        }
    )

    route_record = {
        "rule": rule.rule,
        "endpoint": rule.endpoint,
        "methods": methods,
    }

    WEB_ROUTE_INVENTORY.append(
        route_record
    )

    print(
        f"{','.join(methods):<16} "
        f"{rule.rule:<40} "
        f"endpoint={rule.endpoint}"
    )


if not WEB_ROUTE_INVENTORY:
    raise RuntimeError(
        "The Flask application was found, "
        "but it has no registered routes."
    )


# ---------------------------------------------------------------------
# 5. Locate question-answer route candidates
# ---------------------------------------------------------------------

WEB_ASK_ROUTES = [
    route
    for route in WEB_ROUTE_INVENTORY
    if (
        route["rule"] == "/ask"
        or "ask" in route["rule"].lower()
        or "ask" in route["endpoint"].lower()
        or "question" in route["rule"].lower()
        or "question" in route["endpoint"].lower()
        or "query" in route["endpoint"].lower()
    )
]


print("\n" + "=" * 96)
print("❓ QUESTION-ANSWER ROUTE CANDIDATES")


if not WEB_ASK_ROUTES:
    print(
        "⚠️ No obvious question-answer route was found."
    )

else:
    for route in WEB_ASK_ROUTES:
        print(
            f"{route['methods']} "
            f"{route['rule']} "
            f"→ {route['endpoint']}"
        )


# ---------------------------------------------------------------------
# 6. Inspect each question-answer view function
# ---------------------------------------------------------------------

WEB_ASK_VIEW_FUNCTIONS = {}

INTERESTING_REFERENCE_KEYWORDS = [
    "answer",
    "ask",
    "tenant",
    "organization",
    "membership",
    "query",
    "document",
    "session",
    "render",
    "current",
    "user",
    "source",
    "service",
]


for route in WEB_ASK_ROUTES:
    endpoint = route["endpoint"]

    registered_view = (
        WEB_FLASK_APP
        .view_functions
        .get(endpoint)
    )

    if registered_view is None:
        print(
            f"⚠️ No view function found "
            f"for endpoint: {endpoint}"
        )
        continue

    unwrapped_view = safe_unwrap(
        registered_view
    )

    WEB_ASK_VIEW_FUNCTIONS[
        endpoint
    ] = registered_view

    print("\n" + "-" * 96)

    print(
        f"Endpoint: {endpoint}"
    )

    print(
        f"Registered function: "
        f"{getattr(registered_view, '__name__', type(registered_view).__name__)}"
    )

    print(
        f"Unwrapped function: "
        f"{getattr(unwrapped_view, '__name__', type(unwrapped_view).__name__)}"
    )

    print(
        f"Signature: "
        f"{safe_signature(unwrapped_view)}"
    )

    print(
        f"Defined in: "
        f"{safe_location(unwrapped_view)}"
    )

    referenced_names = (
        collect_referenced_names(
            registered_view
        )
    )

    interesting_names = [
        name
        for name in referenced_names
        if any(
            keyword in name.lower()
            for keyword in (
                INTERESTING_REFERENCE_KEYWORDS
            )
        )
    ]

    print(
        "Referenced names:"
    )

    if not interesting_names:
        print(
            "   No relevant names detected."
        )

    view_global_namespaces = []

    for callable_candidate in [
        registered_view,
        unwrapped_view,
    ]:
        if is_local_proxy(
            callable_candidate
        ):
            continue

        candidate_globals = getattr(
            callable_candidate,
            "__globals__",
            None,
        )

        if (
            isinstance(
                candidate_globals,
                dict,
            )
            and candidate_globals
            not in view_global_namespaces
        ):
            view_global_namespaces.append(
                candidate_globals
            )

    for referenced_name in (
        interesting_names
    ):
        referenced_object = None
        object_found = False

        for global_namespace in (
            view_global_namespaces
        ):
            if referenced_name in global_namespace:
                referenced_object = (
                    global_namespace[
                        referenced_name
                    ]
                )
                object_found = True
                break

        if not object_found:
            object_description = (
                "not found in function globals"
            )

        else:
            object_description = (
                describe_object(
                    referenced_object
                )
            )

        print(
            f"   {referenced_name}: "
            f"{object_description}"
        )

    print(
        "\nSource preview:"
    )

    print_source_preview(
        unwrapped_view
    )


# ---------------------------------------------------------------------
# 7. Inventory relevant backend callables and proxies
# ---------------------------------------------------------------------

RELEVANT_BACKEND_KEYWORDS = [
    "ask",
    "answer",
    "tenant",
    "organization",
    "membership",
    "query",
    "retriev",
    "current",
    "document",
    "create_app",
]


WEB_RELEVANT_OBJECTS = {}


for name in dir(
    WEB_BACKEND_MODULE
):
    if not any(
        keyword in name.lower()
        for keyword in (
            RELEVANT_BACKEND_KEYWORDS
        )
    ):
        continue

    try:
        value = getattr(
            WEB_BACKEND_MODULE,
            name,
        )

    except Exception as error:
        WEB_RELEVANT_OBJECTS[
            name
        ] = {
            "value": None,
            "description": (
                "attribute access failed: "
                f"{type(error).__name__}"
            ),
        }
        continue

    WEB_RELEVANT_OBJECTS[
        name
    ] = {
        "value": value,
        "description": describe_object(
            value
        ),
    }


print("\n" + "=" * 96)
print("🧩 RELEVANT BACKEND OBJECTS")


if not WEB_RELEVANT_OBJECTS:
    print(
        "⚠️ No relevant backend objects were found."
    )


for name in sorted(
    WEB_RELEVANT_OBJECTS
):
    print(
        f"{name}: "
        f"{WEB_RELEVANT_OBJECTS[name]['description']}"
    )


# ---------------------------------------------------------------------
# 8. Identify tenant-resolution helper candidates
# ---------------------------------------------------------------------

TENANT_HELPER_CANDIDATES = {
    name: record
    for name, record
    in WEB_RELEVANT_OBJECTS.items()
    if any(
        keyword in name.lower()
        for keyword in [
            "tenant",
            "organization",
            "membership",
            "current",
        ]
    )
}


print("\n" + "=" * 96)
print("🔐 TENANT-RESOLUTION HELPER CANDIDATES")


if not TENANT_HELPER_CANDIDATES:
    print(
        "⚠️ No named tenant-resolution helper "
        "was detected."
    )


for name, record in sorted(
    TENANT_HELPER_CANDIDATES.items()
):
    print(
        f"{name}: "
        f"{record['description']}"
    )


# ---------------------------------------------------------------------
# 9. Inspect template-facing response fields
# ---------------------------------------------------------------------

print("\n" + "=" * 96)
print("🖼️ TEMPLATE CONTRACT HINTS")


TEMPLATE_FIELD_PATTERN = re.compile(
    r"\b("
    r"answer"
    r"|question"
    r"|sources"
    r"|retrieval_results"
    r"|results"
    r"|error"
    r"|status"
    r"|documents"
    r"|organization"
    r"|upload_status"
    r"|upload_success"
    r"|username"
    r"|tenant_id"
    r")\b",
    flags=re.IGNORECASE,
)


WEB_TEMPLATE_CONTRACT_HINTS = {}


for endpoint, view_function in (
    WEB_ASK_VIEW_FUNCTIONS.items()
):
    source = safe_source(
        view_function
    )

    template_keywords = sorted(
        {
            match.group(0)
            for match in (
                TEMPLATE_FIELD_PATTERN
                .finditer(source)
            )
        },
        key=str.lower,
    )

    WEB_TEMPLATE_CONTRACT_HINTS[
        endpoint
    ] = template_keywords

    print(
        f"{endpoint}: "
        f"{template_keywords}"
    )


# ---------------------------------------------------------------------
# 10. Identify current answer dependencies
# ---------------------------------------------------------------------

print("\n" + "=" * 96)
print("🤖 CURRENT ANSWER DEPENDENCY HINTS")


ANSWER_DEPENDENCY_NAMES = set()


for view_function in (
    WEB_ASK_VIEW_FUNCTIONS.values()
):
    referenced_names = (
        collect_referenced_names(
            view_function
        )
    )

    for referenced_name in (
        referenced_names
    ):
        lowered_name = (
            referenced_name.lower()
        )

        if (
            "answer" in lowered_name
            or "ask" in lowered_name
            or "retriev" in lowered_name
        ):
            ANSWER_DEPENDENCY_NAMES.add(
                referenced_name
            )


if not ANSWER_DEPENDENCY_NAMES:
    print(
        "⚠️ No answer dependency was detected "
        "from the route bytecode."
    )


for dependency_name in sorted(
    ANSWER_DEPENDENCY_NAMES
):
    try:
        dependency_object = getattr(
            WEB_BACKEND_MODULE,
            dependency_name,
            None,
        )

    except Exception as error:
        dependency_description = (
            "attribute access failed: "
            f"{type(error).__name__}"
        )

    else:
        dependency_description = (
            describe_object(
                dependency_object
            )
        )

    print(
        f"{dependency_name}: "
        f"{dependency_description}"
    )


# ---------------------------------------------------------------------
# 11. Save the inspection result
# ---------------------------------------------------------------------

WEB_INTEGRATION_INSPECTION_READY = bool(
    WEB_ASK_VIEW_FUNCTIONS
)


WEB_INTEGRATION_INSPECTION_RESULT = {
    "ready": (
        WEB_INTEGRATION_INSPECTION_READY
    ),
    "backend_module": (
        WEB_BACKEND_MODULE.__name__
    ),
    "backend_module_file": getattr(
        WEB_BACKEND_MODULE,
        "__file__",
        None,
    ),
    "flask_app": (
        WEB_FLASK_APP
    ),
    "flask_app_name": (
        WEB_FLASK_APP.name
    ),
    "flask_app_source": (
        WEB_FLASK_APP_SOURCE
    ),
    "routes": (
        WEB_ROUTE_INVENTORY
    ),
    "ask_routes": (
        WEB_ASK_ROUTES
    ),
    "ask_view_functions": (
        WEB_ASK_VIEW_FUNCTIONS
    ),
    "tenant_helper_candidates": (
        TENANT_HELPER_CANDIDATES
    ),
    "template_contract_hints": (
        WEB_TEMPLATE_CONTRACT_HINTS
    ),
    "answer_dependency_names": sorted(
        ANSWER_DEPENDENCY_NAMES
    ),
}


# ---------------------------------------------------------------------
# 12. Final inspection status
# ---------------------------------------------------------------------

print("\n" + "=" * 96)
print("📊 CELL 22A SUMMARY")


if WEB_INTEGRATION_INSPECTION_READY:
    print(
        "✅ The active Flask application "
        "was resolved."
    )

    print(
        "✅ The active question-answer route "
        "was located."
    )

    print(
        "✅ Flask LocalProxy dependencies were "
        "inspected without resolving request context."
    )

    print(
        "✅ Tenant, answer, and template dependencies "
        "were inventoried."
    )

    print(
        "➡️ Ready to build a security-preserving "
        "web adapter in Cell 22B."
    )

else:
    print(
        "⚠️ The Flask application was resolved, "
        "but no question-answer route was identified."
    )

    print(
        "⛔ Do not patch the web backend blindly."
    )


print(
    "\n⏭️ No Flask route was changed."
)

print(
    "⏭️ No Flask LocalProxy was resolved."
)

print(
    "⏭️ No Groq request was made."
)

print(
    "⏭️ No Qdrant query or write was made."
)

✅ APPROVED ORCHESTRATOR SERVICE
answer_question_v2(tenant_id, question, use_llm=True)
GROUNDED_ANSWER_V1_PASSED: True
UNIT_CONTEXT_PATCH_APPROVED: True

🌐 WEB BACKEND INTEGRATION INVENTORY
Module: vaultify_web_backend
Module file: /content/vaultify_compact_web/vaultify_web_backend.py
Selected Flask app source: notebook global: vaultify_web_app
Flask app name: vaultify_web_backend
Flask app import name: vaultify_web_backend
Detected Flask app candidates: 1
   1. notebook global: vaultify_web_app ← selected

🛣️ FLASK ROUTES
GET              /                                        endpoint=index
POST             /ask                                     endpoint=ask
GET              /dashboard                               endpoint=dashboard
GET              /documents                               endpoint=documents
POST             /documents/<int:document_id>/delete      endpoint=delete_document
POST             /documents/<int:document_id>/retry       endpoint=retry_document
GET,POST 

In [54]:
# PHASE 3.7 - Cell 22B:
# Connect the approved question-answering orchestrator
# to the existing tenant-secure Flask web application.
#
# SECURITY:
# - Keeps the existing login requirement
# - Keeps trusted membership-based tenant resolution
# - Does not accept tenant_id from the browser
# - Does not modify the /ask Flask route
#
# COMPATIBILITY:
# - Converts answer_question_v2 output into the legacy web contract
# - Preserves the existing dashboard template and source serializer
#
# NO LIVE REQUEST:
# - Does not call Groq in this cell
# - Does not query or modify Qdrant
# - Actual behavior will be tested in Cell 22C

from types import SimpleNamespace


# ---------------------------------------------------------------------
# 1. Validate Cell 22A and orchestrator dependencies
# ---------------------------------------------------------------------

required_names = [
    "WEB_FLASK_APP",
    "WEB_INTEGRATION_INSPECTION_READY",
    "answer_question_v2",
    "ORCHESTRATOR_TENANT_ID",
    "GROUNDED_ANSWER_V1_PASSED",
    "UNIT_CONTEXT_PATCH_APPROVED",
]

missing_names = [
    name
    for name in required_names
    if name not in globals()
]

if missing_names:
    raise RuntimeError(
        "Missing required integration components: "
        + ", ".join(missing_names)
        + ". Run Cells 21E, 21E.1, and 22A first."
    )


if not WEB_INTEGRATION_INSPECTION_READY:
    raise RuntimeError(
        "Cell 22A did not identify the active "
        "Flask question-answer route."
    )


if not GROUNDED_ANSWER_V1_PASSED:
    raise RuntimeError(
        "Grounded Answer Generation V1 "
        "did not pass its regression gate."
    )


if not UNIT_CONTEXT_PATCH_APPROVED:
    raise RuntimeError(
        "Context-Aware Unit Resolution "
        "did not pass its regression gate."
    )


if not callable(answer_question_v2):
    raise TypeError(
        "answer_question_v2 exists but is not callable."
    )


# ---------------------------------------------------------------------
# 2. Resolve the active Flask service registry
# ---------------------------------------------------------------------

WEB_SERVICE_REGISTRY = (
    WEB_FLASK_APP
    .config
    .get("VAULTIFY_SERVICES")
)


if not isinstance(
    WEB_SERVICE_REGISTRY,
    dict,
):
    raise RuntimeError(
        "The active Flask application does not contain "
        "a valid VAULTIFY_SERVICES registry."
    )


current_answer_service = (
    WEB_SERVICE_REGISTRY.get(
        "answer_tenant_question"
    )
)


if current_answer_service is None:
    raise RuntimeError(
        "The existing answer_tenant_question "
        "web service was not found."
    )


# Preserve the original service only when it is not
# an adapter from a previous execution of this cell.
if (
    getattr(
        current_answer_service,
        "__name__",
        "",
    )
    != "answer_question_v2_web_adapter"
):
    ORIGINAL_WEB_ANSWER_SERVICE = (
        current_answer_service
    )


if (
    "ORIGINAL_WEB_ANSWER_SERVICE"
    not in globals()
):
    raise RuntimeError(
        "The original web answer service "
        "could not be preserved."
    )


# ---------------------------------------------------------------------
# 3. Define supported orchestrator statuses
# ---------------------------------------------------------------------

WEB_SUPPORTED_ANSWER_STATUSES = {
    "answered",
    "clarification_required",
    "no_answer",
    "insufficient_evidence",
    "unsupported",
}


# ---------------------------------------------------------------------
# 4. Convert verified source cards into the legacy web result shape
# ---------------------------------------------------------------------

def normalize_retrieval_rank(
    retrieval_rank,
):
    """
    Return a safe positive retrieval rank.
    """

    try:
        retrieval_rank = int(
            retrieval_rank
        )

    except (
        TypeError,
        ValueError,
    ):
        return None

    if retrieval_rank < 1:
        return None

    return retrieval_rank


def build_web_compatibility_score(
    retrieval_rank,
):
    """
    Convert a verified retrieval rank into a numeric value
    required by the legacy dashboard source serializer.

    This is a rank-derived compatibility value, not a raw
    vector-similarity score:

        rank 1 -> 1.0000
        rank 2 -> 0.5000
        rank 3 -> 0.3333
    """

    normalized_rank = (
        normalize_retrieval_rank(
            retrieval_rank
        )
    )

    if normalized_rank is None:
        return 0.0

    return 1.0 / normalized_rank


def source_card_to_web_result(
    source_card,
):
    """
    Convert an answer_question_v2 verified source card into
    the object shape expected by serialize_sources().
    """

    if not isinstance(
        source_card,
        dict,
    ):
        raise TypeError(
            "Each verified source must be a dictionary."
        )

    filename = str(
        source_card.get(
            "filename"
        )
        or "Unknown file"
    )

    section = str(
        source_card.get(
            "section"
        )
        or "Unknown section"
    )

    retrieval_rank = (
        normalize_retrieval_rank(
            source_card.get(
                "retrieval_rank"
            )
        )
    )

    payload = {
        "filename": filename,
        "section": section,
        "entity": source_card.get(
            "entity"
        ),
        "metric": source_card.get(
            "metric"
        ),
        "period": source_card.get(
            "period"
        ),
        "value": source_card.get(
            "value"
        ),
        "retrieval_rank": (
            retrieval_rank
        ),
        "verified": True,
    }

    return SimpleNamespace(
        payload=payload,
        score=(
            build_web_compatibility_score(
                retrieval_rank
            )
        ),
    )


# ---------------------------------------------------------------------
# 5. Define the security-preserving web adapter
# ---------------------------------------------------------------------

def answer_question_v2_web_adapter(
    question,
    tenant_id,
    **_ignored_arguments,
):
    """
    Call answer_question_v2 using the trusted tenant supplied
    by the existing Flask membership-resolution flow.

    Returns the legacy web contract expected by the current
    /ask route:

        {
            "answer": str,
            "results": list,
        }

    Additional orchestrator fields are preserved for testing
    and future template upgrades.
    """

    cleaned_tenant_id = str(
        tenant_id or ""
    ).strip()

    cleaned_question = str(
        question or ""
    ).strip()


    if not cleaned_tenant_id:
        raise ValueError(
            "tenant_id is required."
        )


    if not cleaned_question:
        raise ValueError(
            "Question cannot be empty."
        )


    orchestrator_result = (
        answer_question_v2(
            tenant_id=cleaned_tenant_id,
            question=cleaned_question,
            use_llm=True,
        )
    )


    if not isinstance(
        orchestrator_result,
        dict,
    ):
        raise TypeError(
            "answer_question_v2 must return a dictionary."
        )


    required_result_fields = {
        "tenant_id",
        "question",
        "status",
        "answer",
        "sources",
    }

    missing_result_fields = (
        required_result_fields
        - set(
            orchestrator_result
        )
    )

    if missing_result_fields:
        raise RuntimeError(
            "answer_question_v2 returned an incomplete result. "
            "Missing fields: "
            + ", ".join(
                sorted(
                    missing_result_fields
                )
            )
        )


    returned_tenant_id = str(
        orchestrator_result.get(
            "tenant_id"
        )
        or ""
    ).strip()


    if (
        returned_tenant_id
        != cleaned_tenant_id
    ):
        raise PermissionError(
            "The orchestrator returned a different tenant_id "
            "than the trusted web tenant."
        )


    status = str(
        orchestrator_result.get(
            "status"
        )
        or ""
    ).strip()


    if (
        status
        not in WEB_SUPPORTED_ANSWER_STATUSES
    ):
        raise RuntimeError(
            "Unsupported orchestrator status: "
            f"{status!r}"
        )


    answer = str(
        orchestrator_result.get(
            "answer"
        )
        or ""
    ).strip()


    if not answer:
        raise RuntimeError(
            "The orchestrator returned an empty answer."
        )


    verified_source_cards = (
        orchestrator_result.get(
            "sources"
        )
        or []
    )


    if not isinstance(
        verified_source_cards,
        list,
    ):
        raise TypeError(
            "The orchestrator sources field "
            "must be a list."
        )


    web_results = [
        source_card_to_web_result(
            source_card
        )
        for source_card
        in verified_source_cards
    ]


    return {
        # Required by the existing /ask route.
        "answer": answer,
        "results": web_results,

        # Preserved for regression tests and future UI upgrades.
        "tenant_id": returned_tenant_id,
        "question": orchestrator_result[
            "question"
        ],
        "status": status,
        "sources": verified_source_cards,
        "facts": orchestrator_result.get(
            "facts",
            [],
        ),
        "generation_method": (
            orchestrator_result.get(
                "generation_method"
            )
        ),
        "llm_called": (
            orchestrator_result.get(
                "llm_called"
            )
        ),
        "generation_validation": (
            orchestrator_result.get(
                "generation_validation"
            )
        ),
        "verification": (
            orchestrator_result.get(
                "verification"
            )
        ),
    }


# ---------------------------------------------------------------------
# 6. Install the adapter into the active Flask application
# ---------------------------------------------------------------------

WEB_SERVICE_REGISTRY[
    "answer_tenant_question"
] = answer_question_v2_web_adapter


# Keep the notebook-level service registry synchronized when present.
if (
    isinstance(
        globals().get(
            "WEB_SERVICES"
        ),
        dict,
    )
):
    WEB_SERVICES[
        "answer_tenant_question"
    ] = answer_question_v2_web_adapter


# Reassign the registry explicitly for clarity.
WEB_FLASK_APP.config[
    "VAULTIFY_SERVICES"
] = WEB_SERVICE_REGISTRY


# ---------------------------------------------------------------------
# 7. Verify installation without executing a document question
# ---------------------------------------------------------------------

installed_answer_service = (
    WEB_FLASK_APP
    .config[
        "VAULTIFY_SERVICES"
    ]
    .get(
        "answer_tenant_question"
    )
)


WEB_ADAPTER_V2_INSTALLED = (
    installed_answer_service
    is answer_question_v2_web_adapter
)


if not WEB_ADAPTER_V2_INSTALLED:
    raise RuntimeError(
        "The answer_question_v2 web adapter "
        "was not installed successfully."
    )


print("=" * 96)
print("🌐 CELL 22B — WEB ADAPTER INSTALLATION")

print(
    "Original web service: "
    f"{getattr(ORIGINAL_WEB_ANSWER_SERVICE, '__name__', type(ORIGINAL_WEB_ANSWER_SERVICE).__name__)}"
)

print(
    "Installed web service: "
    f"{installed_answer_service.__name__}"
)

print(
    f"Approved orchestrator tenant: "
    f"{ORCHESTRATOR_TENANT_ID}"
)

print(
    "Supported statuses: "
    + ", ".join(
        sorted(
            WEB_SUPPORTED_ANSWER_STATUSES
        )
    )
)


print("\n" + "=" * 96)
print("📊 CELL 22B SUMMARY")

print(
    "✅ answer_question_v2 is connected "
    "to the Flask service registry."
)

print(
    "✅ The existing /ask route was preserved."
)

print(
    "✅ Login, membership, CSRF, and trusted "
    "tenant resolution were preserved."
)

print(
    "✅ Verified source cards are compatible "
    "with the existing dashboard serializer."
)

print(
    "➡️ Ready for controlled web adapter "
    "regression tests in Cell 22C."
)


print(
    "\n⏭️ No Flask route was replaced."
)

print(
    "⏭️ No Groq request was made."
)

print(
    "⏭️ No Qdrant query or write was made."
)

🌐 CELL 22B — WEB ADAPTER INSTALLATION
Original web service: answer_tenant_question
Installed web service: answer_question_v2_web_adapter
Approved orchestrator tenant: demo_apple_tenant
Supported statuses: answered, clarification_required, insufficient_evidence, no_answer, unsupported

📊 CELL 22B SUMMARY
✅ answer_question_v2 is connected to the Flask service registry.
✅ The existing /ask route was preserved.
✅ Login, membership, CSRF, and trusted tenant resolution were preserved.
✅ Verified source cards are compatible with the existing dashboard serializer.
➡️ Ready for controlled web adapter regression tests in Cell 22C.

⏭️ No Flask route was replaced.
⏭️ No Groq request was made.
⏭️ No Qdrant query or write was made.


In [55]:
# PHASE 3.7 - Cell 22C:
# Run controlled regression tests against the installed web adapter.
#
# TEST SCOPE:
# - Calls the web adapter directly
# - Verifies the legacy web response contract
# - Verifies trusted tenant preservation
# - Verifies source-card conversion
# - Verifies compatibility with serialize_sources()
# - Covers single-entity, comparison, ambiguity, and no-answer behavior
#
# RUNTIME EFFECTS:
# - Queries Qdrant
# - May call Groq for verified answerable questions
# - Does not modify Qdrant
# - Does not modify Flask routes
# - Does not create database records

import time


# ---------------------------------------------------------------------
# 1. Validate Cell 22B dependencies
# ---------------------------------------------------------------------

required_names = [
    "WEB_FLASK_APP",
    "WEB_ADAPTER_V2_INSTALLED",
    "answer_question_v2_web_adapter",
    "ORCHESTRATOR_TENANT_ID",
]

missing_names = [
    name
    for name in required_names
    if name not in globals()
]


if missing_names:
    raise RuntimeError(
        "Missing required web-adapter components: "
        + ", ".join(missing_names)
        + ". Run Cell 22B before Cell 22C."
    )


if not WEB_ADAPTER_V2_INSTALLED:
    raise RuntimeError(
        "The answer_question_v2 web adapter "
        "was not installed successfully."
    )


installed_service = (
    WEB_FLASK_APP
    .config[
        "VAULTIFY_SERVICES"
    ]
    .get(
        "answer_tenant_question"
    )
)


if (
    installed_service
    is not answer_question_v2_web_adapter
):
    raise RuntimeError(
        "The active Flask service registry is not "
        "using answer_question_v2_web_adapter."
    )


WEB_SOURCE_SERIALIZER = getattr(
    vaultify_web_backend,
    "serialize_sources",
    None,
)


if not callable(
    WEB_SOURCE_SERIALIZER
):
    raise RuntimeError(
        "vaultify_web_backend.serialize_sources "
        "was not found."
    )


# ---------------------------------------------------------------------
# 2. Define regression scenarios
# ---------------------------------------------------------------------

WEB_ADAPTER_REGRESSION_CASES = [
    {
        "name": "Apple fiscal 2025 total net sales",
        "question": (
            "What were Apple's total net sales "
            "in fiscal year 2025?"
        ),
        "accepted_statuses": {
            "answered",
        },
        "required_answer_fragments": [
            "416,161",
        ],
        "required_source_markers": [
            "apple",
        ],
        "minimum_source_count": 1,
        "maximum_source_count": None,
        "expect_llm_called": True,
        "require_period_warning": False,
    },
    {
        "name": "Tesla Q4 2025 total revenue",
        "question": (
            "What was Tesla's total revenue "
            "in Q4 2025?"
        ),
        "accepted_statuses": {
            "answered",
        },
        "required_answer_fragments": [
            "24,901",
        ],
        "required_source_markers": [
            "tesla",
        ],
        "minimum_source_count": 1,
        "maximum_source_count": None,
        "expect_llm_called": True,
        "require_period_warning": False,
    },
    {
        "name": "Apple and Tesla comparison",
        "question": (
            "Compare Apple's fiscal 2025 net sales "
            "with Tesla's Q4 2025 revenue."
        ),
        "accepted_statuses": {
            "answered",
        },
        "required_answer_fragments": [
            "416,161",
            "24,901",
        ],
        "required_source_markers": [
            "apple",
            "tesla",
        ],
        "minimum_source_count": 2,
        "maximum_source_count": None,
        "expect_llm_called": True,
        "require_period_warning": True,
    },
    {
        "name": "Ambiguous company and period",
        "question": (
            "What was the total revenue in 2025?"
        ),
        "accepted_statuses": {
            "clarification_required",
        },
        "required_answer_fragments": [],
        "required_source_markers": [],
        "minimum_source_count": 0,
        "maximum_source_count": 0,
        "expect_llm_called": False,
        "require_period_warning": False,
    },
    {
        "name": "Question outside uploaded documents",
        "question": (
            "Who wrote Pride and Prejudice?"
        ),
        "accepted_statuses": {
            "no_answer",
            "unsupported",
            "insufficient_evidence",
        },
        "required_answer_fragments": [],
        "required_source_markers": [],
        "minimum_source_count": 0,
        "maximum_source_count": 0,
        "expect_llm_called": False,
        "require_period_warning": False,
    },
]


PERIOD_WARNING_MARKERS = [
    "fiscal year",
    "quarter",
    "different reporting periods",
    "not directly comparable",
    "not directly period-equivalent",
    "not period-equivalent",
    "annual",
    "quarterly",
]


# ---------------------------------------------------------------------
# 3. Define validation helpers
# ---------------------------------------------------------------------

def normalize_text(
    value,
):
    return str(
        value or ""
    ).strip().lower()


def collect_source_search_text(
    source_cards,
):
    source_parts = []

    for source_card in source_cards:
        if not isinstance(
            source_card,
            dict,
        ):
            continue

        for field_name in [
            "filename",
            "section",
            "entity",
            "metric",
            "period",
            "value",
        ]:
            field_value = source_card.get(
                field_name
            )

            if field_value is not None:
                source_parts.append(
                    str(field_value)
                )

    return normalize_text(
        " ".join(source_parts)
    )


def validate_legacy_web_results(
    web_results,
):
    errors = []

    if not isinstance(
        web_results,
        list,
    ):
        return [
            "result['results'] must be a list."
        ]

    for index, web_result in enumerate(
        web_results,
        start=1,
    ):
        if not hasattr(
            web_result,
            "payload",
        ):
            errors.append(
                f"Web result {index} has no payload."
            )
            continue

        if not isinstance(
            web_result.payload,
            dict,
        ):
            errors.append(
                f"Web result {index} payload "
                "is not a dictionary."
            )

        if not hasattr(
            web_result,
            "score",
        ):
            errors.append(
                f"Web result {index} has no score."
            )

        else:
            try:
                float(
                    web_result.score
                )

            except (
                TypeError,
                ValueError,
            ):
                errors.append(
                    f"Web result {index} score "
                    "is not numeric."
                )

    return errors


def validate_serialized_sources(
    web_results,
):
    try:
        serialized_sources = (
            WEB_SOURCE_SERIALIZER(
                web_results
            )
        )

    except Exception as error:
        return (
            None,
            [
                "serialize_sources() failed: "
                f"{type(error).__name__}: {error}"
            ],
        )

    if not isinstance(
        serialized_sources,
        list,
    ):
        return (
            serialized_sources,
            [
                "serialize_sources() did not "
                "return a list."
            ],
        )

    return (
        serialized_sources,
        [],
    )


def validate_web_adapter_case(
    test_case,
    result,
):
    errors = []

    if not isinstance(
        result,
        dict,
    ):
        return [
            "The adapter result is not a dictionary."
        ]


    required_fields = {
        "tenant_id",
        "question",
        "status",
        "answer",
        "results",
        "sources",
    }

    missing_fields = (
        required_fields
        - set(
            result
        )
    )

    if missing_fields:
        errors.append(
            "Missing adapter fields: "
            + ", ".join(
                sorted(
                    missing_fields
                )
            )
        )


    returned_tenant_id = normalize_text(
        result.get(
            "tenant_id"
        )
    )

    expected_tenant_id = normalize_text(
        ORCHESTRATOR_TENANT_ID
    )

    if (
        returned_tenant_id
        != expected_tenant_id
    ):
        errors.append(
            "Trusted tenant preservation failed. "
            f"Expected {ORCHESTRATOR_TENANT_ID!r}, "
            f"received {result.get('tenant_id')!r}."
        )


    returned_question = normalize_text(
        result.get(
            "question"
        )
    )

    expected_question = normalize_text(
        test_case[
            "question"
        ]
    )

    if (
        returned_question
        != expected_question
    ):
        errors.append(
            "The adapter did not preserve "
            "the original question."
        )


    status = str(
        result.get(
            "status"
        )
        or ""
    ).strip()

    if (
        status
        not in test_case[
            "accepted_statuses"
        ]
    ):
        errors.append(
            f"Unexpected status {status!r}. "
            "Accepted statuses: "
            f"{sorted(test_case['accepted_statuses'])}."
        )


    answer = str(
        result.get(
            "answer"
        )
        or ""
    ).strip()

    if not answer:
        errors.append(
            "The adapter returned an empty answer."
        )


    normalized_answer = normalize_text(
        answer
    )

    for required_fragment in (
        test_case[
            "required_answer_fragments"
        ]
    ):
        if (
            normalize_text(
                required_fragment
            )
            not in normalized_answer
        ):
            errors.append(
                "Required answer fragment "
                f"{required_fragment!r} was not found."
            )


    source_cards = (
        result.get(
            "sources"
        )
        or []
    )

    if not isinstance(
        source_cards,
        list,
    ):
        errors.append(
            "The sources field is not a list."
        )
        source_cards = []


    source_count = len(
        source_cards
    )

    minimum_source_count = (
        test_case[
            "minimum_source_count"
        ]
    )

    maximum_source_count = (
        test_case[
            "maximum_source_count"
        ]
    )

    if (
        source_count
        < minimum_source_count
    ):
        errors.append(
            f"Expected at least "
            f"{minimum_source_count} sources, "
            f"received {source_count}."
        )

    if (
        maximum_source_count is not None
        and source_count
        > maximum_source_count
    ):
        errors.append(
            f"Expected at most "
            f"{maximum_source_count} sources, "
            f"received {source_count}."
        )


    source_search_text = (
        collect_source_search_text(
            source_cards
        )
    )

    for required_marker in (
        test_case[
            "required_source_markers"
        ]
    ):
        if (
            normalize_text(
                required_marker
            )
            not in source_search_text
        ):
            errors.append(
                "Required source marker "
                f"{required_marker!r} was not found."
            )


    if test_case[
        "require_period_warning"
    ]:
        period_warning_found = any(
            normalize_text(
                marker
            )
            in normalized_answer
            for marker in (
                PERIOD_WARNING_MARKERS
            )
        )

        if not period_warning_found:
            errors.append(
                "The comparison answer did not "
                "include a reporting-period warning."
            )


    expected_llm_called = (
        test_case[
            "expect_llm_called"
        ]
    )

    returned_llm_called = (
        result.get(
            "llm_called"
        )
    )

    if (
        expected_llm_called is False
        and returned_llm_called is not False
    ):
        errors.append(
            "The LLM should not have been called "
            f"for this scenario. Received: "
            f"{returned_llm_called!r}."
        )

    if (
        expected_llm_called is True
        and returned_llm_called not in {
            True,
            False,
        }
    ):
        errors.append(
            "The answerable scenario did not return "
            "a valid llm_called boolean."
        )


    web_results = (
        result.get(
            "results"
        )
    )

    errors.extend(
        validate_legacy_web_results(
            web_results
        )
    )


    serialized_sources, serializer_errors = (
        validate_serialized_sources(
            web_results
        )
    )

    errors.extend(
        serializer_errors
    )


    if (
        serialized_sources is not None
        and len(
            serialized_sources
        )
        != len(
            web_results
        )
    ):
        errors.append(
            "serialize_sources() returned a different "
            "source count than the adapter results."
        )


    return errors


# ---------------------------------------------------------------------
# 4. Execute controlled adapter regression tests
# ---------------------------------------------------------------------

WEB_ADAPTER_REGRESSION_RESULTS = {}

passed_count = 0
failed_count = 0


print("=" * 96)
print("🧪 CELL 22C — CONTROLLED WEB ADAPTER REGRESSION")


for test_case in (
    WEB_ADAPTER_REGRESSION_CASES
):
    print("\n" + "-" * 96)

    print(
        f"Test: "
        f"{test_case['name']}"
    )

    print(
        f"Question: "
        f"{test_case['question']}"
    )

    started_at = time.perf_counter()

    try:
        result = (
            answer_question_v2_web_adapter(
                tenant_id=(
                    ORCHESTRATOR_TENANT_ID
                ),
                question=(
                    test_case[
                        "question"
                    ]
                ),
            )
        )

        elapsed_seconds = (
            time.perf_counter()
            - started_at
        )

        validation_errors = (
            validate_web_adapter_case(
                test_case,
                result,
            )
        )

    except Exception as error:
        elapsed_seconds = (
            time.perf_counter()
            - started_at
        )

        result = None

        validation_errors = [
            f"{type(error).__name__}: {error}"
        ]


    passed = not validation_errors

    if passed:
        passed_count += 1
        status_label = "PASS"

    else:
        failed_count += 1
        status_label = "FAIL"


    WEB_ADAPTER_REGRESSION_RESULTS[
        test_case["name"]
    ] = {
        "passed": passed,
        "elapsed_seconds": (
            elapsed_seconds
        ),
        "errors": (
            validation_errors
        ),
        "result": result,
    }


    print(
        f"Result: {status_label}"
    )

    print(
        f"Elapsed: "
        f"{elapsed_seconds:.2f} seconds"
    )


    if result is not None:
        print(
            f"Status: "
            f"{result.get('status')}"
        )

        print(
            f"Tenant: "
            f"{result.get('tenant_id')}"
        )

        print(
            f"LLM called: "
            f"{result.get('llm_called')}"
        )

        print(
            f"Verified sources: "
            f"{len(result.get('sources') or [])}"
        )

        print(
            f"Web results: "
            f"{len(result.get('results') or [])}"
        )

        print(
            "Answer:"
        )

        print(
            result.get(
                "answer"
            )
        )


    if validation_errors:
        print(
            "Validation errors:"
        )

        for validation_error in (
            validation_errors
        ):
            print(
                f"   - {validation_error}"
            )


# ---------------------------------------------------------------------
# 5. Final regression gate
# ---------------------------------------------------------------------

WEB_ADAPTER_REGRESSION_PASSED = (
    failed_count == 0
    and passed_count
    == len(
        WEB_ADAPTER_REGRESSION_CASES
    )
)


print("\n" + "=" * 96)
print("📊 CELL 22C REGRESSION SUMMARY")

print(
    f"Passed: "
    f"{passed_count}/"
    f"{len(WEB_ADAPTER_REGRESSION_CASES)}"
)

print(
    f"Failed: "
    f"{failed_count}"
)


if WEB_ADAPTER_REGRESSION_PASSED:
    print(
        "✅ The web adapter preserved "
        "the trusted tenant."
    )

    print(
        "✅ Single-entity answers passed."
    )

    print(
        "✅ Multi-document comparison passed."
    )

    print(
        "✅ Clarification and no-answer "
        "gates passed."
    )

    print(
        "✅ Verified sources remain compatible "
        "with the existing dashboard serializer."
    )

    print(
        "➡️ Ready for real Flask request-flow "
        "regression tests in Cell 22D."
    )

else:
    failed_test_names = [
        test_name
        for test_name, test_result
        in WEB_ADAPTER_REGRESSION_RESULTS.items()
        if not test_result[
            "passed"
        ]
    ]

    print(
        "❌ Web adapter regression failed."
    )

    print(
        "Failed tests: "
        + ", ".join(
            failed_test_names
        )
    )

    raise RuntimeError(
        "Cell 22C regression gate failed. "
        "Review the printed validation errors "
        "before continuing to Cell 22D."
    )


print(
    "\n⏭️ No Flask route was replaced."
)

print(
    "⏭️ No database QueryLog row was created."
)

print(
    "⏭️ No Qdrant data was modified."
)

🧪 CELL 22C — CONTROLLED WEB ADAPTER REGRESSION

------------------------------------------------------------------------------------------------
Test: Apple fiscal 2025 total net sales
Question: What were Apple's total net sales in fiscal year 2025?
Result: PASS
Elapsed: 2.33 seconds
Status: answered
Tenant: demo_apple_tenant
LLM called: True
Verified sources: 1
Web results: 1
Answer:
According to the apple_fy2025_10k.pdf, Note 2 - Revenue, Apple's total net sales in fiscal year 2025 were $416,161 million.

------------------------------------------------------------------------------------------------
Test: Tesla Q4 2025 total revenue
Question: What was Tesla's total revenue in Q4 2025?
Result: PASS
Elapsed: 0.34 seconds
Status: answered
Tenant: demo_apple_tenant
LLM called: True
Verified sources: 1
Web results: 1
Answer:
According to the TSLA-Q4-2025-Update.pdf, specifically the (Unaudited) section, Tesla's total revenue in Q4 2025 was $24,901 million.

------------------------------

In [56]:
# PHASE 3.7 - Cell 22D:
# Run end-to-end Flask request-flow regression tests.
#
# TEST SCOPE:
# - Verifies that /ask requires authentication
# - Logs in through the real Flask login route
# - Verifies membership-based tenant resolution
# - Sends browser-controlled tenant_id values and confirms they are ignored
# - Exercises the real POST /ask route
# - Verifies rendered answers and sources
# - Verifies QueryLog creation
# - Cleans up QueryLog rows created by this test
#
# RUNTIME EFFECTS:
# - Queries Qdrant
# - May call Groq for answerable questions
# - Temporarily disables CSRF only inside the test
# - Temporarily installs a transparent service spy
# - Creates temporary QueryLog rows and removes them afterward
# - Does not modify Qdrant
# - Does not replace Flask routes

import time


# ---------------------------------------------------------------------
# 1. Validate required components
# ---------------------------------------------------------------------

required_names = [
    "WEB_FLASK_APP",
    "WEB_ADAPTER_V2_INSTALLED",
    "answer_question_v2_web_adapter",
    "DEMO_LOGIN_EMAIL",
    "DEMO_LOGIN_PASSWORD",
    "DEMO_TENANT_ID",
    "vaultify_web_backend",
]

missing_names = [
    name
    for name in required_names
    if name not in globals()
]


if missing_names:
    raise RuntimeError(
        "Missing required Flask test components: "
        + ", ".join(missing_names)
        + ". Run Cells 13, 22B, and 22C first."
    )


if not WEB_ADAPTER_V2_INSTALLED:
    raise RuntimeError(
        "The approved web adapter is not installed."
    )


if not globals().get(
    "WEB_ADAPTER_REGRESSION_PASSED",
    False,
):
    raise RuntimeError(
        "Cell 22C did not pass its regression gate."
    )


User = getattr(
    vaultify_web_backend,
    "User",
    None,
)

Organization = getattr(
    vaultify_web_backend,
    "Organization",
    None,
)

Membership = getattr(
    vaultify_web_backend,
    "Membership",
    None,
)

QueryLog = getattr(
    vaultify_web_backend,
    "QueryLog",
    None,
)

db = getattr(
    vaultify_web_backend,
    "db",
    None,
)


required_backend_objects = {
    "User": User,
    "Organization": Organization,
    "Membership": Membership,
    "QueryLog": QueryLog,
    "db": db,
}


missing_backend_objects = [
    name
    for name, value
    in required_backend_objects.items()
    if value is None
]


if missing_backend_objects:
    raise RuntimeError(
        "Missing backend database objects: "
        + ", ".join(
            missing_backend_objects
        )
    )


# ---------------------------------------------------------------------
# 2. Resolve the demo user and trusted organization
# ---------------------------------------------------------------------

with WEB_FLASK_APP.app_context():
    demo_user = (
        User.query.filter_by(
            email=DEMO_LOGIN_EMAIL
        ).first()
    )

    demo_organization = (
        Organization.query.filter_by(
            tenant_id=DEMO_TENANT_ID
        ).first()
    )

    if demo_user is None:
        raise RuntimeError(
            "The Cell 13 demo user was not found."
        )

    if demo_organization is None:
        raise RuntimeError(
            "The Cell 13 demo organization "
            "was not found."
        )

    demo_membership = (
        Membership.query.filter_by(
            user_id=demo_user.id,
            organization_id=(
                demo_organization.id
            ),
        ).first()
    )

    if demo_membership is None:
        raise RuntimeError(
            "The demo user is not a member "
            "of the demo organization."
        )

    starting_query_log_id = (
        db.session.query(
            db.func.coalesce(
                db.func.max(
                    QueryLog.id
                ),
                0,
            )
        )
        .scalar()
    )


# ---------------------------------------------------------------------
# 3. Define real request-flow scenarios
# ---------------------------------------------------------------------

FLASK_REQUEST_REGRESSION_CASES = [
    {
        "name": "Apple fiscal 2025 total net sales",
        "question": (
            "What were Apple's total net sales "
            "in fiscal year 2025?"
        ),
        "required_page_fragments": [
            "416,161",
            "apple_fy2025_10k.pdf",
            "Note 2 - Revenue",
        ],
        "expected_status": "answered",
        "expected_source_count": 1,
    },
    {
        "name": "Tesla Q4 2025 total revenue",
        "question": (
            "What was Tesla's total revenue "
            "in Q4 2025?"
        ),
        "required_page_fragments": [
            "24,901",
            "TSLA-Q4-2025-Update.pdf",
        ],
        "expected_status": "answered",
        "expected_source_count": 1,
    },
    {
        "name": "Apple and Tesla comparison",
        "question": (
            "Compare Apple's fiscal 2025 net sales "
            "with Tesla's Q4 2025 revenue."
        ),
        "required_page_fragments": [
            "416,161",
            "24,901",
            "not directly period-equivalent",
            "apple_fy2025_10k.pdf",
            "TSLA-Q4-2025-Update.pdf",
        ],
        "expected_status": "answered",
        "expected_source_count": 2,
    },
    {
        "name": "Ambiguous company and period",
        "question": (
            "What was the total revenue in 2025?"
        ),
        "required_page_fragments": [
            "Which company do you mean",
            "reporting period",
        ],
        "expected_status": (
            "clarification_required"
        ),
        "expected_source_count": 0,
    },
    {
        "name": "Question outside uploaded documents",
        "question": (
            "Who wrote Pride and Prejudice?"
        ),
        "required_page_fragments": [
            "could not identify relevant evidence",
        ],
        "expected_status": "no_answer",
        "expected_source_count": 0,
    },
]


BROWSER_CONTROLLED_TENANT_ID = (
    "attacker_controlled_tenant"
)


# ---------------------------------------------------------------------
# 4. Install a transparent service spy
# ---------------------------------------------------------------------

service_registry = (
    WEB_FLASK_APP
    .config[
        "VAULTIFY_SERVICES"
    ]
)

original_active_service = (
    service_registry[
        "answer_tenant_question"
    ]
)


if (
    original_active_service
    is not answer_question_v2_web_adapter
):
    raise RuntimeError(
        "The active Flask service is no longer "
        "the approved answer_question_v2 adapter."
    )


FLASK_ROUTE_SERVICE_CALLS = []


def flask_route_answer_service_spy(
    question,
    tenant_id,
    **kwargs,
):
    """
    Record the trusted server-side tenant received by the
    /ask route, then delegate to the approved web adapter.
    """

    result = (
        answer_question_v2_web_adapter(
            question=question,
            tenant_id=tenant_id,
            **kwargs,
        )
    )

    FLASK_ROUTE_SERVICE_CALLS.append(
        {
            "question": question,
            "tenant_id": tenant_id,
            "status": result.get(
                "status"
            ),
            "llm_called": result.get(
                "llm_called"
            ),
            "source_count": len(
                result.get(
                    "sources"
                )
                or []
            ),
        }
    )

    return result


# ---------------------------------------------------------------------
# 5. Preserve Flask test settings
# ---------------------------------------------------------------------

original_testing_setting = (
    WEB_FLASK_APP.config.get(
        "TESTING",
        False,
    )
)

original_csrf_setting = (
    WEB_FLASK_APP.config.get(
        "WTF_CSRF_ENABLED",
        True,
    )
)


FLASK_REQUEST_REGRESSION_RESULTS = {}

created_query_log_records = []

test_failure = None


try:
    WEB_FLASK_APP.config[
        "TESTING"
    ] = True

    WEB_FLASK_APP.config[
        "WTF_CSRF_ENABLED"
    ] = False

    service_registry[
        "answer_tenant_question"
    ] = flask_route_answer_service_spy


    print("=" * 96)
    print("🧪 CELL 22D — REAL FLASK REQUEST-FLOW REGRESSION")


    with WEB_FLASK_APP.test_client() as client:

        # -------------------------------------------------------------
        # 5A. Verify that /ask requires authentication
        # -------------------------------------------------------------

        print("\n" + "-" * 96)
        print("Test: Unauthenticated /ask access")

        unauthenticated_response = (
            client.post(
                "/ask",
                data={
                    "question": (
                        "What were Apple's "
                        "total net sales?"
                    ),
                    "tenant_id": (
                        BROWSER_CONTROLLED_TENANT_ID
                    ),
                },
                follow_redirects=False,
            )
        )

        unauthenticated_location = (
            unauthenticated_response.headers.get(
                "Location",
                "",
            )
        )

        unauthenticated_passed = (
            unauthenticated_response.status_code
            in {
                302,
                303,
            }
            and "/login"
            in unauthenticated_location
            and len(
                FLASK_ROUTE_SERVICE_CALLS
            )
            == 0
        )

        FLASK_REQUEST_REGRESSION_RESULTS[
            "Unauthenticated /ask access"
        ] = {
            "passed": (
                unauthenticated_passed
            ),
            "status_code": (
                unauthenticated_response
                .status_code
            ),
            "location": (
                unauthenticated_location
            ),
        }

        print(
            "Result: "
            + (
                "PASS"
                if unauthenticated_passed
                else "FAIL"
            )
        )

        print(
            f"Status code: "
            f"{unauthenticated_response.status_code}"
        )

        print(
            f"Redirect location: "
            f"{unauthenticated_location}"
        )


        if not unauthenticated_passed:
            raise RuntimeError(
                "The /ask route did not enforce login."
            )


        # -------------------------------------------------------------
        # 5B. Log in through the real Flask login route
        # -------------------------------------------------------------

        print("\n" + "-" * 96)
        print("Test: Demo login")

        login_response = client.post(
            "/login",
            data={
                "email": (
                    DEMO_LOGIN_EMAIL
                ),
                "password": (
                    DEMO_LOGIN_PASSWORD
                ),
            },
            follow_redirects=True,
        )

        login_page = (
            login_response.get_data(
                as_text=True
            )
        )

        login_passed = (
            login_response.status_code
            == 200
            and demo_organization.name
            in login_page
            and DEMO_TENANT_ID
            in login_page
        )

        FLASK_REQUEST_REGRESSION_RESULTS[
            "Demo login"
        ] = {
            "passed": login_passed,
            "status_code": (
                login_response.status_code
            ),
        }

        print(
            "Result: "
            + (
                "PASS"
                if login_passed
                else "FAIL"
            )
        )

        if not login_passed:
            raise RuntimeError(
                "The demo login did not reach "
                "the trusted organization dashboard."
            )


        # -------------------------------------------------------------
        # 5C. Tamper with the browser session organization selection
        # -------------------------------------------------------------

        with client.session_transaction() as flask_session:
            flask_session[
                "organization_id"
            ] = 999999999


        # -------------------------------------------------------------
        # 5D. Verify that an empty question never reaches the adapter
        # -------------------------------------------------------------

        print("\n" + "-" * 96)
        print("Test: Empty question rejection")

        service_call_count_before_empty = len(
            FLASK_ROUTE_SERVICE_CALLS
        )

        empty_response = client.post(
            "/ask",
            data={
                "question": "   ",
                "tenant_id": (
                    BROWSER_CONTROLLED_TENANT_ID
                ),
            },
            follow_redirects=True,
        )

        empty_page = (
            empty_response.get_data(
                as_text=True
            )
        )

        empty_question_passed = (
            empty_response.status_code
            == 200
            and "Enter a question first."
            in empty_page
            and len(
                FLASK_ROUTE_SERVICE_CALLS
            )
            == service_call_count_before_empty
        )

        FLASK_REQUEST_REGRESSION_RESULTS[
            "Empty question rejection"
        ] = {
            "passed": (
                empty_question_passed
            ),
            "status_code": (
                empty_response.status_code
            ),
        }

        print(
            "Result: "
            + (
                "PASS"
                if empty_question_passed
                else "FAIL"
            )
        )

        if not empty_question_passed:
            raise RuntimeError(
                "The empty-question validation failed."
            )


        # -------------------------------------------------------------
        # 5E. Execute the real /ask request cases
        # -------------------------------------------------------------

        for test_case in (
            FLASK_REQUEST_REGRESSION_CASES
        ):
            print("\n" + "-" * 96)

            print(
                f"Test: "
                f"{test_case['name']}"
            )

            print(
                f"Question: "
                f"{test_case['question']}"
            )

            call_count_before_request = len(
                FLASK_ROUTE_SERVICE_CALLS
            )

            started_at = time.perf_counter()

            response = client.post(
                "/ask",
                data={
                    "question": (
                        test_case[
                            "question"
                        ]
                    ),

                    # The Flask route must ignore this value.
                    "tenant_id": (
                        BROWSER_CONTROLLED_TENANT_ID
                    ),
                },
                follow_redirects=False,
            )

            elapsed_seconds = (
                time.perf_counter()
                - started_at
            )

            response_page = (
                response.get_data(
                    as_text=True
                )
            )

            validation_errors = []


            if response.status_code != 200:
                validation_errors.append(
                    "Expected HTTP 200, received "
                    f"{response.status_code}."
                )


            for required_fragment in (
                test_case[
                    "required_page_fragments"
                ]
            ):
                if (
                    required_fragment.lower()
                    not in response_page.lower()
                ):
                    validation_errors.append(
                        "Rendered dashboard is missing "
                        f"{required_fragment!r}."
                    )


            if (
                len(
                    FLASK_ROUTE_SERVICE_CALLS
                )
                != call_count_before_request
                + 1
            ):
                validation_errors.append(
                    "The /ask route did not call the "
                    "answer service exactly once."
                )

                latest_service_call = None

            else:
                latest_service_call = (
                    FLASK_ROUTE_SERVICE_CALLS[
                        -1
                    ]
                )


            if latest_service_call is not None:
                if (
                    latest_service_call[
                        "tenant_id"
                    ]
                    != DEMO_TENANT_ID
                ):
                    validation_errors.append(
                        "Trusted tenant resolution failed. "
                        f"Expected {DEMO_TENANT_ID!r}, "
                        "received "
                        f"{latest_service_call['tenant_id']!r}."
                    )


                if (
                    latest_service_call[
                        "tenant_id"
                    ]
                    == BROWSER_CONTROLLED_TENANT_ID
                ):
                    validation_errors.append(
                        "The browser-controlled tenant_id "
                        "reached the answer service."
                    )


                if (
                    latest_service_call[
                        "question"
                    ]
                    != test_case[
                        "question"
                    ]
                ):
                    validation_errors.append(
                        "The route changed the submitted question."
                    )


                if (
                    latest_service_call[
                        "status"
                    ]
                    != test_case[
                        "expected_status"
                    ]
                ):
                    validation_errors.append(
                        "Unexpected orchestrator status. "
                        f"Expected "
                        f"{test_case['expected_status']!r}, "
                        "received "
                        f"{latest_service_call['status']!r}."
                    )


                if (
                    latest_service_call[
                        "source_count"
                    ]
                    != test_case[
                        "expected_source_count"
                    ]
                ):
                    validation_errors.append(
                        "Unexpected verified source count. "
                        f"Expected "
                        f"{test_case['expected_source_count']}, "
                        "received "
                        f"{latest_service_call['source_count']}."
                    )


            case_passed = not validation_errors

            FLASK_REQUEST_REGRESSION_RESULTS[
                test_case["name"]
            ] = {
                "passed": case_passed,
                "elapsed_seconds": (
                    elapsed_seconds
                ),
                "status_code": (
                    response.status_code
                ),
                "service_call": (
                    latest_service_call
                ),
                "errors": (
                    validation_errors
                ),
            }


            print(
                "Result: "
                + (
                    "PASS"
                    if case_passed
                    else "FAIL"
                )
            )

            print(
                f"Elapsed: "
                f"{elapsed_seconds:.2f} seconds"
            )

            print(
                f"HTTP status: "
                f"{response.status_code}"
            )


            if latest_service_call:
                print(
                    "Trusted tenant received by service: "
                    f"{latest_service_call['tenant_id']}"
                )

                print(
                    "Browser tenant ignored: "
                    f"{latest_service_call['tenant_id'] != BROWSER_CONTROLLED_TENANT_ID}"
                )

                print(
                    f"Orchestrator status: "
                    f"{latest_service_call['status']}"
                )

                print(
                    f"Verified sources: "
                    f"{latest_service_call['source_count']}"
                )


            if validation_errors:
                print(
                    "Validation errors:"
                )

                for validation_error in (
                    validation_errors
                ):
                    print(
                        f"   - {validation_error}"
                    )


        # -------------------------------------------------------------
        # 5F. Verify QueryLog rows produced by the real /ask route
        # -------------------------------------------------------------

        print("\n" + "-" * 96)
        print("Test: QueryLog persistence")

        with WEB_FLASK_APP.app_context():
            created_query_log_records = (
                QueryLog.query
                .filter(
                    QueryLog.id
                    > starting_query_log_id
                )
                .order_by(
                    QueryLog.id.asc()
                )
                .all()
            )

            expected_questions = [
                test_case[
                    "question"
                ]
                for test_case in (
                    FLASK_REQUEST_REGRESSION_CASES
                )
            ]

            logged_questions = [
                record.question
                for record in (
                    created_query_log_records
                )
            ]

            expected_source_counts = [
                test_case[
                    "expected_source_count"
                ]
                for test_case in (
                    FLASK_REQUEST_REGRESSION_CASES
                )
            ]

            logged_source_counts = [
                record.source_count
                for record in (
                    created_query_log_records
                )
            ]

            query_log_passed = (
                len(
                    created_query_log_records
                )
                == len(
                    FLASK_REQUEST_REGRESSION_CASES
                )
                and logged_questions
                == expected_questions
                and logged_source_counts
                == expected_source_counts
                and all(
                    record.user_id
                    == demo_user.id
                    for record in (
                        created_query_log_records
                    )
                )
                and all(
                    record.organization_id
                    == demo_organization.id
                    for record in (
                        created_query_log_records
                    )
                )
            )


        FLASK_REQUEST_REGRESSION_RESULTS[
            "QueryLog persistence"
        ] = {
            "passed": query_log_passed,
            "created_rows": len(
                created_query_log_records
            ),
            "logged_questions": (
                logged_questions
            ),
            "logged_source_counts": (
                logged_source_counts
            ),
        }

        print(
            "Result: "
            + (
                "PASS"
                if query_log_passed
                else "FAIL"
            )
        )

        print(
            f"Created QueryLog rows: "
            f"{len(created_query_log_records)}"
        )


        if not query_log_passed:
            raise RuntimeError(
                "QueryLog persistence validation failed."
            )


        # -------------------------------------------------------------
        # 5G. Verify logout
        # -------------------------------------------------------------

        print("\n" + "-" * 96)
        print("Test: Logout")

        logout_response = client.post(
            "/logout",
            follow_redirects=True,
        )

        logout_page = (
            logout_response.get_data(
                as_text=True
            )
        )

        logout_passed = (
            logout_response.status_code
            == 200
            and (
                "Sign in"
                in logout_page
                or "signed out"
                in logout_page.lower()
            )
        )

        FLASK_REQUEST_REGRESSION_RESULTS[
            "Logout"
        ] = {
            "passed": logout_passed,
            "status_code": (
                logout_response.status_code
            ),
        }

        print(
            "Result: "
            + (
                "PASS"
                if logout_passed
                else "FAIL"
            )
        )


        if not logout_passed:
            raise RuntimeError(
                "The logout regression test failed."
            )


except Exception as error:
    test_failure = error


finally:
    # Restore the approved production service.
    service_registry[
        "answer_tenant_question"
    ] = original_active_service

    # Restore Flask configuration.
    WEB_FLASK_APP.config[
        "TESTING"
    ] = original_testing_setting

    WEB_FLASK_APP.config[
        "WTF_CSRF_ENABLED"
    ] = original_csrf_setting

    # Remove QueryLog rows created by this regression test.
    with WEB_FLASK_APP.app_context():
        try:
            (
                QueryLog.query
                .filter(
                    QueryLog.id
                    > starting_query_log_id
                )
                .delete(
                    synchronize_session=False
                )
            )

            db.session.commit()

        except Exception:
            db.session.rollback()
            raise


# ---------------------------------------------------------------------
# 6. Final regression gate
# ---------------------------------------------------------------------

passed_results = [
    test_name
    for test_name, test_result
    in FLASK_REQUEST_REGRESSION_RESULTS.items()
    if test_result.get(
        "passed"
    )
]

failed_results = [
    test_name
    for test_name, test_result
    in FLASK_REQUEST_REGRESSION_RESULTS.items()
    if not test_result.get(
        "passed"
    )
]


FLASK_REQUEST_FLOW_REGRESSION_PASSED = (
    test_failure is None
    and not failed_results
    and len(
        FLASK_ROUTE_SERVICE_CALLS
    )
    == len(
        FLASK_REQUEST_REGRESSION_CASES
    )
)


print("\n" + "=" * 96)
print("📊 CELL 22D REGRESSION SUMMARY")

print(
    f"Passed checks: "
    f"{len(passed_results)}/"
    f"{len(FLASK_REQUEST_REGRESSION_RESULTS)}"
)

print(
    f"Failed checks: "
    f"{len(failed_results)}"
)

print(
    f"Real /ask service calls: "
    f"{len(FLASK_ROUTE_SERVICE_CALLS)}"
)

print(
    "QueryLog cleanup: completed"
)


if FLASK_REQUEST_FLOW_REGRESSION_PASSED:
    print(
        "✅ Authentication protected the /ask route."
    )

    print(
        "✅ The real login and logout flow passed."
    )

    print(
        "✅ The browser-controlled tenant_id "
        "was ignored."
    )

    print(
        "✅ Tenant identity came from the "
        "authenticated membership."
    )

    print(
        "✅ The real /ask route rendered answers "
        "and verified sources."
    )

    print(
        "✅ QueryLog persistence passed."
    )

    print(
        "✅ Temporary regression logs were removed."
    )

    print(
        "➡️ Phase 3.7 web integration is approved."
    )

else:
    print(
        "❌ Real Flask request-flow regression failed."
    )

    if failed_results:
        print(
            "Failed checks: "
            + ", ".join(
                failed_results
            )
        )

    if test_failure is not None:
        print(
            "Failure: "
            f"{type(test_failure).__name__}: "
            f"{test_failure}"
        )

    raise RuntimeError(
        "Cell 22D regression gate failed. "
        "Review the printed results before continuing."
    )


print(
    "\n⏭️ The approved web adapter was restored."
)

print(
    "⏭️ Flask routes were not replaced."
)

print(
    "⏭️ Qdrant data was not modified."
)

🧪 CELL 22D — REAL FLASK REQUEST-FLOW REGRESSION

------------------------------------------------------------------------------------------------
Test: Unauthenticated /ask access
Result: PASS
Status code: 302
Redirect location: /login?next=%2Fask

------------------------------------------------------------------------------------------------
Test: Demo login
Result: PASS

------------------------------------------------------------------------------------------------
Test: Empty question rejection
Result: PASS

------------------------------------------------------------------------------------------------
Test: Apple fiscal 2025 total net sales
Question: What were Apple's total net sales in fiscal year 2025?
Result: PASS
Elapsed: 1.98 seconds
HTTP status: 200
Trusted tenant received by service: demo_apple_tenant
Browser tenant ignored: True
Orchestrator status: answered
Verified sources: 1

---------------------------------------------------------------------------------------------